# STraT-Net Reproducibility Notebook

**STraT-Net: A Longitudinal Symptom Trajectory Transformer Network for Explainable Neurodegenerative Disease Diagnosis**

This notebook contains the research workflow used for STraT-Net, including data preparation, longitudinal modelling, baseline comparisons, statistical analysis, trajectory-length analysis, and explainability experiments.

## Expected data files

The public repository may provide STraT-Bench as `data/STraT-Bench_v1.0.zip`. This notebook automatically extracts it when the CSV files are not already present. After extraction, the following files are expected:

- `stratnet_v2_sentence_level_dataset.csv`
- `stratnet_v2_year_level_trajectories.csv`
- `stratnet_v2_symptom_vocabulary.txt`

The notebook searches for the data directory in `../data`, `data`, and `/content/STraT-Net/data`. You can also set the environment variable `STRATNET_DATA_DIR` to an alternate location.

> **Research-use disclaimer:** STraT-Bench is a synthetic benchmark. Results from this notebook should not be interpreted as clinically validated diagnostic performance.

### Recommended execution

Run the notebook from top to bottom in a fresh Python/Colab environment. Generated checkpoints and outputs are written to local project-relative folders.


In [ ]:
# Automatic STraT-Bench extraction
from pathlib import Path
import zipfile
import os

def resolve_data_dir():
    override = os.environ.get("STRATNET_DATA_DIR")
    candidates = []
    if override:
        candidates.append(Path(override))
    candidates.extend([
        Path("../data"),
        Path("data"),
        Path("/content/STraT-Net/data"),
    ])
    for candidate in candidates:
        if candidate.exists():
            return candidate.resolve()
    return Path("../data").resolve()

DATA_DIR = resolve_data_dir()
ZIP_PATH = DATA_DIR / "STraT-Bench_v1.0.zip"

required_files = [
    DATA_DIR / "stratnet_v2_sentence_level_dataset.csv",
    DATA_DIR / "stratnet_v2_year_level_trajectories.csv",
]

if not all(path.exists() for path in required_files):
    if ZIP_PATH.exists():
        print(f"Extracting {ZIP_PATH.name} into {DATA_DIR} ...")
        with zipfile.ZipFile(ZIP_PATH, "r") as zf:
            zf.extractall(DATA_DIR)
    else:
        raise FileNotFoundError(
            f"Required STraT-Bench CSV files were not found and {ZIP_PATH.name} is missing. "
            f"Expected data directory: {DATA_DIR}"
        )

missing = [str(path) for path in required_files if not path.exists()]
if missing:
    raise FileNotFoundError(
        "STraT-Bench extraction completed, but required files are still missing:\n" + "\n".join(missing)
    )

print(f"STraT-Bench ready at: {DATA_DIR}")


In [ ]:
# ============================================================
# STraT-Net
# Module 1
# Environment Setup + Dataset Loading
# ============================================================

# ============================================================
# 1. Install Required Libraries (Colab / fresh environment)
# ============================================================

!pip -q install transformers accelerate sentencepiece
!pip -q install scikit-learn seaborn
!pip -q install torchmetrics

# ============================================================
# 2. Import Libraries
# ============================================================

import os
import random
import warnings
import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns

import torch
import torch.nn as nn


from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.utils.class_weight import compute_class_weight

warnings.filterwarnings("ignore")

# ============================================================
# 4. Project Configuration
# ============================================================

class Config:

    # --------------------------------------------------------
    # Dataset folder
    # --------------------------------------------------------

    DATASET_PATH = str(DATA_DIR)

    # --------------------------------------------------------

    # Repository-relative dataset discovery.
    # You may override this with the STRATNET_DATA_DIR
    # environment variable.
    _data_override = os.environ.get("STRATNET_DATA_DIR")

    if _data_override:
        DATASET_PATH = _data_override
    else:
        _data_candidates = [
            "../data",
            "data",
            "/content/STraT-Net/data",
        ]
        DATASET_PATH = next(
            (
                candidate
                for candidate in _data_candidates
                if os.path.isdir(candidate)
            ),
            "../data"
        )

    # Updated STraT-Net V2 dataset files
    SENTENCE_FILE = "stratnet_v2_sentence_level_dataset.csv"

    TRAJECTORY_FILE = "stratnet_v2_year_level_trajectories.csv"

    VOCAB_FILE = "stratnet_v2_symptom_vocabulary.txt"

    # --------------------------------------------------------
    # Model
    # --------------------------------------------------------

    BERT_MODEL = (
        "microsoft/"
        "BiomedNLP-PubMedBERT-base-uncased-abstract-fulltext"
    )

    MAX_LEN = 128

    EMBEDDING_DIM = 768

    TRANSFORMER_DIM = 512

    NUM_HEADS = 8

    NUM_LAYERS = 4

    DROPOUT = 0.2

    # --------------------------------------------------------
    # Training
    # --------------------------------------------------------

    BATCH_SIZE = 16

    LEARNING_RATE = 2e-5

    EPOCHS = 20

    WEIGHT_DECAY = 0.01

    RANDOM_SEED = 42

    DEVICE = torch.device(
        "cuda" if torch.cuda.is_available() else "cpu"
    )


cfg = Config()

print("=" * 60)
print("Running Device :", cfg.DEVICE)
print("=" * 60)

# ============================================================
# 5. Reproducibility
# ============================================================

def seed_everything(seed):

    random.seed(seed)

    np.random.seed(seed)

    torch.manual_seed(seed)

    torch.cuda.manual_seed_all(seed)

    os.environ["PYTHONHASHSEED"] = str(seed)

    torch.backends.cudnn.deterministic = True

    torch.backends.cudnn.benchmark = False


seed_everything(cfg.RANDOM_SEED)

# ============================================================
# 6. Dataset Paths
# ============================================================

sentence_path = os.path.join(
    cfg.DATASET_PATH,
    cfg.SENTENCE_FILE
)

trajectory_path = os.path.join(
    cfg.DATASET_PATH,
    cfg.TRAJECTORY_FILE
)

vocab_path = os.path.join(
    cfg.DATASET_PATH,
    cfg.VOCAB_FILE
)

# ============================================================
# 7. Check Files
# ============================================================

print("\nChecking Dataset Files...\n")

print("Sentence File   :", sentence_path)
print("Trajectory File :", trajectory_path)
print("Vocabulary File :", vocab_path)

assert os.path.exists(sentence_path), \
    f"Missing: {sentence_path}"

assert os.path.exists(trajectory_path), \
    f"Missing: {trajectory_path}"

assert os.path.exists(vocab_path), \
    f"Missing: {vocab_path}"

print("\nAll dataset files found.")

# ============================================================
# 8. Load Dataset
# ============================================================

sentence_df = pd.read_csv(
    sentence_path
)

trajectory_df = pd.read_csv(
    trajectory_path
)

with open(
    vocab_path,
    "r",
    encoding="utf-8"
) as f:

    symptom_vocab = [
        x.strip()
        for x in f.readlines()
        if len(x.strip()) > 0
    ]

print("\nDatasets Loaded Successfully.\n")

# ============================================================
# 9. Dataset Information
# ============================================================

print("=" * 70)
print("Sentence Dataset")
print("=" * 70)

print(
    sentence_df.head()
)

print(
    "\nShape :",
    sentence_df.shape
)

print("\nColumns")

print(
    sentence_df.columns.tolist()
)

print("=" * 70)

print("\nTrajectory Dataset")

print("=" * 70)

print(
    trajectory_df.head()
)

print(
    "\nShape :",
    trajectory_df.shape
)

print("\nColumns")

print(
    trajectory_df.columns.tolist()
)

print("=" * 70)

print(
    "\nSymptom Vocabulary Size :",
    len(symptom_vocab)
)

print(
    "\nFirst 20 Symptoms:"
)

print(
    symptom_vocab[:20]
)

# ============================================================
# 10. Missing Values
# ============================================================

print("\nChecking Missing Values...\n")

print("=" * 60)
print("Sentence Dataset Missing Values")
print("=" * 60)

print(
    sentence_df.isnull().sum()
)

print()

print("=" * 60)
print("Trajectory Dataset Missing Values")
print("=" * 60)

print(
    trajectory_df.isnull().sum()
)

# ============================================================
# 11. Remove Duplicate Records
# ============================================================

sentence_before = len(
    sentence_df
)

trajectory_before = len(
    trajectory_df
)

sentence_df = (
    sentence_df
    .drop_duplicates()
    .reset_index(drop=True)
)

trajectory_df = (
    trajectory_df
    .drop_duplicates()
    .reset_index(drop=True)
)

print("\nDuplicates Removed")

print(
    "Sentence Dataset :",
    sentence_before,
    "->",
    len(sentence_df)
)

print(
    "Trajectory Dataset:",
    trajectory_before,
    "->",
    len(trajectory_df)
)

# ============================================================
# 12. Verify Disease Labels
# ============================================================

print("\nDisease Distribution\n")

if "Diagnosis" in trajectory_df.columns:

    disease_column = "Diagnosis"

elif "diagnosis" in trajectory_df.columns:

    disease_column = "diagnosis"

elif "Disease" in trajectory_df.columns:

    disease_column = "Disease"

else:

    raise ValueError(
        "Disease column not found. "
        "Expected one of: Diagnosis, diagnosis, Disease"
    )

print(
    "Detected Disease Column:",
    disease_column
)

print()

print(
    trajectory_df[disease_column]
    .value_counts()
)

# ============================================================
# 13. Label Encoding
# ============================================================

label_encoder = LabelEncoder()

trajectory_df["Disease_Label"] = (
    label_encoder.fit_transform(
        trajectory_df[disease_column]
    )
)

print("\nDisease Mapping\n")

mapping = pd.DataFrame({

    "Disease":
        label_encoder.classes_,

    "Encoded Label":
        np.arange(
            len(label_encoder.classes_)
        )

})

print(
    mapping.to_string(index=False)
)

NUM_CLASSES = len(
    label_encoder.classes_
)

print(
    "\nNumber of Diseases :",
    NUM_CLASSES
)

# ============================================================
# 14. Compute Class Weights
# ============================================================

class_weights = compute_class_weight(

    class_weight="balanced",

    classes=np.unique(
        trajectory_df["Disease_Label"]
    ),

    y=trajectory_df["Disease_Label"]

)

class_weights = torch.tensor(

    class_weights,

    dtype=torch.float

).to(
    cfg.DEVICE
)

print("\nClass Weights\n")

print(
    class_weights
)

# ============================================================
# 15. Dataset Summary
# ============================================================

print("\n" + "=" * 70)

print("NEW DATASET SUMMARY")

print("=" * 70)

print(
    "Sentence Records    :",
    len(sentence_df)
)

print(
    "Trajectory Records  :",
    len(trajectory_df)
)

print(
    "Number of Symptoms  :",
    len(symptom_vocab)
)

print(
    "Number of Diseases  :",
    NUM_CLASSES
)

print(
    "Disease Column      :",
    disease_column
)

print("=" * 70)

# ============================================================
# 16. Module 1 Completed
# ============================================================

print("\n" + "=" * 70)
print("Module 1 Completed Successfully")
print("=" * 70)

In [ ]:
# ============================================================
# STraT-Net
# Module 2
# Data Preprocessing + Dataset Classes + DataLoaders
# Adapted for STraT-Net V2 Dataset
# ============================================================

import ast
import numpy as np
import torch

from transformers import AutoTokenizer
from torch.utils.data import Dataset, DataLoader

print("=" * 70)
print("STraT-Net : Module 2")
print("New Dataset Preprocessing")
print("=" * 70)

# Adding Config class and cfg instantiation for self-sufficiency
# This is copied from Module 1 (cell S9sYvLZeU4mD) to resolve NameError if Module 1 was not run
class Config:

    # --------------------------------------------------------
    # Dataset folder
    # --------------------------------------------------------

    DATASET_PATH = str(DATA_DIR)

    # --------------------------------------------------------

    # Repository-relative dataset discovery.
    # You may override this with the STRATNET_DATA_DIR
    # environment variable.
    _data_override = os.environ.get("STRATNET_DATA_DIR")

    if _data_override:
        DATASET_PATH = _data_override
    else:
        _data_candidates = [
            "../data",
            "data",
            "/content/STraT-Net/data",
        ]
        DATASET_PATH = next(
            (
                candidate
                for candidate in _data_candidates
                if os.path.isdir(candidate)
            ),
            "../data"
        )

    # Updated STraT-Net V2 dataset files
    SENTENCE_FILE = "stratnet_v2_sentence_level_dataset.csv"

    TRAJECTORY_FILE = "stratnet_v2_year_level_trajectories.csv"

    VOCAB_FILE = "stratnet_v2_symptom_vocabulary.txt"

    # --------------------------------------------------------
    # Model
    # --------------------------------------------------------

    BERT_MODEL = (
        "microsoft/"
        "BiomedNLP-PubMedBERT-base-uncased-abstract-fulltext"
    )

    MAX_LEN = 128

    EMBEDDING_DIM = 768

    TRANSFORMER_DIM = 512

    NUM_HEADS = 8

    NUM_LAYERS = 4

    DROPOUT = 0.2

    # --------------------------------------------------------
    # Training
    # --------------------------------------------------------

    BATCH_SIZE = 16

    LEARNING_RATE = 2e-5

    WEIGHT_DECAY = 0.01

    EPOCHS = 20

    RANDOM_SEED = 42

    DEVICE = torch.device(
        "cuda" if torch.cuda.is_available() else "cpu"
    )


cfg = Config()

# ============================================================
# 1. Load PubMedBERT Tokenizer
# ============================================================

print("\nLoading PubMedBERT tokenizer...")

tokenizer = AutoTokenizer.from_pretrained(
    cfg.BERT_MODEL
)

print("Tokenizer Loaded Successfully")

# ============================================================
# 2. Define Symptom Columns
# ============================================================

symptom_columns = symptom_vocab.copy()

NUM_SYMPTOMS = len(
    symptom_columns
)

print("\nSymptom Columns")
print("-" * 60)

for i, symptom in enumerate(
    symptom_columns
):

    print(
        f"{i:02d} : {symptom}"
    )

print(
    "\nNumber of Symptoms :",
    NUM_SYMPTOMS
)

# ============================================================
# 3. Verify Symptom Columns Exist
# ============================================================

missing_symptom_columns = [

    symptom

    for symptom in symptom_columns

    if symptom not in sentence_df.columns

]

if missing_symptom_columns:

    raise ValueError(

        "Missing symptom columns in "
        "sentence dataset: "
        f"{missing_symptom_columns}"

    )

print(
    "\n✓ All symptom vocabulary entries "
    "exist as sentence-level columns."
)

# ============================================================
# 4. Convert Trajectory Symptom Vectors
# ============================================================

print("\nConverting trajectory symptom vectors...")

def parse_symptom_vector(value):

    # Already converted
    if isinstance(
        value,
        np.ndarray
    ):

        return value.astype(
            np.float32
        )

    # Python list
    if isinstance(
        value,
        list
    ):

        return np.asarray(
            value,
            dtype=np.float32
        )

    # CSV string representation
    if isinstance(
        value,
        str
    ):

        return np.asarray(
            ast.literal_eval(value),
            dtype=np.float32
        )

    raise TypeError(
        f"Unsupported symptom_vector type: "
        f"{type(value)}"
    )


trajectory_df[
    "symptom_vector"
] = trajectory_df[
    "symptom_vector"
].apply(
    parse_symptom_vector
)

print(
    "Trajectory vector conversion completed."
)

# ============================================================
# 5. Validate Trajectory Vector Length
# ============================================================

vector_lengths = trajectory_df[
    "symptom_vector"
].apply(
    len
)

print("\nTrajectory Vector Length Distribution")
print("-" * 60)

print(
    vector_lengths
    .value_counts()
    .sort_index()
)

if not (
    vector_lengths == NUM_SYMPTOMS
).all():

    invalid_rows = trajectory_df.loc[

        vector_lengths != NUM_SYMPTOMS,

        [
            "patient_id",
            "year",
            "symptom_vector"
        ]

    ]

    raise ValueError(

        "Trajectory vector length mismatch.\n"

        f"Expected: {NUM_SYMPTOMS}\n"

        f"Invalid rows: {len(invalid_rows)}"

    )

print(
    f"\n✓ All trajectory vectors contain "
    f"{NUM_SYMPTOMS} symptom values."
)

print("\nExample Trajectory Vector")

print(
    trajectory_df[
        "symptom_vector"
    ].iloc[0]
)

print(
    "Vector Length :",
    len(
        trajectory_df[
            "symptom_vector"
        ].iloc[0]
    )
)

# ============================================================
# 6. Validate Existing Disease Labels
# ============================================================

print("\nValidating Disease Labels...")
print("-" * 60)

expected_sentence_labels = (
    label_encoder.transform(
        sentence_df[
            "diagnosis"
        ]
    )
)

expected_trajectory_labels = (
    label_encoder.transform(
        trajectory_df[
            "diagnosis"
        ]
    )
)

# ------------------------------------------------------------
# Sentence dataset
# ------------------------------------------------------------

if "Disease_Label" in sentence_df.columns:

    sentence_label_match = np.array_equal(

        sentence_df[
            "Disease_Label"
        ].astype(int).values,

        expected_sentence_labels

    )

    print(
        "Sentence label mapping correct :",
        sentence_label_match
    )

else:

    sentence_label_match = False

# ------------------------------------------------------------
# Trajectory dataset
# ------------------------------------------------------------

if "Disease_Label" in trajectory_df.columns:

    trajectory_label_match = np.array_equal(

        trajectory_df[
            "Disease_Label"
        ].astype(int).values,

        expected_trajectory_labels

    )

    print(
        "Trajectory label mapping correct:",
        trajectory_label_match
    )

else:

    trajectory_label_match = False

# ------------------------------------------------------------
# Standardize labels using the fitted encoder
# ------------------------------------------------------------

sentence_df[
    "Disease_Label"
] = expected_sentence_labels

trajectory_df[
    "Disease_Label"
] = expected_trajectory_labels

print(
    "\n✓ Disease labels standardized "
    "using current LabelEncoder."
)

# ============================================================
# 7. Verify Dataset Splits
# ============================================================

print("\nSentence Split Distribution")
print("-" * 60)

print(
    sentence_df[
        "split"
    ].value_counts()
)

print("\nTrajectory Split Distribution")
print("-" * 60)

print(
    trajectory_df[
        "split"
    ].value_counts()
)

required_splits = {
    "train",
    "val",
    "test"
}

sentence_splits = set(

    sentence_df[
        "split"
    ].unique()

)

trajectory_splits = set(

    trajectory_df[
        "split"
    ].unique()

)

if not required_splits.issubset(
    sentence_splits
):

    raise ValueError(
        "Sentence dataset is missing "
        "train/val/test splits."
    )

if not required_splits.issubset(
    trajectory_splits
):

    raise ValueError(
        "Trajectory dataset is missing "
        "train/val/test splits."
    )

print(
    "\n✓ Train / validation / test "
    "splits detected."
)

# ============================================================
# 8. Patient-Level Leakage Check
# ============================================================

print("\nChecking Patient-Level Split Leakage...")
print("-" * 60)

train_patients = set(

    trajectory_df.loc[

        trajectory_df[
            "split"
        ] == "train",

        "patient_id"

    ]

)

val_patients = set(

    trajectory_df.loc[

        trajectory_df[
            "split"
        ] == "val",

        "patient_id"

    ]

)

test_patients = set(

    trajectory_df.loc[

        trajectory_df[
            "split"
        ] == "test",

        "patient_id"

    ]

)

train_val_overlap = (
    train_patients
    & val_patients
)

train_test_overlap = (
    train_patients
    & test_patients
)

val_test_overlap = (
    val_patients
    & test_patients
)

print(
    "Train-Val overlap :",
    len(train_val_overlap)
)

print(
    "Train-Test overlap:",
    len(train_test_overlap)
)

print(
    "Val-Test overlap  :",
    len(val_test_overlap)
)

if (
    len(train_val_overlap) > 0
    or len(train_test_overlap) > 0
    or len(val_test_overlap) > 0
):

    raise ValueError(
        "Patient-level data leakage detected."
    )

print(
    "\n✓ No patient-level split leakage detected."
)

# ============================================================
# 9. Create Train / Validation / Test DataFrames
# ============================================================

train_sentence_df = sentence_df[

    sentence_df[
        "split"
    ] == "train"

].reset_index(
    drop=True
)

val_sentence_df = sentence_df[

    sentence_df[
        "split"
    ] == "val"

].reset_index(
    drop=True
)

test_sentence_df = sentence_df[

    sentence_df[
        "split"
    ] == "test"

].reset_index(
    drop=True
)


train_traj_df = trajectory_df[

    trajectory_df[
        "split"
    ] == "train"

].reset_index(
    drop=True
)

val_traj_df = trajectory_df[

    trajectory_df[
        "split"
    ] == "val"

].reset_index(
    drop=True
)

test_traj_df = trajectory_df[

    trajectory_df[
        "split"
    ] == "test"

].reset_index(
    drop=True
)

# ============================================================
# 10. Dataset Split Sizes
# ============================================================

print("\nSentence Dataset Sizes")
print("-" * 60)

print(
    "Train :",
    len(train_sentence_df)
)

print(
    "Val   :",
    len(val_sentence_df)
)

print(
    "Test  :",
    len(test_sentence_df)
)

print("\nTrajectory Dataset Sizes")
print("-" * 60)

print(
    "Train :",
    len(train_traj_df)
)

print(
    "Val   :",
    len(val_traj_df)
)

print(
    "Test  :",
    len(test_traj_df)
)

# ============================================================
# 11. Sentence Dataset
# ============================================================

class SentenceDataset(
    Dataset
):

    def __init__(
        self,
        dataframe
    ):

        self.df = dataframe.reset_index(
            drop=True
        )

    def __len__(
        self
    ):

        return len(
            self.df
        )

    def __getitem__(
        self,
        idx
    ):

        row = self.df.iloc[
            idx
        ]

        encoding = tokenizer(

            str(
                row[
                    "sentence"
                ]
            ),

            max_length=cfg.MAX_LEN,

            truncation=True,

            padding="max_length",

            return_tensors="pt"

        )

        symptom_targets = torch.tensor(

            row[
                symptom_columns
            ].values.astype(
                np.float32
            ),

            dtype=torch.float32

        )

        disease_label = torch.tensor(

            int(
                row[
                    "Disease_Label"
                ]
            ),

            dtype=torch.long

        )

        return {

            "patient_id":
                row[
                    "patient_id"
                ],

            "year":
                torch.tensor(
                    int(
                        row[
                            "year"
                        ]
                    ),
                    dtype=torch.long
                ),

            "input_ids":
                encoding[
                    "input_ids"
                ].squeeze(0),

            "attention_mask":
                encoding[
                    "attention_mask"
                ].squeeze(0),

            "symptom_labels":
                symptom_targets,

            "disease_labels":
                disease_label

        }

# ============================================================
# 12. Trajectory Dataset
# ============================================================

class TrajectoryDataset(
    Dataset
):

    def __init__(
        self,
        dataframe
    ):

        self.df = dataframe.reset_index(
            drop=True
        )

    def __len__(
        self
    ):

        return len(
            self.df
        )

    def __getitem__(
        self,
        idx
    ):

        row = self.df.iloc[
            idx
        ]

        return {

            "patient_id":
                row[
                    "patient_id"
                ],

            "year":
                torch.tensor(
                    int(
                        row[
                            "year"
                        ]
                    ),
                    dtype=torch.long
                ),

            "trajectory":
                torch.tensor(
                    row[
                        "symptom_vector"
                    ],
                    dtype=torch.float32
                ),

            "disease_labels":
                torch.tensor(
                    int(
                        row[
                            "Disease_Label"
                        ]
                    ),
                    dtype=torch.long
                )

        }

# ============================================================
# 13. Build Dataset Objects
# ============================================================

train_sentence_dataset = SentenceDataset(
    train_sentence_df
)

val_sentence_dataset = SentenceDataset(
    val_sentence_df
)

test_sentence_dataset = SentenceDataset(
    test_sentence_df
)

train_traj_dataset = TrajectoryDataset(
    train_traj_df
)

val_traj_dataset = TrajectoryDataset(
    val_traj_df
)

test_traj_dataset = TrajectoryDataset(
    test_traj_df
)

print(
    "\n✓ Dataset objects created successfully."
)

# ============================================================
# 14. DataLoaders
# ============================================================

train_sentence_loader = DataLoader(

    train_sentence_dataset,

    batch_size=cfg.BATCH_SIZE,

    shuffle=True,

    num_workers=2,

    pin_memory=torch.cuda.is_available()

)

val_sentence_loader = DataLoader(

    val_sentence_dataset,

    batch_size=cfg.BATCH_SIZE,

    shuffle=False,

    num_workers=2,

    pin_memory=torch.cuda.is_available()

)

test_sentence_loader = DataLoader(

    test_sentence_dataset,

    batch_size=cfg.BATCH_SIZE,

    shuffle=False,

    num_workers=2,

    pin_memory=torch.cuda.is_available()

)


train_traj_loader = DataLoader(

    train_traj_dataset,

    batch_size=cfg.BATCH_SIZE,

    shuffle=True,

    num_workers=2,

    pin_memory=torch.cuda.is_available()

)

val_traj_loader = DataLoader(

    val_traj_dataset,

    batch_size=cfg.BATCH_SIZE,

    shuffle=False,

    num_workers=2,

    pin_memory=torch.cuda.is_available()

)

test_traj_loader = DataLoader(

    test_traj_dataset,

    batch_size=cfg.BATCH_SIZE,

    shuffle=False,

    num_workers=2,

    pin_memory=torch.cuda.is_available()

)

print(
    "✓ DataLoaders created successfully."
)

# ============================================================
# 15. Verify Sentence Batch
# ============================================================

sample_batch = next(
    iter(
        train_sentence_loader
    )
)

print("\nSentence Batch Verification")
print("-" * 60)

print(
    "Input IDs Shape      :",
    sample_batch[
        "input_ids"
    ].shape
)

print(
    "Attention Shape      :",
    sample_batch[
        "attention_mask"
    ].shape
)

print(
    "Symptom Labels Shape :",
    sample_batch[
        "symptom_labels"
    ].shape
)

print(
    "Disease Labels Shape :",
    sample_batch[
        "disease_labels"
    ].shape
)

# ============================================================
# 16. Verify Trajectory Batch
# ============================================================

traj_batch = next(
    iter(
        train_traj_loader
    )
)

print("\nTrajectory Batch Verification")
print("-" * 60)

print(
    "Trajectory Shape :",
    traj_batch[
        "trajectory"
    ].shape
)

print(
    "Disease Shape    :",
    traj_batch[
        "disease_labels"
    ].shape
)

# ============================================================
# 17. Final Dynamic Configuration
# ============================================================

NUM_CLASSES = len(
    label_encoder.classes_
)

NUM_SYMPTOMS = len(
    symptom_vocab
)

TRAJECTORY_DIM = NUM_SYMPTOMS

print("\n" + "=" * 70)

print("STraT-Net V2 Dataset Configuration")

print("=" * 70)

print(
    "Number of Diseases   :",
    NUM_CLASSES
)

print(
    "Number of Symptoms   :",
    NUM_SYMPTOMS
)

print(
    "Trajectory Dimension :",
    TRAJECTORY_DIM
)

print("=" * 70)

# ============================================================
# 18. Module Completed
# ============================================================

print("\n" + "=" * 70)
print("Module 2 Completed Successfully")
print("=" * 70)

In [ ]:
# ============================================================
# STraT-Net
# Module 3
# Patient Trajectory Construction
# Variable-Length Sequence Builder
# Adapted for STraT-Net V2 Dataset
# ============================================================

import numpy as np
import pandas as pd
import torch

from torch.utils.data import Dataset, DataLoader

print("=" * 70)
print("STraT-Net : Module 3")
print("Patient Trajectory Construction")
print("=" * 70)

# ============================================================
# 1. Verify Patient-Level Diagnosis Consistency
# ============================================================

print("\nChecking patient-level diagnosis consistency...")
print("-" * 60)

diagnosis_counts_per_patient = (

    trajectory_df
    .groupby("patient_id")["Disease_Label"]
    .nunique()

)

inconsistent_patients = diagnosis_counts_per_patient[

    diagnosis_counts_per_patient > 1

]

print(
    "Patients with multiple disease labels:",
    len(inconsistent_patients)
)

if len(inconsistent_patients) > 0:

    print(
        "\nExample inconsistent patients:"
    )

    print(
        inconsistent_patients.head()
    )

    raise ValueError(
        "Some patients have more than one "
        "Disease_Label across trajectory years."
    )

print(
    "✓ Every patient has one consistent disease label."
)

# ============================================================
# 2. Verify Patient-Level Split Consistency
# ============================================================

print("\nChecking patient-level split consistency...")
print("-" * 60)

split_counts_per_patient = (

    trajectory_df
    .groupby("patient_id")["split"]
    .nunique()

)

inconsistent_split_patients = split_counts_per_patient[

    split_counts_per_patient > 1

]

print(
    "Patients appearing in multiple splits:",
    len(inconsistent_split_patients)
)

if len(inconsistent_split_patients) > 0:

    raise ValueError(
        "Some patients appear in multiple dataset splits."
    )

print(
    "✓ Every patient belongs to exactly one split."
)

# ============================================================
# 3. Visit Count Statistics
# ============================================================

visit_counts = (

    trajectory_df
    .groupby("patient_id")
    .size()

)

MAX_VISITS = int(
    visit_counts.max()
)

MIN_VISITS = int(
    visit_counts.min()
)

MEAN_VISITS = float(
    visit_counts.mean()
)

MEDIAN_VISITS = float(
    visit_counts.median()
)

print("\nPatient Visit Statistics")
print("-" * 60)

print(
    "Total Patients  :",
    len(visit_counts)
)

print(
    "Minimum Visits  :",
    MIN_VISITS
)

print(
    "Maximum Visits  :",
    MAX_VISITS
)

print(
    "Average Visits  :",
    f"{MEAN_VISITS:.2f}"
)

print(
    "Median Visits   :",
    f"{MEDIAN_VISITS:.2f}"
)

print("\nVisit Count Distribution")
print("-" * 60)

print(
    visit_counts
    .value_counts()
    .sort_index()
)

# ============================================================
# 4. Verify Chronological Data
# ============================================================

print("\nChecking duplicate patient-year records...")
print("-" * 60)

duplicate_patient_year = trajectory_df.duplicated(

    subset=[
        "patient_id",
        "year"
    ],

    keep=False

)

duplicate_count = int(
    duplicate_patient_year.sum()
)

print(
    "Duplicate patient-year rows:",
    duplicate_count
)

if duplicate_count > 0:

    print(
        trajectory_df.loc[
            duplicate_patient_year,
            [
                "patient_id",
                "year",
                "diagnosis",
                "split"
            ]
        ].head(20)
    )

    raise ValueError(
        "Duplicate patient-year trajectory records detected."
    )

print(
    "✓ Every patient has at most one trajectory record per year."
)

# ============================================================
# 5. Define Sequence Builder
# ============================================================

def build_patient_sequences(
    dataframe,
    max_visits
):

    patient_sequences = []

    patient_masks = []

    patient_labels = []

    patient_ids = []

    patient_years = []

    patient_lengths = []

    grouped = dataframe.groupby(
        "patient_id",
        sort=False
    )

    for patient_id, group in grouped:

        # --------------------------------------------
        # Sort visits chronologically
        # --------------------------------------------

        group = group.sort_values(
            "year"
        ).reset_index(
            drop=True
        )

        # --------------------------------------------
        # Trajectory vectors
        # --------------------------------------------

        vectors = np.stack(

            group[
                "symptom_vector"
            ].values

        ).astype(
            np.float32
        )

        length = len(
            vectors
        )

        # --------------------------------------------
        # Safety check
        # --------------------------------------------

        if length > max_visits:

            raise ValueError(

                f"Patient {patient_id} has "
                f"{length} visits, exceeding "
                f"MAX_VISITS={max_visits}."

            )

        # --------------------------------------------
        # Years
        # --------------------------------------------

        years = group[
            "year"
        ].astype(
            int
        ).tolist()

        # --------------------------------------------
        # Disease label
        # --------------------------------------------

        unique_labels = group[
            "Disease_Label"
        ].unique()

        if len(unique_labels) != 1:

            raise ValueError(

                f"Patient {patient_id} has "
                "multiple disease labels."

            )

        label = int(
            unique_labels[0]
        )

        # --------------------------------------------
        # Allocate padded sequence
        # Shape:
        # (MAX_VISITS, NUM_SYMPTOMS)
        # --------------------------------------------

        padded_vectors = np.zeros(

            (
                max_visits,
                NUM_SYMPTOMS
            ),

            dtype=np.float32

        )

        padded_vectors[
            :length
        ] = vectors

        # --------------------------------------------
        # Padding mask
        #
        # True  = real visit
        # False = padding
        # --------------------------------------------

        mask = np.zeros(

            max_visits,

            dtype=bool

        )

        mask[
            :length
        ] = True

        # --------------------------------------------
        # Padded year sequence
        # --------------------------------------------

        padded_years = np.zeros(

            max_visits,

            dtype=np.int64

        )

        padded_years[
            :length
        ] = np.asarray(

            years,

            dtype=np.int64

        )

        # --------------------------------------------
        # Store
        # --------------------------------------------

        patient_sequences.append(

            torch.tensor(
                padded_vectors,
                dtype=torch.float32
            )

        )

        patient_masks.append(

            torch.tensor(
                mask,
                dtype=torch.bool
            )

        )

        patient_labels.append(
            label
        )

        patient_ids.append(
            patient_id
        )

        patient_years.append(

            torch.tensor(
                padded_years,
                dtype=torch.long
            )

        )

        patient_lengths.append(
            length
        )

    return {

        "sequences":
            patient_sequences,

        "masks":
            patient_masks,

        "labels":
            patient_labels,

        "patient_ids":
            patient_ids,

        "years":
            patient_years,

        "lengths":
            patient_lengths

    }

# ============================================================
# 6. Build Train / Validation / Test Sequences
# ============================================================

print("\nBuilding patient trajectories...")

train_sequences = build_patient_sequences(

    train_traj_df,

    MAX_VISITS

)

val_sequences = build_patient_sequences(

    val_traj_df,

    MAX_VISITS

)

test_sequences = build_patient_sequences(

    test_traj_df,

    MAX_VISITS

)

print(
    "✓ Patient trajectories built successfully."
)

# ============================================================
# 7. Unpack Sequence Collections
# ============================================================

train_patient_sequences = train_sequences[
    "sequences"
]

train_masks = train_sequences[
    "masks"
]

train_labels = train_sequences[
    "labels"
]

train_patient_ids = train_sequences[
    "patient_ids"
]

train_years = train_sequences[
    "years"
]

train_lengths = train_sequences[
    "lengths"
]


val_patient_sequences = val_sequences[
    "sequences"
]

val_masks = val_sequences[
    "masks"
]

val_labels = val_sequences[
    "labels"
]

val_patient_ids = val_sequences[
    "patient_ids"
]

val_years = val_sequences[
    "years"
]

val_lengths = val_sequences[
    "lengths"
]


test_patient_sequences = test_sequences[
    "sequences"
]

test_masks = test_sequences[
    "masks"
]

test_labels = test_sequences[
    "labels"
]

test_patient_ids = test_sequences[
    "patient_ids"
]

test_years = test_sequences[
    "years"
]

test_lengths = test_sequences[
    "lengths"
]

# ============================================================
# 8. Patient Dataset
# ============================================================

class PatientTrajectoryDataset(
    Dataset
):

    def __init__(

        self,

        sequences,

        masks,

        labels,

        patient_ids,

        years,

        lengths

    ):

        self.sequences = sequences

        self.masks = masks

        self.labels = labels

        self.patient_ids = patient_ids

        self.years = years

        self.lengths = lengths

    def __len__(
        self
    ):

        return len(
            self.labels
        )

    def __getitem__(
        self,
        idx
    ):

        return {

            "patient_id":
                self.patient_ids[
                    idx
                ],

            "trajectory":
                self.sequences[
                    idx
                ],

            "mask":
                self.masks[
                    idx
                ],

            "label":
                torch.tensor(

                    self.labels[
                        idx
                    ],

                    dtype=torch.long

                ),

            "years":
                self.years[
                    idx
                ],

            "length":
                torch.tensor(

                    self.lengths[
                        idx
                    ],

                    dtype=torch.long

                )

        }

# ============================================================
# 9. Create Patient Dataset Objects
# ============================================================

train_patient_dataset = PatientTrajectoryDataset(

    train_patient_sequences,

    train_masks,

    train_labels,

    train_patient_ids,

    train_years,

    train_lengths

)

val_patient_dataset = PatientTrajectoryDataset(

    val_patient_sequences,

    val_masks,

    val_labels,

    val_patient_ids,

    val_years,

    val_lengths

)

test_patient_dataset = PatientTrajectoryDataset(

    test_patient_sequences,

    test_masks,

    test_labels,

    test_patient_ids,

    test_years,

    test_lengths

)

print(
    "\n✓ Patient datasets created."
)

# ============================================================
# 10. DataLoaders
# ============================================================

train_patient_loader = DataLoader(

    train_patient_dataset,

    batch_size=cfg.BATCH_SIZE,

    shuffle=True,

    num_workers=2,

    pin_memory=torch.cuda.is_available()

)

val_patient_loader = DataLoader(

    val_patient_dataset,

    batch_size=cfg.BATCH_SIZE,

    shuffle=False,

    num_workers=2,

    pin_memory=torch.cuda.is_available()

)

test_patient_loader = DataLoader(

    test_patient_dataset,

    batch_size=cfg.BATCH_SIZE,

    shuffle=False,

    num_workers=2,

    pin_memory=torch.cuda.is_available()

)

print(
    "✓ Patient DataLoaders created."
)

# ============================================================
# 11. Patient Counts by Split
# ============================================================

print("\nPatient Counts")
print("-" * 60)

print(
    "Training Patients   :",
    len(
        train_patient_dataset
    )
)

print(
    "Validation Patients :",
    len(
        val_patient_dataset
    )
)

print(
    "Testing Patients    :",
    len(
        test_patient_dataset
    )
)

print(
    "Total Patients      :",
    (
        len(
            train_patient_dataset
        )
        +
        len(
            val_patient_dataset
        )
        +
        len(
            test_patient_dataset
        )
    )
)

# ============================================================
# 12. Verify One Batch
# ============================================================

sample = next(

    iter(
        train_patient_loader
    )

)

print("\nBatch Verification")
print("-" * 60)

print(
    "Trajectory Shape :",
    sample[
        "trajectory"
    ].shape
)

print(
    "Mask Shape       :",
    sample[
        "mask"
    ].shape
)

print(
    "Years Shape      :",
    sample[
        "years"
    ].shape
)

print(
    "Labels Shape     :",
    sample[
        "label"
    ].shape
)

print(
    "Lengths Shape    :",
    sample[
        "length"
    ].shape
)

# ============================================================
# 13. Verify Expected Tensor Dimensions
# ============================================================

expected_trajectory_shape = (

    cfg.BATCH_SIZE,

    MAX_VISITS,

    NUM_SYMPTOMS

)

print("\nExpected Architecture Input")
print("-" * 60)

print(
    "Batch Size           :",
    cfg.BATCH_SIZE
)

print(
    "Maximum Visits       :",
    MAX_VISITS
)

print(
    "Feature Dimension    :",
    NUM_SYMPTOMS
)

print(
    "Expected Batch Shape :",
    expected_trajectory_shape
)

print(
    "Actual Batch Shape   :",
    tuple(
        sample[
            "trajectory"
        ].shape
    )
)

# ============================================================
# 14. Validate Padding Mask
# ============================================================

print("\nPadding Validation")
print("-" * 60)

mask_visit_counts = (

    sample[
        "mask"
    ]
    .sum(
        dim=1
    )

)

stored_lengths = sample[
    "length"
]

mask_length_match = torch.equal(

    mask_visit_counts.cpu(),

    stored_lengths.cpu()

)

print(
    "Mask counts equal stored lengths:",
    mask_length_match
)

if not mask_length_match:

    raise ValueError(
        "Mask and trajectory-length mismatch detected."
    )

print(
    "✓ Padding masks correctly represent real visits."
)

# ============================================================
# 15. Verify Padded Values Are Zero
# ============================================================

trajectory_batch = sample[
    "trajectory"
]

mask_batch = sample[
    "mask"
]

padded_values = trajectory_batch[
    ~mask_batch
]

if padded_values.numel() > 0:

    max_padding_value = (

        padded_values
        .abs()
        .max()
        .item()

    )

else:

    max_padding_value = 0.0

print(
    "Maximum absolute padded value:",
    max_padding_value
)

if max_padding_value != 0.0:

    raise ValueError(
        "Non-zero values detected in padded trajectory positions."
    )

print(
    "✓ All padded trajectory values are zero."
)

# ============================================================
# 16. Verify Chronological Order
# ============================================================

print("\nChronological Order Validation")
print("-" * 60)

chronological_errors = 0

for years_tensor, length in zip(

    train_years,

    train_lengths

):

    valid_years = years_tensor[
        :length
    ].tolist()

    if valid_years != sorted(
        valid_years
    ):

        chronological_errors += 1

print(
    "Training patients with unordered years:",
    chronological_errors
)

if chronological_errors > 0:

    raise ValueError(
        "Chronological ordering error detected."
    )

print(
    "✓ Patient visits are chronologically ordered."
)

# ============================================================
# 17. Final Transformer Input Configuration
# ============================================================

TRAJECTORY_DIM = NUM_SYMPTOMS

SEQUENCE_LENGTH = MAX_VISITS

print("\n" + "=" * 70)

print("Transformer Input Configuration")

print("=" * 70)

print(
    "Sequence Length      :",
    SEQUENCE_LENGTH
)

print(
    "Feature Dimension    :",
    TRAJECTORY_DIM
)

print(
    "Disease Classes      :",
    NUM_CLASSES
)

print(
    "Symptoms             :",
    NUM_SYMPTOMS
)

print("=" * 70)

# ============================================================
# 18. Example Patient
# ============================================================

print("\nExample Patient")
print("-" * 60)

print(
    "Patient ID :",
    sample[
        "patient_id"
    ][0]
)

print(
    "Label      :",
    sample[
        "label"
    ][0].item()
)

print(
    "Visits     :",
    sample[
        "length"
    ][0].item()
)

print(
    "Years      :",
    sample[
        "years"
    ][0][
        :sample[
            "length"
        ][0]
    ].tolist()
)

print(
    "Mask       :",
    sample[
        "mask"
    ][0].tolist()
)

# ============================================================
# 19. Module Complete
# ============================================================

print("\n" + "=" * 70)
print("Module 3 Completed Successfully")
print("=" * 70)

In [ ]:
# ============================================================
# STraT-Net
# Module 4A
# Temporal Representation Network
# Adapted for STraT-Net V2 Dataset
# ============================================================

import math
import torch
import torch.nn as nn

print("=" * 70)
print("STraT-Net : Module 4A")
print("Temporal Representation Network")
print("=" * 70)

# ============================================================
# 1. Model Hyperparameters
# ============================================================

EMBED_DIM = 128

NUM_HEADS = 8

NUM_LAYERS = 4

FF_DIM = 512

DROPOUT = 0.20

print("\nModel Hyperparameters")
print("-" * 60)

print(
    "Trajectory Dimension :",
    TRAJECTORY_DIM
)

print(
    "Embedding Dimension  :",
    EMBED_DIM
)

print(
    "Attention Heads      :",
    NUM_HEADS
)

print(
    "Transformer Layers   :",
    NUM_LAYERS
)

print(
    "Feed Forward Dim     :",
    FF_DIM
)

print(
    "Dropout              :",
    DROPOUT
)

print(
    "Maximum Visits       :",
    MAX_VISITS
)

# ============================================================
# 2. Validate Transformer Dimensions
# ============================================================

if EMBED_DIM % NUM_HEADS != 0:

    raise ValueError(

        f"EMBED_DIM ({EMBED_DIM}) must be divisible "
        f"by NUM_HEADS ({NUM_HEADS})."

    )

if TRAJECTORY_DIM != NUM_SYMPTOMS:

    raise ValueError(

        f"TRAJECTORY_DIM ({TRAJECTORY_DIM}) does not match "
        f"NUM_SYMPTOMS ({NUM_SYMPTOMS})."

    )

print(
    "\n✓ Transformer dimensions validated."
)

# ============================================================
# 3. Determine Year Range Dynamically
# ============================================================

MIN_YEAR = int(
    trajectory_df[
        "year"
    ].min()
)

MAX_YEAR = int(
    trajectory_df[
        "year"
    ].max()
)

NUM_YEAR_VALUES = (
    MAX_YEAR
    - MIN_YEAR
    + 1
)

print("\nYear Range")
print("-" * 60)

print(
    "Minimum Year :",
    MIN_YEAR
)

print(
    "Maximum Year :",
    MAX_YEAR
)

print(
    "Year Values  :",
    NUM_YEAR_VALUES
)

# ============================================================
# 4. Positional Encoding
# ============================================================

class PositionalEncoding(
    nn.Module
):

    def __init__(
        self,
        d_model,
        max_len
    ):

        super().__init__()

        pe = torch.zeros(
            max_len,
            d_model
        )

        position = torch.arange(

            0,
            max_len,
            dtype=torch.float32

        ).unsqueeze(
            1
        )

        div_term = torch.exp(

            torch.arange(

                0,
                d_model,
                2,

                dtype=torch.float32

            )

            *

            (
                -math.log(10000.0)
                / d_model
            )

        )

        pe[
            :,
            0::2
        ] = torch.sin(

            position
            *
            div_term

        )

        pe[
            :,
            1::2
        ] = torch.cos(

            position
            *
            div_term

        )

        pe = pe.unsqueeze(
            0
        )

        self.register_buffer(
            "pe",
            pe
        )

    def forward(
        self,
        x
    ):

        return (

            x

            +

            self.pe[
                :,
                :x.size(1)
            ]

        )

# ============================================================
# 5. Dynamic Year Embedding
# ============================================================

class YearEmbedding(
    nn.Module
):

    def __init__(
        self,
        embedding_dim,
        min_year,
        max_year
    ):

        super().__init__()

        self.min_year = int(
            min_year
        )

        self.max_year = int(
            max_year
        )

        self.num_years = (

            self.max_year
            -
            self.min_year
            +
            1

        )

        # +1 reserved for padding year = 0
        self.padding_index = (
            self.num_years
        )

        self.embedding = nn.Embedding(

            num_embeddings=(
                self.num_years
                +
                1
            ),

            embedding_dim=embedding_dim,

            padding_idx=self.padding_index

        )

    def forward(
        self,
        years
    ):

        years = years.clone()

        # --------------------------------------------
        # Padding positions
        # year == 0
        # --------------------------------------------

        padding_mask = (
            years == 0
        )

        # --------------------------------------------
        # Validate actual years
        # --------------------------------------------

        valid_year_mask = (
            ~padding_mask
        )

        if valid_year_mask.any():

            valid_years = years[
                valid_year_mask
            ]

            if (
                valid_years.min()
                <
                self.min_year
            ):

                raise ValueError(

                    "Observed year below configured "
                    f"minimum year {self.min_year}."

                )

            if (
                valid_years.max()
                >
                self.max_year
            ):

                raise ValueError(

                    "Observed year above configured "
                    f"maximum year {self.max_year}."

                )

        # --------------------------------------------
        # Convert real years to embedding indices
        #
        # MIN_YEAR -> 0
        # MIN_YEAR+1 -> 1
        # ...
        # Padding -> padding_index
        # --------------------------------------------

        indices = (

            years
            -
            self.min_year

        )

        indices[
            padding_mask
        ] = self.padding_index

        return self.embedding(
            indices
        )

# ============================================================
# 6. Feature Projection
# ============================================================

class FeatureProjection(
    nn.Module
):

    def __init__(
        self
    ):

        super().__init__()

        self.network = nn.Sequential(

            nn.Linear(

                TRAJECTORY_DIM,

                EMBED_DIM

            ),

            nn.LayerNorm(
                EMBED_DIM
            ),

            nn.GELU(),

            nn.Dropout(
                DROPOUT
            )

        )

    def forward(
        self,
        x
    ):

        return self.network(
            x
        )

# ============================================================
# 7. Temporal Transformer Encoder
# ============================================================

class TemporalEncoder(
    nn.Module
):

    def __init__(
        self
    ):

        super().__init__()

        # --------------------------------------------
        # Project 11 symptom features -> 128 dimensions
        # --------------------------------------------

        self.feature_projection = (
            FeatureProjection()
        )

        # --------------------------------------------
        # Calendar-year embedding
        # --------------------------------------------

        self.year_embedding = YearEmbedding(

            embedding_dim=EMBED_DIM,

            min_year=MIN_YEAR,

            max_year=MAX_YEAR

        )

        # --------------------------------------------
        # Visit-order positional encoding
        # --------------------------------------------

        self.position_encoding = (
            PositionalEncoding(

                d_model=EMBED_DIM,

                max_len=MAX_VISITS

            )
        )

        # --------------------------------------------
        # Transformer layer
        # --------------------------------------------

        encoder_layer = (
            nn.TransformerEncoderLayer(

                d_model=EMBED_DIM,

                nhead=NUM_HEADS,

                dim_feedforward=FF_DIM,

                dropout=DROPOUT,

                activation="gelu",

                batch_first=True,

                norm_first=False

            )
        )

        # --------------------------------------------
        # Stack Transformer layers
        # --------------------------------------------

        self.transformer = (
            nn.TransformerEncoder(

                encoder_layer,

                num_layers=NUM_LAYERS

            )
        )

        self.output_norm = (
            nn.LayerNorm(
                EMBED_DIM
            )
        )

    def forward(
        self,
        trajectory,
        years,
        mask
    ):

        # --------------------------------------------
        # trajectory:
        # (B, MAX_VISITS, 11)
        #
        # ->
        #
        # (B, MAX_VISITS, 128)
        # --------------------------------------------

        x = self.feature_projection(
            trajectory
        )

        # --------------------------------------------
        # Add calendar-year information
        # --------------------------------------------

        year_features = (
            self.year_embedding(
                years
            )
        )

        x = (
            x
            +
            year_features
        )

        # --------------------------------------------
        # Add relative visit-position information
        # --------------------------------------------

        x = self.position_encoding(
            x
        )

        # --------------------------------------------
        # Transformer
        #
        # src_key_padding_mask:
        #
        # True  = ignore
        # False = attend
        #
        # Our mask:
        #
        # True  = real
        # False = padding
        #
        # Therefore invert using ~mask
        # --------------------------------------------

        x = self.transformer(

            x,

            src_key_padding_mask=(
                ~mask
            )

        )

        x = self.output_norm(
            x
        )

        # Optional safety:
        # zero padded output positions
        x = x.masked_fill(

            (
                ~mask
            ).unsqueeze(
                -1
            ),

            0.0

        )

        return x

# ============================================================
# 8. Initialize Temporal Encoder
# ============================================================

temporal_encoder = (
    TemporalEncoder()
    .to(
        cfg.DEVICE
    )
)

print(
    "\nTemporal Encoder Created Successfully"
)

# ============================================================
# 9. Parameter Count
# ============================================================

total_params = sum(

    p.numel()

    for p in temporal_encoder.parameters()

)

trainable_params = sum(

    p.numel()

    for p in temporal_encoder.parameters()

    if p.requires_grad

)

print("\n" + "=" * 60)

print("Temporal Encoder Parameters")

print("=" * 60)

print(
    f"Total Parameters     : "
    f"{total_params:,}"
)

print(
    f"Trainable Parameters : "
    f"{trainable_params:,}"
)

print("=" * 60)

# ============================================================
# 10. Verify Forward Pass
# ============================================================

sample = next(

    iter(
        train_patient_loader
    )

)

trajectory = sample[
    "trajectory"
].float().to(
    cfg.DEVICE
)

years = sample[
    "years"
].long().to(
    cfg.DEVICE
)

mask = sample[
    "mask"
].bool().to(
    cfg.DEVICE
)

temporal_encoder.eval()

with torch.no_grad():

    output = temporal_encoder(

        trajectory,

        years,

        mask

    )

print("\nForward Pass Verification")
print("-" * 60)

print(
    "Input Trajectory Shape :",
    trajectory.shape
)

print(
    "Year Shape             :",
    years.shape
)

print(
    "Mask Shape             :",
    mask.shape
)

print(
    "Output Shape           :",
    output.shape
)

# ============================================================
# 11. Expected Output Shape Check
# ============================================================

expected_shape = (

    trajectory.shape[0],

    MAX_VISITS,

    EMBED_DIM

)

actual_shape = tuple(
    output.shape
)

print(
    "\nExpected Output Shape :",
    expected_shape
)

print(
    "Actual Output Shape   :",
    actual_shape
)

if actual_shape != expected_shape:

    raise ValueError(

        "Temporal encoder output shape mismatch."

    )

print(
    "✓ Temporal encoder output shape is correct."
)

# ============================================================
# 12. Numerical Stability Check
# ============================================================

print("\nNumerical Stability")
print("-" * 60)

print(
    "NaN Present :",
    torch.isnan(
        output
    ).any().item()
)

print(
    "Inf Present :",
    torch.isinf(
        output
    ).any().item()
)

if (
    torch.isnan(
        output
    ).any()
    or
    torch.isinf(
        output
    ).any()
):

    raise ValueError(
        "NaN or Inf detected in temporal encoder output."
    )

print(
    "✓ No NaN or Inf values detected."
)

# ============================================================
# 13. Padding Output Validation
# ============================================================

padded_output = output[
    ~mask
]

if padded_output.numel() > 0:

    max_padded_output = (

        padded_output
        .abs()
        .max()
        .item()

    )

else:

    max_padded_output = 0.0

print("\nPadding Output Validation")
print("-" * 60)

print(
    "Maximum absolute padded output:",
    max_padded_output
)

if max_padded_output != 0.0:

    raise ValueError(
        "Padded temporal outputs are not zero."
    )

print(
    "✓ Padded temporal outputs correctly zeroed."
)

# ============================================================
# 14. Embedding Statistics
# ============================================================

valid_output = output[
    mask
]

print("\nValid Temporal Embedding Statistics")
print("-" * 60)

print(
    "Mean :",
    valid_output.mean().item()
)

print(
    "Std  :",
    valid_output.std().item()
)

print(
    "Min  :",
    valid_output.min().item()
)

print(
    "Max  :",
    valid_output.max().item()
)

# ============================================================
# 15. Architecture Summary
# ============================================================

print("\n" + "=" * 70)

print("TEMPORAL ENCODER CONFIGURATION")

print("=" * 70)

print(
    "Input Features       :",
    TRAJECTORY_DIM
)

print(
    "Sequence Length      :",
    MAX_VISITS
)

print(
    "Embedding Dimension  :",
    EMBED_DIM
)

print(
    "Transformer Layers   :",
    NUM_LAYERS
)

print(
    "Attention Heads      :",
    NUM_HEADS
)

print(
    "Feed-Forward Dim     :",
    FF_DIM
)

print(
    "Dataset Year Range   :",
    f"{MIN_YEAR} - {MAX_YEAR}"
)

print(
    "Year Embedding Count :",
    NUM_YEAR_VALUES
)

print("=" * 70)

# ============================================================
# 16. Module Complete
# ============================================================

print("\n" + "=" * 70)
print("Module 4A Completed Successfully")
print("=" * 70)

In [ ]:
# ============================================================
# STraT-Net
# Module 4B
# Gated Temporal Attention + Prediction Heads
# Adapted for New STraT-Net V2 Dataset
# ============================================================

import torch
import torch.nn as nn

print("=" * 70)
print("STraT-Net : Module 4B")
print("Gated Temporal Attention + Prediction Heads")
print("=" * 70)

# ============================================================
# 1. Dynamic Output Dimensions
# ============================================================

NUM_DISEASES = len(
    label_encoder.classes_
)

NUM_SYMPTOMS = len(
    symptom_vocab
)

print("\nOutput Configuration")
print("-" * 60)

print(
    "Number of Diseases :",
    NUM_DISEASES
)

print(
    "Number of Symptoms :",
    NUM_SYMPTOMS
)

if NUM_DISEASES != NUM_CLASSES:

    raise ValueError(
        "NUM_DISEASES does not match NUM_CLASSES."
    )

if NUM_SYMPTOMS != TRAJECTORY_DIM:

    raise ValueError(
        "NUM_SYMPTOMS does not match TRAJECTORY_DIM."
    )

print(
    "\n✓ Output dimensions validated."
)

# ============================================================
# 2. Gated Temporal Attention Pooling
# ============================================================

class GatedTemporalAttentionPooling(
    nn.Module
):

    def __init__(
        self,
        embed_dim,
        hidden_dim=128,
        dropout=0.10
    ):

        super().__init__()

        # --------------------------------------------
        # Content representation branch
        # --------------------------------------------

        self.value_branch = nn.Sequential(

            nn.Linear(
                embed_dim,
                hidden_dim
            ),

            nn.Tanh()

        )

        # --------------------------------------------
        # Gating branch
        # --------------------------------------------

        self.gate_branch = nn.Sequential(

            nn.Linear(
                embed_dim,
                hidden_dim
            ),

            nn.Sigmoid()

        )

        # --------------------------------------------
        # Convert gated representation
        # into one scalar attention score
        # per visit
        # --------------------------------------------

        self.score_layer = nn.Linear(

            hidden_dim,

            1

        )

        self.dropout = nn.Dropout(
            dropout
        )

    def forward(
        self,
        x,
        mask
    ):

        """
        x:
            Shape:
            (batch_size,
             visits,
             embed_dim)

        mask:
            Shape:
            (batch_size,
             visits)

            True  = real visit
            False = padded visit
        """

        # --------------------------------------------
        # Gated attention representation
        # --------------------------------------------

        value_features = (
            self.value_branch(
                x
            )
        )

        gate_features = (
            self.gate_branch(
                x
            )
        )

        gated_features = (

            value_features

            *

            gate_features

        )

        gated_features = (
            self.dropout(
                gated_features
            )
        )

        # --------------------------------------------
        # Scalar attention scores
        # --------------------------------------------

        scores = (

            self.score_layer(
                gated_features
            )

            .squeeze(
                -1
            )

        )

        # --------------------------------------------
        # Ignore padded visits
        # --------------------------------------------

        scores = scores.masked_fill(

            ~mask,

            torch.finfo(
                scores.dtype
            ).min

        )

        # --------------------------------------------
        # Normalize across visits
        # --------------------------------------------

        attention_weights = (
            torch.softmax(

                scores,

                dim=1

            )
        )

        # --------------------------------------------
        # Weighted patient representation
        # --------------------------------------------

        pooled_embedding = torch.sum(

            x

            *

            attention_weights.unsqueeze(
                -1
            ),

            dim=1

        )

        return (

            pooled_embedding,

            attention_weights

        )

# ============================================================
# 3. Disease Classification Head
# ============================================================

class DiseaseHead(
    nn.Module
):

    def __init__(
        self
    ):

        super().__init__()

        self.network = nn.Sequential(

            nn.Linear(
                EMBED_DIM,
                256
            ),

            nn.GELU(),

            nn.Dropout(
                0.30
            ),

            nn.Linear(
                256,
                128
            ),

            nn.GELU(),

            nn.Dropout(
                0.20
            ),

            nn.Linear(
                128,
                NUM_DISEASES
            )

        )

    def forward(
        self,
        x
    ):

        return self.network(
            x
        )

# ============================================================
# 4. Auxiliary Symptom Reconstruction Head
# ============================================================

class SymptomHead(
    nn.Module
):

    def __init__(
        self
    ):

        super().__init__()

        self.network = nn.Sequential(

            nn.Linear(
                EMBED_DIM,
                128
            ),

            nn.GELU(),

            nn.Dropout(
                0.20
            ),

            nn.Linear(
                128,
                NUM_SYMPTOMS
            )

        )

    def forward(
        self,
        x
    ):

        return self.network(
            x
        )

# ============================================================
# 5. Prediction Head
# ============================================================

class STraTNetPredictionHead(
    nn.Module
):

    def __init__(
        self
    ):

        super().__init__()

        self.pool = (
            GatedTemporalAttentionPooling(

                embed_dim=EMBED_DIM,

                hidden_dim=128,

                dropout=0.10

            )
        )

        self.disease_head = (
            DiseaseHead()
        )

        self.symptom_head = (
            SymptomHead()
        )

    def forward(

        self,

        temporal_features,

        mask

    ):

        # --------------------------------------------
        # Patient-level temporal pooling
        # --------------------------------------------

        embedding, attention = (
            self.pool(

                temporal_features,

                mask

            )
        )

        # --------------------------------------------
        # Disease prediction
        # --------------------------------------------

        disease_logits = (
            self.disease_head(
                embedding
            )
        )

        # --------------------------------------------
        # Symptom reconstruction
        # --------------------------------------------

        symptom_logits = (
            self.symptom_head(
                embedding
            )
        )

        return {

            "embedding":
                embedding,

            "attention":
                attention,

            "disease_logits":
                disease_logits,

            "symptom_logits":
                symptom_logits

        }

# ============================================================
# 6. Initialize Prediction Head
# ============================================================

prediction_head = (

    STraTNetPredictionHead()

    .to(
        cfg.DEVICE
    )

)

print(
    "\nPrediction Head Created Successfully"
)

# ============================================================
# 7. Forward-Pass Verification
# ============================================================

sample = next(

    iter(
        train_patient_loader
    )

)

trajectory = sample[
    "trajectory"
].float().to(
    cfg.DEVICE
)

years = sample[
    "years"
].long().to(
    cfg.DEVICE
)

mask = sample[
    "mask"
].bool().to(
    cfg.DEVICE
)

temporal_encoder.eval()

prediction_head.eval()

with torch.no_grad():

    temporal_output = temporal_encoder(

        trajectory,

        years,

        mask

    )

    outputs = prediction_head(

        temporal_output,

        mask

    )

# ============================================================
# 8. Output Shape Verification
# ============================================================

print("\nOutput Shape Verification")
print("-" * 60)

print(
    "Temporal Features :",
    temporal_output.shape
)

print(
    "Patient Embedding :",
    outputs[
        "embedding"
    ].shape
)

print(
    "Attention Weights :",
    outputs[
        "attention"
    ].shape
)

print(
    "Disease Logits    :",
    outputs[
        "disease_logits"
    ].shape
)

print(
    "Symptom Logits    :",
    outputs[
        "symptom_logits"
    ].shape
)

expected_embedding_shape = (

    trajectory.shape[0],

    EMBED_DIM

)

expected_attention_shape = (

    trajectory.shape[0],

    MAX_VISITS

)

expected_disease_shape = (

    trajectory.shape[0],

    NUM_DISEASES

)

expected_symptom_shape = (

    trajectory.shape[0],

    NUM_SYMPTOMS

)

assert tuple(
    outputs[
        "embedding"
    ].shape
) == expected_embedding_shape

assert tuple(
    outputs[
        "attention"
    ].shape
) == expected_attention_shape

assert tuple(
    outputs[
        "disease_logits"
    ].shape
) == expected_disease_shape

assert tuple(
    outputs[
        "symptom_logits"
    ].shape
) == expected_symptom_shape

print(
    "\n✓ All output dimensions are correct."
)

# ============================================================
# 9. Attention Sum Validation
# ============================================================

attention = outputs[
    "attention"
]

attention_sums = (
    attention.sum(
        dim=1
    )
)

print("\nAttention Sum Validation")
print("-" * 60)

print(
    "First 5 attention sums:"
)

print(
    attention_sums[:5]
)

attention_sum_valid = torch.allclose(

    attention_sums,

    torch.ones_like(
        attention_sums
    ),

    atol=1e-6

)

print(
    "Attention sums equal 1:",
    attention_sum_valid
)

if not attention_sum_valid:

    raise ValueError(
        "Attention weights do not sum to 1."
    )

print(
    "✓ Attention normalization is correct."
)

# ============================================================
# 10. Padded Attention Validation
# ============================================================

padded_attention = (
    attention.masked_select(
        ~mask
    )
)

if padded_attention.numel() > 0:

    max_padded_attention = (

        padded_attention
        .abs()
        .max()
        .item()

    )

else:

    max_padded_attention = 0.0

print("\nPadded Attention Validation")
print("-" * 60)

print(
    "Maximum padded attention:",
    max_padded_attention
)

if max_padded_attention > 1e-7:

    raise ValueError(
        "Padded visits received non-zero attention."
    )

print(
    "✓ Padded visits receive zero attention."
)

# ============================================================
# 11. Numerical Stability
# ============================================================

print("\nNumerical Stability")
print("-" * 60)

for name in [

    "embedding",

    "attention",

    "disease_logits",

    "symptom_logits"

]:

    tensor = outputs[
        name
    ]

    has_nan = (
        torch.isnan(
            tensor
        )
        .any()
        .item()
    )

    has_inf = (
        torch.isinf(
            tensor
        )
        .any()
        .item()
    )

    print(
        f"{name:20s}"
        f" NaN={has_nan}"
        f" Inf={has_inf}"
    )

    if (
        has_nan
        or
        has_inf
    ):

        raise ValueError(

            f"Numerical instability "
            f"detected in {name}."

        )

print(
    "\n✓ No NaN or Inf values detected."
)

# ============================================================
# 12. Attention Statistics
# ============================================================

valid_attention = (
    attention.masked_select(
        mask
    )
)

print("\nValid Attention Statistics")
print("-" * 60)

print(
    "Minimum :",
    valid_attention.min().item()
)

print(
    "Maximum :",
    valid_attention.max().item()
)

print(
    "Mean    :",
    valid_attention.mean().item()
)

print(
    "Std     :",
    valid_attention.std().item()
)

# ============================================================
# 13. Example Patient Attention
# ============================================================

patient_index = 0

patient_length = int(

    sample[
        "length"
    ][
        patient_index
    ].item()

)

patient_years_example = (

    sample[
        "years"
    ][
        patient_index,
        :patient_length
    ]

    .tolist()

)

patient_attention_example = (

    attention[
        patient_index,
        :patient_length
    ]

    .detach()

    .cpu()

    .tolist()

)

print("\nExample Patient Attention")
print("-" * 60)

print(
    "Patient ID :",
    sample[
        "patient_id"
    ][
        patient_index
    ]
)

print(
    "Years      :",
    patient_years_example
)

print(
    "Attention  :"
)

for year, weight in zip(

    patient_years_example,

    patient_attention_example

):

    print(

        f"  {year} -> "
        f"{weight:.6f}"

    )

# ============================================================
# 14. Parameter Count
# ============================================================

prediction_params = sum(

    p.numel()

    for p in prediction_head.parameters()

)

trainable_prediction_params = sum(

    p.numel()

    for p in prediction_head.parameters()

    if p.requires_grad

)

print("\n" + "=" * 60)

print("Prediction Head Parameters")

print("=" * 60)

print(
    f"Total Parameters     : "
    f"{prediction_params:,}"
)

print(
    f"Trainable Parameters : "
    f"{trainable_prediction_params:,}"
)

print("=" * 60)

# ============================================================
# 15. Architecture Summary
# ============================================================

print("\n" + "=" * 70)

print("MODULE 4B CONFIGURATION")

print("=" * 70)

print(
    "Temporal Input Dim    :",
    EMBED_DIM
)

print(
    "Maximum Visits        :",
    MAX_VISITS
)

print(
    "Attention Type        :",
    "Gated Temporal Attention"
)

print(
    "Attention Hidden Dim  :",
    128
)

print(
    "Patient Embedding Dim :",
    EMBED_DIM
)

print(
    "Disease Outputs       :",
    NUM_DISEASES
)

print(
    "Symptom Outputs       :",
    NUM_SYMPTOMS
)

print("=" * 70)

# ============================================================
# 16. Module Complete
# ============================================================

print("\n" + "=" * 70)
print("Module 4B Completed Successfully")
print("=" * 70)

In [ ]:
# ============================================================
# STraT-Net
# Module 4C
# Complete STraT-Net V2 Model
# Residual Clinical Projection + Gated Attention
# ============================================================

import os
import torch
import torch.nn as nn

print("=" * 70)
print("STraT-Net : Module 4C")
print("Complete STraT-Net V2 Model")
print("=" * 70)

# ============================================================
# 1. Residual Clinical Projection
# ============================================================

class ClinicalProjection(
    nn.Module
):

    def __init__(
        self,
        embed_dim=EMBED_DIM,
        dropout=0.20
    ):

        super().__init__()

        self.block = nn.Sequential(

            nn.Linear(
                embed_dim,
                embed_dim
            ),

            nn.GELU(),

            nn.Dropout(
                dropout
            ),

            nn.Linear(
                embed_dim,
                embed_dim
            )

        )

        self.norm = nn.LayerNorm(
            embed_dim
        )

    def forward(
        self,
        x
    ):

        residual = x

        x = self.block(
            x
        )

        x = (
            x
            +
            residual
        )

        x = self.norm(
            x
        )

        return x

# ============================================================
# 2. Complete STraT-Net V2
# ============================================================

class STraTNetV2(
    nn.Module
):

    def __init__(
        self
    ):

        super().__init__()

        # --------------------------------------------
        # Temporal representation network
        # --------------------------------------------

        self.encoder = (
            TemporalEncoder()
        )

        # --------------------------------------------
        # Gated temporal attention pooling
        # --------------------------------------------

        self.pool = (
            GatedTemporalAttentionPooling(

                embed_dim=EMBED_DIM,

                hidden_dim=128,

                dropout=0.10

            )
        )

        # --------------------------------------------
        # Residual clinical projection
        # --------------------------------------------

        self.projection = (
            ClinicalProjection(

                embed_dim=EMBED_DIM,

                dropout=0.20

            )
        )

        # --------------------------------------------
        # Prediction heads
        # --------------------------------------------

        self.disease_head = (
            DiseaseHead()
        )

        self.symptom_head = (
            SymptomHead()
        )

    def forward(

        self,

        trajectory,

        years,

        mask

    ):

        # ============================================
        # Step 1:
        # Encode longitudinal symptom trajectories
        #
        # Input:
        # (B, 7, 11)
        #
        # Output:
        # (B, 7, 128)
        # ============================================

        temporal_features = (
            self.encoder(

                trajectory,

                years,

                mask

            )
        )

        # ============================================
        # Step 2:
        # Gated temporal attention pooling
        #
        # Output:
        #
        # embedding:
        # (B, 128)
        #
        # attention:
        # (B, 7)
        # ============================================

        pooled_embedding, attention = (
            self.pool(

                temporal_features,

                mask

            )
        )

        # ============================================
        # Step 3:
        # Residual clinical refinement
        # ============================================

        embedding = (
            self.projection(
                pooled_embedding
            )
        )

        # ============================================
        # Step 4:
        # Disease classification
        # ============================================

        disease_logits = (
            self.disease_head(
                embedding
            )
        )

        # ============================================
        # Step 5:
        # Auxiliary symptom reconstruction
        # ============================================

        symptom_logits = (
            self.symptom_head(
                embedding
            )
        )

        return {
            "embedding": embedding,
            "attention": attention,
            "disease_logits": disease_logits,
            "symptom_logits": symptom_logits
        }

In [ ]:
stratnet_model = STraTNetV2()

In [ ]:
stratnet_model.pool
stratnet_model.disease_head
stratnet_model.symptom_head

In [ ]:
stratnet_model.pool
stratnet_model.projection
stratnet_model.disease_head
stratnet_model.symptom_head

In [ ]:
print("=" * 70)
print("MODULE 4C QUICK CHECK")
print("=" * 70)

# Ensure the model is moved to the same device as the input tensors
# This is done by calling .to(cfg.DEVICE) on the model during initialization
stratnet_model.to(cfg.DEVICE)

print("Model class:", stratnet_model.__class__.__name__)
print("Device:", next(stratnet_model.parameters()).device)

sample = next(iter(train_patient_loader))

trajectory = sample["trajectory"].float().to(cfg.DEVICE)
years = sample["years"].long().to(cfg.DEVICE)
mask = sample["mask"].bool().to(cfg.DEVICE)

stratnet_model.eval()

with torch.no_grad():
    outputs = stratnet_model(
        trajectory,
        years,
        mask
    )

print("\nOutput Shapes")

for key, value in outputs.items():
    if torch.is_tensor(value):
        print(f"{key:22s}: {tuple(value.shape)}")

print("\nAttention sums:")
print(outputs["attention"].sum(dim=1)[:5])

print(
    "\nNaN present:",
    any(
        torch.isnan(v).any().item()
        for v in outputs.values()
        if torch.is_tensor(v)
    )
)

print(
    "Inf present:",
    any(
        torch.isinf(v).any().item()
        for v in outputs.values()
        if torch.is_tensor(v)
    )
)

print("\n" + "=" * 70)
print("CHECK COMPLETED")
print("=" * 70)

In [ ]:
# ============================================================
# STraT-Net V2
# Module 5A
# Training Configuration + Utilities
# ============================================================

import os
import json
import random
import numpy as np
import pandas as pd

import torch
import torch.nn as nn

from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)

print("=" * 70)
print("STraT-Net V2 : Module 5A")
print("Training Configuration + Utilities")
print("=" * 70)

# ============================================================
# 1. Training Hyperparameters
# ============================================================

NUM_EPOCHS = 30

LEARNING_RATE = 1e-4

WEIGHT_DECAY = 1e-4

GRAD_CLIP = 1.0

LABEL_SMOOTHING = 0.05

EARLY_STOPPING_PATIENCE = 8

USE_SYMPTOM_LOSS = False

SYMPTOM_LOSS_WEIGHT = 0.30

print("\nTraining Configuration")
print("-" * 60)

print("Epochs                  :", NUM_EPOCHS)
print("Learning Rate           :", LEARNING_RATE)
print("Weight Decay            :", WEIGHT_DECAY)
print("Gradient Clip           :", GRAD_CLIP)
print("Label Smoothing         :", LABEL_SMOOTHING)
print("Early Stopping Patience :", EARLY_STOPPING_PATIENCE)
print("Symptom Loss Enabled    :", USE_SYMPTOM_LOSS)

# ============================================================
# 2. Device
# ============================================================

DEVICE = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)

# Corrected: Use stratnet_model instead of model
stratnet_model = stratnet_model.to(
    DEVICE
)

print("\nDevice :", DEVICE)

if torch.cuda.is_available():

    print(
        "GPU    :",
        torch.cuda.get_device_name(0)
    )

# ============================================================
# 3. Fresh Loss Functions
# ============================================================

disease_loss_fn = nn.CrossEntropyLoss(

    weight=class_weights.to(
        DEVICE
    ),

    label_smoothing=LABEL_SMOOTHING

)

symptom_loss_fn = nn.BCEWithLogitsLoss()

print(
    "\n✓ Loss functions created."
)

# ============================================================
# 4. Fresh Optimizer
# ============================================================

optimizer = AdamW(

    # Corrected: Use stratnet_model.parameters()
    stratnet_model.parameters(),

    lr=LEARNING_RATE,

    weight_decay=WEIGHT_DECAY

)

print(
    "✓ Fresh optimizer created."
)

# ============================================================
# 5. Fresh Scheduler
# ============================================================

scheduler = CosineAnnealingLR(

    optimizer,

    T_max=NUM_EPOCHS,

    eta_min=1e-6

)

print(
    "✓ Cosine scheduler created."
)

# ============================================================
# 6. Mixed Precision
# ============================================================

USE_AMP = torch.cuda.is_available()

if USE_AMP:

    scaler = torch.cuda.amp.GradScaler()

else:

    scaler = None

print(
    "Mixed Precision:",
    USE_AMP
)

# ============================================================
# 7. Experiment Directories
# ============================================================

CHECKPOINT_DIR = (
    "checkpoints_stratnet_v2_newdataset"
)

OUTPUT_DIR = (
    "outputs_stratnet_v2_newdataset"
)

os.makedirs(
    CHECKPOINT_DIR,
    exist_ok=True
)

os.makedirs(
    OUTPUT_DIR,
    exist_ok=True
)

BEST_MODEL_PATH = os.path.join(

    CHECKPOINT_DIR,

    "best_model.pth"

)

LAST_MODEL_PATH = os.path.join(

    CHECKPOINT_DIR,

    "last_model.pth"

)

print(
    "\nCheckpoint Directory:",
    CHECKPOINT_DIR
)

print(
    "Output Directory    :",
    OUTPUT_DIR
)

# ============================================================
# 8. Training History
# ============================================================

history = {

    "epoch": [],

    "train_loss": [],

    "val_loss": [],

    "train_accuracy": [],

    "val_accuracy": [],

    "train_macro_f1": [],

    "val_macro_f1": [],

    "learning_rate": []

}

best_val_f1 = -1.0

best_epoch = -1

early_stop_counter = 0

print(
    "\n✓ Training history initialized."
)

# ============================================================
# 9. Metric Function
# ============================================================

def compute_metrics(
    y_true,
    y_pred
):

    return {

        "accuracy":

            accuracy_score(
                y_true,
                y_pred
            ),

        "precision":

            precision_score(
                y_true,
                y_pred,
                average="macro",
                zero_division=0
            ),

        "recall":

            recall_score(
                y_true,
                y_pred,
                average="macro",
                zero_division=0
            ),

        "macro_f1":

            f1_score(
                y_true,
                y_pred,
                average="macro",
                zero_division=0
            )

    }

print(
    "✓ Metric function ready."
)

# ============================================================
# 10. Save Checkpoint
# ============================================================

def save_checkpoint(

    model_to_save, # Renamed parameter to avoid confusion with global variable

    optimizer,

    scheduler,

    epoch,

    val_macro_f1,

    path

):

    torch.save(

        {

            "epoch":
                epoch,

            "model_state_dict":
                model_to_save.state_dict(), # Corrected: Use model_to_save

            "optimizer_state_dict":
                optimizer.state_dict(),

            "scheduler_state_dict":
                scheduler.state_dict(),

            "val_macro_f1":
                val_macro_f1,

            "label_classes":
                list(
                    label_encoder.classes_
                ),

            "symptom_vocabulary":
                symptom_vocab,

            "num_diseases":
                NUM_DISEASES,

            "num_symptoms":
                NUM_SYMPTOMS,

            "max_visits":
                MAX_VISITS,

            "min_year":
                MIN_YEAR,

            "max_year":
                MAX_YEAR

        },

        path

    )

# ============================================================
# 11. Save History
# ============================================================

def save_history():

    history_df = pd.DataFrame(
        history
    )

    history_df.to_csv(

        os.path.join(

            OUTPUT_DIR,

            "training_history.csv"

        ),

        index=False

    )

# ============================================================
# 12. Save Experiment Configuration
# ============================================================

experiment_config = {

    "architecture":
        "STraT-Net V2",

    "dataset":
        "stratnet_v2",

    "training_patients":
        len(
            train_patient_dataset
        ),

    "validation_patients":
        len(
            val_patient_dataset
        ),

    "testing_patients":
        len(
            test_patient_dataset
        ),

    "num_diseases":
        NUM_DISEASES,

    "num_symptoms":
        NUM_SYMPTOMS,

    "max_visits":
        MAX_VISITS,

    "embedding_dim":
        EMBED_DIM,

    "transformer_layers":
        NUM_LAYERS,

    "attention_heads":
        NUM_HEADS,

    "ff_dim":
        FF_DIM,

    "epochs":
        NUM_EPOCHS,

    "learning_rate":
        LEARNING_RATE,

    "weight_decay":
        WEIGHT_DECAY,

    "label_smoothing":
        LABEL_SMOOTHING,

    "early_stopping_patience":
        EARLY_STOPPING_PATIENCE,

    "use_symptom_loss":
        USE_SYMPTOM_LOSS,

    "device":
        str(
            DEVICE
        )

}

with open(

    os.path.join(

        OUTPUT_DIR,

        "experiment_config.json"

    ),

    "w"

) as f:

    json.dump(

        experiment_config,

        f,

        indent=4

    )

print(
    "\n✓ Experiment configuration saved."
)

# ============================================================
# 13. Final Sanity Check
# ============================================================

print("\n" + "=" * 70)

print("TRAINING SETUP STATUS")

print("=" * 70)

print(
    "Model Device          :",
    # Corrected: Use stratnet_model.parameters()
    next(
        stratnet_model.parameters()
    ).device
)

print(
    "Optimizer             :",
    type(
        optimizer
    ).__name__
)

print(
    "Scheduler             :",
    type(
        scheduler
    ).__name__
)

print(
    "Disease Loss          :",
    type(
        disease_loss_fn
    ).__name__
)

print(
    "Class Weights         :",
    class_weights
)

print(
    "Train Batches         :",
    len(
        train_patient_loader
    )
)

print(
    "Validation Batches    :",
    len(
        val_patient_loader
    )
)

print(
    "Test Batches          :",
    len(
        test_patient_loader
    )
)

print("=" * 70)

print(
    "\nSTraT-Net V2 Training Setup READY"
)

print("=" * 70)

# ============================================================
# 14. Module Complete
# ============================================================

print("\n" + "=" * 70)
print("Module 5A Completed Successfully")
print("=" * 70)


In [ ]:
# ============================================================
# STraT-Net V2
# Module 5B
# Training + Validation Functions
# ============================================================

import numpy as np
import torch

from tqdm.auto import tqdm

print("=" * 70)
print("STraT-Net V2 : Module 5B")
print("Training + Validation Functions")
print("=" * 70)

# ============================================================
# 1. Epoch Storage Helper
# ============================================================

def initialize_epoch_storage():

    return {

        "labels": [],

        "predictions": [],

        "probabilities": [],

        "attention": [],

        "patient_ids": []

    }

# ============================================================
# 2. Train One Epoch
# ============================================================

def train_one_epoch(

    model,

    dataloader,

    optimizer,

    criterion,

    device,

    scaler,

    epoch

):

    model.train()

    storage = initialize_epoch_storage()

    running_loss = 0.0

    progress_bar = tqdm(

        dataloader,

        total=len(dataloader),

        desc=f"Epoch {epoch} [Train]"

    )

    for batch_idx, batch in enumerate(

        progress_bar

    ):

        # --------------------------------------------
        # Move batch to device
        # --------------------------------------------

        trajectory = (

            batch[
                "trajectory"
            ]

            .float()

            .to(
                device,
                non_blocking=True
            )

        )

        years = (

            batch[
                "years"
            ]

            .long()

            .to(
                device,
                non_blocking=True
            )

        )

        mask = (

            batch[
                "mask"
            ]

            .bool()

            .to(
                device,
                non_blocking=True
            )

        )

        labels = (

            batch[
                "label"
            ]

            .long()

            .to(
                device,
                non_blocking=True
            )

        )

        # --------------------------------------------
        # Reset gradients
        # --------------------------------------------

        optimizer.zero_grad(
            set_to_none=True
        )

        # --------------------------------------------
        # Forward pass with AMP
        # --------------------------------------------

        if scaler is not None:

            with torch.cuda.amp.autocast():

                outputs = model(

                    trajectory,

                    years,

                    mask

                )

                logits = outputs[
                    "disease_logits"
                ]

                loss = criterion(

                    logits,

                    labels

                )

            # ----------------------------------------
            # Backward pass
            # ----------------------------------------

            scaler.scale(
                loss
            ).backward()

            # ----------------------------------------
            # Unscale before clipping
            # IMPORTANT for correct AMP gradient clip
            # ----------------------------------------

            scaler.unscale_(
                optimizer
            )

            torch.nn.utils.clip_grad_norm_(

                model.parameters(),

                GRAD_CLIP

            )

            scaler.step(
                optimizer
            )

            scaler.update()

        else:

            outputs = model(

                trajectory,

                years,

                mask

            )

            logits = outputs[
                "disease_logits"
            ]

            loss = criterion(

                logits,

                labels

            )

            loss.backward()

            torch.nn.utils.clip_grad_norm_(

                model.parameters(),

                GRAD_CLIP

            )

            optimizer.step()

        # --------------------------------------------
        # Running loss
        # --------------------------------------------

        running_loss += (
            loss.item()
        )

        # --------------------------------------------
        # Probabilities + predictions
        # --------------------------------------------

        probabilities = torch.softmax(

            logits,

            dim=1

        )

        predictions = torch.argmax(

            probabilities,

            dim=1

        )

        # --------------------------------------------
        # Store outputs
        # --------------------------------------------

        storage[
            "labels"
        ].extend(

            labels
            .detach()
            .cpu()
            .numpy()
            .tolist()

        )

        storage[
            "predictions"
        ].extend(

            predictions
            .detach()
            .cpu()
            .numpy()
            .tolist()

        )

        storage[
            "probabilities"
        ].extend(

            probabilities
            .detach()
            .cpu()
            .numpy()

        )

        storage[
            "attention"
        ].extend(

            outputs[
                "attention"
            ]
            .detach()
            .cpu()
            .numpy()

        )

        storage[
            "patient_ids"
        ].extend(

            list(
                batch[
                    "patient_id"
                ]
            )

        )

        # --------------------------------------------
        # Progress display
        # --------------------------------------------

        average_loss = (

            running_loss

            /

            (
                batch_idx
                +
                1
            )

        )

        progress_bar.set_postfix({

            "loss":
                f"{average_loss:.4f}"

        })

    # ========================================================
    # Epoch Metrics
    # ========================================================

    metrics = compute_metrics(

        storage[
            "labels"
        ],

        storage[
            "predictions"
        ]

    )

    epoch_loss = (

        running_loss

        /

        len(
            dataloader
        )

    )

    results = {

        "loss":
            epoch_loss,

        "accuracy":
            metrics[
                "accuracy"
            ],

        "precision":
            metrics[
                "precision"
            ],

        "recall":
            metrics[
                "recall"
            ],

        "macro_f1":
            metrics[
                "macro_f1"
            ],

        "labels":
            np.asarray(
                storage[
                    "labels"
                ]
            ),

        "predictions":
            np.asarray(
                storage[
                    "predictions"
                ]
            ),

        "probabilities":
            np.asarray(
                storage[
                    "probabilities"
                ]
            ),

        "attention":
            np.asarray(
                storage[
                    "attention"
                ]
            ),

        "patient_ids":
            storage[
                "patient_ids"
            ]

    }

    print("\nTraining Summary")
    print("-" * 60)

    print(
        f"Loss      : "
        f"{results['loss']:.4f}"
    )

    print(
        f"Accuracy  : "
        f"{results['accuracy']:.4f}"
    )

    print(
        f"Precision : "
        f"{results['precision']:.4f}"
    )

    print(
        f"Recall    : "
        f"{results['recall']:.4f}"
    )

    print(
        f"Macro-F1  : "
        f"{results['macro_f1']:.4f}"
    )

    return results

# ============================================================
# 3. Validate One Epoch
# ============================================================

def validate_one_epoch(

    model,

    dataloader,

    criterion,

    device,

    epoch

):

    model.eval()

    storage = initialize_epoch_storage()

    running_loss = 0.0

    progress_bar = tqdm(

        dataloader,

        total=len(dataloader),

        desc=f"Epoch {epoch} [Validation]"

    )

    with torch.no_grad():

        for batch_idx, batch in enumerate(

            progress_bar

        ):

            # ----------------------------------------
            # Move batch to device
            # ----------------------------------------

            trajectory = (

                batch[
                    "trajectory"
                ]

                .float()

                .to(
                    device,
                    non_blocking=True
                )

            )

            years = (

                batch[
                    "years"
                ]

                .long()

                .to(
                    device,
                    non_blocking=True
                )

            )

            mask = (

                batch[
                    "mask"
                ]

                .bool()

                .to(
                    device,
                    non_blocking=True
                )

            )

            labels = (

                batch[
                    "label"
                ]

                .long()

                .to(
                    device,
                    non_blocking=True
                )

            )

            # ----------------------------------------
            # Forward pass
            # ----------------------------------------

            if USE_AMP:

                with torch.cuda.amp.autocast():

                    outputs = model(

                        trajectory,

                        years,

                        mask

                    )

                    logits = outputs[
                        "disease_logits"
                    ]

                    loss = criterion(

                        logits,

                        labels

                    )

            else:

                outputs = model(

                    trajectory,

                    years,

                    mask

                )

                logits = outputs[
                    "disease_logits"
                ]

                loss = criterion(

                    logits,

                    labels

                )

            # ----------------------------------------
            # Running loss
            # ----------------------------------------

            running_loss += (
                loss.item()
            )

            probabilities = torch.softmax(

                logits,

                dim=1

            )

            predictions = torch.argmax(

                probabilities,

                dim=1

            )

            # ----------------------------------------
            # Store results
            # ----------------------------------------

            storage[
                "labels"
            ].extend(

                labels
                .cpu()
                .numpy()
                .tolist()

            )

            storage[
                "predictions"
            ].extend(

                predictions
                .cpu()
                .numpy()
                .tolist()

            )

            storage[
                "probabilities"
            ].extend(

                probabilities
                .cpu()
                .numpy()

            )

            storage[
                "attention"
            ].extend(

                outputs[
                    "attention"
                ]
                .cpu()
                .numpy()

            )

            storage[
                "patient_ids"
            ].extend(

                list(
                    batch[
                        "patient_id"
                    ]
                )

            )

            # ----------------------------------------
            # Progress display
            # ----------------------------------------

            average_loss = (

                running_loss

                /

                (
                    batch_idx
                    +
                    1
                )

            )

            progress_bar.set_postfix({

                "loss":
                    f"{average_loss:.4f}"

            })

    # ========================================================
    # Metrics
    # ========================================================

    metrics = compute_metrics(

        storage[
            "labels"
        ],

        storage[
            "predictions"
        ]

    )

    epoch_loss = (

        running_loss

        /

        len(
            dataloader
        )

    )

    results = {

        "loss":
            epoch_loss,

        "accuracy":
            metrics[
                "accuracy"
            ],

        "precision":
            metrics[
                "precision"
            ],

        "recall":
            metrics[
                "recall"
            ],

        "macro_f1":
            metrics[
                "macro_f1"
            ],

        "labels":
            np.asarray(
                storage[
                    "labels"
                ]
            ),

        "predictions":
            np.asarray(
                storage[
                    "predictions"
                ]
            ),

        "probabilities":
            np.asarray(
                storage[
                    "probabilities"
                ]
            ),

        "attention":
            np.asarray(
                storage[
                    "attention"
                ]
            ),

        "patient_ids":
            storage[
                "patient_ids"
            ]

    }

    print("\nValidation Summary")
    print("-" * 60)

    print(
        f"Loss      : "
        f"{results['loss']:.4f}"
    )

    print(
        f"Accuracy  : "
        f"{results['accuracy']:.4f}"
    )

    print(
        f"Precision : "
        f"{results['precision']:.4f}"
    )

    print(
        f"Recall    : "
        f"{results['recall']:.4f}"
    )

    print(
        f"Macro-F1  : "
        f"{results['macro_f1']:.4f}"
    )

    return results

# ============================================================
# 4. Function Status
# ============================================================

print("\n" + "=" * 70)

print("FUNCTION STATUS")

print("=" * 70)

print(
    "train_one_epoch()    ✓"
)

print(
    "validate_one_epoch() ✓"
)

print("=" * 70)

print("\n" + "=" * 70)
print("Module 5B Completed Successfully")
print("=" * 70)

In [ ]:
# ============================================================
# STraT-Net V2
# Module 5C
# Complete Controlled Training Loop
# ============================================================

import os
import pandas as pd
import torch

print("=" * 70)
print("STraT-Net V2 : Module 5C")
print("Complete Controlled Training Loop")
print("=" * 70)

# ============================================================
# 1. Confirm Fresh Training State
# ============================================================

print("\nPre-Training Check")
print("-" * 60)

print(
    "Model Device       :",
    next(stratnet_model.parameters()).device
)

print(
    "Current LR         :",
    optimizer.param_groups[0]["lr"]
)

print(
    "Best Val Macro-F1  :",
    best_val_f1
)

print(
    "Best Epoch         :",
    best_epoch
)

print(
    "History Epochs     :",
    len(history["epoch"])
)

if len(history["epoch"]) != 0:

    raise RuntimeError(
        "Training history is not empty. "
        "Restart from Module 5A before beginning "
        "a fresh controlled experiment."
    )

# ============================================================
# 2. Complete Training Function
# ============================================================

def train_model():

    global best_val_f1
    global best_epoch
    global early_stop_counter
    global history

    print("\n" + "=" * 70)

    print("Starting STraT-Net V2 Training")

    print("=" * 70)

    for epoch in range(
        1,
        NUM_EPOCHS + 1
    ):

        print("\n" + "=" * 70)

        print(
            f"EPOCH {epoch}/{NUM_EPOCHS}"
        )

        print("=" * 70)

        # ====================================================
        # TRAINING
        # ====================================================

        train_results = train_one_epoch(

            model=stratnet_model,

            dataloader=train_patient_loader,

            optimizer=optimizer,

            criterion=disease_loss_fn,

            device=DEVICE,

            scaler=scaler,

            epoch=epoch

        )

        # ====================================================
        # VALIDATION
        # ====================================================

        val_results = validate_one_epoch(

            model=stratnet_model,

            dataloader=val_patient_loader,

            criterion=disease_loss_fn,

            device=DEVICE,

            epoch=epoch

        )

        # ====================================================
        # Capture current LR before scheduler update
        # ====================================================

        current_lr = (
            optimizer
            .param_groups[0]["lr"]
        )

        # ====================================================
        # Update Training History
        # ====================================================

        history[
            "epoch"
        ].append(
            epoch
        )

        history[
            "train_loss"
        ].append(
            train_results["loss"]
        )

        history[
            "val_loss"
        ].append(
            val_results["loss"]
        )

        history[
            "train_accuracy"
        ].append(
            train_results["accuracy"]
        )

        history[
            "val_accuracy"
        ].append(
            val_results["accuracy"]
        )

        history[
            "train_macro_f1"
        ].append(
            train_results["macro_f1"]
        )

        history[
            "val_macro_f1"
        ].append(
            val_results["macro_f1"]
        )

        history[
            "learning_rate"
        ].append(
            current_lr
        )

        # ====================================================
        # Save History
        # ====================================================

        save_history()

        # ====================================================
        # Best Checkpoint Selection
        #
        # Validation Macro-F1 ONLY
        # ====================================================

        current_val_f1 = (
            val_results[
                "macro_f1"
            ]
        )

        improved = (
            current_val_f1
            >
            best_val_f1
        )

        if improved:

            best_val_f1 = (
                current_val_f1
            )

            best_epoch = (
                epoch
            )

            early_stop_counter = 0

            save_checkpoint(

                model_to_save=stratnet_model,

                optimizer=optimizer,

                scheduler=scheduler,

                epoch=epoch,

                val_macro_f1=(
                    current_val_f1
                ),

                path=BEST_MODEL_PATH

            )

            print(
                "\n✓ New best model saved."
            )

            print(
                f"Best Validation Macro-F1: "
                f"{best_val_f1:.4f}"
            )

        else:

            early_stop_counter += 1

            print(
                "\nNo validation improvement."
            )

            print(
                "Early stopping counter:",
                f"{early_stop_counter}/"
                f"{EARLY_STOPPING_PATIENCE}"
            )

        # ====================================================
        # Save Last Checkpoint
        # ====================================================

        save_checkpoint(

            model_to_save=stratnet_model,

            optimizer=optimizer,

            scheduler=scheduler,

            epoch=epoch,

            val_macro_f1=(
                current_val_f1
            ),

            path=LAST_MODEL_PATH

        )

        # ====================================================
        # Scheduler Step
        # ====================================================

        scheduler.step()

        next_lr = (
            optimizer
            .param_groups[0]["lr"]
        )

        # ====================================================
        # Epoch Summary
        # ====================================================

        print("\n" + "-" * 60)

        print("EPOCH SUMMARY")

        print("-" * 60)

        print(
            f"Train Loss       : "
            f"{train_results['loss']:.4f}"
        )

        print(
            f"Validation Loss  : "
            f"{val_results['loss']:.4f}"
        )

        print()

        print(
            f"Train Accuracy   : "
            f"{train_results['accuracy']:.4f}"
        )

        print(
            f"Val Accuracy     : "
            f"{val_results['accuracy']:.4f}"
        )

        print()

        print(
            f"Train Macro-F1   : "
            f"{train_results['macro_f1']:.4f}"
        )

        print(
            f"Val Macro-F1     : "
            f"{val_results['macro_f1']:.4f}"
        )

        print()

        print(
            f"Current LR       : "
            f"{current_lr:.8f}"
        )

        print(
            f"Next Epoch LR    : "
            f"{next_lr:.8f}"
        )

        print()

        print(
            f"Best Val Macro-F1: "
            f"{best_val_f1:.4f}"
        )

        print(
            f"Best Epoch       : "
            f"{best_epoch}"
        )

        print("-" * 60)

        # ====================================================
        # Early Stopping
        # ====================================================

        if (
            early_stop_counter
            >=
            EARLY_STOPPING_PATIENCE
        ):

            print(
                "\n" + "=" * 70
            )

            print(
                "EARLY STOPPING TRIGGERED"
            )

            print("=" * 70)

            print(
                "No improvement in validation "
                f"Macro-F1 for "
                f"{EARLY_STOPPING_PATIENCE} epochs."
            )

            break

    # ========================================================
    # Final Training Summary
    # ========================================================

    history_df = pd.DataFrame(
        history
    )

    print("\n" + "=" * 70)

    print("TRAINING FINISHED")

    print("=" * 70)

    print(
        "Epochs Completed :",
        len(
            history_df
        )
    )

    print(
        "Best Epoch       :",
        best_epoch
    )

    print(
        "Best Val Macro-F1:",
        f"{best_val_f1:.4f}"
    )

    print(
        "\nBest Checkpoint:"
    )

    print(
        BEST_MODEL_PATH
    )

    print(
        "Exists:",
        os.path.exists(
            BEST_MODEL_PATH
        )
    )

    print(
        "\nLast Checkpoint:"
    )

    print(
        LAST_MODEL_PATH
    )

    print(
        "Exists:",
        os.path.exists(
            LAST_MODEL_PATH
        )
    )

    print("=" * 70)

    return history_df

# ============================================================
# 3. RUN THE CONTROLLED TRAINING EXPERIMENT
# ============================================================

history_df = train_model()

# ============================================================
# 4. Display Final History
# ============================================================

print("\nFinal Training History")
print("-" * 60)

display(
    history_df
)

print("\n" + "=" * 70)
print("Module 5C Completed Successfully")
print("=" * 70)

In [ ]:
# ============================================================
# Fix DataLoader Multiprocessing Issue
# Use Stable Single-Process Loading for Evaluation
# ============================================================

from torch.utils.data import DataLoader

train_patient_loader = DataLoader(
    train_patient_dataset,
    batch_size=cfg.BATCH_SIZE,
    shuffle=True,
    num_workers=0,
    pin_memory=torch.cuda.is_available()
)

val_patient_loader = DataLoader(
    val_patient_dataset,
    batch_size=cfg.BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    pin_memory=torch.cuda.is_available()
)

test_patient_loader = DataLoader(
    test_patient_dataset,
    batch_size=cfg.BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    pin_memory=torch.cuda.is_available()
)

print("=" * 60)
print("Stable DataLoaders Recreated")
print("=" * 60)

print(
    "Train batches :",
    len(train_patient_loader)
)

print(
    "Val batches   :",
    len(val_patient_loader)
)

print(
    "Test batches  :",
    len(test_patient_loader)
)

print(
    "num_workers   : 0"
)

print("=" * 60)

In [ ]:
# ============================================================
# STraT-Net V2
# Module 5D
# Final Held-Out Test Evaluation
# ============================================================

import os
import numpy as np
import pandas as pd
import torch

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report,
    roc_auc_score,
    average_precision_score
)

from sklearn.preprocessing import label_binarize

print("=" * 70)
print("STraT-Net V2 : Module 5D")
print("FINAL HELD-OUT TEST EVALUATION")
print("=" * 70)

# ============================================================
# 1. Load Best Validation-Selected Checkpoint
# ============================================================

print("\nLoading best checkpoint...")
print("-" * 60)

checkpoint = torch.load(
    BEST_MODEL_PATH,
    map_location=DEVICE
)

print(
    "Checkpoint Epoch       :",
    checkpoint["epoch"]
)

print(
    "Checkpoint Val Macro-F1:",
    checkpoint["val_macro_f1"]
)

stratnet_model.load_state_dict(
    checkpoint[
        "model_state_dict"
    ]
)

stratnet_model = stratnet_model.to(
    DEVICE
)

stratnet_model.eval()

print(
    "\n✓ Best validation-selected model loaded."
)

# ============================================================
# 2. Confirm Expected Best Epoch
# ============================================================

if checkpoint["epoch"] != best_epoch:

    print(
        "\nWARNING:"
    )

    print(
        "Checkpoint epoch differs from "
        "in-memory best_epoch."
    )

else:

    print(
        "✓ Checkpoint epoch matches best epoch."
    )

# ============================================================
# 3. Test Prediction Storage
# ============================================================

all_patient_ids = []

all_labels = []

all_predictions = []

all_probabilities = []

all_attention = []

all_years = []

all_masks = []

all_lengths = []

# ============================================================
# 4. Run Test Inference
# ============================================================

print("\nRunning inference on held-out test set...")
print("-" * 60)

with torch.no_grad():

    for batch in test_patient_loader:

        trajectory = (
            batch["trajectory"]
            .float()
            .to(
                DEVICE,
                non_blocking=True
            )
        )

        years = (
            batch["years"]
            .long()
            .to(
                DEVICE,
                non_blocking=True
            )
        )

        mask = (
            batch["mask"]
            .bool()
            .to(
                DEVICE,
                non_blocking=True
            )
        )

        labels = (
            batch["label"]
            .long()
            .to(
                DEVICE,
                non_blocking=True
            )
        )

        outputs = stratnet_model(
            trajectory,
            years,
            mask
        )

        logits = outputs[
            "disease_logits"
        ]

        probabilities = torch.softmax(
            logits,
            dim=1
        )

        predictions = torch.argmax(
            probabilities,
            dim=1
        )

        # --------------------------------------------
        # Store results
        # --------------------------------------------

        all_patient_ids.extend(
            list(
                batch["patient_id"]
            )
        )

        all_labels.extend(
            labels
            .cpu()
            .numpy()
            .tolist()
        )

        all_predictions.extend(
            predictions
            .cpu()
            .numpy()
            .tolist()
        )

        all_probabilities.extend(
            probabilities
            .cpu()
            .numpy()
        )

        all_attention.extend(
            outputs[
                "attention"
            ]
            .cpu()
            .numpy()
        )

        all_years.extend(
            batch[
                "years"
            ]
            .cpu()
            .numpy()
        )

        all_masks.extend(
            batch[
                "mask"
            ]
            .cpu()
            .numpy()
        )

        all_lengths.extend(
            batch[
                "length"
            ]
            .cpu()
            .numpy()
            .tolist()
        )

# ============================================================
# 5. Convert to NumPy
# ============================================================

y_true = np.asarray(
    all_labels
)

y_pred = np.asarray(
    all_predictions
)

y_prob = np.asarray(
    all_probabilities
)

attention_array = np.asarray(
    all_attention
)

years_array = np.asarray(
    all_years
)

masks_array = np.asarray(
    all_masks
)

lengths_array = np.asarray(
    all_lengths
)

print(
    "\nTest Patients Evaluated:",
    len(y_true)
)

if len(y_true) != len(
    test_patient_dataset
):

    raise ValueError(
        "Test prediction count does not "
        "match test dataset size."
    )

print(
    "✓ All test patients evaluated exactly once."
)

# ============================================================
# 6. Overall Classification Metrics
# ============================================================

test_accuracy = accuracy_score(
    y_true,
    y_pred
)

test_macro_precision = precision_score(
    y_true,
    y_pred,
    average="macro",
    zero_division=0
)

test_macro_recall = recall_score(
    y_true,
    y_pred,
    average="macro",
    zero_division=0
)

test_macro_f1 = f1_score(
    y_true,
    y_pred,
    average="macro",
    zero_division=0
)

test_weighted_f1 = f1_score(
    y_true,
    y_pred,
    average="weighted",
    zero_division=0
)

print("\n" + "=" * 70)

print("FINAL TEST METRICS")

print("=" * 70)

print(
    f"Accuracy        : "
    f"{test_accuracy:.4f}"
)

print(
    f"Macro Precision : "
    f"{test_macro_precision:.4f}"
)

print(
    f"Macro Recall    : "
    f"{test_macro_recall:.4f}"
)

print(
    f"Macro F1        : "
    f"{test_macro_f1:.4f}"
)

print(
    f"Weighted F1     : "
    f"{test_weighted_f1:.4f}"
)

print("=" * 70)

# ============================================================
# 7. Confusion Matrix
# ============================================================

cm = confusion_matrix(
    y_true,
    y_pred,
    labels=np.arange(
        NUM_DISEASES
    )
)

print("\nConfusion Matrix")
print("-" * 60)

print(
    cm
)

# ============================================================
# 8. Classification Report
# ============================================================

class_names = list(
    label_encoder.classes_
)

print("\nDisease Mapping")
print("-" * 60)

for i, disease in enumerate(
    class_names
):

    print(
        f"{i} -> {disease}"
    )

print("\nClassification Report")
print("-" * 60)

report_text = classification_report(
    y_true,
    y_pred,
    labels=np.arange(
        NUM_DISEASES
    ),
    target_names=class_names,
    digits=4,
    zero_division=0
)

print(
    report_text
)

# ============================================================
# 9. Per-Class Sensitivity + Specificity
# ============================================================

per_class_results = []

for class_index, disease_name in enumerate(
    class_names
):

    TP = cm[
        class_index,
        class_index
    ]

    FN = (
        cm[
            class_index,
            :
        ].sum()
        -
        TP
    )

    FP = (
        cm[
            :,
            class_index
        ].sum()
        -
        TP
    )

    TN = (
        cm.sum()
        -
        TP
        -
        FN
        -
        FP
    )

    sensitivity = (
        TP
        /
        (
            TP
            +
            FN
        )
        if (
            TP
            +
            FN
        ) > 0
        else 0
    )

    specificity = (
        TN
        /
        (
            TN
            +
            FP
        )
        if (
            TN
            +
            FP
        ) > 0
        else 0
    )

    class_precision = (
        TP
        /
        (
            TP
            +
            FP
        )
        if (
            TP
            +
            FP
        ) > 0
        else 0
    )

    class_f1 = (

        2
        *
        class_precision
        *
        sensitivity
        /
        (
            class_precision
            +
            sensitivity
        )

        if (
            class_precision
            +
            sensitivity
        ) > 0

        else 0

    )

    per_class_results.append({

        "Disease":
            disease_name,

        "Sensitivity":
            sensitivity,

        "Specificity":
            specificity,

        "Precision":
            class_precision,

        "F1":
            class_f1,

        "Support":
            int(
                cm[
                    class_index,
                    :
                ].sum()
            )

    })

per_class_df = pd.DataFrame(
    per_class_results
)

print(
    "\nPer-Disease Metrics"
)

print("-" * 60)

display(
    per_class_df
)

# ============================================================
# 10. Multiclass ROC-AUC
# ============================================================

y_true_binary = label_binarize(

    y_true,

    classes=np.arange(
        NUM_DISEASES
    )

)

try:

    macro_roc_auc = roc_auc_score(

        y_true_binary,

        y_prob,

        average="macro",

        multi_class="ovr"

    )

    weighted_roc_auc = roc_auc_score(

        y_true_binary,

        y_prob,

        average="weighted",

        multi_class="ovr"

    )

except ValueError:

    macro_roc_auc = np.nan

    weighted_roc_auc = np.nan

print("\nROC-AUC")
print("-" * 60)

print(
    f"Macro ROC-AUC    : "
    f"{macro_roc_auc:.4f}"
)

print(
    f"Weighted ROC-AUC : "
    f"{weighted_roc_auc:.4f}"
)

# ============================================================
# 11. Precision-Recall AUC
# ============================================================

try:

    macro_pr_auc = (
        average_precision_score(

            y_true_binary,

            y_prob,

            average="macro"

        )
    )

    weighted_pr_auc = (
        average_precision_score(

            y_true_binary,

            y_prob,

            average="weighted"

        )
    )

except ValueError:

    macro_pr_auc = np.nan

    weighted_pr_auc = np.nan

print("\nPrecision-Recall AUC")
print("-" * 60)

print(
    f"Macro PR-AUC    : "
    f"{macro_pr_auc:.4f}"
)

print(
    f"Weighted PR-AUC : "
    f"{weighted_pr_auc:.4f}"
)

# ============================================================
# 12. Prediction Confidence
# ============================================================

prediction_confidence = (
    y_prob.max(
        axis=1
    )
)

correct_mask = (
    y_pred
    ==
    y_true
)

incorrect_mask = (
    y_pred
    !=
    y_true
)

print("\nPrediction Confidence")
print("-" * 60)

print(
    "Mean Confidence Overall :",
    f"{prediction_confidence.mean():.4f}"
)

if correct_mask.any():

    print(
        "Mean Confidence Correct :",
        f"{prediction_confidence[correct_mask].mean():.4f}"
    )

if incorrect_mask.any():

    print(
        "Mean Confidence Errors  :",
        f"{prediction_confidence[incorrect_mask].mean():.4f}"
    )

# ============================================================
# 13. Build Patient-Level Prediction Table
# ============================================================

prediction_df = pd.DataFrame({

    "patient_id":
        all_patient_ids,

    "true_label":
        y_true,

    "true_disease":
        [
            class_names[i]
            for i in y_true
        ],

    "predicted_label":
        y_pred,

    "predicted_disease":
        [
            class_names[i]
            for i in y_pred
        ],

    "confidence":
        prediction_confidence,

    "correct":
        correct_mask,

    "trajectory_length":
        lengths_array

})

# Add class probabilities

for class_index, disease_name in enumerate(
    class_names
):

    safe_name = (
        disease_name
        .lower()
        .replace(
            " ",
            "_"
        )
        .replace(
            "'",
            ""
        )
    )

    prediction_df[
        f"prob_{safe_name}"
    ] = y_prob[
        :,
        class_index
    ]

print("\nPatient Prediction Table")
print("-" * 60)

display(
    prediction_df.head(
        10
    )
)

# ============================================================
# 14. Misclassification Table
# ============================================================

misclassified_df = (

    prediction_df[
        ~prediction_df[
            "correct"
        ]
    ]

    .sort_values(

        "confidence",

        ascending=False

    )

    .reset_index(
        drop=True
    )

)

print(
    "\nMisclassified Patients:",
    len(
        misclassified_df
    )
)

print(
    "Misclassification Rate:",
    f"{len(misclassified_df) / len(prediction_df):.4f}"
)

display(
    misclassified_df.head(
        20
    )
)

# ============================================================
# 15. Accuracy by Trajectory Length
# ============================================================

length_analysis = (

    prediction_df

    .groupby(
        "trajectory_length"
    )

    .agg(

        patients=(
            "patient_id",
            "count"
        ),

        accuracy=(
            "correct",
            "mean"
        ),

        mean_confidence=(
            "confidence",
            "mean"
        )

    )

    .reset_index()

)

print("\nPerformance by Trajectory Length")
print("-" * 60)

display(
    length_analysis
)

# ============================================================
# 16. Save All Results
# ============================================================

prediction_df.to_csv(

    os.path.join(

        OUTPUT_DIR,

        "test_patient_predictions.csv"

    ),

    index=False

)

misclassified_df.to_csv(

    os.path.join(

        OUTPUT_DIR,

        "misclassified_patients.csv"

    ),

    index=False

)

per_class_df.to_csv(

    os.path.join(

        OUTPUT_DIR,

        "per_disease_metrics.csv"

    ),

    index=False

)

length_analysis.to_csv(

    os.path.join(

        OUTPUT_DIR,

        "performance_by_trajectory_length.csv"

    ),

    index=False

)

np.save(

    os.path.join(

        OUTPUT_DIR,

        "confusion_matrix.npy"

    ),

    cm

)

# ============================================================
# 17. Save Overall Metrics
# ============================================================

overall_metrics = pd.DataFrame({

    "Metric": [

        "Accuracy",

        "Macro Precision",

        "Macro Recall",

        "Macro F1",

        "Weighted F1",

        "Macro ROC-AUC",

        "Weighted ROC-AUC",

        "Macro PR-AUC",

        "Weighted PR-AUC"

    ],

    "Value": [

        test_accuracy,

        test_macro_precision,

        test_macro_recall,

        test_macro_f1,

        test_weighted_f1,

        macro_roc_auc,

        weighted_roc_auc,

        macro_pr_auc,

        weighted_pr_auc

    ]

})

overall_metrics.to_csv(

    os.path.join(

        OUTPUT_DIR,

        "final_test_metrics.csv"

    ),

    index=False

)

# ============================================================
# 18. Final Summary
# ============================================================

print("\n" + "=" * 70)

print("FINAL STraT-Net V2 TEST SUMMARY")

print("=" * 70)

print(
    "Best Validation Epoch :",
    checkpoint[
        "epoch"
    ]
)

print(
    "Best Validation F1    :",
    f"{checkpoint['val_macro_f1']:.4f}"
)

print()

print(
    "Test Accuracy         :",
    f"{test_accuracy:.4f}"
)

print(
    "Test Macro-F1         :",
    f"{test_macro_f1:.4f}"
)

print(
    "Test Weighted-F1      :",
    f"{test_weighted_f1:.4f}"
)

print(
    "Test Macro ROC-AUC    :",
    f"{macro_roc_auc:.4f}"
)

print(
    "Test Macro PR-AUC     :",
    f"{macro_pr_auc:.4f}"
)

print()

print(
    "Test Patients         :",
    len(
        y_true
    )
)

print("=" * 70)

print(
    "\n✓ Final held-out test evaluation complete."
)

print(
    "✓ Results saved to:",
    OUTPUT_DIR
)

print("\n" + "=" * 70)

print("Module 5D Completed Successfully")

print("=" * 70)

In [ ]:
# ============================================================
# STraT-Net V2
# Experiment E1 Final Record
# ============================================================

import json
import os
import pandas as pd

E1_SUMMARY = {

    "experiment_id":
        "E1",

    "architecture":
        "STraT-Net V2",

    "attention":
        "Gated Temporal Attention",

    "input_features":
        11,

    "input_description":
        "Longitudinal symptom trajectory only",

    "training_patients":
        2100,

    "validation_patients":
        450,

    "test_patients":
        450,

    "best_epoch":
        5,

    "best_validation_macro_f1":
        0.4746261729016495,

    "test_accuracy":
        0.4644,

    "test_macro_precision":
        0.4676,

    "test_macro_recall":
        0.4644,

    "test_macro_f1":
        0.4595,

    "test_weighted_f1":
        0.4595,

    "test_macro_roc_auc":
        0.7864,

    "test_macro_pr_auc":
        0.5014,

    "notes":
        (
            "Disease-only supervised training. "
            "Overall severity not used as model input."
        )

}

E1_PATH = os.path.join(
    OUTPUT_DIR,
    "experiment_E1_summary.json"
)

with open(
    E1_PATH,
    "w"
) as f:

    json.dump(
        E1_SUMMARY,
        f,
        indent=4
    )

print("=" * 70)
print("EXPERIMENT E1 SAVED")
print("=" * 70)

for key, value in E1_SUMMARY.items():

    print(
        f"{key:<30}: {value}"
    )

print("\nSaved to:")
print(E1_PATH)

print("=" * 70)

### Grouping `trajectory_df` by `patient_id`

The `NameError: name 'group' is not defined` in your previous attempt occurred because the variable `group` is only instantiated when you iterate through the results of a `groupby()` operation.

Below, I will show you how to group `trajectory_df` by `patient_id` and then demonstrate how `group` is used within a loop. The previous code snippet, intended for `E2 CHANGE`, would typically be placed inside such a loop to process each patient's data individually.

In [ ]:
# Group the trajectory_df by 'patient_id'
grouped_trajectory_df = trajectory_df.groupby('patient_id', sort=False)

print(f"Number of patient groups: {len(grouped_trajectory_df)}")

# To access a specific group or iterate through them, you would do the following:
# Get the first patient's ID and their corresponding group
first_patient_id, first_group = next(iter(grouped_trajectory_df))

print(f"\nFirst patient ID: {first_patient_id}")
print("First patient's trajectory data (first 5 rows):")
display(first_group.head())

# You can then apply the logic from 'E2 CHANGE' to 'first_group' or any 'group' within a loop.
# For example, to process the 'symptom_vector' and 'overall_severity' for the first patient:
symptom_vectors_first_patient = np.stack(
    first_group["symptom_vector"].values
).astype(np.float32)

severity_values_first_patient = (
    first_group["overall_severity"]
    .astype(np.float32)
    .values
    .reshape(-1, 1)
)

vectors_first_patient = np.concatenate(
    [
        symptom_vectors_first_patient,
        severity_values_first_patient
    ],
    axis=1
)

length_first_patient = len(vectors_first_patient)

print(f"\nCombined vectors shape for first patient: {vectors_first_patient.shape}")
print(f"Length of combined vectors for first patient: {length_first_patient}")

# If 'E2 CHANGE' were part of a function or loop, 'group' would be the DataFrame for a single patient.
# For example:
# for patient_id, group in grouped_trajectory_df:
#     # Your 'E2 CHANGE' code would go here, using 'group' for the current patient
#     symptom_vectors = np.stack(group["symptom_vector"].values).astype(np.float32)
#     severity_values = group["overall_severity"].astype(np.float32).values.reshape(-1, 1)
#     vectors = np.concatenate([symptom_vectors, severity_values], axis=1)
#     length = len(vectors)
#     # ... further processing ...


In [ ]:
# ============================================================
# STraT-Net
# Experiment E2
# Module 3 — Severity-Augmented Patient Trajectories
# 11 Symptoms + Overall Severity
# ============================================================

import numpy as np
import torch

from torch.utils.data import Dataset, DataLoader

print("=" * 70)
print("STraT-Net : Experiment E2")
print("Severity-Augmented Patient Trajectory Construction")
print("=" * 70)

# ============================================================
# 1. E2 Feature Dimension
# ============================================================

E2_TRAJECTORY_DIM = (
    NUM_SYMPTOMS + 1
)

print(
    "\nSymptom Features :",
    NUM_SYMPTOMS
)

print(
    "Severity Features:",
    1
)

print(
    "Total E2 Features:",
    E2_TRAJECTORY_DIM
)

# ============================================================
# 2. Maximum Visits
# ============================================================

visit_counts = (
    trajectory_df
    .groupby("patient_id")
    .size()
)

MAX_VISITS = int(
    visit_counts.max()
)

print(
    "\nMaximum Visits:",
    MAX_VISITS
)

# ============================================================
# 3. Build E2 Patient Sequences
# ============================================================

def build_patient_sequences_e2(
    dataframe,
    max_visits
):

    patient_sequences = []

    patient_masks = []

    patient_labels = []

    patient_ids = []

    patient_years = []

    patient_lengths = []

    grouped = dataframe.groupby(
        "patient_id",
        sort=False
    )

    for patient_id, group in grouped:

        group = (
            group
            .sort_values("year")
            .reset_index(drop=True)
        )

        # --------------------------------------------
        # 11 symptom trajectory features
        # --------------------------------------------

        symptom_vectors = np.stack(

            group[
                "symptom_vector"
            ].values

        ).astype(
            np.float32
        )

        # --------------------------------------------
        # 1 overall severity feature
        # --------------------------------------------

        severity_values = (

            group[
                "overall_severity"
            ]

            .astype(
                np.float32
            )

            .values

            .reshape(
                -1,
                1
            )

        )

        # --------------------------------------------
        # Combine:
        #
        # 11 symptoms + 1 severity = 12 features
        # --------------------------------------------

        vectors = np.concatenate(

            [
                symptom_vectors,
                severity_values
            ],

            axis=1

        )

        length = len(
            vectors
        )

        # --------------------------------------------
        # Dimension safety check
        # --------------------------------------------

        if vectors.shape[1] != E2_TRAJECTORY_DIM:

            raise ValueError(

                f"Feature dimension mismatch "
                f"for patient {patient_id}. "
                f"Expected {E2_TRAJECTORY_DIM}, "
                f"got {vectors.shape[1]}."

            )

        if length > max_visits:

            raise ValueError(

                f"Patient {patient_id} has "
                f"{length} visits, exceeding "
                f"MAX_VISITS={max_visits}."

            )

        # --------------------------------------------
        # Years
        # --------------------------------------------

        years = (

            group[
                "year"
            ]

            .astype(
                int
            )

            .tolist()

        )

        # --------------------------------------------
        # Disease label
        # --------------------------------------------

        unique_labels = (

            group[
                "Disease_Label"
            ]

            .unique()

        )

        if len(unique_labels) != 1:

            raise ValueError(

                f"Patient {patient_id} "
                "has multiple labels."

            )

        label = int(
            unique_labels[0]
        )

        # --------------------------------------------
        # Allocate padded trajectory
        #
        # Shape:
        # (MAX_VISITS, 12)
        # --------------------------------------------

        padded_vectors = np.zeros(

            (
                max_visits,
                E2_TRAJECTORY_DIM
            ),

            dtype=np.float32

        )

        padded_vectors[
            :length
        ] = vectors

        # --------------------------------------------
        # Padding mask
        # --------------------------------------------

        mask = np.zeros(

            max_visits,

            dtype=bool

        )

        mask[
            :length
        ] = True

        # --------------------------------------------
        # Padded years
        # --------------------------------------------

        padded_years = np.zeros(

            max_visits,

            dtype=np.int64

        )

        padded_years[
            :length
        ] = np.asarray(

            years,

            dtype=np.int64

        )

        # --------------------------------------------
        # Store
        # --------------------------------------------

        patient_sequences.append(

            torch.tensor(

                padded_vectors,

                dtype=torch.float32

            )

        )

        patient_masks.append(

            torch.tensor(

                mask,

                dtype=torch.bool

            )

        )

        patient_labels.append(
            label
        )

        patient_ids.append(
            patient_id
        )

        patient_years.append(

            torch.tensor(

                padded_years,

                dtype=torch.long

            )

        )

        patient_lengths.append(
            length
        )

    return {

        "sequences":
            patient_sequences,

        "masks":
            patient_masks,

        "labels":
            patient_labels,

        "patient_ids":
            patient_ids,

        "years":
            patient_years,

        "lengths":
            patient_lengths

    }

# ============================================================
# 4. Build Train / Validation / Test E2 Sequences
# ============================================================

train_sequences_e2 = (
    build_patient_sequences_e2(

        train_traj_df,

        MAX_VISITS

    )
)

val_sequences_e2 = (
    build_patient_sequences_e2(

        val_traj_df,

        MAX_VISITS

    )
)

test_sequences_e2 = (
    build_patient_sequences_e2(

        test_traj_df,

        MAX_VISITS

    )
)

print(
    "\n✓ E2 patient trajectories built successfully."
)

# ============================================================
# 5. Patient Dataset
# ============================================================

class PatientTrajectoryDatasetE2(
    Dataset
):

    def __init__(
        self,
        data
    ):

        self.sequences = data[
            "sequences"
        ]

        self.masks = data[
            "masks"
        ]

        self.labels = data[
            "labels"
        ]

        self.patient_ids = data[
            "patient_ids"
        ]

        self.years = data[
            "years"
        ]

        self.lengths = data[
            "lengths"
        ]

    def __len__(
        self
    ):

        return len(
            self.labels
        )

    def __getitem__(
        self,
        idx
    ):

        return {

            "patient_id":
                self.patient_ids[
                    idx
                ],

            "trajectory":
                self.sequences[
                    idx
                ],

            "mask":
                self.masks[
                    idx
                ],

            "label":
                torch.tensor(

                    self.labels[
                        idx
                    ],

                    dtype=torch.long

                ),

            "years":
                self.years[
                    idx
                ],

            "length":
                torch.tensor(

                    self.lengths[
                        idx
                    ],

                    dtype=torch.long

                )

        }

# ============================================================
# 6. Create E2 Dataset Objects
# ============================================================

train_patient_dataset_e2 = (
    PatientTrajectoryDatasetE2(
        train_sequences_e2
    )
)

val_patient_dataset_e2 = (
    PatientTrajectoryDatasetE2(
        val_sequences_e2
    )
)

test_patient_dataset_e2 = (
    PatientTrajectoryDatasetE2(
        test_sequences_e2
    )
)

# ============================================================
# 7. Stable DataLoaders
# ============================================================

train_patient_loader_e2 = DataLoader(

    train_patient_dataset_e2,

    batch_size=cfg.BATCH_SIZE,

    shuffle=True,

    num_workers=0,

    pin_memory=torch.cuda.is_available()

)

val_patient_loader_e2 = DataLoader(

    val_patient_dataset_e2,

    batch_size=cfg.BATCH_SIZE,

    shuffle=False,

    num_workers=0,

    pin_memory=torch.cuda.is_available()

)

test_patient_loader_e2 = DataLoader(

    test_patient_dataset_e2,

    batch_size=cfg.BATCH_SIZE,

    shuffle=False,

    num_workers=0,

    pin_memory=torch.cuda.is_available()

)

print(
    "✓ E2 DataLoaders created."
)

# ============================================================
# 8. Verify E2 Batch
# ============================================================

sample_e2 = next(

    iter(
        train_patient_loader_e2
    )

)

print("\nE2 Batch Verification")
print("-" * 60)

print(
    "Trajectory Shape :",
    sample_e2[
        "trajectory"
    ].shape
)

print(
    "Mask Shape       :",
    sample_e2[
        "mask"
    ].shape
)

print(
    "Years Shape      :",
    sample_e2[
        "years"
    ].shape
)

print(
    "Labels Shape     :",
    sample_e2[
        "label"
    ].shape
)

print(
    "Lengths Shape    :",
    sample_e2[
        "length"
    ].shape
)

# ============================================================
# 9. Verify Severity Feature
# ============================================================

first_patient_length = int(

    sample_e2[
        "length"
    ][0].item()

)

first_patient_data = (

    sample_e2[
        "trajectory"
    ][
        0,
        :first_patient_length
    ]

)

print("\nFirst E2 Patient")
print("-" * 60)

print(
    "Patient ID:",
    sample_e2[
        "patient_id"
    ][0]
)

print(
    "Visits:",
    first_patient_length
)

print(
    "Feature Shape:",
    first_patient_data.shape
)

print(
    "\nSeverity column:"
)

print(
    first_patient_data[
        :,
        -1
    ]
)

# ============================================================
# 10. Final E2 Architecture Configuration
# ============================================================

TRAJECTORY_DIM = (
    E2_TRAJECTORY_DIM
)

SEQUENCE_LENGTH = (
    MAX_VISITS
)

print("\n" + "=" * 70)

print("EXPERIMENT E2 INPUT CONFIGURATION")

print("=" * 70)

print(
    "Symptoms             :",
    NUM_SYMPTOMS
)

print(
    "Severity Features    :",
    1
)

print(
    "Trajectory Dimension :",
    TRAJECTORY_DIM
)

print(
    "Maximum Visits       :",
    MAX_VISITS
)

print(
    "Expected Model Input :",
    f"(batch, {MAX_VISITS}, {TRAJECTORY_DIM})"
)

print("=" * 70)

# ============================================================
# 11. Assertions
# ============================================================

assert (
    sample_e2[
        "trajectory"
    ].shape[-1]
    ==
    12
)

assert (
    TRAJECTORY_DIM
    ==
    12
)

print(
    "\n✓ E2 trajectory dimension verified as 12."
)

print("\n" + "=" * 70)
print("E2 Module 3 Completed Successfully")
print("=" * 70)

In [ ]:
# ============================================================
# STraT-Net
# Experiment E2
# Module 4A
# Severity-Augmented Temporal Representation Network
# ============================================================

import math
import torch
import torch.nn as nn

print("=" * 70)
print("STraT-Net : Experiment E2")
print("Module 4A — Severity-Augmented Temporal Encoder")
print("=" * 70)

# ============================================================
# 1. E2 Model Hyperparameters
# Keep identical to E1 for fair comparison
# ============================================================

EMBED_DIM = 128
NUM_HEADS = 8
NUM_LAYERS = 4
FF_DIM = 512
DROPOUT = 0.20

# TRAJECTORY_DIM should already be 12
assert TRAJECTORY_DIM == 12

print("\nE2 Model Configuration")
print("-" * 60)

print("Trajectory Dimension :", TRAJECTORY_DIM)
print("Embedding Dimension  :", EMBED_DIM)
print("Attention Heads      :", NUM_HEADS)
print("Transformer Layers   :", NUM_LAYERS)
print("Feed Forward Dim     :", FF_DIM)
print("Dropout              :", DROPOUT)
print("Maximum Visits       :", MAX_VISITS)

if EMBED_DIM % NUM_HEADS != 0:
    raise ValueError(
        "EMBED_DIM must be divisible by NUM_HEADS."
    )

# ============================================================
# 2. Dynamic Year Range
# Same dataset, so should remain 2015–2025
# ============================================================

MIN_YEAR = int(
    trajectory_df["year"].min()
)

MAX_YEAR = int(
    trajectory_df["year"].max()
)

NUM_YEAR_VALUES = (
    MAX_YEAR - MIN_YEAR + 1
)

print("\nYear Range")
print("-" * 60)

print("Minimum Year :", MIN_YEAR)
print("Maximum Year :", MAX_YEAR)
print("Year Values  :", NUM_YEAR_VALUES)

# ============================================================
# 3. Positional Encoding
# ============================================================

class PositionalEncoding(nn.Module):

    def __init__(
        self,
        d_model,
        max_len
    ):

        super().__init__()

        pe = torch.zeros(
            max_len,
            d_model
        )

        position = torch.arange(
            0,
            max_len,
            dtype=torch.float32
        ).unsqueeze(1)

        div_term = torch.exp(
            torch.arange(
                0,
                d_model,
                2,
                dtype=torch.float32
            )
            *
            (
                -math.log(10000.0)
                / d_model
            )
        )

        pe[:, 0::2] = torch.sin(
            position * div_term
        )

        pe[:, 1::2] = torch.cos(
            position * div_term
        )

        pe = pe.unsqueeze(0)

        self.register_buffer(
            "pe",
            pe
        )

    def forward(
        self,
        x
    ):

        return (
            x
            +
            self.pe[
                :,
                :x.size(1)
            ]
        )

# ============================================================
# 4. Dynamic Year Embedding
# ============================================================

class YearEmbedding(nn.Module):

    def __init__(
        self,
        embedding_dim,
        min_year,
        max_year
    ):

        super().__init__()

        self.min_year = int(
            min_year
        )

        self.max_year = int(
            max_year
        )

        self.num_years = (
            self.max_year
            -
            self.min_year
            +
            1
        )

        # Last index reserved for padding year = 0
        self.padding_index = (
            self.num_years
        )

        self.embedding = nn.Embedding(
            num_embeddings=(
                self.num_years
                +
                1
            ),
            embedding_dim=embedding_dim,
            padding_idx=self.padding_index
        )

    def forward(
        self,
        years
    ):

        years = years.clone()

        padding_mask = (
            years == 0
        )

        valid_mask = (
            ~padding_mask
        )

        if valid_mask.any():

            valid_years = years[
                valid_mask
            ]

            if valid_years.min() < self.min_year:

                raise ValueError(
                    "Observed year below configured minimum."
                )

            if valid_years.max() > self.max_year:

                raise ValueError(
                    "Observed year above configured maximum."
                )

        indices = (
            years
            -
            self.min_year
        )

        indices[
            padding_mask
        ] = self.padding_index

        return self.embedding(
            indices
        )

# ============================================================
# 5. E2 Feature Projection
#
# IMPORTANT:
# 12 features -> 128
# ============================================================

class FeatureProjectionE2(nn.Module):

    def __init__(
        self
    ):

        super().__init__()

        self.network = nn.Sequential(

            nn.Linear(
                TRAJECTORY_DIM,
                EMBED_DIM
            ),

            nn.LayerNorm(
                EMBED_DIM
            ),

            nn.GELU(),

            nn.Dropout(
                DROPOUT
            )

        )

    def forward(
        self,
        x
    ):

        return self.network(
            x
        )

# ============================================================
# 6. E2 Temporal Encoder
# ============================================================

class TemporalEncoderE2(nn.Module):

    def __init__(
        self
    ):

        super().__init__()

        self.feature_projection = (
            FeatureProjectionE2()
        )

        self.year_embedding = YearEmbedding(
            embedding_dim=EMBED_DIM,
            min_year=MIN_YEAR,
            max_year=MAX_YEAR
        )

        self.position_encoding = (
            PositionalEncoding(
                d_model=EMBED_DIM,
                max_len=MAX_VISITS
            )
        )

        encoder_layer = nn.TransformerEncoderLayer(

            d_model=EMBED_DIM,

            nhead=NUM_HEADS,

            dim_feedforward=FF_DIM,

            dropout=DROPOUT,

            activation="gelu",

            batch_first=True,

            norm_first=False

        )

        self.transformer = nn.TransformerEncoder(

            encoder_layer,

            num_layers=NUM_LAYERS

        )

        self.output_norm = nn.LayerNorm(
            EMBED_DIM
        )

    def forward(
        self,
        trajectory,
        years,
        mask
    ):

        # --------------------------------------------
        # Input:
        # (B, 7, 12)
        #
        # Output projection:
        # (B, 7, 128)
        # --------------------------------------------

        x = self.feature_projection(
            trajectory
        )

        year_features = (
            self.year_embedding(
                years
            )
        )

        x = (
            x
            +
            year_features
        )

        x = self.position_encoding(
            x
        )

        x = self.transformer(

            x,

            src_key_padding_mask=(
                ~mask
            )

        )

        x = self.output_norm(
            x
        )

        # Force padded temporal positions to zero
        x = x.masked_fill(

            (
                ~mask
            ).unsqueeze(-1),

            0.0

        )

        return x

# ============================================================
# 7. Initialize Fresh E2 Encoder
# ============================================================

temporal_encoder_e2 = (
    TemporalEncoderE2()
    .to(cfg.DEVICE)
)

print(
    "\n✓ Fresh E2 Temporal Encoder created."
)

# ============================================================
# 8. Parameter Count
# ============================================================

total_params_e2_encoder = sum(

    p.numel()

    for p in temporal_encoder_e2.parameters()

)

trainable_params_e2_encoder = sum(

    p.numel()

    for p in temporal_encoder_e2.parameters()

    if p.requires_grad

)

print("\nParameter Statistics")
print("-" * 60)

print(
    f"Total Parameters     : "
    f"{total_params_e2_encoder:,}"
)

print(
    f"Trainable Parameters : "
    f"{trainable_params_e2_encoder:,}"
)

# ============================================================
# 9. Forward Pass Verification
# ============================================================

sample_e2 = next(
    iter(
        train_patient_loader_e2
    )
)

trajectory_e2 = (
    sample_e2[
        "trajectory"
    ]
    .float()
    .to(cfg.DEVICE)
)

years_e2 = (
    sample_e2[
        "years"
    ]
    .long()
    .to(cfg.DEVICE)
)

mask_e2 = (
    sample_e2[
        "mask"
    ]
    .bool()
    .to(cfg.DEVICE)
)

temporal_encoder_e2.eval()

with torch.no_grad():

    output_e2 = temporal_encoder_e2(

        trajectory_e2,

        years_e2,

        mask_e2

    )

print("\nForward Pass Verification")
print("-" * 60)

print(
    "Input Shape  :",
    trajectory_e2.shape
)

print(
    "Output Shape :",
    output_e2.shape
)

expected_shape = (
    trajectory_e2.shape[0],
    MAX_VISITS,
    EMBED_DIM
)

if tuple(
    output_e2.shape
) != expected_shape:

    raise ValueError(
        "E2 temporal output shape mismatch."
    )

print(
    "✓ E2 temporal output shape is correct."
)

# ============================================================
# 10. Numerical Stability
# ============================================================

has_nan = torch.isnan(
    output_e2
).any().item()

has_inf = torch.isinf(
    output_e2
).any().item()

print("\nNumerical Stability")
print("-" * 60)

print(
    "NaN Present :",
    has_nan
)

print(
    "Inf Present :",
    has_inf
)

if has_nan or has_inf:

    raise ValueError(
        "NaN or Inf detected in E2 encoder."
    )

# ============================================================
# 11. Padding Validation
# ============================================================

padded_output = output_e2[
    ~mask_e2
]

if padded_output.numel() > 0:

    max_padded_value = (
        padded_output
        .abs()
        .max()
        .item()
    )

else:

    max_padded_value = 0.0

print("\nPadding Validation")
print("-" * 60)

print(
    "Maximum padded output:",
    max_padded_value
)

if max_padded_value != 0.0:

    raise ValueError(
        "Non-zero padded encoder output detected."
    )

print(
    "✓ E2 padded outputs are zero."
)

# ============================================================
# 12. Embedding Statistics
# ============================================================

valid_output = output_e2[
    mask_e2
]

print("\nValid Embedding Statistics")
print("-" * 60)

print(
    "Mean :",
    valid_output.mean().item()
)

print(
    "Std  :",
    valid_output.std().item()
)

print(
    "Min  :",
    valid_output.min().item()
)

print(
    "Max  :",
    valid_output.max().item()
)

# ============================================================
# 13. Verify Input Projection
# ============================================================

first_linear = (
    temporal_encoder_e2
    .feature_projection
    .network[0]
)

print("\nE2 Input Projection")
print("-" * 60)

print(
    first_linear
)

print(
    "Input Features  :",
    first_linear.in_features
)

print(
    "Output Features :",
    first_linear.out_features
)

assert (
    first_linear.in_features
    ==
    12
)

assert (
    first_linear.out_features
    ==
    128
)

print(
    "✓ E2 projection confirmed: 12 -> 128"
)

# ============================================================
# 14. Module Summary
# ============================================================

print("\n" + "=" * 70)

print("EXPERIMENT E2 TEMPORAL ENCODER")

print("=" * 70)

print(
    "Input Features        :",
    TRAJECTORY_DIM
)

print(
    "Feature Composition   :",
    "11 symptoms + 1 overall severity"
)

print(
    "Maximum Visits        :",
    MAX_VISITS
)

print(
    "Embedding Dimension   :",
    EMBED_DIM
)

print(
    "Transformer Layers    :",
    NUM_LAYERS
)

print(
    "Attention Heads       :",
    NUM_HEADS
)

print(
    "Year Range            :",
    f"{MIN_YEAR} - {MAX_YEAR}"
)

print("=" * 70)

print("\n" + "=" * 70)
print("E2 Module 4A Completed Successfully")
print("=" * 70)

In [ ]:
# ============================================================
# STraT-Net
# Experiment E2
# Module 4B/4C
# Complete Severity-Augmented STraT-Net Model
# ============================================================

import os
import torch
import torch.nn as nn

print("=" * 70)
print("STraT-Net : Experiment E2")
print("Complete Severity-Augmented Model")
print("=" * 70)

# ============================================================
# 1. Output Dimensions
# ============================================================

NUM_DISEASES = len(
    label_encoder.classes_
)

NUM_SYMPTOMS = len(
    symptom_vocab
)

print("\nOutput Configuration")
print("-" * 60)

print(
    "Disease Classes :",
    NUM_DISEASES
)

print(
    "Symptom Outputs :",
    NUM_SYMPTOMS
)

print(
    "Input Features  :",
    TRAJECTORY_DIM
)

assert NUM_DISEASES == 5
assert NUM_SYMPTOMS == 11
assert TRAJECTORY_DIM == 12

# ============================================================
# 2. Gated Temporal Attention Pooling
# Same architecture as E1
# ============================================================

class GatedTemporalAttentionPoolingE2(
    nn.Module
):

    def __init__(
        self,
        embed_dim,
        hidden_dim=128,
        dropout=0.10
    ):

        super().__init__()

        self.value_branch = nn.Sequential(

            nn.Linear(
                embed_dim,
                hidden_dim
            ),

            nn.Tanh()

        )

        self.gate_branch = nn.Sequential(

            nn.Linear(
                embed_dim,
                hidden_dim
            ),

            nn.Sigmoid()

        )

        self.score_layer = nn.Linear(

            hidden_dim,

            1

        )

        self.dropout = nn.Dropout(
            dropout
        )

    def forward(
        self,
        x,
        mask
    ):

        value_features = (
            self.value_branch(
                x
            )
        )

        gate_features = (
            self.gate_branch(
                x
            )
        )

        gated_features = (

            value_features

            *

            gate_features

        )

        gated_features = (
            self.dropout(
                gated_features
            )
        )

        scores = (

            self.score_layer(
                gated_features
            )

            .squeeze(-1)

        )

        scores = scores.masked_fill(

            ~mask,

            torch.finfo(
                scores.dtype
            ).min

        )

        attention = torch.softmax(

            scores,

            dim=1

        )

        pooled = torch.sum(

            x

            *

            attention.unsqueeze(-1),

            dim=1

        )

        return (
            pooled,
            attention
        )

# ============================================================
# 3. Residual Clinical Projection
# Same as E1
# ============================================================

class ClinicalProjectionE2(
    nn.Module
):

    def __init__(
        self,
        embed_dim=EMBED_DIM,
        dropout=0.20
    ):

        super().__init__()

        self.block = nn.Sequential(

            nn.Linear(
                embed_dim,
                embed_dim
            ),

            nn.GELU(),

            nn.Dropout(
                dropout
            ),

            nn.Linear(
                embed_dim,
                embed_dim
            )

        )

        self.norm = nn.LayerNorm(
            embed_dim
        )

    def forward(
        self,
        x
    ):

        residual = x

        x = self.block(
            x
        )

        x = (
            x
            +
            residual
        )

        return self.norm(
            x
        )

# ============================================================
# 4. Disease Classification Head
# ============================================================

class DiseaseHeadE2(
    nn.Module
):

    def __init__(
        self
    ):

        super().__init__()

        self.network = nn.Sequential(

            nn.Linear(
                EMBED_DIM,
                256
            ),

            nn.GELU(),

            nn.Dropout(
                0.30
            ),

            nn.Linear(
                256,
                128
            ),

            nn.GELU(),

            nn.Dropout(
                0.20
            ),

            nn.Linear(
                128,
                NUM_DISEASES
            )

        )

    def forward(
        self,
        x
    ):

        return self.network(
            x
        )

# ============================================================
# 5. Symptom Reconstruction Head
# Kept for architectural parity with E1
# Not used in training loss yet
# ============================================================

class SymptomHeadE2(
    nn.Module
):

    def __init__(
        self
    ):

        super().__init__()

        self.network = nn.Sequential(

            nn.Linear(
                EMBED_DIM,
                128
            ),

            nn.GELU(),

            nn.Dropout(
                0.20
            ),

            nn.Linear(
                128,
                NUM_SYMPTOMS
            )

        )

    def forward(
        self,
        x
    ):

        return self.network(
            x
        )

# ============================================================
# 6. Complete STraT-Net E2
# ============================================================

class STraTNetE2(
    nn.Module
):

    def __init__(
        self
    ):

        super().__init__()

        self.encoder = (
            TemporalEncoderE2()
        )

        self.pool = (
            GatedTemporalAttentionPoolingE2(

                embed_dim=EMBED_DIM,

                hidden_dim=128,

                dropout=0.10

            )
        )

        self.projection = (
            ClinicalProjectionE2(
                embed_dim=EMBED_DIM
            )
        )

        self.disease_head = (
            DiseaseHeadE2()
        )

        self.symptom_head = (
            SymptomHeadE2()
        )

    def forward(
        self,
        trajectory,
        years,
        mask
    ):

        # --------------------------------------------
        # Input:
        # (B, 7, 12)
        # --------------------------------------------

        temporal_features = (
            self.encoder(

                trajectory,

                years,

                mask

            )
        )

        # --------------------------------------------
        # Temporal attention pooling
        # --------------------------------------------

        pooled_embedding, attention = (
            self.pool(

                temporal_features,

                mask

            )
        )

        # --------------------------------------------
        # Clinical projection
        # --------------------------------------------

        embedding = (
            self.projection(
                pooled_embedding
            )
        )

        # --------------------------------------------
        # Prediction heads
        # --------------------------------------------

        disease_logits = (
            self.disease_head(
                embedding
            )
        )

        symptom_logits = (
            self.symptom_head(
                embedding
            )
        )

        return {

            "embedding":
                embedding,

            "attention":
                attention,

            "disease_logits":
                disease_logits,

            "symptom_logits":
                symptom_logits

        }

# ============================================================
# 7. Weight Initialization
# ============================================================

def initialize_weights_e2(
    model
):

    for module in model.modules():

        if isinstance(
            module,
            nn.Linear
        ):

            nn.init.xavier_uniform_(
                module.weight
            )

            if module.bias is not None:

                nn.init.zeros_(
                    module.bias
                )

        elif isinstance(
            module,
            nn.LayerNorm
        ):

            nn.init.ones_(
                module.weight
            )

            nn.init.zeros_(
                module.bias
            )

        elif isinstance(
            module,
            nn.Embedding
        ):

            nn.init.normal_(

                module.weight,

                mean=0.0,

                std=0.02

            )

            if module.padding_idx is not None:

                with torch.no_grad():

                    module.weight[
                        module.padding_idx
                    ].zero_()

# ============================================================
# 8. Instantiate Fresh E2 Model
# ============================================================

model_e2 = (
    STraTNetE2()
)

initialize_weights_e2(
    model_e2
)

model_e2 = model_e2.to(
    cfg.DEVICE
)

print(
    "\n✓ Fresh STraT-Net E2 initialized."
)

print(
    "Device:",
    next(
        model_e2.parameters()
    ).device
)

# ============================================================
# 9. Parameter Count
# ============================================================

total_params_e2 = sum(

    p.numel()

    for p in model_e2.parameters()

)

trainable_params_e2 = sum(

    p.numel()

    for p in model_e2.parameters()

    if p.requires_grad

)

print("\nParameter Statistics")
print("-" * 60)

print(
    f"Total Parameters     : "
    f"{total_params_e2:,}"
)

print(
    f"Trainable Parameters : "
    f"{trainable_params_e2:,}"
)

# ============================================================
# 10. Full Forward Pass
# ============================================================

sample_e2 = next(

    iter(
        train_patient_loader_e2
    )

)

trajectory_e2 = (
    sample_e2[
        "trajectory"
    ]
    .float()
    .to(cfg.DEVICE)
)

years_e2 = (
    sample_e2[
        "years"
    ]
    .long()
    .to(cfg.DEVICE)
)

mask_e2 = (
    sample_e2[
        "mask"
    ]
    .bool()
    .to(cfg.DEVICE)
)

model_e2.eval()

with torch.no_grad():

    outputs_e2 = model_e2(

        trajectory_e2,

        years_e2,

        mask_e2

    )

# ============================================================
# 11. Output Shape Verification
# ============================================================

print("\nOutput Shape Verification")
print("-" * 60)

for key, value in outputs_e2.items():

    print(

        f"{key:20s}: "
        f"{tuple(value.shape)}"

    )

expected_shapes = {

    "embedding":
        (
            trajectory_e2.shape[0],
            EMBED_DIM
        ),

    "attention":
        (
            trajectory_e2.shape[0],
            MAX_VISITS
        ),

    "disease_logits":
        (
            trajectory_e2.shape[0],
            NUM_DISEASES
        ),

    "symptom_logits":
        (
            trajectory_e2.shape[0],
            NUM_SYMPTOMS
        )

}

for key, expected_shape in expected_shapes.items():

    if tuple(
        outputs_e2[
            key
        ].shape
    ) != expected_shape:

        raise ValueError(

            f"{key} shape mismatch."

        )

print(
    "\n✓ All E2 output shapes are correct."
)

# ============================================================
# 12. Attention Validation
# ============================================================

attention_e2 = outputs_e2[
    "attention"
]

attention_sums = attention_e2.sum(
    dim=1
)

print("\nAttention Validation")
print("-" * 60)

print(
    "First 5 sums:"
)

print(
    attention_sums[:5]
)

if not torch.allclose(

    attention_sums,

    torch.ones_like(
        attention_sums
    ),

    atol=1e-6

):

    raise ValueError(
        "E2 attention does not sum to 1."
    )

padded_attention = (
    attention_e2.masked_select(
        ~mask_e2
    )
)

if padded_attention.numel() > 0:

    max_padding_attention = (

        padded_attention
        .abs()
        .max()
        .item()

    )

else:

    max_padding_attention = 0.0

print(
    "Maximum padded attention:",
    max_padding_attention
)

if max_padding_attention > 1e-7:

    raise ValueError(
        "Padded visits received attention."
    )

print(
    "✓ E2 attention validated."
)

# ============================================================
# 13. Numerical Stability
# ============================================================

print("\nNumerical Stability")
print("-" * 60)

for key, value in outputs_e2.items():

    has_nan = torch.isnan(
        value
    ).any().item()

    has_inf = torch.isinf(
        value
    ).any().item()

    print(

        f"{key:20s}"
        f" NaN={has_nan}"
        f" Inf={has_inf}"

    )

    if has_nan or has_inf:

        raise ValueError(

            f"Numerical instability "
            f"in {key}."

        )

# ============================================================
# 14. Confirm Severity Actually Influences Input
# ============================================================

print("\nSeverity Feature Check")
print("-" * 60)

print(
    "Input feature dimension:",
    trajectory_e2.shape[-1]
)

print(
    "Severity values, first patient:"
)

first_length = int(
    sample_e2[
        "length"
    ][0]
)

print(

    trajectory_e2[
        0,
        :first_length,
        -1
    ]

    .detach()

    .cpu()

)

assert trajectory_e2.shape[-1] == 12

# ============================================================
# 15. Save Initialized E2 Model
# ============================================================

E2_CHECKPOINT_DIR = (
    "checkpoints_stratnet_e2_severity"
)

E2_OUTPUT_DIR = (
    "outputs_stratnet_e2_severity"
)

os.makedirs(
    E2_CHECKPOINT_DIR,
    exist_ok=True
)

os.makedirs(
    E2_OUTPUT_DIR,
    exist_ok=True
)

E2_INITIAL_MODEL_PATH = os.path.join(

    E2_CHECKPOINT_DIR,

    "STraTNet_E2_Initialized.pth"

)

torch.save(

    {

        "model_state_dict":
            model_e2.state_dict(),

        "experiment":
            "E2",

        "input_features":
            12,

        "feature_description":
            "11 symptoms + overall severity",

        "num_diseases":
            NUM_DISEASES,

        "num_symptoms":
            NUM_SYMPTOMS,

        "max_visits":
            MAX_VISITS,

        "label_classes":
            list(
                label_encoder.classes_
            )

    },

    E2_INITIAL_MODEL_PATH

)

print("\nInitialized E2 Checkpoint")
print("-" * 60)

print(
    E2_INITIAL_MODEL_PATH
)

print(
    "Exists:",
    os.path.exists(
        E2_INITIAL_MODEL_PATH
    )
)

# ============================================================
# 16. Final Status
# ============================================================

print("\n" + "=" * 70)

print("EXPERIMENT E2 ARCHITECTURE STATUS")

print("=" * 70)

print("12-D Severity-Augmented Input   ✓")
print("Temporal Transformer            ✓")
print("Dynamic Year Embedding          ✓")
print("Gated Temporal Attention        ✓")
print("Residual Clinical Projection    ✓")
print("Disease Classification Head     ✓")
print("Symptom Reconstruction Head     ✓")
print("Fresh Weight Initialization     ✓")
print("Forward Pass                    ✓")
print("Attention Validation            ✓")
print("Numerical Stability             ✓")

print()

print(
    "E2 MODEL READY FOR TRAINING"
)

print("=" * 70)

In [ ]:
# ============================================================
# STraT-Net
# Experiment E2
# Module 5B/5C
# Controlled Training Loop
# ============================================================

import os
import numpy as np
import pandas as pd
import torch

from tqdm.auto import tqdm

print("=" * 70)
print("STraT-Net : Experiment E2")
print("Controlled Training Loop")
print("=" * 70)

# ============================================================
# 1. Epoch Storage
# ============================================================

def initialize_epoch_storage_e2():

    return {

        "labels": [],
        "predictions": [],
        "probabilities": [],
        "attention": [],
        "patient_ids": []

    }

# ============================================================
# 2. Train One Epoch
# ============================================================

def train_one_epoch_e2(

    model,

    dataloader,

    optimizer,

    criterion,

    device,

    scaler,

    epoch

):

    model.train()

    storage = initialize_epoch_storage_e2()

    running_loss = 0.0

    progress_bar = tqdm(

        dataloader,

        total=len(dataloader),

        desc=f"E2 Epoch {epoch} [Train]"

    )

    for batch_idx, batch in enumerate(
        progress_bar
    ):

        trajectory = (
            batch["trajectory"]
            .float()
            .to(
                device,
                non_blocking=True
            )
        )

        years = (
            batch["years"]
            .long()
            .to(
                device,
                non_blocking=True
            )
        )

        mask = (
            batch["mask"]
            .bool()
            .to(
                device,
                non_blocking=True
            )
        )

        labels = (
            batch["label"]
            .long()
            .to(
                device,
                non_blocking=True
            )
        )

        optimizer.zero_grad(
            set_to_none=True
        )

        # --------------------------------------------
        # AMP training
        # --------------------------------------------

        if scaler is not None:

            with torch.cuda.amp.autocast():

                outputs = model(

                    trajectory,

                    years,

                    mask

                )

                logits = outputs[
                    "disease_logits"
                ]

                loss = criterion(

                    logits,

                    labels

                )

            scaler.scale(
                loss
            ).backward()

            # Correct clipping sequence for AMP
            scaler.unscale_(
                optimizer
            )

            torch.nn.utils.clip_grad_norm_(

                model.parameters(),

                E2_GRAD_CLIP

            )

            scaler.step(
                optimizer
            )

            scaler.update()

        else:

            outputs = model(

                trajectory,

                years,

                mask

            )

            logits = outputs[
                "disease_logits"
            ]

            loss = criterion(

                logits,

                labels

            )

            loss.backward()

            torch.nn.utils.clip_grad_norm_(

                model.parameters(),

                E2_GRAD_CLIP

            )

            optimizer.step()

        running_loss += (
            loss.item()
        )

        probabilities = torch.softmax(

            logits,

            dim=1

        )

        predictions = torch.argmax(

            probabilities,

            dim=1

        )

        storage[
            "labels"
        ].extend(

            labels
            .detach()
            .cpu()
            .numpy()
            .tolist()

        )

        storage[
            "predictions"
        ].extend(

            predictions
            .detach()
            .cpu()
            .numpy()
            .tolist()

        )

        storage[
            "probabilities"
        ].extend(

            probabilities
            .detach()
            .cpu()
            .numpy()

        )

        storage[
            "attention"
        ].extend(

            outputs[
                "attention"
            ]
            .detach()
            .cpu()
            .numpy()

        )

        storage[
            "patient_ids"
        ].extend(

            list(
                batch[
                    "patient_id"
                ]
            )

        )

        average_loss = (

            running_loss

            /

            (
                batch_idx
                +
                1
            )

        )

        progress_bar.set_postfix({

            "loss":
                f"{average_loss:.4f}"

        })

    metrics = compute_metrics(

        storage[
            "labels"
        ],

        storage[
            "predictions"
        ]

    )

    results = {

        "loss":

            running_loss
            /
            len(
                dataloader
            ),

        "accuracy":

            metrics[
                "accuracy"
            ],

        "precision":

            metrics[
                "precision"
            ],

        "recall":

            metrics[
                "recall"
            ],

        "macro_f1":

            metrics[
                "macro_f1"
            ],

        "labels":

            np.asarray(
                storage[
                    "labels"
                ]
            ),

        "predictions":

            np.asarray(
                storage[
                    "predictions"
                ]
            ),

        "probabilities":

            np.asarray(
                storage[
                    "probabilities"
                ]
            ),

        "attention":

            np.asarray(
                storage[
                    "attention"
                ]
            ),

        "patient_ids":

            storage[
                "patient_ids"
            ]

    }

    print("\nE2 Training Summary")
    print("-" * 60)

    print(
        f"Loss      : "
        f"{results['loss']:.4f}"
    )

    print(
        f"Accuracy  : "
        f"{results['accuracy']:.4f}"
    )

    print(
        f"Macro-F1  : "
        f"{results['macro_f1']:.4f}"
    )

    return results

# ============================================================
# 3. Validate One Epoch
# ============================================================

def validate_one_epoch_e2(

    model,

    dataloader,

    criterion,

    device,

    epoch

):

    model.eval()

    storage = initialize_epoch_storage_e2()

    running_loss = 0.0

    progress_bar = tqdm(

        dataloader,

        total=len(dataloader),

        desc=f"E2 Epoch {epoch} [Validation]"

    )

    with torch.no_grad():

        for batch_idx, batch in enumerate(
            progress_bar
        ):

            trajectory = (
                batch["trajectory"]
                .float()
                .to(
                    device,
                    non_blocking=True
                )
            )

            years = (
                batch["years"]
                .long()
                .to(
                    device,
                    non_blocking=True
                )
            )

            mask = (
                batch["mask"]
                .bool()
                .to(
                    device,
                    non_blocking=True
                )
            )

            labels = (
                batch["label"]
                .long()
                .to(
                    device,
                    non_blocking=True
                )
            )

            if E2_USE_AMP:

                with torch.cuda.amp.autocast():

                    outputs = model(

                        trajectory,

                        years,

                        mask

                    )

                    logits = outputs[
                        "disease_logits"
                    ]

                    loss = criterion(

                        logits,

                        labels

                    )

            else:

                outputs = model(

                    trajectory,

                    years,

                    mask

                )

                logits = outputs[
                    "disease_logits"
                ]

                loss = criterion(

                    logits,

                    labels

                )

            running_loss += (
                loss.item()
            )

            probabilities = torch.softmax(

                logits,

                dim=1

            )

            predictions = torch.argmax(

                probabilities,

                dim=1

            )

            storage[
                "labels"
            ].extend(

                labels
                .cpu()
                .numpy()
                .tolist()

            )

            storage[
                "predictions"
            ].extend(

                predictions
                .cpu()
                .numpy()
                .tolist()

            )

            storage[
                "probabilities"
            ].extend(

                probabilities
                .cpu()
                .numpy()

            )

            storage[
                "attention"
            ].extend(

                outputs[
                    "attention"
                ]
                .cpu()
                .numpy()

            )

            storage[
                "patient_ids"
            ].extend(

                list(
                    batch[
                        "patient_id"
                    ]
                )

            )

            average_loss = (

                running_loss

                /

                (
                    batch_idx
                    +
                    1
                )

            )

            progress_bar.set_postfix({

                "loss":
                    f"{average_loss:.4f}"

            })

    metrics = compute_metrics(

        storage[
            "labels"
        ],

        storage[
            "predictions"
        ]

    )

    results = {

        "loss":

            running_loss
            /
            len(
                dataloader
            ),

        "accuracy":

            metrics[
                "accuracy"
            ],

        "precision":

            metrics[
                "precision"
            ],

        "recall":

            metrics[
                "recall"
            ],

        "macro_f1":

            metrics[
                "macro_f1"
            ]

    }

    print("\nE2 Validation Summary")
    print("-" * 60)

    print(
        f"Loss      : "
        f"{results['loss']:.4f}"
    )

    print(
        f"Accuracy  : "
        f"{results['accuracy']:.4f}"
    )

    print(
        f"Macro-F1  : "
        f"{results['macro_f1']:.4f}"
    )

    return results

# ============================================================
# 4. Controlled Full Training Loop
# ============================================================

def train_model_e2():

    global best_val_f1_e2
    global best_epoch_e2
    global early_stop_counter_e2
    global history_e2

    print("\n" + "=" * 70)

    print("STARTING EXPERIMENT E2 TRAINING")

    print("=" * 70)

    for epoch in range(

        1,

        E2_NUM_EPOCHS + 1

    ):

        print("\n" + "=" * 70)

        print(
            f"E2 EPOCH "
            f"{epoch}/"
            f"{E2_NUM_EPOCHS}"
        )

        print("=" * 70)

        # --------------------------------------------
        # Training
        # --------------------------------------------

        train_results = train_one_epoch_e2(

            model=model_e2,

            dataloader=train_patient_loader_e2,

            optimizer=optimizer_e2,

            criterion=criterion_e2,

            device=cfg.DEVICE,

            scaler=scaler_e2,

            epoch=epoch

        )

        # --------------------------------------------
        # Validation
        # --------------------------------------------

        val_results = validate_one_epoch_e2(

            model=model_e2,

            dataloader=val_patient_loader_e2,

            criterion=criterion_e2,

            device=cfg.DEVICE,

            epoch=epoch

        )

        current_lr = (

            optimizer_e2
            .param_groups[0]["lr"]

        )

        # --------------------------------------------
        # History
        # --------------------------------------------

        history_e2[
            "epoch"
        ].append(
            epoch
        )

        history_e2[
            "train_loss"
        ].append(
            train_results[
                "loss"
            ]
        )

        history_e2[
            "val_loss"
        ].append(
            val_results[
                "loss"
            ]
        )

        history_e2[
            "train_accuracy"
        ].append(
            train_results[
                "accuracy"
            ]
        )

        history_e2[
            "val_accuracy"
        ].append(
            val_results[
                "accuracy"
            ]
        )

        history_e2[
            "train_macro_f1"
        ].append(
            train_results[
                "macro_f1"
            ]
        )

        history_e2[
            "val_macro_f1"
        ].append(
            val_results[
                "macro_f1"
            ]
        )

        history_e2[
            "learning_rate"
        ].append(
            current_lr
        )

        save_history_e2()

        # --------------------------------------------
        # Best checkpoint
        # --------------------------------------------

        current_val_f1 = (
            val_results[
                "macro_f1"
            ]
        )

        if (
            current_val_f1
            >
            best_val_f1_e2
        ):

            best_val_f1_e2 = (
                current_val_f1
            )

            best_epoch_e2 = (
                epoch
            )

            early_stop_counter_e2 = 0

            save_checkpoint_e2(

                model=model_e2,

                optimizer=optimizer_e2,

                scheduler=scheduler_e2,

                epoch=epoch,

                val_macro_f1=(
                    current_val_f1
                ),

                path=E2_BEST_MODEL_PATH

            )

            print(
                "\n✓ New best E2 model saved."
            )

        else:

            early_stop_counter_e2 += 1

            print(
                "\nNo validation improvement."
            )

            print(

                "Early stopping counter:",

                f"{early_stop_counter_e2}/"
                f"{E2_EARLY_STOPPING_PATIENCE}"

            )

        # --------------------------------------------
        # Save last checkpoint
        # --------------------------------------------

        save_checkpoint_e2(

            model=model_e2,

            optimizer=optimizer_e2,

            scheduler=scheduler_e2,

            epoch=epoch,

            val_macro_f1=(
                current_val_f1
            ),

            path=E2_LAST_MODEL_PATH

        )

        # --------------------------------------------
        # Scheduler
        # --------------------------------------------

        scheduler_e2.step()

        next_lr = (

            optimizer_e2
            .param_groups[0]["lr"]

        )

        # --------------------------------------------
        # Summary
        # --------------------------------------------

        print("\n" + "-" * 60)

        print("E2 EPOCH SUMMARY")

        print("-" * 60)

        print(
            f"Train Loss       : "
            f"{train_results['loss']:.4f}"
        )

        print(
            f"Validation Loss  : "
            f"{val_results['loss']:.4f}"
        )

        print()

        print(
            f"Train Accuracy   : "
            f"{train_results['accuracy']:.4f}"
        )

        print(
            f"Val Accuracy     : "
            f"{val_results['accuracy']:.4f}"
        )

        print()

        print(
            f"Train Macro-F1   : "
            f"{train_results['macro_f1']:.4f}"
        )

        print(
            f"Val Macro-F1     : "
            f"{val_results['macro_f1']:.4f}"
        )

        print()

        print(
            f"Current LR       : "
            f"{current_lr:.8f}"
        )

        print(
            f"Next LR          : "
            f"{next_lr:.8f}"
        )

        print()

        print(
            f"Best Val F1      : "
            f"{best_val_f1_e2:.4f}"
        )

        print(
            f"Best Epoch       : "
            f"{best_epoch_e2}"
        )

        print("-" * 60)

        # --------------------------------------------
        # Early stopping
        # --------------------------------------------

        if (

            early_stop_counter_e2

            >=

            E2_EARLY_STOPPING_PATIENCE

        ):

            print("\n" + "=" * 70)

            print(
                "E2 EARLY STOPPING TRIGGERED"
            )

            print("=" * 70)

            break

    history_e2_df = pd.DataFrame(
        history_e2
    )

    print("\n" + "=" * 70)

    print("E2 TRAINING FINISHED")

    print("=" * 70)

    print(
        "Epochs Completed :",
        len(
            history_e2_df
        )
    )

    print(
        "Best Epoch       :",
        best_epoch_e2
    )

    print(
        "Best Val Macro-F1:",
        f"{best_val_f1_e2:.4f}"
    )

    print()

    print(
        "Best Checkpoint:",
        E2_BEST_MODEL_PATH
    )

    print(
        "Exists:",
        os.path.exists(
            E2_BEST_MODEL_PATH
        )
    )

    print("=" * 70)

    return history_e2_df

# ============================================================
# 5. RUN E2 TRAINING
# ============================================================

# ERROR: E2_NUM_EPOCHS is not defined
# The required E2-specific training configuration variables, optimizer,
# scheduler, scaler, and history objects must be initialized first.
# This will be done in a separate cell, similar to Module 5A.
history_e2_df = train_model_e2()

# ============================================================
# 6. Display Training History
# ============================================================

display(
    history_e2_df
)

print("\n" + "=" * 70)
print("E2 Training Module Completed Successfully")
print("=" * 70)


In [ ]:
# ============================================================
# STraT-Net V2
# Experiment E2
# Module 5A - Training Configuration + Utilities
# ============================================================

import os
import json
import random
import numpy as np
import pandas as pd

import torch
import torch.nn as nn

from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR

# E2-specific training hyperparameters
E2_NUM_EPOCHS = 30
E2_LEARNING_RATE = 1e-4
E2_WEIGHT_DECAY = 1e-4
E2_GRAD_CLIP = 1.0
E2_LABEL_SMOOTHING = 0.05
E2_EARLY_STOPPING_PATIENCE = 8
E2_USE_SYMPTOM_LOSS = False  # For E2, focusing on disease prediction first
E2_SYMPTOM_LOSS_WEIGHT = 0.30

print("=" * 70)
print("STraT-Net V2 : Experiment E2 - Module 5A")
print("Training Configuration + Utilities")
print("=" * 70)

print("\nE2 Training Configuration")
print("-" * 60)
print("Epochs                  :", E2_NUM_EPOCHS)
print("Learning Rate           :", E2_LEARNING_RATE)
print("Weight Decay            :", E2_WEIGHT_DECAY)
print("Gradient Clip           :", E2_GRAD_CLIP)
print("Label Smoothing         :", E2_LABEL_SMOOTHING)
print("Early Stopping Patience :", E2_EARLY_STOPPING_PATIENCE)
print("Symptom Loss Enabled    :", E2_USE_SYMPTOM_LOSS)

# E2-specific device configuration
E2_DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)
model_e2 = model_e2.to(E2_DEVICE)  # Ensure the E2 model is on the correct device

print("\nE2 Device :", E2_DEVICE)
if torch.cuda.is_available():
    print("E2 GPU    :", torch.cuda.get_device_name(0))

# E2-specific loss functions (using class_weights from Module 1)
criterion_e2 = nn.CrossEntropyLoss(
    weight=class_weights.to(E2_DEVICE),
    label_smoothing=E2_LABEL_SMOOTHING
)
# If E2_USE_SYMPTOM_LOSS is True, you might need a separate criterion for it
# For now, let's keep it simple as in E1.
symptom_criterion_e2 = nn.BCEWithLogitsLoss()

print("\n✓ E2 Loss functions created.")

# E2-specific optimizer
optimizer_e2 = AdamW(
    model_e2.parameters(),
    lr=E2_LEARNING_RATE,
    weight_decay=E2_WEIGHT_DECAY
)
print("✓ E2 Optimizer created.")

# E2-specific scheduler
scheduler_e2 = CosineAnnealingLR(
    optimizer_e2,
    T_max=E2_NUM_EPOCHS,
    eta_min=1e-6
)
print("✓ E2 Cosine scheduler created.")

# E2-specific mixed precision scaler
E2_USE_AMP = torch.cuda.is_available()
if E2_USE_AMP:
    scaler_e2 = torch.cuda.amp.GradScaler()
else:
    scaler_e2 = None
print("E2 Mixed Precision:", E2_USE_AMP)

# E2-specific experiment directories
E2_CHECKPOINT_DIR = "checkpoints_stratnet_e2_severity_run"
E2_OUTPUT_DIR = "outputs_stratnet_e2_severity_run"

os.makedirs(E2_CHECKPOINT_DIR, exist_ok=True)
os.makedirs(E2_OUTPUT_DIR, exist_ok=True)

E2_BEST_MODEL_PATH = os.path.join(
    E2_CHECKPOINT_DIR,
    "best_model_e2.pth"
)
E2_LAST_MODEL_PATH = os.path.join(
    E2_CHECKPOINT_DIR,
    "last_model_e2.pth"
)
print("\nE2 Checkpoint Directory:", E2_CHECKPOINT_DIR)
print("E2 Output Directory    :", E2_OUTPUT_DIR)

# E2-specific training history
history_e2 = {
    "epoch": [],
    "train_loss": [],
    "val_loss": [],
    "train_accuracy": [],
    "val_accuracy": [],
    "train_macro_f1": [],
    "val_macro_f1": [],
    "learning_rate": []
}
best_val_f1_e2 = -1.0
best_epoch_e2 = -1
early_stop_counter_e2 = 0
print("✓ E2 Training history initialized.")

# Save checkpoint function for E2
def save_checkpoint_e2(
    model,
    optimizer,
    scheduler,
    epoch,
    val_macro_f1,
    path
):
    torch.save(
        {
            "experiment":
                "E2",

            "epoch": epoch,
            "model_state_dict": model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "scheduler_state_dict": scheduler.state_dict(),
            "val_macro_f1": val_macro_f1,
            "label_classes": list(label_encoder.classes_),
            "symptom_vocabulary": symptom_vocab,
            "num_diseases": NUM_DISEASES,
            "num_symptoms": NUM_SYMPTOMS,
            "max_visits": MAX_VISITS,
            "min_year": MIN_YEAR,
            "max_year": MAX_YEAR,
            "e2_trajectory_dim": E2_TRAJECTORY_DIM  # Add E2 specific info
        },
        path
    )

# Save history function for E2
def save_history_e2():
    history_df_e2 = pd.DataFrame(history_e2)
    history_df_e2.to_csv(
        os.path.join(E2_OUTPUT_DIR, "training_history_e2.csv"),
        index=False
    )

# Save Experiment Configuration for E2
experiment_config_e2 = {
    "architecture": "STraT-Net V2 E2 (Severity-Augmented)",
    "dataset": "stratnet_v2",
    "training_patients": len(train_patient_dataset_e2),
    "validation_patients": len(val_patient_dataset_e2),
    "testing_patients": len(test_patient_dataset_e2),
    "num_diseases": NUM_DISEASES,
    "num_symptoms": NUM_SYMPTOMS,
    "e2_trajectory_dimension": E2_TRAJECTORY_DIM,
    "max_visits": MAX_VISITS,
    "embedding_dim": EMBED_DIM,
    "transformer_layers": NUM_LAYERS,
    "attention_heads": NUM_HEADS,
    "ff_dim": FF_DIM,
    "epochs": E2_NUM_EPOCHS,
    "learning_rate": E2_LEARNING_RATE,
    "weight_decay": E2_WEIGHT_DECAY,
    "label_smoothing": E2_LABEL_SMOOTHING,
    "early_stopping_patience": E2_EARLY_STOPPING_PATIENCE,
    "use_symptom_loss": E2_USE_SYMPTOM_LOSS,
    "device": str(E2_DEVICE)
}

with open(
    os.path.join(E2_OUTPUT_DIR, "experiment_config_e2.json"),
    "w"
) as f:
    json.dump(experiment_config_e2, f, indent=4)

print("\n✓ E2 Experiment configuration saved.")

print("\n" + "=" * 70)
print("E2 TRAINING SETUP STATUS")
print("=" * 70)
print("Model Device          :", next(model_e2.parameters()).device)
print("Optimizer             :", type(optimizer_e2).__name__)
print("Scheduler             :", type(scheduler_e2).__name__)
print("Disease Loss          :", type(criterion_e2).__name__)
print("Class Weights         :", class_weights)
print("Train Batches         :", len(train_patient_loader_e2))
print("Validation Batches    :", len(val_patient_loader_e2))
print("Test Batches          :", len(test_patient_loader_e2))
print("=" * 70)
print("\nSTraT-Net V2 E2 Training Setup READY")
print("=" * 70)


In [ ]:
print("\n============================================================")
print("Resetting E2 training history for fresh run")
print("============================================================")

history_e2 = {
    "epoch": [],
    "train_loss": [],
    "val_loss": [],
    "train_accuracy": [],
    "val_accuracy": [],
    "train_macro_f1": [],
    "val_macro_f1": [],
    "learning_rate": []
}
best_val_f1_e2 = -1.0
best_epoch_e2 = -1
early_stop_counter_e2 = 0

print("✓ E2 Training history and tracking variables reset.")

In [ ]:
# ============================================================
# RUN EXPERIMENT E2 TRAINING
# ============================================================

print("=" * 70)
print("STARTING E2 TRAINING")
print("=" * 70)

# Safety check: training must start fresh
print(
    "History epochs before training:",
    len(history_e2["epoch"])
)

if len(history_e2["epoch"]) != 0:

    raise RuntimeError(
        "E2 history is not empty. "
        "Do not start a second mixed training run."
    )

history_e2_df = train_model_e2()

print("\n" + "=" * 70)
print("E2 TRAINING RESULT")
print("=" * 70)

print(
    "Epochs Completed :",
    len(history_e2_df)
)

print(
    "Best Epoch       :",
    best_epoch_e2
)

print(
    "Best Val Macro-F1:",
    f"{best_val_f1_e2:.4f}"
)

print(
    "Best Model Exists:",
    os.path.exists(
        E2_BEST_MODEL_PATH
    )
)

print("\nTraining History")

display(
    history_e2_df
)

print("=" * 70)


In [ ]:
# ============================================================
# E2 TRAINING COMPLETION CHECK
# ============================================================

import os
import pandas as pd

print("=" * 70)
print("E2 TRAINING COMPLETION CHECK")
print("=" * 70)

print(
    "History epochs:",
    len(history_e2["epoch"])
)

print(
    "Best Epoch:",
    best_epoch_e2
)

print(
    "Best Validation Macro-F1:",
    best_val_f1_e2
)

print(
    "Best checkpoint path:",
    E2_BEST_MODEL_PATH
)

print(
    "Best checkpoint exists:",
    os.path.exists(
        E2_BEST_MODEL_PATH
    )
)

print(
    "Last checkpoint exists:",
    os.path.exists(
        E2_LAST_MODEL_PATH
    )
)

print("\nHistory Summary")
print("-" * 70)

history_e2_df = pd.DataFrame(
    history_e2
)

display(
    history_e2_df
)

# Show best row directly
if len(history_e2_df) > 0:

    best_row_index = (
        history_e2_df[
            "val_macro_f1"
        ].idxmax()
    )

    print("\nBest Validation Epoch Row")
    print("-" * 70)

    display(
        history_e2_df.loc[
            [best_row_index]
        ]
    )

print("=" * 70)

In [ ]:
# ============================================================
# STraT-Net
# Experiment E2
# Final Held-Out Test Evaluation
# ============================================================

import os
import numpy as np
import pandas as pd
import torch

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report,
    roc_auc_score,
    average_precision_score
)

from sklearn.preprocessing import label_binarize

print("=" * 70)
print("STraT-Net : Experiment E2")
print("FINAL HELD-OUT TEST EVALUATION")
print("=" * 70)

# ============================================================
# 1. Load Best E2 Checkpoint
# ============================================================

checkpoint_e2 = torch.load(
    E2_BEST_MODEL_PATH,
    map_location=cfg.DEVICE
)

print("\nBest Checkpoint")
print("-" * 60)

print(
    "Epoch:",
    checkpoint_e2["epoch"]
)

print(
    "Validation Macro-F1:",
    checkpoint_e2["val_macro_f1"]
)

model_e2.load_state_dict(
    checkpoint_e2[
        "model_state_dict"
    ]
)

model_e2 = model_e2.to(
    cfg.DEVICE
)

model_e2.eval()

print(
    "\n✓ Best E2 model loaded."
)

# ============================================================
# 2. Prediction Storage
# ============================================================

all_patient_ids = []

all_labels = []

all_predictions = []

all_probabilities = []

all_lengths = []

all_attention = []

# ============================================================
# 3. Test Inference
# ============================================================

with torch.no_grad():

    for batch in test_patient_loader_e2:

        trajectory = (
            batch["trajectory"]
            .float()
            .to(cfg.DEVICE)
        )

        years = (
            batch["years"]
            .long()
            .to(cfg.DEVICE)
        )

        mask = (
            batch["mask"]
            .bool()
            .to(cfg.DEVICE)
        )

        labels = (
            batch["label"]
            .long()
            .to(cfg.DEVICE)
        )

        outputs = model_e2(

            trajectory,

            years,

            mask

        )

        logits = outputs[
            "disease_logits"
        ]

        probabilities = torch.softmax(

            logits,

            dim=1

        )

        predictions = torch.argmax(

            probabilities,

            dim=1

        )

        all_patient_ids.extend(
            list(
                batch[
                    "patient_id"
                ]
            )
        )

        all_labels.extend(
            labels
            .cpu()
            .numpy()
            .tolist()
        )

        all_predictions.extend(
            predictions
            .cpu()
            .numpy()
            .tolist()
        )

        all_probabilities.extend(
            probabilities
            .cpu()
            .numpy()
        )

        all_lengths.extend(
            batch[
                "length"
            ]
            .cpu()
            .numpy()
            .tolist()
        )

        all_attention.extend(
            outputs[
                "attention"
            ]
            .cpu()
            .numpy()
        )

# ============================================================
# 4. Arrays
# ============================================================

y_true_e2 = np.asarray(
    all_labels
)

y_pred_e2 = np.asarray(
    all_predictions
)

y_prob_e2 = np.asarray(
    all_probabilities
)

lengths_e2 = np.asarray(
    all_lengths
)

attention_e2 = np.asarray(
    all_attention
)

print(
    "\nTest Patients:",
    len(
        y_true_e2
    )
)

assert (
    len(
        y_true_e2
    )
    ==
    len(
        test_patient_dataset_e2
    )
)

# ============================================================
# 5. Overall Metrics
# ============================================================

e2_test_accuracy = accuracy_score(
    y_true_e2,
    y_pred_e2
)

e2_macro_precision = precision_score(
    y_true_e2,
    y_pred_e2,
    average="macro",
    zero_division=0
)

e2_macro_recall = recall_score(
    y_true_e2,
    y_pred_e2,
    average="macro",
    zero_division=0
)

e2_macro_f1 = f1_score(
    y_true_e2,
    y_pred_e2,
    average="macro",
    zero_division=0
)

e2_weighted_f1 = f1_score(
    y_true_e2,
    y_pred_e2,
    average="weighted",
    zero_division=0
)

print("\n" + "=" * 70)

print("E2 FINAL TEST METRICS")

print("=" * 70)

print(
    f"Accuracy        : "
    f"{e2_test_accuracy:.4f}"
)

print(
    f"Macro Precision : "
    f"{e2_macro_precision:.4f}"
)

print(
    f"Macro Recall    : "
    f"{e2_macro_recall:.4f}"
)

print(
    f"Macro F1        : "
    f"{e2_macro_f1:.4f}"
)

print(
    f"Weighted F1     : "
    f"{e2_weighted_f1:.4f}"
)

# ============================================================
# 6. Confusion Matrix
# ============================================================

class_names = list(
    label_encoder.classes_
)

cm_e2 = confusion_matrix(

    y_true_e2,

    y_pred_e2,

    labels=np.arange(
        NUM_DISEASES
    )

)

print("\nConfusion Matrix")
print("-" * 60)

print(
    cm_e2
)

# ============================================================
# 7. Classification Report
# ============================================================

print("\nClassification Report")
print("-" * 60)

print(

    classification_report(

        y_true_e2,

        y_pred_e2,

        labels=np.arange(
            NUM_DISEASES
        ),

        target_names=class_names,

        digits=4,

        zero_division=0

    )

)

# ============================================================
# 8. Per-Disease Metrics
# ============================================================

per_class_e2 = []

for i, disease in enumerate(
    class_names
):

    TP = cm_e2[i, i]

    FN = (
        cm_e2[i, :].sum()
        -
        TP
    )

    FP = (
        cm_e2[:, i].sum()
        -
        TP
    )

    TN = (
        cm_e2.sum()
        -
        TP
        -
        FN
        -
        FP
    )

    sensitivity = (
        TP / (TP + FN)
        if (TP + FN) > 0
        else 0
    )

    specificity = (
        TN / (TN + FP)
        if (TN + FP) > 0
        else 0
    )

    precision = (
        TP / (TP + FP)
        if (TP + FP) > 0
        else 0
    )

    f1 = (
        2
        *
        precision
        *
        sensitivity
        /
        (
            precision
            +
            sensitivity
        )
        if (
            precision
            +
            sensitivity
        ) > 0
        else 0
    )

    per_class_e2.append({

        "Disease":
            disease,

        "Sensitivity":
            sensitivity,

        "Specificity":
            specificity,

        "Precision":
            precision,

        "F1":
            f1,

        "Support":
            int(
                cm_e2[
                    i,
                    :
                ].sum()
            )

    })

per_class_e2_df = pd.DataFrame(
    per_class_e2
)

print("\nPer-Disease Metrics")
print("-" * 60)

display(
    per_class_e2_df
)

# ============================================================
# 9. ROC-AUC + PR-AUC
# ============================================================

y_true_binary_e2 = label_binarize(

    y_true_e2,

    classes=np.arange(
        NUM_DISEASES
    )

)

e2_macro_roc_auc = roc_auc_score(

    y_true_binary_e2,

    y_prob_e2,

    average="macro",

    multi_class="ovr"

)

e2_macro_pr_auc = average_precision_score(

    y_true_binary_e2,

    y_prob_e2,

    average="macro"

)

print("\nRanking Metrics")
print("-" * 60)

print(
    f"Macro ROC-AUC : "
    f"{e2_macro_roc_auc:.4f}"
)

print(
    f"Macro PR-AUC  : "
    f"{e2_macro_pr_auc:.4f}"
)

# ============================================================
# 10. Trajectory-Length Analysis
# ============================================================

prediction_e2_df = pd.DataFrame({

    "patient_id":
        all_patient_ids,

    "true_label":
        y_true_e2,

    "predicted_label":
        y_pred_e2,

    "correct":
        (
            y_true_e2
            ==
            y_pred_e2
        ),

    "trajectory_length":
        lengths_e2,

    "confidence":
        y_prob_e2.max(
            axis=1
        )

})

length_analysis_e2 = (

    prediction_e2_df

    .groupby(
        "trajectory_length"
    )

    .agg(

        patients=(
            "patient_id",
            "count"
        ),

        accuracy=(
            "correct",
            "mean"
        ),

        mean_confidence=(
            "confidence",
            "mean"
        )

    )

    .reset_index()

)

print("\nPerformance by Trajectory Length")
print("-" * 60)

display(
    length_analysis_e2
)

# ============================================================
# 11. Direct E1 vs E2 Comparison
# ============================================================

comparison_df = pd.DataFrame({

    "Experiment": [
        "E1",
        "E2"
    ],

    "Input": [
        "11 symptoms",
        "11 symptoms + severity"
    ],

    "Best_Val_Macro_F1": [
        0.4746261729016495,
        best_val_f1_e2
    ],

    "Test_Accuracy": [
        0.4644,
        e2_test_accuracy
    ],

    "Test_Macro_F1": [
        0.4595,
        e2_macro_f1
    ],

    "Macro_ROC_AUC": [
        0.7864,
        e2_macro_roc_auc
    ],

    "Macro_PR_AUC": [
        0.5014,
        e2_macro_pr_auc
    ]

})

print("\n" + "=" * 70)

print("E1 vs E2")

print("=" * 70)

display(
    comparison_df
)

# ============================================================
# 12. Improvements
# ============================================================

print("\nE2 Improvement Over E1")
print("-" * 60)

print(
    "Validation Macro-F1 change:",
    f"{best_val_f1_e2 - 0.4746261729016495:+.4f}"
)

print(
    "Test Accuracy change:",
    f"{e2_test_accuracy - 0.4644:+.4f}"
)

print(
    "Test Macro-F1 change:",
    f"{e2_macro_f1 - 0.4595:+.4f}"
)

print(
    "Macro ROC-AUC change:",
    f"{e2_macro_roc_auc - 0.7864:+.4f}"
)

print(
    "Macro PR-AUC change:",
    f"{e2_macro_pr_auc - 0.5014:+.4f}"
)

# ============================================================
# 13. Save Results
# ============================================================

comparison_df.to_csv(

    os.path.join(

        E2_OUTPUT_DIR,

        "E1_vs_E2_comparison.csv"

    ),

    index=False

)

per_class_e2_df.to_csv(

    os.path.join(

        E2_OUTPUT_DIR,

        "E2_per_disease_metrics.csv"

    ),

    index=False

)

prediction_e2_df.to_csv(

    os.path.join(

        E2_OUTPUT_DIR,

        "E2_test_predictions.csv"

    ),

    index=False

)

length_analysis_e2.to_csv(

    os.path.join(

        E2_OUTPUT_DIR,

        "E2_performance_by_trajectory_length.csv"

    ),

    index=False

)

# ============================================================
# 14. Final Summary
# ============================================================

print("\n" + "=" * 70)

print("FINAL E2 TEST SUMMARY")

print("=" * 70)

print(
    "Best Epoch       :",
    checkpoint_e2[
        "epoch"
    ]
)

print(
    "Best Val F1      :",
    f"{checkpoint_e2['val_macro_f1']:.4f}"
)

print(
    "Test Accuracy    :",
    f"{e2_test_accuracy:.4f}"
)

print(
    "Test Macro-F1    :",
    f"{e2_macro_f1:.4f}"
)

print(
    "Macro ROC-AUC    :",
    f"{e2_macro_roc_auc:.4f}"
)

print(
    "Macro PR-AUC     :",
    f"{e2_macro_pr_auc:.4f}"
)

print("=" * 70)


In [ ]:
# ============================================================
# STraT-Net
# Experiment E2
# Final Experiment Record
# ============================================================

import os
import json

E2_SUMMARY = {

    "experiment_id":
        "E2",

    "architecture":
        "STraT-Net V2",

    "attention":
        "Gated Temporal Attention",

    "input_features":
        12,

    "input_description":
        "11 longitudinal symptom features + overall severity",

    "training_patients":
        2100,

    "validation_patients":
        450,

    "test_patients":
        450,

    "best_epoch":
        25,

    "best_validation_macro_f1":
        0.534787017366394,

    "test_accuracy":
        0.506667,

    "test_macro_precision":
        0.5086,

    "test_macro_recall":
        0.5067,

    "test_macro_f1":
        0.505231,

    "test_weighted_f1":
        0.505231,

    "test_macro_roc_auc":
        0.812457,

    "test_macro_pr_auc":
        0.536556,

    "baseline_E1_test_macro_f1":
        0.4595,

    "macro_f1_improvement_over_E1":
        0.045731,

    "notes":
        (
            "Severity-augmented longitudinal model. "
            "Disease-only supervised training. "
            "Auxiliary symptom loss disabled."
        )

}

E2_SUMMARY_PATH = os.path.join(

    E2_OUTPUT_DIR,

    "experiment_E2_summary.json"

)

with open(
    E2_SUMMARY_PATH,
    "w"
) as f:

    json.dump(
        E2_SUMMARY,
        f,
        indent=4
    )

print("=" * 70)
print("EXPERIMENT E2 SAVED")
print("=" * 70)

for key, value in E2_SUMMARY.items():

    print(
        f"{key:<35}: {value}"
    )

print("\nSaved to:")
print(E2_SUMMARY_PATH)

print("=" * 70)

In [ ]:
# ============================================================
# STraT-Net
# Experiment E3
# Module 3A
# Add Patient-Level Symptom Targets
# ============================================================

import torch
from torch.utils.data import Dataset, DataLoader

print("=" * 70)
print("STraT-Net : Experiment E3")
print("Patient-Level Symptom Target Construction")
print("=" * 70)

# ============================================================
# 1. E3 Dataset Wrapper
#
# Keeps the same E2 trajectory:
# 11 symptoms + overall severity = 12 features
#
# Adds:
# symptom_target = mean of 11 symptom features
# across valid visits only
# ============================================================

class PatientTrajectoryDatasetE3(Dataset):

    def __init__(self, base_dataset):

        self.base_dataset = base_dataset

    def __len__(self):

        return len(
            self.base_dataset
        )

    def __getitem__(self, idx):

        sample = self.base_dataset[
            idx
        ]

        trajectory = sample[
            "trajectory"
        ]

        mask = sample[
            "mask"
        ]

        # --------------------------------------------
        # First 11 columns = symptom features
        # Last column = overall severity
        # --------------------------------------------

        valid_symptoms = trajectory[
            mask,
            :NUM_SYMPTOMS
        ]

        # Patient-level continuous symptom target
        symptom_target = valid_symptoms.mean(
            dim=0
        )

        return {

            "patient_id":
                sample[
                    "patient_id"
                ],

            "trajectory":
                sample[
                    "trajectory"
                ],

            "mask":
                sample[
                    "mask"
                ],

            "label":
                sample[
                    "label"
                ],

            "years":
                sample[
                    "years"
                ],

            "length":
                sample[
                    "length"
                ],

            "symptom_target":
                symptom_target

        }


# ============================================================
# 2. Create E3 Dataset Objects
# ============================================================

train_patient_dataset_e3 = (
    PatientTrajectoryDatasetE3(
        train_patient_dataset_e2
    )
)

val_patient_dataset_e3 = (
    PatientTrajectoryDatasetE3(
        val_patient_dataset_e2
    )
)

test_patient_dataset_e3 = (
    PatientTrajectoryDatasetE3(
        test_patient_dataset_e2
    )
)

print(
    "\n✓ E3 datasets created."
)

# ============================================================
# 3. Create Stable E3 DataLoaders
# ============================================================

train_patient_loader_e3 = DataLoader(

    train_patient_dataset_e3,

    batch_size=cfg.BATCH_SIZE,

    shuffle=True,

    num_workers=0,

    pin_memory=torch.cuda.is_available()

)

val_patient_loader_e3 = DataLoader(

    val_patient_dataset_e3,

    batch_size=cfg.BATCH_SIZE,

    shuffle=False,

    num_workers=0,

    pin_memory=torch.cuda.is_available()

)

test_patient_loader_e3 = DataLoader(

    test_patient_dataset_e3,

    batch_size=cfg.BATCH_SIZE,

    shuffle=False,

    num_workers=0,

    pin_memory=torch.cuda.is_available()

)

print(
    "✓ E3 DataLoaders created."
)

# ============================================================
# 4. Verify One Batch
# ============================================================

sample_e3 = next(
    iter(
        train_patient_loader_e3
    )
)

print("\nE3 Batch Verification")
print("-" * 60)

print(
    "Trajectory Shape     :",
    sample_e3[
        "trajectory"
    ].shape
)

print(
    "Mask Shape           :",
    sample_e3[
        "mask"
    ].shape
)

print(
    "Disease Labels Shape :",
    sample_e3[
        "label"
    ].shape
)

print(
    "Symptom Target Shape :",
    sample_e3[
        "symptom_target"
    ].shape
)

print(
    "Years Shape          :",
    sample_e3[
        "years"
    ].shape
)

print(
    "Lengths Shape        :",
    sample_e3[
        "length"
    ].shape
)

# ============================================================
# 5. Verify Target Range
# ============================================================

targets = sample_e3[
    "symptom_target"
]

print("\nSymptom Target Statistics")
print("-" * 60)

print(
    "Minimum:",
    targets.min().item()
)

print(
    "Maximum:",
    targets.max().item()
)

print(
    "Mean   :",
    targets.mean().item()
)

# ============================================================
# 6. Assertions
# ============================================================

assert (
    sample_e3[
        "trajectory"
    ].shape[-1]
    ==
    12
)

assert (
    sample_e3[
        "symptom_target"
    ].shape[-1]
    ==
    11
)

assert not torch.isnan(
    sample_e3[
        "symptom_target"
    ]
).any()

assert not torch.isinf(
    sample_e3[
        "symptom_target"
    ]
).any()

print(
    "\n✓ E3 symptom targets verified."
)

print("\n" + "=" * 70)
print("E3 Module 3A Completed Successfully")
print("=" * 70)

In [ ]:
# ============================================================
# STraT-Net
# Experiment E3
# Module 4/5A
# Fresh Multi-Task Model + Training Setup
# ============================================================

import os
import json
import copy

import torch
import torch.nn as nn

from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR

print("=" * 70)
print("STraT-Net : Experiment E3")
print("Fresh Multi-Task Training Setup")
print("=" * 70)

# ============================================================
# 1. Fresh E3 Model
#
# Same architecture as E2
# Fresh weights — do NOT continue training E2 weights
# ============================================================

model_e3 = STraTNetE2()

initialize_weights_e2(
    model_e3
)

model_e3 = model_e3.to(
    cfg.DEVICE
)

print("\n✓ Fresh E3 model created.")

print(
    "Model Device:",
    next(
        model_e3.parameters()
    ).device
)

# ============================================================
# 2. Verify E3 Architecture Matches E2
# ============================================================

total_params_e3 = sum(

    p.numel()

    for p in model_e3.parameters()

)

print(
    "Total Parameters:",
    f"{total_params_e3:,}"
)

assert total_params_e3 == 947729

print(
    "✓ E3 architecture matches E2."
)

# ============================================================
# 3. Training Hyperparameters
#
# Keep identical to E2 except auxiliary loss
# ============================================================

E3_NUM_EPOCHS = 30

E3_LEARNING_RATE = 1e-4

E3_WEIGHT_DECAY = 1e-4

E3_GRAD_CLIP = 1.0

E3_LABEL_SMOOTHING = 0.05

E3_EARLY_STOPPING_PATIENCE = 8

# New experimental variable
E3_SYMPTOM_LOSS_WEIGHT = 0.20

print("\nE3 Training Configuration")
print("-" * 60)

print(
    "Epochs                  :",
    E3_NUM_EPOCHS
)

print(
    "Learning Rate           :",
    E3_LEARNING_RATE
)

print(
    "Weight Decay            :",
    E3_WEIGHT_DECAY
)

print(
    "Gradient Clip           :",
    E3_GRAD_CLIP
)

print(
    "Label Smoothing         :",
    E3_LABEL_SMOOTHING
)

print(
    "Early Stopping Patience :",
    E3_EARLY_STOPPING_PATIENCE
)

print(
    "Symptom Loss Weight     :",
    E3_SYMPTOM_LOSS_WEIGHT
)

# ============================================================
# 4. Loss Functions
# ============================================================

disease_loss_fn_e3 = nn.CrossEntropyLoss(

    weight=class_weights.to(
        cfg.DEVICE
    ),

    label_smoothing=E3_LABEL_SMOOTHING

)

# Continuous patient-level symptom targets
symptom_loss_fn_e3 = nn.MSELoss()

print(
    "\n✓ Disease loss created."
)

print(
    "✓ Symptom MSE loss created."
)

# ============================================================
# 5. Fresh Optimizer
# ============================================================

optimizer_e3 = AdamW(

    model_e3.parameters(),

    lr=E3_LEARNING_RATE,

    weight_decay=E3_WEIGHT_DECAY

)

print(
    "✓ Fresh E3 optimizer created."
)

# ============================================================
# 6. Fresh Scheduler
# ============================================================

scheduler_e3 = CosineAnnealingLR(

    optimizer_e3,

    T_max=E3_NUM_EPOCHS,

    eta_min=1e-6

)

print(
    "✓ Fresh E3 scheduler created."
)

# ============================================================
# 7. Mixed Precision
# ============================================================

E3_USE_AMP = torch.cuda.is_available()

if E3_USE_AMP:

    scaler_e3 = torch.cuda.amp.GradScaler()

else:

    scaler_e3 = None

print(
    "Mixed Precision:",
    E3_USE_AMP
)

# ============================================================
# 8. Experiment Directories
# ============================================================

E3_CHECKPOINT_DIR = (
    "checkpoints_stratnet_e3_multitask"
)

E3_OUTPUT_DIR = (
    "outputs_stratnet_e3_multitask"
)

os.makedirs(
    E3_CHECKPOINT_DIR,
    exist_ok=True
)

os.makedirs(
    E3_OUTPUT_DIR,
    exist_ok=True
)

E3_BEST_MODEL_PATH = os.path.join(

    E3_CHECKPOINT_DIR,

    "best_model_e3.pth"

)

E3_LAST_MODEL_PATH = os.path.join(

    E3_CHECKPOINT_DIR,

    "last_model_e3.pth"

)

# ============================================================
# 9. Fresh Training State
# ============================================================

history_e3 = {

    "epoch": [],

    "train_total_loss": [],

    "train_disease_loss": [],

    "train_symptom_loss": [],

    "val_total_loss": [],

    "val_disease_loss": [],

    "val_symptom_loss": [],

    "train_accuracy": [],

    "val_accuracy": [],

    "train_macro_f1": [],

    "val_macro_f1": [],

    "learning_rate": []

}

best_val_f1_e3 = -1.0

best_epoch_e3 = -1

early_stop_counter_e3 = 0

print(
    "\n✓ Fresh E3 history initialized."
)

# ============================================================
# 10. Save Checkpoint
# ============================================================

def save_checkpoint_e3(

    model,

    optimizer,

    scheduler,

    epoch,

    val_macro_f1,

    path

):

    torch.save(

        {

            "experiment":
                "E3",

            "epoch":
                epoch,

            "model_state_dict":
                model.state_dict(),

            "optimizer_state_dict":
                optimizer.state_dict(),

            "scheduler_state_dict":
                scheduler.state_dict(),

            "val_macro_f1":
                val_macro_f1,

            "symptom_loss_weight":
                E3_SYMPTOM_LOSS_WEIGHT,

            "input_features":
                12,

            "feature_description":
                (
                    "11 symptoms + overall severity "
                    "+ auxiliary symptom supervision"
                ),

            "label_classes":
                list(
                    label_encoder.classes_
                ),

            "symptom_vocabulary":
                symptom_vocab

        },

        path

    )

# ============================================================
# 11. Save History
# ============================================================

def save_history_e3():

    import pandas as pd

    pd.DataFrame(
        history_e3
    ).to_csv(

        os.path.join(

            E3_OUTPUT_DIR,

            "training_history_e3.csv"

        ),

        index=False

    )

# ============================================================
# 12. Quick Forward/Loss Test
# ============================================================

sample_e3 = next(
    iter(
        train_patient_loader_e3
    )
)

trajectory = (
    sample_e3[
        "trajectory"
    ]
    .float()
    .to(cfg.DEVICE)
)

years = (
    sample_e3[
        "years"
    ]
    .long()
    .to(cfg.DEVICE)
)

mask = (
    sample_e3[
        "mask"
    ]
    .bool()
    .to(cfg.DEVICE)
)

labels = (
    sample_e3[
        "label"
    ]
    .long()
    .to(cfg.DEVICE)
)

symptom_targets = (
    sample_e3[
        "symptom_target"
    ]
    .float()
    .to(cfg.DEVICE)
)

model_e3.eval()

with torch.no_grad():

    outputs = model_e3(

        trajectory,

        years,

        mask

    )

    disease_loss_test = disease_loss_fn_e3(

        outputs[
            "disease_logits"
        ],

        labels

    )

    symptom_loss_test = symptom_loss_fn_e3(

        outputs[
            "symptom_logits"
        ],

        symptom_targets

    )

    total_loss_test = (

        disease_loss_test

        +

        E3_SYMPTOM_LOSS_WEIGHT
        *
        symptom_loss_test

    )

print("\nLoss Verification")
print("-" * 60)

print(
    "Disease Loss :",
    disease_loss_test.item()
)

print(
    "Symptom Loss :",
    symptom_loss_test.item()
)

print(
    "Total Loss   :",
    total_loss_test.item()
)

print(
    "\nDisease logits shape:",
    outputs[
        "disease_logits"
    ].shape
)

print(
    "Symptom logits shape:",
    outputs[
        "symptom_logits"
    ].shape
)

assert (
    outputs[
        "disease_logits"
    ].shape
    ==
    torch.Size(
        [
            trajectory.shape[0],
            5
        ]
    )
)

assert (
    outputs[
        "symptom_logits"
    ].shape
    ==
    torch.Size(
        [
            trajectory.shape[0],
            11
        ]
    )
)

# ============================================================
# 13. Final Status
# ============================================================

print("\n" + "=" * 70)

print("E3 TRAINING SETUP STATUS")

print("=" * 70)

print(
    "Architecture       : Same as E2"
)

print(
    "Input Features     : 12"
)

print(
    "Disease Loss       : CrossEntropyLoss"
)

print(
    "Symptom Loss       : MSELoss"
)

print(
    "Symptom Loss Weight:",
    E3_SYMPTOM_LOSS_WEIGHT
)

print(
    "Train Batches      :",
    len(
        train_patient_loader_e3
    )
)

print(
    "Validation Batches :",
    len(
        val_patient_loader_e3
    )
)

print(
    "Test Batches       :",
    len(
        test_patient_loader_e3
    )
)

print("=" * 70)

print(
    "\nE3 READY FOR MULTI-TASK TRAINING"
)

print("=" * 70)

In [ ]:
# ============================================================
# STraT-Net
# Experiment E3
# Multi-Task Training Loop
# ============================================================

import os
import numpy as np
import pandas as pd
import torch

from tqdm.auto import tqdm

print("=" * 70)
print("STraT-Net : Experiment E3")
print("Multi-Task Training Loop")
print("=" * 70)

# ============================================================
# 1. Train One Epoch
# ============================================================

def train_one_epoch_e3(
    model,
    dataloader,
    optimizer,
    disease_criterion,
    symptom_criterion,
    device,
    scaler,
    epoch
):

    model.train()

    running_total_loss = 0.0
    running_disease_loss = 0.0
    running_symptom_loss = 0.0

    all_labels = []
    all_predictions = []

    progress_bar = tqdm(
        dataloader,
        total=len(dataloader),
        desc=f"E3 Epoch {epoch} [Train]"
    )

    for batch_idx, batch in enumerate(progress_bar):

        trajectory = batch["trajectory"].float().to(device)
        years = batch["years"].long().to(device)
        mask = batch["mask"].bool().to(device)
        labels = batch["label"].long().to(device)

        symptom_targets = (
            batch["symptom_target"]
            .float()
            .to(device)
        )

        optimizer.zero_grad(set_to_none=True)

        if scaler is not None:

            with torch.cuda.amp.autocast():

                outputs = model(
                    trajectory,
                    years,
                    mask
                )

                disease_loss = disease_criterion(
                    outputs["disease_logits"],
                    labels
                )

                symptom_loss = symptom_criterion(
                    outputs["symptom_logits"],
                    symptom_targets
                )

                total_loss = (
                    disease_loss
                    +
                    E3_SYMPTOM_LOSS_WEIGHT
                    *
                    symptom_loss
                )

            scaler.scale(
                total_loss
            ).backward()

            scaler.unscale_(
                optimizer
            )

            torch.nn.utils.clip_grad_norm_(
                model.parameters(),
                E3_GRAD_CLIP
            )

            scaler.step(
                optimizer
            )

            scaler.update()

        else:

            outputs = model(
                trajectory,
                years,
                mask
            )

            disease_loss = disease_criterion(
                outputs["disease_logits"],
                labels
            )

            symptom_loss = symptom_criterion(
                outputs["symptom_logits"],
                symptom_targets
            )

            total_loss = (
                disease_loss
                +
                E3_SYMPTOM_LOSS_WEIGHT
                *
                symptom_loss
            )

            total_loss.backward()

            torch.nn.utils.clip_grad_norm_(
                model.parameters(),
                E3_GRAD_CLIP
            )

            optimizer.step()

        running_total_loss += total_loss.item()
        running_disease_loss += disease_loss.item()
        running_symptom_loss += symptom_loss.item()

        probabilities = torch.softmax(
            outputs["disease_logits"],
            dim=1
        )

        predictions = torch.argmax(
            probabilities,
            dim=1
        )

        all_labels.extend(
            labels.detach().cpu().numpy().tolist()
        )

        all_predictions.extend(
            predictions.detach().cpu().numpy().tolist()
        )

        progress_bar.set_postfix({
            "loss":
                f"{running_total_loss / (batch_idx + 1):.4f}"
        })

    metrics = compute_metrics(
        all_labels,
        all_predictions
    )

    return {

        "total_loss":
            running_total_loss / len(dataloader),

        "disease_loss":
            running_disease_loss / len(dataloader),

        "symptom_loss":
            running_symptom_loss / len(dataloader),

        "accuracy":
            metrics["accuracy"],

        "macro_f1":
            metrics["macro_f1"]

    }

# ============================================================
# 2. Validate One Epoch
# ============================================================

def validate_one_epoch_e3(
    model,
    dataloader,
    disease_criterion,
    symptom_criterion,
    device,
    epoch
):

    model.eval()

    running_total_loss = 0.0
    running_disease_loss = 0.0
    running_symptom_loss = 0.0

    all_labels = []
    all_predictions = []

    progress_bar = tqdm(
        dataloader,
        total=len(dataloader),
        desc=f"E3 Epoch {epoch} [Validation]"
    )

    with torch.no_grad():

        for batch_idx, batch in enumerate(progress_bar):

            trajectory = batch["trajectory"].float().to(device)
            years = batch["years"].long().to(device)
            mask = batch["mask"].bool().to(device)
            labels = batch["label"].long().to(device)

            symptom_targets = (
                batch["symptom_target"]
                .float()
                .to(device)
            )

            if E3_USE_AMP:

                with torch.cuda.amp.autocast():

                    outputs = model(
                        trajectory,
                        years,
                        mask
                    )

                    disease_loss = disease_criterion(
                        outputs["disease_logits"],
                        labels
                    )

                    symptom_loss = symptom_criterion(
                        outputs["symptom_logits"],
                        symptom_targets
                    )

                    total_loss = (
                        disease_loss
                        +
                        E3_SYMPTOM_LOSS_WEIGHT
                        *
                        symptom_loss
                    )

            else:

                outputs = model(
                    trajectory,
                    years,
                    mask
                )

                disease_loss = disease_criterion(
                    outputs["disease_logits"],
                    labels
                )

                symptom_loss = symptom_criterion(
                    outputs["symptom_logits"],
                    symptom_targets
                )

                total_loss = (
                    disease_loss
                    +
                    E3_SYMPTOM_LOSS_WEIGHT
                    *
                    symptom_loss
                )

            running_total_loss += total_loss.item()
            running_disease_loss += disease_loss.item()
            running_symptom_loss += symptom_loss.item()

            probabilities = torch.softmax(
                outputs["disease_logits"],
                dim=1
            )

            predictions = torch.argmax(
                probabilities,
                dim=1
            )

            all_labels.extend(
                labels.cpu().numpy().tolist()
            )

            all_predictions.extend(
                predictions.cpu().numpy().tolist()
            )

            progress_bar.set_postfix({
                "loss":
                    f"{running_total_loss / (batch_idx + 1):.4f}"
            })

    metrics = compute_metrics(
        all_labels,
        all_predictions
    )

    return {

        "total_loss":
            running_total_loss / len(dataloader),

        "disease_loss":
            running_disease_loss / len(dataloader),

        "symptom_loss":
            running_symptom_loss / len(dataloader),

        "accuracy":
            metrics["accuracy"],

        "macro_f1":
            metrics["macro_f1"]

    }

# ============================================================
# 3. Full E3 Training Loop
# ============================================================

def train_model_e3():

    global best_val_f1_e3
    global best_epoch_e3
    global early_stop_counter_e3
    global history_e3

    print("\n" + "=" * 70)
    print("STARTING E3 MULTI-TASK TRAINING")
    print("=" * 70)

    for epoch in range(
        1,
        E3_NUM_EPOCHS + 1
    ):

        print("\n" + "=" * 70)
        print(
            f"E3 EPOCH {epoch}/{E3_NUM_EPOCHS}"
        )
        print("=" * 70)

        train_results = train_one_epoch_e3(
            model=model_e3,
            dataloader=train_patient_loader_e3,
            optimizer=optimizer_e3,
            disease_criterion=disease_loss_fn_e3,
            symptom_criterion=symptom_loss_fn_e3,
            device=cfg.DEVICE,
            scaler=scaler_e3,
            epoch=epoch
        )

        val_results = validate_one_epoch_e3(
            model=model_e3,
            dataloader=val_patient_loader_e3,
            disease_criterion=disease_loss_fn_e3,
            symptom_criterion=symptom_loss_fn_e3,
            device=cfg.DEVICE,
            epoch=epoch
        )

        current_lr = (
            optimizer_e3
            .param_groups[0]["lr"]
        )

        history_e3["epoch"].append(epoch)

        history_e3[
            "train_total_loss"
        ].append(
            train_results["total_loss"]
        )

        history_e3[
            "train_disease_loss"
        ].append(
            train_results["disease_loss"]
        )

        history_e3[
            "train_symptom_loss"
        ].append(
            train_results["symptom_loss"]
        )

        history_e3[
            "val_total_loss"
        ].append(
            val_results["total_loss"]
        )

        history_e3[
            "val_disease_loss"
        ].append(
            val_results["disease_loss"]
        )

        history_e3[
            "val_symptom_loss"
        ].append(
            val_results["symptom_loss"]
        )

        history_e3[
            "train_accuracy"
        ].append(
            train_results["accuracy"]
        )

        history_e3[
            "val_accuracy"
        ].append(
            val_results["accuracy"]
        )

        history_e3[
            "train_macro_f1"
        ].append(
            train_results["macro_f1"]
        )

        history_e3[
            "val_macro_f1"
        ].append(
            val_results["macro_f1"]
        )

        history_e3[
            "learning_rate"
        ].append(
            current_lr
        )

        save_history_e3()

        current_val_f1 = (
            val_results["macro_f1"]
        )

        if (
            current_val_f1
            >
            best_val_f1_e3
        ):

            best_val_f1_e3 = (
                current_val_f1
            )

            best_epoch_e3 = epoch

            early_stop_counter_e3 = 0

            save_checkpoint_e3(
                model=model_e3,
                optimizer=optimizer_e3,
                scheduler=scheduler_e3,
                epoch=epoch,
                val_macro_f1=current_val_f1,
                path=E3_BEST_MODEL_PATH
            )

            print(
                "\n✓ New best E3 model saved."
            )

        else:

            early_stop_counter_e3 += 1

            print(
                "\nNo validation improvement."
            )

            print(
                "Early stopping counter:",
                f"{early_stop_counter_e3}/"
                f"{E3_EARLY_STOPPING_PATIENCE}"
            )

        save_checkpoint_e3(
            model=model_e3,
            optimizer=optimizer_e3,
            scheduler=scheduler_e3,
            epoch=epoch,
            val_macro_f1=current_val_f1,
            path=E3_LAST_MODEL_PATH
        )

        scheduler_e3.step()

        print("\n" + "-" * 60)
        print("E3 EPOCH SUMMARY")
        print("-" * 60)

        print(
            f"Train Total Loss   : "
            f"{train_results['total_loss']:.4f}"
        )

        print(
            f"Val Total Loss     : "
            f"{val_results['total_loss']:.4f}"
        )

        print(
            f"Train Disease Loss : "
            f"{train_results['disease_loss']:.4f}"
        )

        print(
            f"Val Disease Loss   : "
            f"{val_results['disease_loss']:.4f}"
        )

        print(
            f"Train Symptom Loss : "
            f"{train_results['symptom_loss']:.4f}"
        )

        print(
            f"Val Symptom Loss   : "
            f"{val_results['symptom_loss']:.4f}"
        )

        print()

        print(
            f"Train Accuracy     : "
            f"{train_results['accuracy']:.4f}"
        )

        print(
            f"Val Accuracy       : "
            f"{val_results['accuracy']:.4f}"
        )

        print(
            f"Train Macro-F1     : "
            f"{train_results['macro_f1']:.4f}"
        )

        print(
            f"Val Macro-F1       : "
            f"{val_results['macro_f1']:.4f}"
        )

        print()

        print(
            f"Best Val Macro-F1  : "
            f"{best_val_f1_e3:.4f}"
        )

        print(
            f"Best Epoch         : "
            f"{best_epoch_e3}"
        )

        print("-" * 60)

        if (
            early_stop_counter_e3
            >=
            E3_EARLY_STOPPING_PATIENCE
        ):

            print("\n" + "=" * 70)
            print(
                "E3 EARLY STOPPING TRIGGERED"
            )
            print("=" * 70)

            break

    history_e3_df = pd.DataFrame(
        history_e3
    )

    print("\n" + "=" * 70)
    print("E3 TRAINING FINISHED")
    print("=" * 70)

    print(
        "Epochs Completed :",
        len(history_e3_df)
    )

    print(
        "Best Epoch       :",
        best_epoch_e3
    )

    print(
        "Best Val Macro-F1:",
        f"{best_val_f1_e3:.4f}"
    )

    print(
        "Best Checkpoint Exists:",
        os.path.exists(
            E3_BEST_MODEL_PATH
        )
    )

    print("=" * 70)

    return history_e3_df

# ============================================================
# 4. RUN E3 TRAINING
# ============================================================

print(
    "History epochs before training:",
    len(
        history_e3["epoch"]
    )
)

if len(
    history_e3["epoch"]
) != 0:

    raise RuntimeError(
        "E3 history is not empty. "
        "Do not start a mixed second run."
    )

history_e3_df = train_model_e3()

display(
    history_e3_df
)

print("\n" + "=" * 70)
print("E3 Training Module Completed Successfully")
print("=" * 70)

In [ ]:
# ============================================================
# STraT-Net
# Experiment E3
# Final Held-Out Test Evaluation
# ============================================================

import os
import numpy as np
import pandas as pd
import torch

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report,
    roc_auc_score,
    average_precision_score
)

from sklearn.preprocessing import label_binarize

print("=" * 70)
print("STraT-Net : Experiment E3")
print("FINAL HELD-OUT TEST EVALUATION")
print("=" * 70)

# ============================================================
# 1. Load Best Validation-Selected E3 Checkpoint
# ============================================================

checkpoint_e3 = torch.load(
    E3_BEST_MODEL_PATH,
    map_location=cfg.DEVICE
)

print("\nBest E3 Checkpoint")
print("-" * 60)

print(
    "Epoch:",
    checkpoint_e3["epoch"]
)

print(
    "Validation Macro-F1:",
    checkpoint_e3["val_macro_f1"]
)

print(
    "Symptom Loss Weight:",
    checkpoint_e3[
        "symptom_loss_weight"
    ]
)

model_e3.load_state_dict(
    checkpoint_e3[
        "model_state_dict"
    ]
)

model_e3 = model_e3.to(
    cfg.DEVICE
)

model_e3.eval()

print(
    "\n✓ Best E3 model loaded."
)

# ============================================================
# 2. Prediction Storage
# ============================================================

all_patient_ids = []

all_labels = []

all_predictions = []

all_probabilities = []

all_lengths = []

all_attention = []

all_symptom_targets = []

all_symptom_predictions = []

# ============================================================
# 3. Held-Out Test Inference
# ============================================================

with torch.no_grad():

    for batch in test_patient_loader_e3:

        trajectory = (
            batch[
                "trajectory"
            ]
            .float()
            .to(cfg.DEVICE)
        )

        years = (
            batch[
                "years"
            ]
            .long()
            .to(cfg.DEVICE)
        )

        mask = (
            batch[
                "mask"
            ]
            .bool()
            .to(cfg.DEVICE)
        )

        labels = (
            batch[
                "label"
            ]
            .long()
            .to(cfg.DEVICE)
        )

        symptom_targets = (
            batch[
                "symptom_target"
            ]
            .float()
            .to(cfg.DEVICE)
        )

        outputs = model_e3(
            trajectory,
            years,
            mask
        )

        logits = outputs[
            "disease_logits"
        ]

        probabilities = torch.softmax(
            logits,
            dim=1
        )

        predictions = torch.argmax(
            probabilities,
            dim=1
        )

        all_patient_ids.extend(
            list(
                batch[
                    "patient_id"
                ]
            )
        )

        all_labels.extend(
            labels
            .cpu()
            .numpy()
            .tolist()
        )

        all_predictions.extend(
            predictions
            .cpu()
            .numpy()
            .tolist()
        )

        all_probabilities.extend(
            probabilities
            .cpu()
            .numpy()
        )

        all_lengths.extend(
            batch[
                "length"
            ]
            .cpu()
            .numpy()
            .tolist()
        )

        all_attention.extend(
            outputs[
                "attention"
            ]
            .cpu()
            .numpy()
        )

        all_symptom_targets.extend(
            symptom_targets
            .cpu()
            .numpy()
        )

        all_symptom_predictions.extend(
            outputs[
                "symptom_logits"
            ]
            .cpu()
            .numpy()
        )

# ============================================================
# 4. Convert Arrays
# ============================================================

y_true_e3 = np.asarray(
    all_labels
)

y_pred_e3 = np.asarray(
    all_predictions
)

y_prob_e3 = np.asarray(
    all_probabilities
)

lengths_e3 = np.asarray(
    all_lengths
)

attention_e3 = np.asarray(
    all_attention
)

symptom_targets_e3 = np.asarray(
    all_symptom_targets
)

symptom_predictions_e3 = np.asarray(
    all_symptom_predictions
)

print(
    "\nTest Patients:",
    len(
        y_true_e3
    )
)

assert (
    len(
        y_true_e3
    )
    ==
    len(
        test_patient_dataset_e3
    )
)

print(
    "✓ All test patients evaluated exactly once."
)

# ============================================================
# 5. Disease Classification Metrics
# ============================================================

e3_accuracy = accuracy_score(
    y_true_e3,
    y_pred_e3
)

e3_macro_precision = precision_score(
    y_true_e3,
    y_pred_e3,
    average="macro",
    zero_division=0
)

e3_macro_recall = recall_score(
    y_true_e3,
    y_pred_e3,
    average="macro",
    zero_division=0
)

e3_macro_f1 = f1_score(
    y_true_e3,
    y_pred_e3,
    average="macro",
    zero_division=0
)

e3_weighted_f1 = f1_score(
    y_true_e3,
    y_pred_e3,
    average="weighted",
    zero_division=0
)

print("\n" + "=" * 70)

print("E3 FINAL TEST METRICS")

print("=" * 70)

print(
    f"Accuracy        : "
    f"{e3_accuracy:.4f}"
)

print(
    f"Macro Precision : "
    f"{e3_macro_precision:.4f}"
)

print(
    f"Macro Recall    : "
    f"{e3_macro_recall:.4f}"
)

print(
    f"Macro F1        : "
    f"{e3_macro_f1:.4f}"
)

print(
    f"Weighted F1     : "
    f"{e3_weighted_f1:.4f}"
)

# ============================================================
# 6. Confusion Matrix
# ============================================================

class_names = list(
    label_encoder.classes_
)

cm_e3 = confusion_matrix(

    y_true_e3,

    y_pred_e3,

    labels=np.arange(
        NUM_DISEASES
    )

)

print("\nConfusion Matrix")
print("-" * 60)

print(
    cm_e3
)

# ============================================================
# 7. Classification Report
# ============================================================

print("\nClassification Report")
print("-" * 60)

print(

    classification_report(

        y_true_e3,

        y_pred_e3,

        labels=np.arange(
            NUM_DISEASES
        ),

        target_names=class_names,

        digits=4,

        zero_division=0

    )

)

# ============================================================
# 8. ROC-AUC + PR-AUC
# ============================================================

y_true_binary_e3 = label_binarize(

    y_true_e3,

    classes=np.arange(
        NUM_DISEASES
    )

)

e3_macro_roc_auc = roc_auc_score(

    y_true_binary_e3,

    y_prob_e3,

    average="macro",

    multi_class="ovr"

)

e3_macro_pr_auc = average_precision_score(

    y_true_binary_e3,

    y_prob_e3,

    average="macro"

)

print("\nRanking Metrics")
print("-" * 60)

print(
    f"Macro ROC-AUC : "
    f"{e3_macro_roc_auc:.4f}"
)

print(
    f"Macro PR-AUC  : "
    f"{e3_macro_pr_auc:.4f}"
)

# ============================================================
# 9. Auxiliary Symptom Reconstruction Metric
# ============================================================

symptom_mse_e3 = np.mean(

    (
        symptom_predictions_e3
        -
        symptom_targets_e3
    )
    ** 2

)

symptom_mae_e3 = np.mean(

    np.abs(
        symptom_predictions_e3
        -
        symptom_targets_e3
    )

)

print("\nAuxiliary Symptom Reconstruction")
print("-" * 60)

print(
    f"Test Symptom MSE : "
    f"{symptom_mse_e3:.6f}"
)

print(
    f"Test Symptom MAE : "
    f"{symptom_mae_e3:.6f}"
)

# ============================================================
# 10. E1 vs E2 vs E3
# ============================================================

comparison_df = pd.DataFrame({

    "Experiment": [

        "E1",
        "E2",
        "E3"

    ],

    "Configuration": [

        "11 symptoms",

        "11 symptoms + severity",

        (
            "11 symptoms + severity "
            "+ auxiliary symptom loss"
        )

    ],

    "Best_Val_Macro_F1": [

        0.4746261729016495,

        0.534787017366394,

        checkpoint_e3[
            "val_macro_f1"
        ]

    ],

    "Test_Accuracy": [

        0.4644,

        0.506667,

        e3_accuracy

    ],

    "Test_Macro_F1": [

        0.4595,

        0.505231,

        e3_macro_f1

    ],

    "Macro_ROC_AUC": [

        0.7864,

        0.812457,

        e3_macro_roc_auc

    ],

    "Macro_PR_AUC": [

        0.5014,

        0.536556,

        e3_macro_pr_auc

    ]

})

print("\n" + "=" * 70)

print("E1 vs E2 vs E3")

print("=" * 70)

display(
    comparison_df
)

# ============================================================
# 11. E3 vs E2 Difference
# ============================================================

print("\nE3 Change Relative to E2")
print("-" * 60)

print(
    "Validation Macro-F1:",
    f"{checkpoint_e3['val_macro_f1'] - 0.534787017366394:+.4f}"
)

print(
    "Test Accuracy:",
    f"{e3_accuracy - 0.506667:+.4f}"
)

print(
    "Test Macro-F1:",
    f"{e3_macro_f1 - 0.505231:+.4f}"
)

print(
    "Macro ROC-AUC:",
    f"{e3_macro_roc_auc - 0.812457:+.4f}"
)

print(
    "Macro PR-AUC:",
    f"{e3_macro_pr_auc - 0.536556:+.4f}"
)

# ============================================================
# 12. Save Results
# ============================================================

comparison_df.to_csv(

    os.path.join(

        E3_OUTPUT_DIR,

        "E1_E2_E3_comparison.csv"

    ),

    index=False

)

# ============================================================
# 13. Final Summary
# ============================================================

print("\n" + "=" * 70)

print("FINAL E3 TEST SUMMARY")

print("=" * 70)

print(
    "Best Epoch       :",
    checkpoint_e3[
        "epoch"
    ]
)

print(
    "Best Val F1      :",
    f"{checkpoint_e3['val_macro_f1']:.4f}"
)

print(
    "Test Accuracy    :",
    f"{e3_accuracy:.4f}"
)

print(
    "Test Macro-F1    :",
    f"{e3_macro_f1:.4f}"
)

print(
    "Macro ROC-AUC    :",
    f"{e3_macro_roc_auc:.4f}"
)

print(
    "Macro PR-AUC     :",
    f"{e3_macro_pr_auc:.4f}"
)

print("=" * 70)

## Statistical Significance Analysis (E2 vs E3)

This section evaluates whether the E3 auxiliary-task formulation consistently improves over E2 across the same five random seeds. It reports paired tests, paired-bootstrap 95% confidence intervals for the mean improvement, Cohen's *d*<sub>z</sub>, Holm-adjusted p-values, and exports publication-ready tables and a confidence-interval figure.

> **Scope:** These tests support the E2–E3 ablation comparison. Significance against external baselines requires matched patient-level predictions or repeated runs for those baselines.


In [ ]:
# ============================================================
# STraT-Net E3
# Statistical Significance Analysis
# Paired Multi-Seed E2 vs E3 Comparison
# ============================================================

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy.stats import ttest_rel, wilcoxon

print("=" * 70)
print("STraT-Net E3 : STATISTICAL SIGNIFICANCE ANALYSIS")
print("=" * 70)

STATS_OUTPUT_DIR = "outputs_statistics_e3"
os.makedirs(STATS_OUTPUT_DIR, exist_ok=True)

# Same seeds must be used for a valid paired comparison.
e2_stats = pd.DataFrame({
    "seed": [42, 123, 2026, 3407, 7777],
    "Macro-F1": [0.495457, 0.501637, 0.526529, 0.509151, 0.496170],
    "Accuracy": [0.495556, 0.504444, 0.526667, 0.511111, 0.495556],
    "ROC-AUC": [0.811519, 0.813556, 0.810790, 0.811685, 0.799284],
    "PR-AUC": [0.535697, 0.542398, 0.541322, 0.537826, 0.521823],
})

e3_stats = pd.DataFrame({
    "seed": [42, 123, 2026, 3407, 7777],
    "Macro-F1": [0.501447, 0.514469, 0.532969, 0.523500, 0.513284],
    "Accuracy": [0.502222, 0.517778, 0.535556, 0.524444, 0.513333],
    "ROC-AUC": [0.811969, 0.814710, 0.815296, 0.811938, 0.811259],
    "PR-AUC": [0.536564, 0.543842, 0.551169, 0.540238, 0.539767],
})

paired_stats = e2_stats.merge(e3_stats, on="seed", suffixes=("_E2", "_E3"))
metrics_stats = ["Macro-F1", "Accuracy", "ROC-AUC", "PR-AUC"]

for metric in metrics_stats:
    paired_stats[f"{metric}_difference"] = (
        paired_stats[f"{metric}_E3"] - paired_stats[f"{metric}_E2"]
    )

print("\nSeed-wise paired results")
print("-" * 70)
display(paired_stats)


def paired_bootstrap_mean_ci(differences, n_bootstrap=10000, confidence=0.95, seed=2026):
    """Percentile CI for the paired mean difference across seeds."""
    differences = np.asarray(differences, dtype=float)
    rng = np.random.default_rng(seed)
    samples = rng.choice(
        differences,
        size=(n_bootstrap, len(differences)),
        replace=True,
    )
    boot_means = samples.mean(axis=1)
    alpha = 1.0 - confidence
    lower, upper = np.quantile(boot_means, [alpha / 2, 1 - alpha / 2])
    return float(lower), float(upper), boot_means


def holm_adjust(p_values):
    """Holm family-wise error correction without external dependencies."""
    p_values = np.asarray(p_values, dtype=float)
    adjusted = np.full_like(p_values, np.nan)
    valid = np.where(np.isfinite(p_values))[0]
    if len(valid) == 0:
        return adjusted
    order = valid[np.argsort(p_values[valid])]
    running = 0.0
    m = len(order)
    for rank, idx_p in enumerate(order):
        candidate = min(1.0, (m - rank) * p_values[idx_p])
        running = max(running, candidate)
        adjusted[idx_p] = running
    return adjusted

rows_stats = []
bootstrap_distributions = {}

for metric_index, metric in enumerate(metrics_stats):
    e2_values = paired_stats[f"{metric}_E2"].to_numpy(dtype=float)
    e3_values = paired_stats[f"{metric}_E3"].to_numpy(dtype=float)
    differences = e3_values - e2_values

    t_stat, t_p = ttest_rel(e3_values, e2_values)
    try:
        w_stat, w_p = wilcoxon(e3_values, e2_values, alternative="two-sided")
    except ValueError:
        w_stat, w_p = np.nan, np.nan

    mean_difference = float(differences.mean())
    sd_difference = float(differences.std(ddof=1))
    cohens_dz = mean_difference / sd_difference if sd_difference > 0 else np.nan

    ci_low, ci_high, boot_means = paired_bootstrap_mean_ci(
        differences,
        n_bootstrap=10000,
        confidence=0.95,
        seed=2026 + metric_index,
    )
    bootstrap_distributions[metric] = boot_means

    rows_stats.append({
        "Metric": metric,
        "E2_mean": float(e2_values.mean()),
        "E2_sd": float(e2_values.std(ddof=1)),
        "E3_mean": float(e3_values.mean()),
        "E3_sd": float(e3_values.std(ddof=1)),
        "Mean_improvement_E3_minus_E2": mean_difference,
        "Bootstrap_95CI_lower": ci_low,
        "Bootstrap_95CI_upper": ci_high,
        "Paired_t_statistic": float(t_stat),
        "Paired_t_p": float(t_p),
        "Wilcoxon_statistic": float(w_stat) if np.isfinite(w_stat) else np.nan,
        "Wilcoxon_p": float(w_p) if np.isfinite(w_p) else np.nan,
        "Cohens_dz": float(cohens_dz) if np.isfinite(cohens_dz) else np.nan,
        "E3_better_seeds": int((differences > 0).sum()),
        "Number_of_pairs": int(len(differences)),
    })

stats_results_df = pd.DataFrame(rows_stats)
stats_results_df["Paired_t_p_Holm"] = holm_adjust(stats_results_df["Paired_t_p"].to_numpy())
stats_results_df["Wilcoxon_p_Holm"] = holm_adjust(stats_results_df["Wilcoxon_p"].to_numpy())
stats_results_df["Significant_t_Holm_0.05"] = stats_results_df["Paired_t_p_Holm"] < 0.05
stats_results_df["Significant_Wilcoxon_Holm_0.05"] = stats_results_df["Wilcoxon_p_Holm"] < 0.05

print("\nStatistical significance results")
print("-" * 70)
display(stats_results_df)

# Save publication-ready tables.
paired_stats.to_csv(
    os.path.join(STATS_OUTPUT_DIR, "E2_vs_E3_seedwise_results.csv"),
    index=False,
)
stats_results_df.to_csv(
    os.path.join(STATS_OUTPUT_DIR, "E2_vs_E3_statistical_significance.csv"),
    index=False,
)

bootstrap_long = pd.concat(
    [
        pd.DataFrame({"Metric": metric, "Bootstrap_mean_improvement": values})
        for metric, values in bootstrap_distributions.items()
    ],
    ignore_index=True,
)
bootstrap_long.to_csv(
    os.path.join(STATS_OUTPUT_DIR, "paired_bootstrap_distributions.csv"),
    index=False,
)

# Confidence-interval figure.
plot_stats = stats_results_df.copy()
x = np.arange(len(plot_stats))
means = plot_stats["Mean_improvement_E3_minus_E2"].to_numpy()
lower_errors = means - plot_stats["Bootstrap_95CI_lower"].to_numpy()
upper_errors = plot_stats["Bootstrap_95CI_upper"].to_numpy() - means

plt.figure(figsize=(9, 5.5))
plt.errorbar(
    x,
    means,
    yerr=np.vstack([lower_errors, upper_errors]),
    fmt="o",
    capsize=5,
)
plt.axhline(0.0, linewidth=1, linestyle="--")
plt.xticks(x, plot_stats["Metric"])
plt.ylabel("Mean paired improvement (E3 − E2)")
plt.title("E3 Improvement over E2 with Paired-Bootstrap 95% CIs")
plt.tight_layout()
plt.savefig(
    os.path.join(STATS_OUTPUT_DIR, "E2_vs_E3_bootstrap_confidence_intervals.png"),
    dpi=300,
    bbox_inches="tight",
)
plt.show()

print("\nInterpretation")
print("-" * 70)
for _, row in stats_results_df.iterrows():
    significance = (
        "statistically significant after Holm correction"
        if row["Paired_t_p_Holm"] < 0.05
        else "not statistically significant after Holm correction"
    )
    print(
        f"{row['Metric']}: mean E3−E2 = {row['Mean_improvement_E3_minus_E2']:.6f}, "
        f"95% bootstrap CI [{row['Bootstrap_95CI_lower']:.6f}, "
        f"{row['Bootstrap_95CI_upper']:.6f}], "
        f"paired t-test Holm p = {row['Paired_t_p_Holm']:.4g}; {significance}."
    )

print("\nImportant reporting note")
print("-" * 70)
print(
    "This analysis is a paired five-seed E2-versus-E3 ablation test. "
    "Because n=5 is small, report the effect sizes and confidence intervals "
    "alongside p-values. Do not claim significance against external baselines "
    "unless matched patient-level predictions or repeated baseline runs are available."
)

print("\n" + "=" * 70)
print("STATISTICAL SIGNIFICANCE ANALYSIS COMPLETE")
print("=" * 70)
print("Saved files:")
print("- E2_vs_E3_seedwise_results.csv")
print("- E2_vs_E3_statistical_significance.csv")
print("- paired_bootstrap_distributions.csv")
print("- E2_vs_E3_bootstrap_confidence_intervals.png")


## Explainability Analysis (E3)

This section explains the final validation-selected E3 model at two complementary levels:

1. **Temporal attention** identifies which visits/years received the greatest model attention.
2. **Gradient × input attribution** estimates which symptom features most influenced each predicted disease logit.

> Attention is presented as a temporal importance signal, not as a complete causal explanation. Gradient-based feature attribution is included to provide a complementary input-level explanation.


## Longitudinal Trajectory Analysis

This section characterizes how the held-out E3 patient trajectories evolve across visits and how the model's predictions change as additional visits become available. It reports:

- trajectory-length coverage;
- disease-specific symptom and overall-severity progression;
- early-to-late within-patient changes;
- prediction confidence and accuracy using progressively longer prefixes;
- temporal attention alongside the observed trajectory.

All symptom values are reported on the **model-input scale** used by E3. The analysis is descriptive and does not imply causal disease progression. Later visit positions contain fewer patients, so visit counts are always reported with trajectory summaries.

In [ ]:
# ============================================================
# STraT-Net E3
# Longitudinal Trajectory Analysis
# Disease Progression + Prefix-Based Model Behaviour
# ============================================================

import os
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch

try:
    from scipy.stats import wilcoxon
except Exception:
    wilcoxon = None

print("=" * 70)
print("STraT-Net E3 : LONGITUDINAL TRAJECTORY ANALYSIS")
print("=" * 70)

# ============================================================
# 1. Preconditions and configuration
# ============================================================

assert "model_e3" in globals(), "Run or restore the trained E3 model first."
assert "test_patient_loader_e3" in globals(), "Run the E3 test DataLoader cell first."
assert "label_encoder" in globals(), "Disease label encoder is unavailable."
assert "cfg" in globals(), "Configuration object cfg is unavailable."

LONG_OUTPUT_DIR = "outputs_longitudinal_e3"
os.makedirs(LONG_OUTPUT_DIR, exist_ok=True)

model_e3 = model_e3.to(cfg.DEVICE)
model_e3.eval()

class_names_long = list(label_encoder.classes_)
first_long_batch = next(iter(test_patient_loader_e3))
num_long_features = int(first_long_batch["trajectory"].shape[-1])

base_long_feature_names = list(symptom_columns) if "symptom_columns" in globals() else []
if len(base_long_feature_names) == num_long_features:
    feature_names_long = base_long_feature_names
elif len(base_long_feature_names) + 1 == num_long_features:
    feature_names_long = base_long_feature_names + ["Overall Severity"]
else:
    feature_names_long = [f"Feature_{i + 1}" for i in range(num_long_features)]

severity_candidates = [
    i for i, name in enumerate(feature_names_long)
    if str(name).strip().lower() in {"overall severity", "severity", "overall_severity"}
]
severity_index_long = severity_candidates[0] if severity_candidates else None

print("\nLongitudinal analysis configuration")
print("-" * 60)
print("Device             :", cfg.DEVICE)
print("Disease classes    :", len(class_names_long))
print("Input features     :", num_long_features)
print("Severity feature   :", feature_names_long[severity_index_long] if severity_index_long is not None else "Not explicitly identified")
print("Output folder      :", LONG_OUTPUT_DIR)

# ============================================================
# 2. Collect visit-level trajectories and patient summaries
# ============================================================

visit_rows_long = []
patient_rows_long = []
prefix_rows_long = []

with torch.no_grad():
    for batch in test_patient_loader_e3:
        trajectory = batch["trajectory"].float().to(cfg.DEVICE)
        years = batch["years"].long().to(cfg.DEVICE)
        mask = batch["mask"].bool().to(cfg.DEVICE)
        labels = batch["label"].long().to(cfg.DEVICE)
        lengths = batch["length"].cpu().numpy().astype(int)
        patient_ids = list(batch["patient_id"])

        full_outputs = model_e3(trajectory, years, mask)
        full_logits = full_outputs["disease_logits"]
        full_prob = torch.softmax(full_logits, dim=1)
        full_pred = full_prob.argmax(dim=1)
        full_attention = full_outputs.get("attention", None)

        batch_size, max_visits, _ = trajectory.shape

        for i, patient_id in enumerate(patient_ids):
            length_i = int(lengths[i])
            true_i = int(labels[i].item())
            pred_i = int(full_pred[i].item())
            conf_i = float(full_prob[i, pred_i].item())
            correct_i = int(pred_i == true_i)

            patient_rows_long.append({
                "patient_id": patient_id,
                "true_disease": class_names_long[true_i],
                "full_prediction": class_names_long[pred_i],
                "full_prediction_confidence": conf_i,
                "full_prediction_correct": correct_i,
                "trajectory_length": length_i,
                "first_year": int(years[i, 0].item()),
                "last_year": int(years[i, length_i - 1].item()),
                "follow_up_span": int(years[i, length_i - 1].item() - years[i, 0].item()),
            })

            for visit_idx in range(length_i):
                values = trajectory[i, visit_idx].detach().cpu().numpy()
                row = {
                    "patient_id": patient_id,
                    "true_disease": class_names_long[true_i],
                    "visit_index": visit_idx + 1,
                    "year": int(years[i, visit_idx].item()),
                    "trajectory_length": length_i,
                    "attention_weight": (
                        float(full_attention[i, visit_idx].item())
                        if full_attention is not None else np.nan
                    ),
                }
                for j, feature_name in enumerate(feature_names_long):
                    row[feature_name] = float(values[j])
                visit_rows_long.append(row)

        # Prefix analysis: evaluate each available trajectory prefix in a vectorized way.
        # The same padded tensors are reused, but visits after prefix k are masked out.
        max_observed_length = int(lengths.max())
        for prefix_len in range(1, max_observed_length + 1):
            eligible = torch.as_tensor(lengths >= prefix_len, device=cfg.DEVICE)
            if not bool(eligible.any()):
                continue

            prefix_mask = mask.clone()
            prefix_mask[:, prefix_len:] = False

            prefix_outputs = model_e3(trajectory, years, prefix_mask)
            prefix_logits = prefix_outputs["disease_logits"]
            prefix_prob = torch.softmax(prefix_logits, dim=1)
            prefix_pred = prefix_prob.argmax(dim=1)
            prefix_conf = prefix_prob.max(dim=1).values

            eligible_indices = torch.where(eligible)[0]
            for idx_tensor in eligible_indices:
                i = int(idx_tensor.item())
                true_i = int(labels[i].item())
                pred_i = int(prefix_pred[i].item())
                prefix_rows_long.append({
                    "patient_id": patient_ids[i],
                    "true_disease": class_names_long[true_i],
                    "prefix_length": prefix_len,
                    "predicted_disease": class_names_long[pred_i],
                    "prediction_confidence": float(prefix_conf[i].item()),
                    "prediction_correct": int(pred_i == true_i),
                })

visit_trajectory_df = pd.DataFrame(visit_rows_long)
patient_trajectory_df = pd.DataFrame(patient_rows_long)
prefix_predictions_df = pd.DataFrame(prefix_rows_long)

visit_trajectory_df.to_csv(os.path.join(LONG_OUTPUT_DIR, "patient_visit_trajectories.csv"), index=False)
patient_trajectory_df.to_csv(os.path.join(LONG_OUTPUT_DIR, "patient_trajectory_summary.csv"), index=False)
prefix_predictions_df.to_csv(os.path.join(LONG_OUTPUT_DIR, "prefix_predictions.csv"), index=False)

# ============================================================
# 3. Trajectory-length and follow-up coverage
# ============================================================

trajectory_length_df = (
    patient_trajectory_df
    .groupby("trajectory_length", as_index=False)
    .agg(
        n_patients=("patient_id", "size"),
        mean_follow_up_span=("follow_up_span", "mean"),
        median_follow_up_span=("follow_up_span", "median")
    )
    .sort_values("trajectory_length")
)
trajectory_length_df["percentage"] = 100 * trajectory_length_df["n_patients"] / trajectory_length_df["n_patients"].sum()
trajectory_length_df.to_csv(os.path.join(LONG_OUTPUT_DIR, "trajectory_length_distribution.csv"), index=False)

print("\nTrajectory-length distribution")
print("-" * 60)
display(trajectory_length_df)

plt.figure(figsize=(8, 5))
plt.bar(trajectory_length_df["trajectory_length"], trajectory_length_df["n_patients"])
plt.title("E3 Held-Out Cohort: Trajectory-Length Distribution")
plt.xlabel("Number of observed visits")
plt.ylabel("Number of patients")
plt.xticks(trajectory_length_df["trajectory_length"])
plt.tight_layout()
plt.savefig(os.path.join(LONG_OUTPUT_DIR, "trajectory_length_distribution.png"), dpi=300, bbox_inches="tight")
plt.show()

# ============================================================
# 4. Disease-specific feature trajectories
# ============================================================

feature_trajectory_rows = []
for disease_name in class_names_long:
    disease_df = visit_trajectory_df[visit_trajectory_df["true_disease"] == disease_name]
    if disease_df.empty:
        continue
    for visit_idx, visit_df in disease_df.groupby("visit_index"):
        for feature_name in feature_names_long:
            values = visit_df[feature_name].dropna().to_numpy(dtype=float)
            if len(values) == 0:
                continue
            feature_trajectory_rows.append({
                "true_disease": disease_name,
                "visit_index": int(visit_idx),
                "feature": feature_name,
                "mean": float(np.mean(values)),
                "std": float(np.std(values, ddof=1)) if len(values) > 1 else np.nan,
                "sem": float(np.std(values, ddof=1) / np.sqrt(len(values))) if len(values) > 1 else np.nan,
                "n_patients": int(len(values)),
            })

disease_feature_trajectory_df = pd.DataFrame(feature_trajectory_rows)
disease_feature_trajectory_df.to_csv(os.path.join(LONG_OUTPUT_DIR, "disease_feature_trajectories.csv"), index=False)

# Plot one panel per feature as separate figures to keep them readable.
for feature_name in feature_names_long:
    plot_df = disease_feature_trajectory_df[disease_feature_trajectory_df["feature"] == feature_name]
    if plot_df.empty:
        continue
    plt.figure(figsize=(8, 5))
    for disease_name, disease_plot in plot_df.groupby("true_disease"):
        disease_plot = disease_plot.sort_values("visit_index")
        plt.plot(disease_plot["visit_index"], disease_plot["mean"], marker="o", label=disease_name)
    plt.title(f"E3 Longitudinal Trajectory: {feature_name}")
    plt.xlabel("Visit position")
    plt.ylabel("Mean value on model-input scale")
    plt.legend(fontsize=8)
    plt.tight_layout()
    safe_name = "".join(ch if ch.isalnum() else "_" for ch in feature_name).strip("_")
    plt.savefig(os.path.join(LONG_OUTPUT_DIR, f"trajectory_{safe_name}.png"), dpi=300, bbox_inches="tight")
    plt.show()

# Compact heatmap of mean overall feature level across disease and visit.
# This averages input features and is shown only as a descriptive trajectory index.
visit_trajectory_df["mean_feature_level"] = visit_trajectory_df[feature_names_long].mean(axis=1)
trajectory_heatmap_df = (
    visit_trajectory_df
    .pivot_table(index="true_disease", columns="visit_index", values="mean_feature_level", aggfunc="mean")
    .reindex(class_names_long)
)
trajectory_heatmap_df.to_csv(os.path.join(LONG_OUTPUT_DIR, "disease_visit_mean_feature_heatmap.csv"))

plt.figure(figsize=(9, max(5, 0.7 * len(trajectory_heatmap_df))))
plt.imshow(trajectory_heatmap_df.values, aspect="auto")
plt.colorbar(label="Mean input-feature level")
plt.xticks(np.arange(len(trajectory_heatmap_df.columns)), trajectory_heatmap_df.columns)
plt.yticks(np.arange(len(trajectory_heatmap_df.index)), trajectory_heatmap_df.index)
plt.title("E3 Disease-by-Visit Longitudinal Profile")
plt.xlabel("Visit position")
plt.ylabel("True disease")
plt.tight_layout()
plt.savefig(os.path.join(LONG_OUTPUT_DIR, "disease_visit_trajectory_heatmap.png"), dpi=300, bbox_inches="tight")
plt.show()

# ============================================================
# 5. Overall severity progression, when available
# ============================================================

if severity_index_long is not None:
    severity_name = feature_names_long[severity_index_long]
    severity_summary_df = (
        visit_trajectory_df
        .groupby(["true_disease", "visit_index"], as_index=False)
        .agg(
            mean_severity=(severity_name, "mean"),
            std_severity=(severity_name, "std"),
            n_patients=(severity_name, "size")
        )
    )
    severity_summary_df.to_csv(os.path.join(LONG_OUTPUT_DIR, "severity_progression.csv"), index=False)

    print("\nDisease-specific severity progression")
    print("-" * 60)
    display(severity_summary_df.head(20))

    plt.figure(figsize=(8, 5))
    for disease_name, disease_plot in severity_summary_df.groupby("true_disease"):
        disease_plot = disease_plot.sort_values("visit_index")
        plt.plot(disease_plot["visit_index"], disease_plot["mean_severity"], marker="o", label=disease_name)
    plt.title("E3 Overall Severity Progression")
    plt.xlabel("Visit position")
    plt.ylabel("Mean overall severity on model-input scale")
    plt.legend(fontsize=8)
    plt.tight_layout()
    plt.savefig(os.path.join(LONG_OUTPUT_DIR, "severity_progression.png"), dpi=300, bbox_inches="tight")
    plt.show()
else:
    severity_summary_df = pd.DataFrame()
    warnings.warn("No explicit Overall Severity feature was identified; severity-specific analysis was skipped.")

# ============================================================
# 6. Early-versus-late within-patient change
# ============================================================

change_rows = []
for patient_id, patient_visits in visit_trajectory_df.groupby("patient_id"):
    patient_visits = patient_visits.sort_values("visit_index")
    if len(patient_visits) < 2:
        continue
    first_row = patient_visits.iloc[0]
    last_row = patient_visits.iloc[-1]
    for feature_name in feature_names_long:
        change_rows.append({
            "patient_id": patient_id,
            "true_disease": first_row["true_disease"],
            "feature": feature_name,
            "first_value": float(first_row[feature_name]),
            "last_value": float(last_row[feature_name]),
            "change_last_minus_first": float(last_row[feature_name] - first_row[feature_name]),
            "trajectory_length": int(first_row["trajectory_length"]),
        })

patient_change_df = pd.DataFrame(change_rows)
patient_change_df.to_csv(os.path.join(LONG_OUTPUT_DIR, "early_late_patient_changes.csv"), index=False)

change_summary_rows = []
for (disease_name, feature_name), grp in patient_change_df.groupby(["true_disease", "feature"]):
    changes = grp["change_last_minus_first"].to_numpy(dtype=float)
    p_value = np.nan
    statistic = np.nan
    if wilcoxon is not None and len(changes) >= 2 and not np.allclose(changes, 0):
        try:
            statistic, p_value = wilcoxon(changes, alternative="two-sided", zero_method="wilcox")
        except Exception:
            pass
    change_summary_rows.append({
        "true_disease": disease_name,
        "feature": feature_name,
        "n_patients": int(len(changes)),
        "mean_change": float(np.mean(changes)),
        "median_change": float(np.median(changes)),
        "std_change": float(np.std(changes, ddof=1)) if len(changes) > 1 else np.nan,
        "wilcoxon_statistic": statistic,
        "wilcoxon_p_uncorrected": p_value,
    })

early_late_change_summary_df = pd.DataFrame(change_summary_rows)
if not early_late_change_summary_df.empty:
    # Benjamini-Hochberg FDR correction across all disease-feature tests.
    p = early_late_change_summary_df["wilcoxon_p_uncorrected"].to_numpy(dtype=float)
    valid = np.isfinite(p)
    adjusted = np.full_like(p, np.nan)
    if valid.any():
        valid_p = p[valid]
        order = np.argsort(valid_p)
        ranked = valid_p[order]
        m = len(ranked)
        corrected_ranked = ranked * m / np.arange(1, m + 1)
        corrected_ranked = np.minimum.accumulate(corrected_ranked[::-1])[::-1]
        corrected_ranked = np.clip(corrected_ranked, 0, 1)
        back = np.empty_like(corrected_ranked)
        back[order] = corrected_ranked
        adjusted[valid] = back
    early_late_change_summary_df["wilcoxon_p_fdr"] = adjusted
    early_late_change_summary_df["significant_fdr_0.05"] = adjusted < 0.05

early_late_change_summary_df.to_csv(os.path.join(LONG_OUTPUT_DIR, "early_late_change_summary.csv"), index=False)

print("\nEarly-to-late within-patient changes")
print("-" * 60)
display(
    early_late_change_summary_df
    .sort_values(["significant_fdr_0.05", "wilcoxon_p_fdr"], ascending=[False, True])
    .head(25)
)

# ============================================================
# 7. Prediction confidence and accuracy as visits accumulate
# ============================================================

confidence_by_prefix_df = (
    prefix_predictions_df
    .groupby("prefix_length", as_index=False)
    .agg(
        mean_confidence=("prediction_confidence", "mean"),
        std_confidence=("prediction_confidence", "std"),
        accuracy=("prediction_correct", "mean"),
        n_patients=("patient_id", "size")
    )
)
confidence_by_prefix_df.to_csv(os.path.join(LONG_OUTPUT_DIR, "prediction_performance_by_prefix.csv"), index=False)

confidence_by_disease_prefix_df = (
    prefix_predictions_df
    .groupby(["true_disease", "prefix_length"], as_index=False)
    .agg(
        mean_confidence=("prediction_confidence", "mean"),
        accuracy=("prediction_correct", "mean"),
        n_patients=("patient_id", "size")
    )
)
confidence_by_disease_prefix_df.to_csv(os.path.join(LONG_OUTPUT_DIR, "prediction_performance_by_disease_and_prefix.csv"), index=False)

print("\nPrediction performance as visits accumulate")
print("-" * 60)
display(confidence_by_prefix_df)

plt.figure(figsize=(8, 5))
plt.plot(confidence_by_prefix_df["prefix_length"], confidence_by_prefix_df["mean_confidence"], marker="o", label="Mean confidence")
plt.plot(confidence_by_prefix_df["prefix_length"], confidence_by_prefix_df["accuracy"], marker="s", label="Accuracy")
plt.title("E3 Prediction Behaviour as Visits Accumulate")
plt.xlabel("Number of visits available to the model")
plt.ylabel("Proportion")
plt.xticks(confidence_by_prefix_df["prefix_length"])
plt.ylim(0, 1)
plt.legend()
plt.tight_layout()
plt.savefig(os.path.join(LONG_OUTPUT_DIR, "confidence_and_accuracy_by_prefix.png"), dpi=300, bbox_inches="tight")
plt.show()

# ============================================================
# 8. Temporal attention aligned with observed trajectories
# ============================================================

if visit_trajectory_df["attention_weight"].notna().any():
    longitudinal_attention_df = (
        visit_trajectory_df
        .groupby("visit_index", as_index=False)
        .agg(
            mean_attention=("attention_weight", "mean"),
            std_attention=("attention_weight", "std"),
            n_visits=("attention_weight", "size"),
            mean_feature_level=("mean_feature_level", "mean")
        )
    )
    longitudinal_attention_df.to_csv(os.path.join(LONG_OUTPUT_DIR, "attention_and_trajectory_by_visit.csv"), index=False)

    plt.figure(figsize=(8, 5))
    plt.plot(longitudinal_attention_df["visit_index"], longitudinal_attention_df["mean_attention"], marker="o")
    plt.title("E3 Temporal Attention Across the Longitudinal Trajectory")
    plt.xlabel("Visit position")
    plt.ylabel("Mean attention weight")
    plt.xticks(longitudinal_attention_df["visit_index"])
    plt.tight_layout()
    plt.savefig(os.path.join(LONG_OUTPUT_DIR, "longitudinal_attention.png"), dpi=300, bbox_inches="tight")
    plt.show()
else:
    longitudinal_attention_df = pd.DataFrame()

# ============================================================
# 9. Reporting summary
# ============================================================

print("\nInterpretation safeguards")
print("-" * 60)
print("1. Symptom trajectories are descriptive summaries on the model-input scale.")
print("2. Later visits include fewer patients; n_patients should accompany every trajectory estimate.")
print("3. Prefix results compare different eligible subsets at longer lengths and are not a randomized causal test.")
print("4. Attention weights describe model allocation, not clinical causality.")

print("\n" + "=" * 70)
print("LONGITUDINAL TRAJECTORY ANALYSIS COMPLETE")
print("=" * 70)
print("Saved core files:")
print("- patient_visit_trajectories.csv")
print("- patient_trajectory_summary.csv")
print("- trajectory_length_distribution.csv")
print("- disease_feature_trajectories.csv")
print("- severity_progression.csv (when available)")
print("- early_late_patient_changes.csv")
print("- early_late_change_summary.csv")
print("- prefix_predictions.csv")
print("- prediction_performance_by_prefix.csv")
print("- prediction_performance_by_disease_and_prefix.csv")
print("- attention_and_trajectory_by_visit.csv (when attention is available)")
print("- publication-quality PNG figures in", LONG_OUTPUT_DIR)


In [ ]:
# ============================================================
# STraT-Net E3
# Explainability Analysis
# Temporal Attention + Gradient × Input Feature Attribution
# ============================================================

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch

print("=" * 70)
print("STraT-Net E3 : EXPLAINABILITY ANALYSIS")
print("=" * 70)

# ============================================================
# 1. Preconditions and Output Directory
# ============================================================

assert "model_e3" in globals(), "Run the E3 model-definition/training cells first."
assert "test_patient_loader_e3" in globals(), "E3 test DataLoader is unavailable."
assert "label_encoder" in globals(), "Disease label encoder is unavailable."

XAI_OUTPUT_DIR = "outputs_explainability_e3"
os.makedirs(XAI_OUTPUT_DIR, exist_ok=True)

model_e3 = model_e3.to(cfg.DEVICE)
model_e3.eval()

class_names_xai = list(label_encoder.classes_)

# Determine feature names dynamically so this remains compatible with E3.
first_xai_batch = next(iter(test_patient_loader_e3))
num_xai_features = int(first_xai_batch["trajectory"].shape[-1])

base_feature_names = list(symptom_columns) if "symptom_columns" in globals() else []

if len(base_feature_names) == num_xai_features:
    feature_names_xai = base_feature_names
elif len(base_feature_names) + 1 == num_xai_features:
    feature_names_xai = base_feature_names + ["Overall Severity"]
else:
    feature_names_xai = [f"Feature_{i + 1}" for i in range(num_xai_features)]

print("\nExplainability configuration")
print("-" * 60)
print("Device          :", cfg.DEVICE)
print("Disease classes :", len(class_names_xai))
print("Input features  :", num_xai_features)
print("Output folder   :", XAI_OUTPUT_DIR)

# ============================================================
# 2. Collect Patient-Level Explanations
#
# Gradient × input is calculated for the predicted class logit.
# Absolute values are used for global importance aggregation,
# while signed values are retained in patient-level outputs.
# ============================================================

patient_rows_xai = []
visit_rows_xai = []
absolute_feature_attributions = []
signed_feature_attributions = []

for batch in test_patient_loader_e3:

    trajectory = (
        batch["trajectory"]
        .float()
        .to(cfg.DEVICE)
    )

    years = (
        batch["years"]
        .long()
        .to(cfg.DEVICE)
    )

    mask = (
        batch["mask"]
        .bool()
        .to(cfg.DEVICE)
    )

    labels = (
        batch["label"]
        .long()
        .to(cfg.DEVICE)
    )

    # Make the input differentiable without changing model parameters.
    trajectory = trajectory.detach().clone().requires_grad_(True)

    model_e3.zero_grad(set_to_none=True)

    outputs = model_e3(
        trajectory,
        years,
        mask
    )

    logits = outputs["disease_logits"]
    probabilities = torch.softmax(logits, dim=1)
    predicted_classes = probabilities.argmax(dim=1)

    selected_logits = logits.gather(
        1,
        predicted_classes.unsqueeze(1)
    ).sum()

    input_gradients = torch.autograd.grad(
        selected_logits,
        trajectory,
        retain_graph=False,
        create_graph=False
    )[0]

    signed_attr = input_gradients * trajectory

    # Exclude padded visits and aggregate each feature across real visits.
    valid_mask = mask.unsqueeze(-1).float()
    signed_attr = signed_attr * valid_mask

    feature_attr_signed = signed_attr.sum(dim=1)
    feature_attr_abs = signed_attr.abs().sum(dim=1)

    attention = outputs["attention"].detach()

    patient_ids = list(batch["patient_id"])
    lengths = batch["length"].cpu().numpy().astype(int)

    for i, patient_id in enumerate(patient_ids):

        true_idx = int(labels[i].item())
        pred_idx = int(predicted_classes[i].item())
        confidence = float(probabilities[i, pred_idx].item())
        valid_length = int(lengths[i])

        abs_values = feature_attr_abs[i].detach().cpu().numpy()
        signed_values = feature_attr_signed[i].detach().cpu().numpy()

        absolute_feature_attributions.append(abs_values)
        signed_feature_attributions.append(signed_values)

        top_feature_idx = np.argsort(abs_values)[::-1][:5]

        patient_row = {
            "patient_id": patient_id,
            "true_disease": class_names_xai[true_idx],
            "predicted_disease": class_names_xai[pred_idx],
            "prediction_confidence": confidence,
            "trajectory_length": valid_length,
            "top_features": "; ".join(
                f"{feature_names_xai[j]} ({signed_values[j]:.5f})"
                for j in top_feature_idx
            )
        }

        for j, feature_name in enumerate(feature_names_xai):
            patient_row[f"signed_attr__{feature_name}"] = float(signed_values[j])
            patient_row[f"abs_attr__{feature_name}"] = float(abs_values[j])

        patient_rows_xai.append(patient_row)

        patient_attention = attention[i, :valid_length].cpu().numpy()
        patient_years = years[i, :valid_length].detach().cpu().numpy()

        for visit_index in range(valid_length):
            visit_rows_xai.append({
                "patient_id": patient_id,
                "visit_index": visit_index + 1,
                "year": int(patient_years[visit_index]),
                "attention_weight": float(patient_attention[visit_index]),
                "true_disease": class_names_xai[true_idx],
                "predicted_disease": class_names_xai[pred_idx]
            })

# ============================================================
# 3. Save Patient- and Visit-Level Explanations
# ============================================================

patient_explanations_df = pd.DataFrame(patient_rows_xai)
visit_attention_df = pd.DataFrame(visit_rows_xai)

patient_explanations_df.to_csv(
    os.path.join(XAI_OUTPUT_DIR, "patient_feature_attributions.csv"),
    index=False
)

visit_attention_df.to_csv(
    os.path.join(XAI_OUTPUT_DIR, "patient_visit_attention.csv"),
    index=False
)

print("\nPatient-level explanation preview")
print("-" * 60)
display(
    patient_explanations_df[
        [
            "patient_id",
            "true_disease",
            "predicted_disease",
            "prediction_confidence",
            "top_features"
        ]
    ].head(10)
)

# ============================================================
# 4. Global Feature Importance
# ============================================================

absolute_feature_attributions = np.asarray(absolute_feature_attributions)
signed_feature_attributions = np.asarray(signed_feature_attributions)

global_abs_importance = absolute_feature_attributions.mean(axis=0)
global_signed_importance = signed_feature_attributions.mean(axis=0)

global_feature_importance_df = pd.DataFrame({
    "feature": feature_names_xai,
    "mean_absolute_gradient_x_input": global_abs_importance,
    "mean_signed_gradient_x_input": global_signed_importance
}).sort_values(
    "mean_absolute_gradient_x_input",
    ascending=False
).reset_index(drop=True)

global_feature_importance_df.to_csv(
    os.path.join(XAI_OUTPUT_DIR, "global_feature_importance.csv"),
    index=False
)

print("\nGlobal feature importance")
print("-" * 60)
display(global_feature_importance_df)

plt.figure(figsize=(9, 6))
plot_df = global_feature_importance_df.sort_values(
    "mean_absolute_gradient_x_input",
    ascending=True
)
plt.barh(
    plot_df["feature"],
    plot_df["mean_absolute_gradient_x_input"]
)
plt.title("E3 Global Symptom Importance\nMean Absolute Gradient × Input")
plt.xlabel("Mean absolute attribution")
plt.ylabel("Input feature")
plt.tight_layout()
plt.savefig(
    os.path.join(XAI_OUTPUT_DIR, "global_feature_importance.png"),
    dpi=300,
    bbox_inches="tight"
)
plt.show()

# ============================================================
# 5. Disease-Specific Feature Importance
# ============================================================

predicted_labels_xai = patient_explanations_df[
    "predicted_disease"
].to_numpy()

disease_feature_rows = []

for disease_name in class_names_xai:

    disease_mask = predicted_labels_xai == disease_name

    if not disease_mask.any():
        continue

    disease_importance = absolute_feature_attributions[disease_mask].mean(axis=0)

    for feature_name, importance in zip(
        feature_names_xai,
        disease_importance
    ):
        disease_feature_rows.append({
            "predicted_disease": disease_name,
            "feature": feature_name,
            "mean_absolute_gradient_x_input": float(importance),
            "n_patients": int(disease_mask.sum())
        })

disease_feature_importance_df = pd.DataFrame(disease_feature_rows)

disease_feature_importance_df.to_csv(
    os.path.join(XAI_OUTPUT_DIR, "disease_specific_feature_importance.csv"),
    index=False
)

# Matrix for a compact disease-by-feature heatmap.
if not disease_feature_importance_df.empty:

    disease_feature_matrix = disease_feature_importance_df.pivot(
        index="predicted_disease",
        columns="feature",
        values="mean_absolute_gradient_x_input"
    ).reindex(columns=feature_names_xai)

    plt.figure(
        figsize=(max(10, 0.8 * len(feature_names_xai)),
                 max(5, 0.6 * len(disease_feature_matrix)))
    )
    plt.imshow(disease_feature_matrix.values, aspect="auto")
    plt.colorbar(label="Mean absolute attribution")
    plt.xticks(
        np.arange(len(feature_names_xai)),
        feature_names_xai,
        rotation=45,
        ha="right"
    )
    plt.yticks(
        np.arange(len(disease_feature_matrix.index)),
        disease_feature_matrix.index
    )
    plt.title("E3 Disease-Specific Feature Attribution")
    plt.xlabel("Input feature")
    plt.ylabel("Predicted disease")
    plt.tight_layout()
    plt.savefig(
        os.path.join(XAI_OUTPUT_DIR, "disease_specific_feature_heatmap.png"),
        dpi=300,
        bbox_inches="tight"
    )
    plt.show()

# ============================================================
# 6. Temporal Attention Summary
# ============================================================

attention_by_visit_df = (
    visit_attention_df
    .groupby("visit_index", as_index=False)
    .agg(
        mean_attention=("attention_weight", "mean"),
        std_attention=("attention_weight", "std"),
        n_visits=("attention_weight", "size")
    )
)

attention_by_visit_df.to_csv(
    os.path.join(XAI_OUTPUT_DIR, "attention_by_visit_position.csv"),
    index=False
)

print("\nTemporal attention by visit position")
print("-" * 60)
display(attention_by_visit_df)

plt.figure(figsize=(8, 5))
plt.plot(
    attention_by_visit_df["visit_index"],
    attention_by_visit_df["mean_attention"],
    marker="o"
)
plt.fill_between(
    attention_by_visit_df["visit_index"],
    attention_by_visit_df["mean_attention"] - attention_by_visit_df["std_attention"].fillna(0),
    attention_by_visit_df["mean_attention"] + attention_by_visit_df["std_attention"].fillna(0),
    alpha=0.2
)
plt.title("E3 Mean Temporal Attention by Visit Position")
plt.xlabel("Visit position")
plt.ylabel("Mean attention weight")
plt.xticks(attention_by_visit_df["visit_index"])
plt.tight_layout()
plt.savefig(
    os.path.join(XAI_OUTPUT_DIR, "temporal_attention_by_visit.png"),
    dpi=300,
    bbox_inches="tight"
)
plt.show()

print("\n" + "=" * 70)
print("EXPLAINABILITY ANALYSIS COMPLETE")
print("=" * 70)
print("Saved files:")
print("- patient_feature_attributions.csv")
print("- patient_visit_attention.csv")
print("- global_feature_importance.csv")
print("- disease_specific_feature_importance.csv")
print("- attention_by_visit_position.csv")
print("- global_feature_importance.png")
print("- disease_specific_feature_heatmap.png")
print("- temporal_attention_by_visit.png")


In [ ]:
# ============================================================
# STraT-Net
# Experiment E3
# Final Experiment Record
# ============================================================

import os
import json

E3_SUMMARY = {

    "experiment_id":
        "E3",

    "architecture":
        "STraT-Net V2 Multi-Task",

    "attention":
        "Gated Temporal Attention",

    "input_features":
        12,

    "input_description":
        (
            "11 longitudinal symptom features "
            "+ overall severity"
        ),

    "auxiliary_task":
        "Patient-level mean symptom profile reconstruction",

    "auxiliary_loss":
        "MSELoss",

    "symptom_loss_weight":
        0.20,

    "training_patients":
        2100,

    "validation_patients":
        450,

    "test_patients":
        450,

    "best_epoch":
        25,

    "best_validation_macro_f1":
        0.5170759487063142,

    "test_accuracy":
        0.517778,

    "test_macro_precision":
        0.5188,

    "test_macro_recall":
        0.5178,

    "test_macro_f1":
        0.516582,

    "test_weighted_f1":
        0.516582,

    "test_macro_roc_auc":
        0.816049,

    "test_macro_pr_auc":
        0.547010,

    "test_symptom_mse":
        0.019792,

    "test_symptom_mae":
        0.113162,

    "E2_test_macro_f1":
        0.505231,

    "test_macro_f1_change_vs_E2":
        0.011351,

    "notes":
        (
            "Severity-augmented multi-task model. "
            "Auxiliary patient-level mean symptom-profile "
            "reconstruction used with loss weight 0.20."
        )

}

E3_SUMMARY_PATH = os.path.join(

    E3_OUTPUT_DIR,

    "experiment_E3_summary.json"

)

with open(
    E3_SUMMARY_PATH,
    "w"
) as f:

    json.dump(
        E3_SUMMARY,
        f,
        indent=4
    )

print("=" * 70)
print("EXPERIMENT E3 SAVED")
print("=" * 70)

for key, value in E3_SUMMARY.items():

    print(
        f"{key:<38}: {value}"
    )

print("\nSaved to:")
print(E3_SUMMARY_PATH)

print("=" * 70)

In [ ]:
# ============================================================
# STraT-Net
# Experiment E3
# Multi-Seed Robustness Evaluation
# ============================================================

import os
import random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn

from tqdm.auto import tqdm

from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score
)

from sklearn.preprocessing import label_binarize

print("=" * 70)
print("STraT-Net : E3 Multi-Seed Robustness Experiment")
print("=" * 70)

# ============================================================
# 1. Seeds
# ============================================================

SEEDS = [
    42,
    123,
    2026,
    3407,
    7777
]

print(
    "\nSeeds:",
    SEEDS
)

# ============================================================
# 2. Fixed Experimental Configuration
#
# Must remain identical across seeds
# ============================================================

MULTISEED_EPOCHS = 30

MULTISEED_LR = 1e-4

MULTISEED_WEIGHT_DECAY = 1e-4

MULTISEED_GRAD_CLIP = 1.0

MULTISEED_LABEL_SMOOTHING = 0.05

MULTISEED_PATIENCE = 8

MULTISEED_SYMPTOM_WEIGHT = 0.20

DEVICE = cfg.DEVICE

# ============================================================
# 3. Output Directories
# ============================================================

MULTISEED_CHECKPOINT_DIR = (
    "checkpoints_stratnet_e3_multiseed"
)

MULTISEED_OUTPUT_DIR = (
    "outputs_stratnet_e3_multiseed"
)

os.makedirs(
    MULTISEED_CHECKPOINT_DIR,
    exist_ok=True
)

os.makedirs(
    MULTISEED_OUTPUT_DIR,
    exist_ok=True
)

# ============================================================
# 4. Reproducibility Function
# ============================================================

def set_seed_multiseed(seed):

    random.seed(seed)

    np.random.seed(seed)

    torch.manual_seed(seed)

    torch.cuda.manual_seed_all(seed)

    os.environ[
        "PYTHONHASHSEED"
    ] = str(seed)

    torch.backends.cudnn.deterministic = True

    torch.backends.cudnn.benchmark = False

# ============================================================
# 5. Metric Function
# ============================================================

def compute_seed_metrics(
    y_true,
    y_pred
):

    return {

        "accuracy":
            accuracy_score(
                y_true,
                y_pred
            ),

        "macro_precision":
            precision_score(
                y_true,
                y_pred,
                average="macro",
                zero_division=0
            ),

        "macro_recall":
            recall_score(
                y_true,
                y_pred,
                average="macro",
                zero_division=0
            ),

        "macro_f1":
            f1_score(
                y_true,
                y_pred,
                average="macro",
                zero_division=0
            )

    }

# ============================================================
# 6. Train One Epoch
# ============================================================

def train_seed_epoch(
    model,
    dataloader,
    optimizer,
    disease_loss_fn,
    symptom_loss_fn,
    scaler
):

    model.train()

    total_loss_sum = 0.0

    all_labels = []

    all_predictions = []

    for batch in dataloader:

        trajectory = (
            batch[
                "trajectory"
            ]
            .float()
            .to(
                DEVICE,
                non_blocking=True
            )
        )

        years = (
            batch[
                "years"
            ]
            .long()
            .to(
                DEVICE,
                non_blocking=True
            )
        )

        mask = (
            batch[
                "mask"
            ]
            .bool()
            .to(
                DEVICE,
                non_blocking=True
            )
        )

        labels = (
            batch[
                "label"
            ]
            .long()
            .to(
                DEVICE,
                non_blocking=True
            )
        )

        symptom_targets = (
            batch[
                "symptom_target"
            ]
            .float()
            .to(
                DEVICE,
                non_blocking=True
            )
        )

        optimizer.zero_grad(
            set_to_none=True
        )

        # ----------------------------------------------------
        # AMP
        # ----------------------------------------------------

        if scaler is not None:

            with torch.cuda.amp.autocast():

                outputs = model(
                    trajectory,
                    years,
                    mask
                )

                disease_loss = (
                    disease_loss_fn(
                        outputs[
                            "disease_logits"
                        ],
                        labels
                    )
                )

                symptom_loss = (
                    symptom_loss_fn(
                        outputs[
                            "symptom_logits"
                        ],
                        symptom_targets
                    )
                )

                total_loss = (

                    disease_loss

                    +

                    MULTISEED_SYMPTOM_WEIGHT
                    *
                    symptom_loss

                )

            scaler.scale(
                total_loss
            ).backward()

            scaler.unscale_(
                optimizer
            )

            torch.nn.utils.clip_grad_norm_(
                model.parameters(),
                MULTISEED_GRAD_CLIP
            )

            scaler.step(
                optimizer
            )

            scaler.update()

        else:

            outputs = model(
                trajectory,
                years,
                mask
            )

            disease_loss = (
                disease_loss_fn(
                    outputs[
                        "disease_logits"
                    ],
                    labels
                )
            )

            symptom_loss = (
                symptom_loss_fn(
                    outputs[
                        "symptom_logits"
                    ],
                    symptom_targets
                )
            )

            total_loss = (

                disease_loss

                +

                MULTISEED_SYMPTOM_WEIGHT
                *
                symptom_loss

            )

            total_loss.backward()

            torch.nn.utils.clip_grad_norm_(
                model.parameters(),
                MULTISEED_GRAD_CLIP
            )

            optimizer.step()

        total_loss_sum += (
            total_loss.item()
        )

        predictions = torch.argmax(
            outputs[
                "disease_logits"
            ],
            dim=1
        )

        all_labels.extend(
            labels
            .detach()
            .cpu()
            .numpy()
            .tolist()
        )

        all_predictions.extend(
            predictions
            .detach()
            .cpu()
            .numpy()
            .tolist()
        )

    metrics = compute_seed_metrics(
        all_labels,
        all_predictions
    )

    return {

        "loss":
            total_loss_sum
            /
            len(
                dataloader
            ),

        "accuracy":
            metrics[
                "accuracy"
            ],

        "macro_f1":
            metrics[
                "macro_f1"
            ]

    }

# ============================================================
# 7. Validate One Epoch
# ============================================================

def validate_seed_epoch(
    model,
    dataloader,
    disease_loss_fn,
    symptom_loss_fn
):

    model.eval()

    total_loss_sum = 0.0

    all_labels = []

    all_predictions = []

    with torch.no_grad():

        for batch in dataloader:

            trajectory = (
                batch[
                    "trajectory"
                ]
                .float()
                .to(DEVICE)
            )

            years = (
                batch[
                    "years"
                ]
                .long()
                .to(DEVICE)
            )

            mask = (
                batch[
                    "mask"
                ]
                .bool()
                .to(DEVICE)
            )

            labels = (
                batch[
                    "label"
                ]
                .long()
                .to(DEVICE)
            )

            symptom_targets = (
                batch[
                    "symptom_target"
                ]
                .float()
                .to(DEVICE)
            )

            with torch.cuda.amp.autocast(
                enabled=torch.cuda.is_available()
            ):

                outputs = model(
                    trajectory,
                    years,
                    mask
                )

                disease_loss = (
                    disease_loss_fn(
                        outputs[
                            "disease_logits"
                        ],
                        labels
                    )
                )

                symptom_loss = (
                    symptom_loss_fn(
                        outputs[
                            "symptom_logits"
                        ],
                        symptom_targets
                    )
                )

                total_loss = (

                    disease_loss

                    +

                    MULTISEED_SYMPTOM_WEIGHT
                    *
                    symptom_loss

                )

            total_loss_sum += (
                total_loss.item()
            )

            predictions = torch.argmax(
                outputs[
                    "disease_logits"
                ],
                dim=1
            )

            all_labels.extend(
                labels
                .cpu()
                .numpy()
                .tolist()
            )

            all_predictions.extend(
                predictions
                .cpu()
                .numpy()
                .tolist()
            )

    metrics = compute_seed_metrics(
        all_labels,
        all_predictions
    )

    return {

        "loss":
            total_loss_sum
            /
            len(
                dataloader
            ),

        "accuracy":
            metrics[
                "accuracy"
            ],

        "macro_f1":
            metrics[
                "macro_f1"
            ]

    }

# ============================================================
# 8. Test Best Model
# ============================================================

def evaluate_seed_test(
    model,
    dataloader
):

    model.eval()

    all_labels = []

    all_predictions = []

    all_probabilities = []

    with torch.no_grad():

        for batch in dataloader:

            trajectory = (
                batch[
                    "trajectory"
                ]
                .float()
                .to(DEVICE)
            )

            years = (
                batch[
                    "years"
                ]
                .long()
                .to(DEVICE)
            )

            mask = (
                batch[
                    "mask"
                ]
                .bool()
                .to(DEVICE)
            )

            labels = (
                batch[
                    "label"
                ]
                .long()
                .to(DEVICE)
            )

            outputs = model(
                trajectory,
                years,
                mask
            )

            probabilities = torch.softmax(
                outputs[
                    "disease_logits"
                ],
                dim=1
            )

            predictions = torch.argmax(
                probabilities,
                dim=1
            )

            all_labels.extend(
                labels
                .cpu()
                .numpy()
                .tolist()
            )

            all_predictions.extend(
                predictions
                .cpu()
                .numpy()
                .tolist()
            )

            all_probabilities.extend(
                probabilities
                .cpu()
                .numpy()
            )

    y_true = np.asarray(
        all_labels
    )

    y_pred = np.asarray(
        all_predictions
    )

    y_prob = np.asarray(
        all_probabilities
    )

    metrics = compute_seed_metrics(
        y_true,
        y_pred
    )

    y_true_binary = label_binarize(

        y_true,

        classes=np.arange(
            NUM_DISEASES
        )

    )

    roc_auc = roc_auc_score(

        y_true_binary,

        y_prob,

        average="macro",

        multi_class="ovr"

    )

    pr_auc = average_precision_score(

        y_true_binary,

        y_prob,

        average="macro"

    )

    return {

        "test_accuracy":
            metrics[
                "accuracy"
            ],

        "test_macro_precision":
            metrics[
                "macro_precision"
            ],

        "test_macro_recall":
            metrics[
                "macro_recall"
            ],

        "test_macro_f1":
            metrics[
                "macro_f1"
            ],

        "test_macro_roc_auc":
            roc_auc,

        "test_macro_pr_auc":
            pr_auc

    }

# ============================================================
# 9. Run One Seed
# ============================================================

def run_single_seed_e3(seed):

    print("\n" + "=" * 70)

    print(
        f"E3 MULTI-SEED RUN — SEED {seed}"
    )

    print("=" * 70)

    # --------------------------------------------------------
    # Seed
    # --------------------------------------------------------

    set_seed_multiseed(
        seed
    )

    # --------------------------------------------------------
    # Fresh model
    # --------------------------------------------------------

    model = STraTNetE2()

    initialize_weights_e2(
        model
    )

    model = model.to(
        DEVICE
    )

    # --------------------------------------------------------
    # Fresh losses
    # --------------------------------------------------------

    disease_loss_fn = nn.CrossEntropyLoss(

        weight=class_weights.to(
            DEVICE
        ),

        label_smoothing=(
            MULTISEED_LABEL_SMOOTHING
        )

    )

    symptom_loss_fn = (
        nn.MSELoss()
    )

    # --------------------------------------------------------
    # Fresh optimizer
    # --------------------------------------------------------

    optimizer = AdamW(

        model.parameters(),

        lr=MULTISEED_LR,

        weight_decay=(
            MULTISEED_WEIGHT_DECAY
        )

    )

    # --------------------------------------------------------
    # Fresh scheduler
    # --------------------------------------------------------

    scheduler = CosineAnnealingLR(

        optimizer,

        T_max=MULTISEED_EPOCHS,

        eta_min=1e-6

    )

    # --------------------------------------------------------
    # Fresh scaler
    # --------------------------------------------------------

    scaler = (

        torch.cuda.amp.GradScaler()

        if torch.cuda.is_available()

        else None

    )

    # --------------------------------------------------------
    # Seed-specific checkpoint
    # --------------------------------------------------------

    checkpoint_path = os.path.join(

        MULTISEED_CHECKPOINT_DIR,

        f"E3_seed_{seed}_best.pth"

    )

    best_val_f1 = -1.0

    best_epoch = -1

    early_stop_counter = 0

    seed_history = []

    # ========================================================
    # Training Loop
    # ========================================================

    for epoch in range(
        1,
        MULTISEED_EPOCHS + 1
    ):

        train_result = train_seed_epoch(

            model=model,

            dataloader=(
                train_patient_loader_e3
            ),

            optimizer=optimizer,

            disease_loss_fn=(
                disease_loss_fn
            ),

            symptom_loss_fn=(
                symptom_loss_fn
            ),

            scaler=scaler

        )

        val_result = validate_seed_epoch(

            model=model,

            dataloader=(
                val_patient_loader_e3
            ),

            disease_loss_fn=(
                disease_loss_fn
            ),

            symptom_loss_fn=(
                symptom_loss_fn
            )

        )

        seed_history.append({

            "seed":
                seed,

            "epoch":
                epoch,

            "train_loss":
                train_result[
                    "loss"
                ],

            "val_loss":
                val_result[
                    "loss"
                ],

            "train_accuracy":
                train_result[
                    "accuracy"
                ],

            "val_accuracy":
                val_result[
                    "accuracy"
                ],

            "train_macro_f1":
                train_result[
                    "macro_f1"
                ],

            "val_macro_f1":
                val_result[
                    "macro_f1"
                ]

        })

        current_f1 = (
            val_result[
                "macro_f1"
            ]
        )

        print(

            f"Seed {seed} | "
            f"Epoch {epoch:02d} | "
            f"Train F1 "
            f"{train_result['macro_f1']:.4f} | "
            f"Val F1 "
            f"{current_f1:.4f}"

        )

        # ----------------------------------------------------
        # Best validation checkpoint
        # ----------------------------------------------------

        if current_f1 > best_val_f1:

            best_val_f1 = (
                current_f1
            )

            best_epoch = (
                epoch
            )

            early_stop_counter = 0

            torch.save(

                {

                    "seed":
                        seed,

                    "epoch":
                        epoch,

                    "val_macro_f1":
                        best_val_f1,

                    "model_state_dict":
                        model.state_dict()

                },

                checkpoint_path

            )

        else:

            early_stop_counter += 1

        scheduler.step()

        # ----------------------------------------------------
        # Early stopping
        # ----------------------------------------------------

        if (
            early_stop_counter
            >=
            MULTISEED_PATIENCE
        ):

            print(

                f"Seed {seed}: "
                f"early stopping at "
                f"epoch {epoch}"

            )

            break

    # ========================================================
    # Save Seed History
    # ========================================================

    seed_history_df = pd.DataFrame(
        seed_history
    )

    seed_history_df.to_csv(

        os.path.join(

            MULTISEED_OUTPUT_DIR,

            f"E3_seed_{seed}_history.csv"

        ),

        index=False

    )

    # ========================================================
    # Load Best Checkpoint
    # ========================================================

    checkpoint = torch.load(

        checkpoint_path,

        map_location=DEVICE

    )

    model.load_state_dict(

        checkpoint[
            "model_state_dict"
        ]

    )

    # ========================================================
    # Test Evaluation
    # ========================================================

    test_result = evaluate_seed_test(

        model,

        test_patient_loader_e3

    )

    result = {

        "seed":
            seed,

        "best_epoch":
            best_epoch,

        "best_val_macro_f1":
            best_val_f1,

        **test_result

    }

    print("\nSeed Result")
    print("-" * 60)

    for key, value in result.items():

        if isinstance(
            value,
            float
        ):

            print(
                f"{key:<25}: "
                f"{value:.6f}"
            )

        else:

            print(
                f"{key:<25}: "
                f"{value}"
            )

    # --------------------------------------------------------
    # Cleanup GPU memory
    # --------------------------------------------------------

    del model

    del optimizer

    del scheduler

    del scaler

    torch.cuda.empty_cache()

    return result

# ============================================================
# 10. Run All Seeds
# ============================================================

all_seed_results = []

for seed in SEEDS:

    result = run_single_seed_e3(
        seed
    )

    all_seed_results.append(
        result
    )

# ============================================================
# 11. Results Table
# ============================================================

seed_results_df = pd.DataFrame(
    all_seed_results
)

print("\n" + "=" * 70)

print("E3 MULTI-SEED RESULTS")

print("=" * 70)

display(
    seed_results_df
)

# ============================================================
# 12. Mean + Standard Deviation
# ============================================================

metrics_to_summarize = [

    "best_val_macro_f1",

    "test_accuracy",

    "test_macro_precision",

    "test_macro_recall",

    "test_macro_f1",

    "test_macro_roc_auc",

    "test_macro_pr_auc"

]

summary_rows = []

for metric in metrics_to_summarize:

    values = seed_results_df[
        metric
    ]

    summary_rows.append({

        "Metric":
            metric,

        "Mean":
            values.mean(),

        "Std":
            values.std(
                ddof=1
            ),

        "Min":
            values.min(),

        "Max":
            values.max()

    })

summary_df = pd.DataFrame(
    summary_rows
)

print("\n" + "=" * 70)

print("E3 MULTI-SEED SUMMARY")

print("=" * 70)

display(
    summary_df
)

# ============================================================
# 13. Publication-Style Summary
# ============================================================

print("\nPublication-Style Results")
print("-" * 70)

for _, row in summary_df.iterrows():

    print(

        f"{row['Metric']:<25}: "
        f"{row['Mean']:.4f} "
        f"± "
        f"{row['Std']:.4f}"

    )

# ============================================================
# 14. Save Results
# ============================================================

seed_results_df.to_csv(

    os.path.join(

        MULTISEED_OUTPUT_DIR,

        "E3_multiseed_results.csv"

    ),

    index=False

)

summary_df.to_csv(

    os.path.join(

        MULTISEED_OUTPUT_DIR,

        "E3_multiseed_summary.csv"

    ),

    index=False

)

print("\n" + "=" * 70)

print("E3 MULTI-SEED EXPERIMENT COMPLETE")

print("=" * 70)

print(
    "Results saved to:",
    MULTISEED_OUTPUT_DIR
)

print("=" * 70)

In [ ]:
# ============================================================
# STraT-Net
# Experiment E2
# Multi-Seed Robustness Evaluation
# ============================================================

import os
import random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn

from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score
)

from sklearn.preprocessing import label_binarize

print("=" * 70)
print("STraT-Net : E2 Multi-Seed Robustness Experiment")
print("=" * 70)

# ============================================================
# 1. Seeds
# ============================================================

SEEDS_E2 = [
    42,
    123,
    2026,
    3407,
    7777
]

# ============================================================
# 2. Fixed Configuration
# Same as E3 except NO auxiliary symptom loss
# ============================================================

E2_MS_EPOCHS = 30
E2_MS_LR = 1e-4
E2_MS_WEIGHT_DECAY = 1e-4
E2_MS_GRAD_CLIP = 1.0
E2_MS_LABEL_SMOOTHING = 0.05
E2_MS_PATIENCE = 8

DEVICE = cfg.DEVICE

# ============================================================
# 3. Output Directories
# ============================================================

E2_MS_CHECKPOINT_DIR = (
    "checkpoints_stratnet_e2_multiseed"
)

E2_MS_OUTPUT_DIR = (
    "outputs_stratnet_e2_multiseed"
)

os.makedirs(
    E2_MS_CHECKPOINT_DIR,
    exist_ok=True
)

os.makedirs(
    E2_MS_OUTPUT_DIR,
    exist_ok=True
)

# ============================================================
# 4. Seed Function
# ============================================================

def set_seed_e2(seed):

    random.seed(seed)

    np.random.seed(seed)

    torch.manual_seed(seed)

    torch.cuda.manual_seed_all(seed)

    os.environ[
        "PYTHONHASHSEED"
    ] = str(seed)

    torch.backends.cudnn.deterministic = True

    torch.backends.cudnn.benchmark = False

# ============================================================
# 5. Metric Helper
# ============================================================

def compute_metrics_e2(
    y_true,
    y_pred
):

    return {

        "accuracy":
            accuracy_score(
                y_true,
                y_pred
            ),

        "macro_precision":
            precision_score(
                y_true,
                y_pred,
                average="macro",
                zero_division=0
            ),

        "macro_recall":
            recall_score(
                y_true,
                y_pred,
                average="macro",
                zero_division=0
            ),

        "macro_f1":
            f1_score(
                y_true,
                y_pred,
                average="macro",
                zero_division=0
            )

    }

# ============================================================
# 6. Train One Epoch
# ============================================================

def train_one_epoch_e2_ms(
    model,
    dataloader,
    optimizer,
    criterion,
    scaler
):

    model.train()

    running_loss = 0.0

    all_labels = []
    all_predictions = []

    for batch in dataloader:

        trajectory = (
            batch["trajectory"]
            .float()
            .to(DEVICE)
        )

        years = (
            batch["years"]
            .long()
            .to(DEVICE)
        )

        mask = (
            batch["mask"]
            .bool()
            .to(DEVICE)
        )

        labels = (
            batch["label"]
            .long()
            .to(DEVICE)
        )

        optimizer.zero_grad(
            set_to_none=True
        )

        if scaler is not None:

            with torch.cuda.amp.autocast():

                outputs = model(
                    trajectory,
                    years,
                    mask
                )

                loss = criterion(
                    outputs[
                        "disease_logits"
                    ],
                    labels
                )

            scaler.scale(
                loss
            ).backward()

            scaler.unscale_(
                optimizer
            )

            torch.nn.utils.clip_grad_norm_(
                model.parameters(),
                E2_MS_GRAD_CLIP
            )

            scaler.step(
                optimizer
            )

            scaler.update()

        else:

            outputs = model(
                trajectory,
                years,
                mask
            )

            loss = criterion(
                outputs[
                    "disease_logits"
                ],
                labels
            )

            loss.backward()

            torch.nn.utils.clip_grad_norm_(
                model.parameters(),
                E2_MS_GRAD_CLIP
            )

            optimizer.step()

        running_loss += (
            loss.item()
        )

        predictions = torch.argmax(
            outputs[
                "disease_logits"
            ],
            dim=1
        )

        all_labels.extend(
            labels
            .detach()
            .cpu()
            .numpy()
            .tolist()
        )

        all_predictions.extend(
            predictions
            .detach()
            .cpu()
            .numpy()
            .tolist()
        )

    metrics = compute_metrics_e2(
        all_labels,
        all_predictions
    )

    return {

        "loss":
            running_loss
            /
            len(dataloader),

        "accuracy":
            metrics["accuracy"],

        "macro_f1":
            metrics["macro_f1"]

    }

# ============================================================
# 7. Validate One Epoch
# ============================================================

def validate_one_epoch_e2_ms(
    model,
    dataloader,
    criterion
):

    model.eval()

    running_loss = 0.0

    all_labels = []
    all_predictions = []

    with torch.no_grad():

        for batch in dataloader:

            trajectory = (
                batch["trajectory"]
                .float()
                .to(DEVICE)
            )

            years = (
                batch["years"]
                .long()
                .to(DEVICE)
            )

            mask = (
                batch["mask"]
                .bool()
                .to(DEVICE)
            )

            labels = (
                batch["label"]
                .long()
                .to(DEVICE)
            )

            with torch.cuda.amp.autocast(
                enabled=torch.cuda.is_available()
            ):

                outputs = model(
                    trajectory,
                    years,
                    mask
                )

                loss = criterion(
                    outputs[
                        "disease_logits"
                    ],
                    labels
                )

            running_loss += (
                loss.item()
            )

            predictions = torch.argmax(
                outputs[
                    "disease_logits"
                ],
                dim=1
            )

            all_labels.extend(
                labels
                .cpu()
                .numpy()
                .tolist()
            )

            all_predictions.extend(
                predictions
                .cpu()
                .numpy()
                .tolist()
            )

    metrics = compute_metrics_e2(
        all_labels,
        all_predictions
    )

    return {

        "loss":
            running_loss
            /
            len(dataloader),

        "accuracy":
            metrics["accuracy"],

        "macro_f1":
            metrics["macro_f1"]

    }

# ============================================================
# 8. Test Evaluation
# ============================================================

def evaluate_test_e2_ms(
    model,
    dataloader
):

    model.eval()

    all_labels = []
    all_predictions = []
    all_probabilities = []

    with torch.no_grad():

        for batch in dataloader:

            trajectory = (
                batch["trajectory"]
                .float()
                .to(DEVICE)
            )

            years = (
                batch["years"]
                .long()
                .to(DEVICE)
            )

            mask = (
                batch["mask"]
                .bool()
                .to(DEVICE)
            )

            labels = (
                batch["label"]
                .long()
                .to(DEVICE)
            )

            outputs = model(
                trajectory,
                years,
                mask
            )

            probabilities = torch.softmax(
                outputs[
                    "disease_logits"
                ],
                dim=1
            )

            predictions = torch.argmax(
                probabilities,
                dim=1
            )

            all_labels.extend(
                labels
                .cpu()
                .numpy()
                .tolist()
            )

            all_predictions.extend(
                predictions
                .cpu()
                .numpy()
                .tolist()
            )

            all_probabilities.extend(
                probabilities
                .cpu()
                .numpy()
            )

    y_true = np.asarray(
        all_labels
    )

    y_pred = np.asarray(
        all_predictions
    )

    y_prob = np.asarray(
        all_probabilities
    )

    metrics = compute_metrics_e2(
        y_true,
        y_pred
    )

    y_binary = label_binarize(
        y_true,
        classes=np.arange(
            NUM_DISEASES
        )
    )

    macro_roc_auc = roc_auc_score(
        y_binary,
        y_prob,
        average="macro",
        multi_class="ovr"
    )

    macro_pr_auc = average_precision_score(
        y_binary,
        y_prob,
        average="macro"
    )

    return {

        "test_accuracy":
            metrics["accuracy"],

        "test_macro_precision":
            metrics["macro_precision"],

        "test_macro_recall":
            metrics["macro_recall"],

        "test_macro_f1":
            metrics["macro_f1"],

        "test_macro_roc_auc":
            macro_roc_auc,

        "test_macro_pr_auc":
            macro_pr_auc

    }

# ============================================================
# 9. Run One Seed
# ============================================================

def run_single_seed_e2(seed):

    print("\n" + "=" * 70)
    print(
        f"E2 MULTI-SEED RUN — SEED {seed}"
    )
    print("=" * 70)

    set_seed_e2(
        seed
    )

    # --------------------------------------------
    # Fresh E2 model
    # Same 12-D severity architecture
    # --------------------------------------------

    model = STraTNetE2()

    initialize_weights_e2(
        model
    )

    model = model.to(
        DEVICE
    )

    # --------------------------------------------
    # Fresh loss
    # --------------------------------------------

    criterion = nn.CrossEntropyLoss(

        weight=class_weights.to(
            DEVICE
        ),

        label_smoothing=(
            E2_MS_LABEL_SMOOTHING
        )

    )

    # --------------------------------------------
    # Fresh optimizer
    # --------------------------------------------

    optimizer = AdamW(

        model.parameters(),

        lr=E2_MS_LR,

        weight_decay=(
            E2_MS_WEIGHT_DECAY
        )

    )

    # --------------------------------------------
    # Fresh scheduler
    # --------------------------------------------

    scheduler = CosineAnnealingLR(

        optimizer,

        T_max=E2_MS_EPOCHS,

        eta_min=1e-6

    )

    # --------------------------------------------
    # Fresh scaler
    # --------------------------------------------

    scaler = (

        torch.cuda.amp.GradScaler()

        if torch.cuda.is_available()

        else None

    )

    checkpoint_path = os.path.join(

        E2_MS_CHECKPOINT_DIR,

        f"E2_seed_{seed}_best.pth"

    )

    best_val_f1 = -1.0
    best_epoch = -1
    early_stop_counter = 0

    seed_history = []

    # ========================================================
    # Training
    # ========================================================

    for epoch in range(
        1,
        E2_MS_EPOCHS + 1
    ):

        train_result = train_one_epoch_e2_ms(

            model,
            train_patient_loader_e2,
            optimizer,
            criterion,
            scaler

        )

        val_result = validate_one_epoch_e2_ms(

            model,
            val_patient_loader_e2,
            criterion

        )

        seed_history.append({

            "seed":
                seed,

            "epoch":
                epoch,

            "train_loss":
                train_result[
                    "loss"
                ],

            "val_loss":
                val_result[
                    "loss"
                ],

            "train_accuracy":
                train_result[
                    "accuracy"
                ],

            "val_accuracy":
                val_result[
                    "accuracy"
                ],

            "train_macro_f1":
                train_result[
                    "macro_f1"
                ],

            "val_macro_f1":
                val_result[
                    "macro_f1"
                ]

        })

        current_val_f1 = (
            val_result[
                "macro_f1"
            ]
        )

        print(

            f"Seed {seed} | "
            f"Epoch {epoch:02d} | "
            f"Train F1 "
            f"{train_result['macro_f1']:.4f} | "
            f"Val F1 "
            f"{current_val_f1:.4f}"

        )

        if current_val_f1 > best_val_f1:

            best_val_f1 = (
                current_val_f1
            )

            best_epoch = (
                epoch
            )

            early_stop_counter = 0

            torch.save(

                {

                    "seed":
                        seed,

                    "epoch":
                        epoch,

                    "val_macro_f1":
                        best_val_f1,

                    "model_state_dict":
                        model.state_dict()

                },

                checkpoint_path

            )

        else:

            early_stop_counter += 1

        scheduler.step()

        if (
            early_stop_counter
            >=
            E2_MS_PATIENCE
        ):

            print(

                f"Seed {seed}: "
                f"early stopping at "
                f"epoch {epoch}"

            )

            break

    # ========================================================
    # Save History
    # ========================================================

    pd.DataFrame(
        seed_history
    ).to_csv(

        os.path.join(

            E2_MS_OUTPUT_DIR,

            f"E2_seed_{seed}_history.csv"

        ),

        index=False

    )

    # ========================================================
    # Load Best Model
    # ========================================================

    checkpoint = torch.load(
        checkpoint_path,
        map_location=DEVICE
    )

    model.load_state_dict(
        checkpoint[
            "model_state_dict"
        ]
    )

    # ========================================================
    # Test Evaluation
    # ========================================================

    test_result = evaluate_test_e2_ms(

        model,

        test_patient_loader_e2

    )

    result = {

        "seed":
            seed,

        "best_epoch":
            best_epoch,

        "best_val_macro_f1":
            best_val_f1,

        **test_result

    }

    print("\nSeed Result")
    print("-" * 60)

    for key, value in result.items():

        if isinstance(
            value,
            float
        ):

            print(
                f"{key:<25}: "
                f"{value:.6f}"
            )

        else:

            print(
                f"{key:<25}: "
                f"{value}"
            )

    del model
    del optimizer
    del scheduler
    del scaler

    torch.cuda.empty_cache()

    return result

# ============================================================
# 10. Run All Seeds
# ============================================================

all_e2_seed_results = []

for seed in SEEDS_E2:

    result = run_single_seed_e2(
        seed
    )

    all_e2_seed_results.append(
        result
    )

# ============================================================
# 11. Results
# ============================================================

e2_seed_results_df = pd.DataFrame(
    all_e2_seed_results
)

print("\n" + "=" * 70)
print("E2 MULTI-SEED RESULTS")
print("=" * 70)

display(
    e2_seed_results_df
)

# ============================================================
# 12. Summary Statistics
# ============================================================

metrics_to_summarize = [

    "best_val_macro_f1",

    "test_accuracy",

    "test_macro_precision",

    "test_macro_recall",

    "test_macro_f1",

    "test_macro_roc_auc",

    "test_macro_pr_auc"

]

summary_rows = []

for metric in metrics_to_summarize:

    values = e2_seed_results_df[
        metric
    ]

    summary_rows.append({

        "Metric":
            metric,

        "Mean":
            values.mean(),

        "Std":
            values.std(
                ddof=1
            ),

        "Min":
            values.min(),

        "Max":
            values.max()

    })

e2_summary_df = pd.DataFrame(
    summary_rows
)

print("\n" + "=" * 70)
print("E2 MULTI-SEED SUMMARY")
print("=" * 70)

display(
    e2_summary_df
)

# ============================================================
# 13. Publication Style
# ============================================================

print("\nPublication-Style Results")
print("-" * 70)

for _, row in e2_summary_df.iterrows():

    print(

        f"{row['Metric']:<25}: "
        f"{row['Mean']:.4f} "
        f"± "
        f"{row['Std']:.4f}"

    )

# ============================================================
# 14. Save
# ============================================================

e2_seed_results_df.to_csv(

    os.path.join(

        E2_MS_OUTPUT_DIR,

        "E2_multiseed_results.csv"

    ),

    index=False

)

e2_summary_df.to_csv(

    os.path.join(

        E2_MS_OUTPUT_DIR,

        "E2_multiseed_summary.csv"

    ),

    index=False

)

print("\n" + "=" * 70)
print("E2 MULTI-SEED EXPERIMENT COMPLETE")
print("=" * 70)

In [ ]:
# ============================================================
# STraT-Net
# Final Research Analysis Module
# Tables + Figures + Ablation Summary
# ============================================================

import os
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.metrics import (
    confusion_matrix,
    classification_report,
    roc_curve,
    auc,
    precision_recall_curve,
    average_precision_score
)

print("=" * 70)
print("STraT-Net : FINAL RESEARCH ANALYSIS")
print("=" * 70)

# ============================================================
# 1. Output Directory
# ============================================================

FINAL_ANALYSIS_DIR = "final_analysis"

os.makedirs(
    FINAL_ANALYSIS_DIR,
    exist_ok=True
)

print(
    "\nFinal analysis directory:",
    FINAL_ANALYSIS_DIR
)

# ============================================================
# 2. Frozen Experiment Results
# ============================================================

# E1 single controlled run
E1_RESULTS = {

    "Experiment": "E1",

    "Configuration":
        "11 longitudinal symptom features",

    "Best_Val_Macro_F1":
        0.4746261729016495,

    "Test_Accuracy":
        0.4644,

    "Test_Macro_F1":
        0.4595,

    "Macro_ROC_AUC":
        0.7864,

    "Macro_PR_AUC":
        0.5014

}

# E2 five-seed robustness results
E2_MULTI = {

    "Experiment": "E2",

    "Configuration":
        "11 symptoms + overall severity",

    "Best_Val_Macro_F1_Mean":
        0.503467,

    "Best_Val_Macro_F1_SD":
        0.008351,

    "Test_Accuracy_Mean":
        0.506667,

    "Test_Accuracy_SD":
        0.012958,

    "Test_Macro_F1_Mean":
        0.505789,

    "Test_Macro_F1_SD":
        0.012825,

    "Macro_ROC_AUC_Mean":
        0.809367,

    "Macro_ROC_AUC_SD":
        0.005728,

    "Macro_PR_AUC_Mean":
        0.535813,

    "Macro_PR_AUC_SD":
        0.008269

}

# E3 five-seed robustness results
E3_MULTI = {

    "Experiment": "E3",

    "Configuration":
        (
            "11 symptoms + overall severity "
            "+ auxiliary symptom reconstruction"
        ),

    "Best_Val_Macro_F1_Mean":
        0.511176,

    "Best_Val_Macro_F1_SD":
        0.002769,

    "Test_Accuracy_Mean":
        0.518667,

    "Test_Accuracy_SD":
        0.012433,

    "Test_Macro_F1_Mean":
        0.517134,

    "Test_Macro_F1_SD":
        0.011824,

    "Macro_ROC_AUC_Mean":
        0.813035,

    "Macro_ROC_AUC_SD":
        0.001831,

    "Macro_PR_AUC_Mean":
        0.542316,

    "Macro_PR_AUC_SD":
        0.005581

}

# ============================================================
# 3. Main Ablation Table
# ============================================================

ablation_df = pd.DataFrame({

    "Experiment": [
        "E1",
        "E2",
        "E3"
    ],

    "Configuration": [

        "11 symptoms",

        "11 symptoms + severity",

        (
            "11 symptoms + severity "
            "+ auxiliary symptom loss"
        )

    ],

    "Best Validation Macro-F1": [

        0.474626,

        0.503467,

        0.511176

    ],

    "Test Accuracy": [

        0.4644,

        0.506667,

        0.518667

    ],

    "Test Macro-F1": [

        0.4595,

        0.505789,

        0.517134

    ],

    "Macro ROC-AUC": [

        0.7864,

        0.809367,

        0.813035

    ],

    "Macro PR-AUC": [

        0.5014,

        0.535813,

        0.542316

    ]

})

print("\n" + "=" * 70)
print("FINAL ABLATION TABLE")
print("=" * 70)

display(
    ablation_df
)

ablation_df.to_csv(

    os.path.join(
        FINAL_ANALYSIS_DIR,
        "final_experiment_table.csv"
    ),

    index=False

)

# ============================================================
# 4. Publication-Style Mean ± SD Table
# ============================================================

robustness_df = pd.DataFrame({

    "Model": [
        "E2",
        "E3"
    ],

    "Val Macro-F1": [

        "0.5035 ± 0.0084",

        "0.5112 ± 0.0028"

    ],

    "Test Accuracy": [

        "0.5067 ± 0.0130",

        "0.5187 ± 0.0124"

    ],

    "Test Macro-F1": [

        "0.5058 ± 0.0128",

        "0.5171 ± 0.0118"

    ],

    "Macro ROC-AUC": [

        "0.8094 ± 0.0057",

        "0.8130 ± 0.0018"

    ],

    "Macro PR-AUC": [

        "0.5358 ± 0.0083",

        "0.5423 ± 0.0056"

    ]

})

print("\nMulti-Seed Robustness Table")
print("-" * 70)

display(
    robustness_df
)

robustness_df.to_csv(

    os.path.join(
        FINAL_ANALYSIS_DIR,
        "multiseed_robustness_table.csv"
    ),

    index=False

)

# ============================================================
# 5. Statistical Comparison Table
# ============================================================

stats_df = pd.DataFrame({

    "Metric": [

        "Test Macro-F1",

        "Test Accuracy",

        "Macro ROC-AUC",

        "Macro PR-AUC"

    ],

    "E2 Mean": [

        0.505789,

        0.506667,

        0.809367,

        0.535813

    ],

    "E3 Mean": [

        0.517134,

        0.518667,

        0.813034,

        0.542316

    ],

    "Mean Difference": [

        0.011345,

        0.012000,

        0.003668,

        0.006503

    ],

    "Paired t-test p": [

        0.006769,

        0.003454,

        0.172970,

        0.119081

    ],

    "Wilcoxon p": [

        0.0625,

        0.0625,

        0.0625,

        0.0625

    ],

    "Cohen dz": [

        2.300838,

        2.770126,

        0.740792,

        0.884565

    ]

})

print("\nStatistical Comparison")
print("-" * 70)

display(
    stats_df
)

stats_df.to_csv(

    os.path.join(
        FINAL_ANALYSIS_DIR,
        "statistical_comparison.csv"
    ),

    index=False

)

# ============================================================
# 6. Ablation Figure
# ============================================================

experiments = [
    "E1",
    "E2",
    "E3"
]

macro_f1_values = [
    0.4595,
    0.505789,
    0.517134
]

plt.figure(
    figsize=(7, 5)
)

plt.bar(
    experiments,
    macro_f1_values
)

plt.ylabel(
    "Test Macro-F1"
)

plt.xlabel(
    "Experiment"
)

plt.title(
    "STraT-Net Ablation Study"
)

plt.ylim(
    0,
    0.60
)

for i, value in enumerate(
    macro_f1_values
):

    plt.text(

        i,

        value + 0.01,

        f"{value:.4f}",

        ha="center"

    )

plt.tight_layout()

plt.savefig(

    os.path.join(
        FINAL_ANALYSIS_DIR,
        "E1_E2_E3_ablation.png"
    ),

    dpi=300,

    bbox_inches="tight"

)

plt.show()

# ============================================================
# 7. E2 vs E3 Multi-Seed Macro-F1
# ============================================================

seeds = [
    42,
    123,
    2026,
    3407,
    7777
]

e2_f1 = np.array([

    0.495457,

    0.501637,

    0.526529,

    0.509151,

    0.496170

])

e3_f1 = np.array([

    0.501447,

    0.514469,

    0.532969,

    0.523500,

    0.513284

])

plt.figure(
    figsize=(8, 5)
)

plt.plot(
    seeds,
    e2_f1,
    marker="o",
    label="E2"
)

plt.plot(
    seeds,
    e3_f1,
    marker="o",
    label="E3"
)

plt.xlabel(
    "Random Seed"
)

plt.ylabel(
    "Test Macro-F1"
)

plt.title(
    "E2 vs E3 Across Random Seeds"
)

plt.legend()

plt.tight_layout()

plt.savefig(

    os.path.join(
        FINAL_ANALYSIS_DIR,
        "E2_E3_multiseed_macro_f1.png"
    ),

    dpi=300,

    bbox_inches="tight"

)

plt.show()

# ============================================================
# 8. Paired Improvement Figure
# ============================================================

differences = (
    e3_f1
    -
    e2_f1
)

plt.figure(
    figsize=(8, 5)
)

plt.bar(
    [str(x) for x in seeds],
    differences
)

plt.axhline(
    0,
    linewidth=1
)

plt.xlabel(
    "Random Seed"
)

plt.ylabel(
    "E3 - E2 Test Macro-F1"
)

plt.title(
    "Paired Macro-F1 Improvement from Auxiliary Supervision"
)

for i, value in enumerate(
    differences
):

    plt.text(

        i,

        value + 0.0005,

        f"+{value:.4f}",

        ha="center",

        fontsize=9

    )

plt.tight_layout()

plt.savefig(

    os.path.join(
        FINAL_ANALYSIS_DIR,
        "E3_paired_improvement.png"
    ),

    dpi=300,

    bbox_inches="tight"

)

plt.show()

# ============================================================
# 9. Final E3 Confusion Matrix
#
# Uses the previously evaluated single best E3 checkpoint
# if cm_e3 exists in memory.
# ============================================================

if "cm_e3" in globals():

    plt.figure(
        figsize=(8, 7)
    )

    plt.imshow(
        cm_e3
    )

    plt.title(
        "E3 Confusion Matrix"
    )

    plt.xlabel(
        "Predicted Disease"
    )

    plt.ylabel(
        "True Disease"
    )

    plt.xticks(

        np.arange(
            len(class_names)
        ),

        class_names,

        rotation=45,

        ha="right"

    )

    plt.yticks(

        np.arange(
            len(class_names)
        ),

        class_names

    )

    for i in range(
        cm_e3.shape[0]
    ):

        for j in range(
            cm_e3.shape[1]
        ):

            plt.text(

                j,

                i,

                str(
                    cm_e3[i, j]
                ),

                ha="center",

                va="center"

            )

    plt.tight_layout()

    plt.savefig(

        os.path.join(
            FINAL_ANALYSIS_DIR,
            "confusion_matrix_E3.png"
        ),

        dpi=300,

        bbox_inches="tight"

    )

    plt.show()

else:

    print(
        "\ncm_e3 not found in memory."
    )

    print(
        "Confusion matrix figure skipped."
    )

# ============================================================
# 10. Per-Disease E3 Metrics
# ============================================================

e3_per_disease_df = pd.DataFrame({

    "Disease": [

        "Alzheimer's disease",

        "Frontotemporal dementia",

        "Lewy body dementia",

        "Multiple system atrophy",

        "Parkinson's disease"

    ],

    "Precision": [

        0.5867,

        0.5263,

        0.5192,

        0.5161,

        0.4458

    ],

    "Recall": [

        0.4889,

        0.5556,

        0.6000,

        0.5333,

        0.4111

    ],

    "F1": [

        0.5333,

        0.5405,

        0.5567,

        0.5246,

        0.4277

    ]

})

print("\nE3 Per-Disease Metrics")
print("-" * 70)

display(
    e3_per_disease_df
)

e3_per_disease_df.to_csv(

    os.path.join(
        FINAL_ANALYSIS_DIR,
        "per_disease_metrics_E3.csv"
    ),

    index=False

)

# ============================================================
# 11. Per-Disease F1 Figure
# ============================================================

plt.figure(
    figsize=(9, 5)
)

plt.bar(

    e3_per_disease_df[
        "Disease"
    ],

    e3_per_disease_df[
        "F1"
    ]

)

plt.ylabel(
    "F1 Score"
)

plt.xlabel(
    "Disease"
)

plt.title(
    "E3 Per-Disease F1 Performance"
)

plt.xticks(
    rotation=45,
    ha="right"
)

plt.ylim(
    0,
    0.65
)

plt.tight_layout()

plt.savefig(

    os.path.join(
        FINAL_ANALYSIS_DIR,
        "per_disease_f1_E3.png"
    ),

    dpi=300,

    bbox_inches="tight"

)

plt.show()

# ============================================================
# 12. Trajectory-Length Performance
#
# Uses E2/E3 results already observed.
# ============================================================

trajectory_df_final = pd.DataFrame({

    "Trajectory Length": [
        3,
        4,
        5,
        6,
        7
    ],

    "E1 Accuracy": [

        0.440476,

        0.354545,

        0.403670,

        0.620253,

        0.588235

    ],

    "E2 Accuracy": [

        0.416667,

        0.463636,

        0.458716,

        0.582278,

        0.676471

    ]

})

print("\nTrajectory-Length Performance")
print("-" * 70)

display(
    trajectory_df_final
)

trajectory_df_final.to_csv(

    os.path.join(
        FINAL_ANALYSIS_DIR,
        "trajectory_length_performance.csv"
    ),

    index=False

)

plt.figure(
    figsize=(8, 5)
)

plt.plot(

    trajectory_df_final[
        "Trajectory Length"
    ],

    trajectory_df_final[
        "E1 Accuracy"
    ],

    marker="o",

    label="E1"

)

plt.plot(

    trajectory_df_final[
        "Trajectory Length"
    ],

    trajectory_df_final[
        "E2 Accuracy"
    ],

    marker="o",

    label="E2"

)

plt.xlabel(
    "Number of Longitudinal Visits"
)

plt.ylabel(
    "Test Accuracy"
)

plt.title(
    "Performance by Trajectory Length"
)

plt.legend()

plt.tight_layout()

plt.savefig(

    os.path.join(
        FINAL_ANALYSIS_DIR,
        "trajectory_length_performance.png"
    ),

    dpi=300,

    bbox_inches="tight"

)

plt.show()

# ============================================================
# 13. Key Findings File
# ============================================================

key_findings = {

    "E1_Test_Macro_F1":
        0.4595,

    "E2_Test_Macro_F1_Mean":
        0.505789,

    "E3_Test_Macro_F1_Mean":
        0.517134,

    "E2_to_E3_Mean_F1_Gain":
        0.011345,

    "E3_Better_Seeds":
        "5/5",

    "Paired_t_p_Macro_F1":
        0.006769,

    "Wilcoxon_p_Macro_F1":
        0.0625,

    "Cohens_dz_Macro_F1":
        2.300838,

    "Main_Finding":
        (
            "Overall severity substantially improved "
            "classification over symptom-only trajectories, "
            "while auxiliary symptom reconstruction produced "
            "a smaller but consistent additional improvement."
        ),

    "Main_Limitation":
        (
            "Only five random seeds were used, "
            "and Parkinson's disease remained frequently "
            "confused with multiple system atrophy."
        )

}

with open(

    os.path.join(
        FINAL_ANALYSIS_DIR,
        "key_findings.json"
    ),

    "w"

) as f:

    json.dump(
        key_findings,
        f,
        indent=4
    )

# ============================================================
# 14. Final File Check
# ============================================================

print("\n" + "=" * 70)

print("FINAL ANALYSIS FILES")

print("=" * 70)

for filename in sorted(
    os.listdir(
        FINAL_ANALYSIS_DIR
    )
):

    print(
        "✓",
        filename
    )

print("\n" + "=" * 70)

print("FINAL RESEARCH ANALYSIS COMPLETED")

print("=" * 70)

In [ ]:
# ============================================================
# STraT-Net
# Publication-Quality Final Figure Generation
# 1200 DPI | Bold Typography | Scientific Palette
# ============================================================

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib as mpl

from sklearn.metrics import confusion_matrix

print("=" * 70)
print("STraT-Net : PUBLICATION FIGURE GENERATION")
print("=" * 70)

# ============================================================
# 1. Output Directory
# ============================================================

FIGURE_DIR = os.path.join(
    "final_analysis",
    "publication_figures"
)

os.makedirs(
    FIGURE_DIR,
    exist_ok=True
)

print(
    "\nFigure output directory:",
    FIGURE_DIR
)

# ============================================================
# 2. Publication Style
# ============================================================

SCI_COLORS = {

    "E1": "#4477AA",   # scientific blue

    "E2": "#EE6677",   # muted scientific red

    "E3": "#228833",   # scientific green

    "Gray": "#666666",

    "LightGray": "#D9D9D9",

    "Dark": "#222222"

}

mpl.rcParams.update({

    "font.family":
        "sans-serif",

    "font.sans-serif":
        [
            "Arial",
            "DejaVu Sans"
        ],

    "font.size":
        8,

    "font.weight":
        "bold",

    "axes.titlesize":
        10,

    "axes.titleweight":
        "bold",

    "axes.labelsize":
        8,

    "axes.labelweight":
        "bold",

    "xtick.labelsize":
        8,

    "ytick.labelsize":
        8,

    "legend.fontsize":
        8,

    "axes.linewidth":
        1.0,

    "lines.linewidth":
        1.6,

    "lines.markersize":
        5,

    "figure.facecolor":
        "white",

    "axes.facecolor":
        "white",

    "savefig.facecolor":
        "white",

    "savefig.dpi":
        1200,

    "savefig.bbox":
        "tight",

    "savefig.pad_inches":
        0.05

})

# ============================================================
# 3. Helper Functions
# ============================================================

def make_text_bold(ax):

    for label in ax.get_xticklabels():

        label.set_fontweight(
            "bold"
        )

        label.set_fontsize(
            8
        )

    for label in ax.get_yticklabels():

        label.set_fontweight(
            "bold"
        )

        label.set_fontsize(
            8
        )

    legend = ax.get_legend()

    if legend is not None:

        for text in legend.get_texts():

            text.set_fontweight(
                "bold"
            )

            text.set_fontsize(
                8
            )


def clean_axes(ax):

    ax.spines[
        "top"
    ].set_visible(
        False
    )

    ax.spines[
        "right"
    ].set_visible(
        False
    )

    ax.tick_params(
        width=1.0
    )

    make_text_bold(
        ax
    )


def save_figure(
    fig,
    filename
):

    png_path = os.path.join(
        FIGURE_DIR,
        filename + ".png"
    )

    tiff_path = os.path.join(
        FIGURE_DIR,
        filename + ".tiff"
    )

    pdf_path = os.path.join(
        FIGURE_DIR,
        filename + ".pdf"
    )

    fig.savefig(
        png_path,
        dpi=1200,
        bbox_inches="tight",
        facecolor="white"
    )

    fig.savefig(
        tiff_path,
        dpi=1200,
        bbox_inches="tight",
        facecolor="white"
    )

    fig.savefig(
        pdf_path,
        bbox_inches="tight",
        facecolor="white"
    )

    print(
        f"✓ Saved {filename}"
    )

# ============================================================
# FIGURE 1
# E1 → E2 → E3 Ablation Performance
# ============================================================

experiments = [
    "E1",
    "E2",
    "E3"
]

macro_f1 = [
    0.4595,
    0.505789,
    0.517134
]

colors = [
    SCI_COLORS["E1"],
    SCI_COLORS["E2"],
    SCI_COLORS["E3"]
]

fig, ax = plt.subplots(
    figsize=(3.5, 2.8)
)

bars = ax.bar(

    experiments,

    macro_f1,

    color=colors,

    edgecolor=SCI_COLORS[
        "Dark"
    ],

    linewidth=0.8,

    width=0.62

)

ax.set_title(
    "STraT-Net Ablation Performance",
    fontsize=10,
    fontweight="bold"
)

ax.set_xlabel(
    "Experiment",
    fontsize=8,
    fontweight="bold"
)

ax.set_ylabel(
    "Test Macro-F1",
    fontsize=8,
    fontweight="bold"
)

ax.set_ylim(
    0.40,
    0.55
)

for bar, value in zip(
    bars,
    macro_f1
):

    ax.text(

        bar.get_x()
        +
        bar.get_width() / 2,

        value + 0.003,

        f"{value:.4f}",

        ha="center",

        va="bottom",

        fontsize=8,

        fontweight="bold"

    )

clean_axes(
    ax
)

fig.tight_layout()

save_figure(
    fig,
    "Figure_1_E1_E2_E3_Ablation"
)

plt.show()

# ============================================================
# FIGURE 2
# E2 vs E3 Across Five Random Seeds
# ============================================================

seeds = np.array([
    42,
    123,
    2026,
    3407,
    7777
])

e2_f1 = np.array([
    0.495457,
    0.501637,
    0.526529,
    0.509151,
    0.496170
])

e3_f1 = np.array([
    0.501447,
    0.514469,
    0.532969,
    0.523500,
    0.513284
])

fig, ax = plt.subplots(
    figsize=(4.2, 3.0)
)

x = np.arange(
    len(seeds)
)

ax.plot(

    x,

    e2_f1,

    marker="o",

    color=SCI_COLORS[
        "E2"
    ],

    label="E2",

    linewidth=1.6

)

ax.plot(

    x,

    e3_f1,

    marker="s",

    color=SCI_COLORS[
        "E3"
    ],

    label="E3",

    linewidth=1.6

)

ax.set_xticks(
    x
)

ax.set_xticklabels(
    [str(s) for s in seeds]
)

ax.set_title(
    "Multi-Seed Test Macro-F1",
    fontsize=10,
    fontweight="bold"
)

ax.set_xlabel(
    "Random Seed",
    fontsize=8,
    fontweight="bold"
)

ax.set_ylabel(
    "Test Macro-F1",
    fontsize=8,
    fontweight="bold"
)

ax.legend(
    frameon=False
)

ax.set_ylim(
    0.48,
    0.545
)

clean_axes(
    ax
)

fig.tight_layout()

save_figure(
    fig,
    "Figure_2_E2_E3_Multiseed_MacroF1"
)

plt.show()

# ============================================================
# FIGURE 3
# Paired Improvement E3 - E2
# ============================================================

paired_difference = (
    e3_f1
    -
    e2_f1
)

fig, ax = plt.subplots(
    figsize=(4.2, 3.0)
)

bars = ax.bar(

    x,

    paired_difference,

    color=SCI_COLORS[
        "E3"
    ],

    edgecolor=SCI_COLORS[
        "Dark"
    ],

    linewidth=0.8,

    width=0.60

)

ax.axhline(

    0,

    color=SCI_COLORS[
        "Dark"
    ],

    linewidth=1.0

)

ax.set_xticks(
    x
)

ax.set_xticklabels(
    [str(s) for s in seeds]
)

ax.set_title(
    "Paired E3 Improvement over E2",
    fontsize=10,
    fontweight="bold"
)

ax.set_xlabel(
    "Random Seed",
    fontsize=8,
    fontweight="bold"
)

ax.set_ylabel(
    "Macro-F1 Difference",
    fontsize=8,
    fontweight="bold"
)

for bar, value in zip(
    bars,
    paired_difference
):

    ax.text(

        bar.get_x()
        +
        bar.get_width() / 2,

        value + 0.0004,

        f"+{value:.4f}",

        ha="center",

        va="bottom",

        fontsize=8,

        fontweight="bold"

    )

clean_axes(
    ax
)

fig.tight_layout()

save_figure(
    fig,
    "Figure_3_E3_Paired_Improvement"
)

plt.show()

# ============================================================
# FIGURE 4
# Final E3 Confusion Matrix
# ============================================================

cm_e3_final = np.array([

    [44, 20, 15,  4,  7],

    [15, 50, 12,  8,  5],

    [ 9,  9, 54,  8, 10],

    [ 3,  5, 10, 48, 24],

    [ 4, 11, 13, 25, 37]

])

disease_short = [

    "AD",

    "FTD",

    "LBD",

    "MSA",

    "PD"

]

fig, ax = plt.subplots(
    figsize=(3.8, 3.4)
)

im = ax.imshow(

    cm_e3_final,

    cmap="Blues",

    aspect="equal"

)

ax.set_title(
    "E3 Confusion Matrix",
    fontsize=10,
    fontweight="bold"
)

ax.set_xlabel(
    "Predicted Disease",
    fontsize=8,
    fontweight="bold"
)

ax.set_ylabel(
    "True Disease",
    fontsize=8,
    fontweight="bold"
)

ax.set_xticks(
    np.arange(
        len(
            disease_short
        )
    )
)

ax.set_yticks(
    np.arange(
        len(
            disease_short
        )
    )
)

ax.set_xticklabels(
    disease_short
)

ax.set_yticklabels(
    disease_short
)

threshold = (
    cm_e3_final.max()
    /
    2
)

for i in range(
    cm_e3_final.shape[0]
):

    for j in range(
        cm_e3_final.shape[1]
    ):

        value = (
            cm_e3_final[
                i,
                j
            ]
        )

        ax.text(

            j,

            i,

            str(
                value
            ),

            ha="center",

            va="center",

            fontsize=8,

            fontweight="bold",

            color=(
                "white"
                if value > threshold
                else "black"
            )

        )

cbar = fig.colorbar(

    im,

    ax=ax,

    fraction=0.046,

    pad=0.04

)

cbar.set_label(

    "Number of Patients",

    fontsize=8,

    fontweight="bold"

)

for tick in cbar.ax.get_yticklabels():

    tick.set_fontsize(
        8
    )

    tick.set_fontweight(
        "bold"
    )

make_text_bold(
    ax
)

fig.tight_layout()

save_figure(
    fig,
    "Figure_4_E3_Confusion_Matrix"
)

plt.show()

# ============================================================
# FIGURE 5
# E3 Per-Disease F1
# ============================================================

disease_names = [

    "Alzheimer's",

    "Frontotemporal",

    "Lewy body",

    "Multiple system\natrophy",

    "Parkinson's"

]

disease_f1 = np.array([

    0.5333,

    0.5405,

    0.5567,

    0.5246,

    0.4277

])

fig, ax = plt.subplots(
    figsize=(4.6, 3.1)
)

bars = ax.bar(

    np.arange(
        len(
            disease_names
        )
    ),

    disease_f1,

    color=SCI_COLORS[
        "E3"
    ],

    edgecolor=SCI_COLORS[
        "Dark"
    ],

    linewidth=0.8

)

ax.set_xticks(

    np.arange(
        len(
            disease_names
        )
    )

)

ax.set_xticklabels(

    disease_names,

    rotation=25,

    ha="right"

)

ax.set_title(
    "E3 Per-Disease F1 Performance",
    fontsize=10,
    fontweight="bold"
)

ax.set_xlabel(
    "Disease",
    fontsize=8,
    fontweight="bold"
)

ax.set_ylabel(
    "F1 Score",
    fontsize=8,
    fontweight="bold"
)

ax.set_ylim(
    0.35,
    0.60
)

for bar, value in zip(
    bars,
    disease_f1
):

    ax.text(

        bar.get_x()
        +
        bar.get_width() / 2,

        value + 0.005,

        f"{value:.3f}",

        ha="center",

        va="bottom",

        fontsize=8,

        fontweight="bold"

    )

clean_axes(
    ax
)

fig.tight_layout()

save_figure(
    fig,
    "Figure_5_E3_Per_Disease_F1"
)

plt.show()

# ============================================================
# FIGURE 6
# Trajectory Length Performance
# Exploratory Single-Run Analysis
# ============================================================

trajectory_length = np.array([
    3,
    4,
    5,
    6,
    7
])

e1_accuracy = np.array([
    0.440476,
    0.354545,
    0.403670,
    0.620253,
    0.588235
])

e2_accuracy = np.array([
    0.416667,
    0.463636,
    0.458716,
    0.582278,
    0.676471
])

fig, ax = plt.subplots(
    figsize=(4.2, 3.0)
)

ax.plot(

    trajectory_length,

    e1_accuracy,

    marker="o",

    color=SCI_COLORS[
        "E1"
    ],

    label="E1"

)

ax.plot(

    trajectory_length,

    e2_accuracy,

    marker="s",

    color=SCI_COLORS[
        "E2"
    ],

    label="E2"

)

ax.set_title(
    "Performance by Trajectory Length",
    fontsize=10,
    fontweight="bold"
)

ax.set_xlabel(
    "Number of Longitudinal Visits",
    fontsize=8,
    fontweight="bold"
)

ax.set_ylabel(
    "Test Accuracy",
    fontsize=8,
    fontweight="bold"
)

ax.set_xticks(
    trajectory_length
)

ax.legend(
    frameon=False
)

clean_axes(
    ax
)

fig.tight_layout()

save_figure(
    fig,
    "Figure_6_Trajectory_Length_Performance"
)

plt.show()

# ============================================================
# FIGURE 7
# Multi-Seed Summary with Error Bars
# ============================================================

models = [
    "E2",
    "E3"
]

mean_f1 = np.array([
    0.505789,
    0.517134
])

sd_f1 = np.array([
    0.012825,
    0.011824
])

fig, ax = plt.subplots(
    figsize=(3.5, 2.9)
)

bars = ax.bar(

    models,

    mean_f1,

    yerr=sd_f1,

    capsize=4,

    color=[
        SCI_COLORS[
            "E2"
        ],
        SCI_COLORS[
            "E3"
        ]
    ],

    edgecolor=SCI_COLORS[
        "Dark"
    ],

    linewidth=0.8,

    width=0.60

)

ax.set_title(
    "Five-Seed Robustness Comparison",
    fontsize=10,
    fontweight="bold"
)

ax.set_xlabel(
    "Model",
    fontsize=8,
    fontweight="bold"
)

ax.set_ylabel(
    "Test Macro-F1 (Mean ± SD)",
    fontsize=8,
    fontweight="bold"
)

ax.set_ylim(
    0.47,
    0.55
)

for bar, mean, sd in zip(
    bars,
    mean_f1,
    sd_f1
):

    ax.text(

        bar.get_x()
        +
        bar.get_width() / 2,

        mean + sd + 0.003,

        f"{mean:.4f}\n±{sd:.4f}",

        ha="center",

        va="bottom",

        fontsize=8,

        fontweight="bold"

    )

clean_axes(
    ax
)

fig.tight_layout()

save_figure(
    fig,
    "Figure_7_Multiseed_Mean_SD"
)

plt.show()

# ============================================================
# 4. Save Supporting Figure Data
# ============================================================

figure_data_df = pd.DataFrame({

    "Seed":
        seeds,

    "E2_Test_Macro_F1":
        e2_f1,

    "E3_Test_Macro_F1":
        e3_f1,

    "Paired_Difference":
        paired_difference

})

figure_data_df.to_csv(

    os.path.join(
        FIGURE_DIR,
        "figure_source_data_multiseed.csv"
    ),

    index=False

)

# ============================================================
# 5. Final File Check
# ============================================================

print("\n" + "=" * 70)
print("PUBLICATION FIGURES GENERATED")
print("=" * 70)

for filename in sorted(
    os.listdir(
        FIGURE_DIR
    )
):

    print(
        "✓",
        filename
    )

print("\n" + "=" * 70)

print("Formatting Standard")

print("=" * 70)

print("Title            : 10 pt bold")
print("Other text       : 8 pt bold")
print("Resolution       : 1200 DPI")
print("Background       : White")
print("Palette          : Scientific/colorblind-friendly")
print("PNG              : Generated")
print("TIFF             : Generated")
print("PDF              : Generated")

print("=" * 70)
print("PUBLICATION FIGURE MODULE COMPLETED")
print("=" * 70)

In [ ]:
# ============================================================
# STraT-Net E3
# Final Per-Disease Performance Table
# F1 + One-vs-Rest AUC + Misdiagnosis Rate
# ============================================================

import numpy as np
import pandas as pd

from sklearn.metrics import (
    roc_auc_score,
    f1_score,
    recall_score
)

print("=" * 70)
print("STraT-Net E3 : PER-DISEASE PERFORMANCE TABLE")
print("=" * 70)

# ============================================================
# 1. Disease Names
# ============================================================

class_names = list(
    label_encoder.classes_
)

print("\nDisease Mapping")
print("-" * 60)

for i, name in enumerate(class_names):

    print(
        f"{i} -> {name}"
    )

# ============================================================
# 2. Use Final E3 Test Outputs
#
# These should already exist from E3 evaluation:
#
# y_true_e3
# y_pred_e3
# y_prob_e3
# ============================================================

assert "y_true_e3" in globals()
assert "y_pred_e3" in globals()
assert "y_prob_e3" in globals()

print(
    "\nTest patients:",
    len(y_true_e3)
)

# ============================================================
# 3. Calculate Per-Disease Metrics
# ============================================================

rows = []

for class_index, disease_name in enumerate(
    class_names
):

    # --------------------------------------------
    # Binary one-vs-rest true labels
    # --------------------------------------------

    y_true_binary = (
        y_true_e3
        ==
        class_index
    ).astype(int)

    # --------------------------------------------
    # Probability for current disease
    # --------------------------------------------

    y_score = y_prob_e3[
        :,
        class_index
    ]

    # --------------------------------------------
    # Disease-specific F1
    # --------------------------------------------

    y_pred_binary = (
        y_pred_e3
        ==
        class_index
    ).astype(int)

    disease_f1 = f1_score(

        y_true_binary,

        y_pred_binary,

        zero_division=0

    )

    # --------------------------------------------
    # Disease-specific recall / sensitivity
    # --------------------------------------------

    disease_recall = recall_score(

        y_true_binary,

        y_pred_binary,

        zero_division=0

    )

    # --------------------------------------------
    # Misdiagnosis rate
    #
    # = proportion of true disease cases
    # predicted as another disease
    # = 1 - recall
    # --------------------------------------------

    misdiagnosis_rate = (
        1.0
        -
        disease_recall
    )

    # --------------------------------------------
    # One-vs-rest ROC-AUC
    # --------------------------------------------

    disease_auc = roc_auc_score(

        y_true_binary,

        y_score

    )

    rows.append({

        "Neurological Disease":
            disease_name,

        "F1-score":
            disease_f1,

        "One-vs-Rest AUC":
            disease_auc,

        "Misdiagnosis Rate (%)":
            misdiagnosis_rate
            *
            100

    })

# ============================================================
# 4. Final Table
# ============================================================

per_disease_final_df = pd.DataFrame(
    rows
)

print("\n" + "=" * 70)
print("FINAL PER-DISEASE TABLE")
print("=" * 70)

display(
    per_disease_final_df
)

# ============================================================
# 5. Rounded Publication Version
# ============================================================

per_disease_publication_df = (
    per_disease_final_df.copy()
)

per_disease_publication_df[
    "F1-score"
] = (

    per_disease_publication_df[
        "F1-score"
    ]

    .round(3)

)

per_disease_publication_df[
    "One-vs-Rest AUC"
] = (

    per_disease_publication_df[
        "One-vs-Rest AUC"
    ]

    .round(3)

)

per_disease_publication_df[
    "Misdiagnosis Rate (%)"
] = (

    per_disease_publication_df[
        "Misdiagnosis Rate (%)"
    ]

    .round(2)

)

print("\nPublication Version")
print("-" * 70)

display(
    per_disease_publication_df
)

# ============================================================
# 6. Save
# ============================================================

per_disease_final_df.to_csv(

    "final_analysis/"
    "E3_per_disease_performance_full.csv",

    index=False

)

per_disease_publication_df.to_csv(

    "final_analysis/"
    "E3_per_disease_performance_publication.csv",

    index=False

)

print("\nSaved:")
print(
    "final_analysis/"
    "E3_per_disease_performance_publication.csv"
)

print("=" * 70)

#1)Baseline B1: TF-IDF + SVM.

In [ ]:
# ============================================================
# STraT-Net Baseline B1
# TF-IDF + SVM
# Step 1: Patient-Level Longitudinal Text Construction
# ============================================================

import os
import numpy as np
import pandas as pd

print("=" * 70)
print("Baseline B1 : TF-IDF + SVM")
print("Patient-Level Longitudinal Text Construction")
print("=" * 70)

# ============================================================
# 1. Verify Required Columns
# ============================================================

required_columns = {
    "patient_id",
    "year",
    "sentence",
    "diagnosis",
    "split"
}

missing_columns = (
    required_columns
    -
    set(sentence_df.columns)
)

if missing_columns:

    raise ValueError(
        f"Missing sentence-level columns: {missing_columns}"
    )

print(
    "\n✓ Required sentence-level columns found."
)

# ============================================================
# 2. Sort Chronologically
# ============================================================

sentence_text_df = (

    sentence_df[
        [
            "patient_id",
            "year",
            "sentence",
            "diagnosis",
            "split"
        ]
    ]

    .copy()

    .sort_values(
        [
            "patient_id",
            "year"
        ]
    )

)

# ============================================================
# 3. Build One Document Per Patient
#
# Each patient's sentences are concatenated in chronological
# year order.
# ============================================================

patient_text_df = (

    sentence_text_df

    .groupby(
        "patient_id",
        as_index=False
    )

    .agg({

        "sentence":
            lambda x: " ".join(
                x.astype(str)
            ),

        "diagnosis":
            "first",

        "split":
            "first"

    })

    .rename(
        columns={
            "sentence":
                "patient_document"
        }
    )

)

print(
    "\nPatient-level documents created:",
    len(patient_text_df)
)

# ============================================================
# 4. Verify One Diagnosis / Split Per Patient
# ============================================================

diagnosis_check = (

    sentence_df

    .groupby(
        "patient_id"
    )["diagnosis"]

    .nunique()

)

split_check = (

    sentence_df

    .groupby(
        "patient_id"
    )["split"]

    .nunique()

)

print(
    "Patients with multiple diagnoses:",
    int(
        (
            diagnosis_check > 1
        ).sum()
    )
)

print(
    "Patients in multiple splits:",
    int(
        (
            split_check > 1
        ).sum()
    )
)

if (
    diagnosis_check.max() > 1
    or
    split_check.max() > 1
):

    raise ValueError(
        "Patient-level diagnosis/split inconsistency detected."
    )

# ============================================================
# 5. Encode Labels Using Existing Encoder
# ============================================================

patient_text_df[
    "Disease_Label"
] = label_encoder.transform(

    patient_text_df[
        "diagnosis"
    ]

)

# ============================================================
# 6. Split Data
# ============================================================

train_text_df = patient_text_df[
    patient_text_df[
        "split"
    ] == "train"
].reset_index(
    drop=True
)

val_text_df = patient_text_df[
    patient_text_df[
        "split"
    ] == "val"
].reset_index(
    drop=True
)

test_text_df = patient_text_df[
    patient_text_df[
        "split"
    ] == "test"
].reset_index(
    drop=True
)

print("\nPatient Split Sizes")
print("-" * 60)

print(
    "Train:",
    len(train_text_df)
)

print(
    "Val  :",
    len(val_text_df)
)

print(
    "Test :",
    len(test_text_df)
)

# ============================================================
# 7. Expected Split Verification
# ============================================================

assert len(
    train_text_df
) == 2100

assert len(
    val_text_df
) == 450

assert len(
    test_text_df
) == 450

print(
    "\n✓ Patient split sizes match STraT-Net."
)

# ============================================================
# 8. Document Statistics
# ============================================================

patient_text_df[
    "document_length_words"
] = patient_text_df[
    "patient_document"
].str.split().str.len()

print("\nPatient Document Statistics")
print("-" * 60)

print(
    patient_text_df[
        "document_length_words"
    ].describe()
)

# ============================================================
# 9. Example
# ============================================================

print("\nExample Patient")
print("-" * 60)

print(
    "Patient ID:",
    train_text_df.loc[
        0,
        "patient_id"
    ]
)

print(
    "Diagnosis:",
    train_text_df.loc[
        0,
        "diagnosis"
    ]
)

print(
    "Document preview:"
)

print(
    train_text_df.loc[
        0,
        "patient_document"
    ][
        :700
    ]
)

print("\n" + "=" * 70)
print("B1 Step 1 Completed Successfully")
print("=" * 70)

#actual TF-IDF + SVM training/

In [ ]:
# ============================================================
# STraT-Net Baseline B1
# TF-IDF + Linear SVM
# Training + Validation Selection + Final Test
# ============================================================

import os
import json
import numpy as np
import pandas as pd

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC

from sklearn.calibration import CalibratedClassifierCV

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report,
    roc_auc_score,
    average_precision_score
)

from sklearn.preprocessing import label_binarize

print("=" * 70)
print("Baseline B1 : TF-IDF + SVM")
print("Training + Validation Selection + Final Test")
print("=" * 70)

# ============================================================
# 1. Output Directory
# ============================================================

B1_OUTPUT_DIR = (
    "outputs_baseline_tfidf_svm"
)

os.makedirs(
    B1_OUTPUT_DIR,
    exist_ok=True
)

# ============================================================
# 2. TF-IDF Configuration
#
# Fit ONLY on training patient documents.
# ============================================================

tfidf = TfidfVectorizer(

    lowercase=True,

    ngram_range=(
        1,
        2
    ),

    min_df=2,

    max_df=0.98,

    max_features=50000,

    sublinear_tf=True,

    norm="l2"

)

print(
    "\nFitting TF-IDF on training patients only..."
)

X_train = tfidf.fit_transform(

    train_text_df[
        "patient_document"
    ]

)

X_val = tfidf.transform(

    val_text_df[
        "patient_document"
    ]

)

X_test = tfidf.transform(

    test_text_df[
        "patient_document"
    ]

)

y_train = train_text_df[
    "Disease_Label"
].values

y_val = val_text_df[
    "Disease_Label"
].values

y_test = test_text_df[
    "Disease_Label"
].values

print(
    "Train TF-IDF shape:",
    X_train.shape
)

print(
    "Validation shape:",
    X_val.shape
)

print(
    "Test shape:",
    X_test.shape
)

# ============================================================
# 3. Validation-Based C Selection
# ============================================================

C_VALUES = [
    0.01,
    0.1,
    0.5,
    1.0,
    2.0,
    5.0,
    10.0
]

validation_results = []

best_c = None
best_val_f1 = -1.0

print("\nHyperparameter Search")
print("-" * 60)

for c_value in C_VALUES:

    svm = LinearSVC(

        C=c_value,

        class_weight="balanced",

        random_state=42,

        max_iter=10000

    )

    svm.fit(
        X_train,
        y_train
    )

    val_pred = svm.predict(
        X_val
    )

    val_accuracy = accuracy_score(
        y_val,
        val_pred
    )

    val_macro_f1 = f1_score(
        y_val,
        val_pred,
        average="macro",
        zero_division=0
    )

    validation_results.append({

        "C":
            c_value,

        "Val_Accuracy":
            val_accuracy,

        "Val_Macro_F1":
            val_macro_f1

    })

    print(

        f"C={c_value:<5} | "
        f"Val Accuracy={val_accuracy:.4f} | "
        f"Val Macro-F1={val_macro_f1:.4f}"

    )

    if val_macro_f1 > best_val_f1:

        best_val_f1 = (
            val_macro_f1
        )

        best_c = (
            c_value
        )

print("\nBest C:", best_c)

print(
    "Best Validation Macro-F1:",
    f"{best_val_f1:.4f}"
)

validation_results_df = pd.DataFrame(
    validation_results
)

validation_results_df.to_csv(

    os.path.join(

        B1_OUTPUT_DIR,

        "validation_C_search.csv"

    ),

    index=False

)

# ============================================================
# 4. Train Best SVM
#
# IMPORTANT:
# Keep training on train only.
# Validation was used for model selection.
# Test remains untouched until now.
# ============================================================

best_svm = LinearSVC(

    C=best_c,

    class_weight="balanced",

    random_state=42,

    max_iter=10000

)

best_svm.fit(
    X_train,
    y_train
)

# ============================================================
# 5. Calibrate Scores for Probabilities
#
# Needed for ROC-AUC / PR-AUC.
# Calibration uses training data only.
# ============================================================

calibrated_svm = CalibratedClassifierCV(

    estimator=LinearSVC(

        C=best_c,

        class_weight="balanced",

        random_state=42,

        max_iter=10000

    ),

    method="sigmoid",

    cv=5

)

print(
    "\nCalibrating SVM probabilities..."
)

calibrated_svm.fit(
    X_train,
    y_train
)

# ============================================================
# 6. Final Test Predictions
# ============================================================

test_pred = best_svm.predict(
    X_test
)

test_prob = calibrated_svm.predict_proba(
    X_test
)

# ============================================================
# 7. Overall Metrics
# ============================================================

b1_accuracy = accuracy_score(
    y_test,
    test_pred
)

b1_macro_precision = precision_score(
    y_test,
    test_pred,
    average="macro",
    zero_division=0
)

b1_macro_recall = recall_score(
    y_test,
    test_pred,
    average="macro",
    zero_division=0
)

b1_macro_f1 = f1_score(
    y_test,
    test_pred,
    average="macro",
    zero_division=0
)

b1_error_rate = (
    1.0
    -
    b1_accuracy
) * 100

y_test_binary = label_binarize(

    y_test,

    classes=np.arange(
        NUM_DISEASES
    )

)

b1_macro_roc_auc = roc_auc_score(

    y_test_binary,

    test_prob,

    average="macro",

    multi_class="ovr"

)

b1_macro_pr_auc = average_precision_score(

    y_test_binary,

    test_prob,

    average="macro"

)

print("\n" + "=" * 70)
print("TF-IDF + SVM FINAL TEST METRICS")
print("=" * 70)

print(
    f"Accuracy        : {b1_accuracy:.4f}"
)

print(
    f"Macro Precision : {b1_macro_precision:.4f}"
)

print(
    f"Macro Recall    : {b1_macro_recall:.4f}"
)

print(
    f"Macro F1        : {b1_macro_f1:.4f}"
)

print(
    f"Macro ROC-AUC   : {b1_macro_roc_auc:.4f}"
)

print(
    f"Macro PR-AUC    : {b1_macro_pr_auc:.4f}"
)

print(
    f"Error Rate (%)  : {b1_error_rate:.2f}"
)

# ============================================================
# 8. Confusion Matrix
# ============================================================

cm_b1 = confusion_matrix(
    y_test,
    test_pred
)

print("\nConfusion Matrix")
print("-" * 60)

print(
    cm_b1
)

# ============================================================
# 9. Classification Report
# ============================================================

class_names = list(
    label_encoder.classes_
)

print("\nClassification Report")
print("-" * 60)

print(

    classification_report(

        y_test,

        test_pred,

        target_names=class_names,

        digits=4,

        zero_division=0

    )

)

# ============================================================
# 10. Per-Disease Metrics
# ============================================================

per_disease_rows = []

for class_index, disease_name in enumerate(
    class_names
):

    y_true_binary = (
        y_test
        ==
        class_index
    ).astype(int)

    y_pred_binary = (
        test_pred
        ==
        class_index
    ).astype(int)

    disease_f1 = f1_score(

        y_true_binary,

        y_pred_binary,

        zero_division=0

    )

    disease_recall = recall_score(

        y_true_binary,

        y_pred_binary,

        zero_division=0

    )

    disease_auc = roc_auc_score(

        y_true_binary,

        test_prob[
            :,
            class_index
        ]

    )

    per_disease_rows.append({

        "Neurological Disease":
            disease_name,

        "F1-score":
            disease_f1,

        "One-vs-Rest AUC":
            disease_auc,

        "Misdiagnosis Rate (%)":
            (
                1.0
                -
                disease_recall
            )
            *
            100

    })

b1_per_disease_df = pd.DataFrame(
    per_disease_rows
)

print("\nPer-Disease Results")
print("-" * 60)

display(
    b1_per_disease_df
)

# ============================================================
# 11. Patient Prediction File
# ============================================================

b1_predictions_df = pd.DataFrame({

    "patient_id":
        test_text_df[
            "patient_id"
        ].values,

    "true_label":
        y_test,

    "predicted_label":
        test_pred,

    "true_disease":
        [
            class_names[i]
            for i in y_test
        ],

    "predicted_disease":
        [
            class_names[i]
            for i in test_pred
        ],

    "correct":
        (
            y_test
            ==
            test_pred
        )

})

for i, disease_name in enumerate(
    class_names
):

    safe_name = (
        disease_name
        .lower()
        .replace(
            " ",
            "_"
        )
        .replace(
            "'",
            ""
        )
    )

    b1_predictions_df[
        f"prob_{safe_name}"
    ] = test_prob[
        :,
        i
    ]

# ============================================================
# 12. Save Final Results
# ============================================================

b1_summary = {

    "Model":
        "TF-IDF + Linear SVM",

    "Best_C":
        float(
            best_c
        ),

    "Best_Validation_Macro_F1":
        float(
            best_val_f1
        ),

    "Test_Accuracy":
        float(
            b1_accuracy
        ),

    "Test_Macro_Precision":
        float(
            b1_macro_precision
        ),

    "Test_Macro_Recall":
        float(
            b1_macro_recall
        ),

    "Test_Macro_F1":
        float(
            b1_macro_f1
        ),

    "Test_Macro_ROC_AUC":
        float(
            b1_macro_roc_auc
        ),

    "Test_Macro_PR_AUC":
        float(
            b1_macro_pr_auc
        ),

    "Error_Rate_Percent":
        float(
            b1_error_rate
        )

}

with open(

    os.path.join(

        B1_OUTPUT_DIR,

        "TFIDF_SVM_summary.json"

    ),

    "w"

) as f:

    json.dump(
        b1_summary,
        f,
        indent=4
    )

b1_per_disease_df.to_csv(

    os.path.join(

        B1_OUTPUT_DIR,

        "TFIDF_SVM_per_disease_metrics.csv"

    ),

    index=False

)

b1_predictions_df.to_csv(

    os.path.join(

        B1_OUTPUT_DIR,

        "TFIDF_SVM_test_predictions.csv"

    ),

    index=False

)

# ============================================================
# 13. Baseline Comparison Preview
# ============================================================

comparison_preview_df = pd.DataFrame({

    "Model": [

        "TF-IDF + SVM",

        "STraT-Net E3"

    ],

    "Accuracy": [

        b1_accuracy,

        0.518667

    ],

    "Macro-F1": [

        b1_macro_f1,

        0.517134

    ],

    "Macro ROC-AUC": [

        b1_macro_roc_auc,

        0.813035

    ],

    "Macro PR-AUC": [

        b1_macro_pr_auc,

        0.542316

    ],

    "Error Rate (%)": [

        b1_error_rate,

        (
            1.0
            -
            0.518667
        )
        *
        100

    ]

})

print("\n" + "=" * 70)
print("BASELINE COMPARISON PREVIEW")
print("=" * 70)

display(
    comparison_preview_df
)

print("\n" + "=" * 70)
print("TF-IDF + SVM BASELINE COMPLETED")
print("=" * 70)

#2)Baseline 2 CNN

In [ ]:
# ============================================================
# STraT-Net Baseline B2
# TextCNN
# Step 1: Vocabulary + Tokenization + DataLoaders
# ============================================================

import os
import re
import random
import numpy as np
import pandas as pd

import torch
from torch.utils.data import Dataset, DataLoader

from collections import Counter

print("=" * 70)
print("Baseline B2 : TextCNN")
print("Vocabulary + Tokenization + DataLoaders")
print("=" * 70)

# ============================================================
# 1. Reproducibility
# ============================================================

CNN_SEED = 42

random.seed(
    CNN_SEED
)

np.random.seed(
    CNN_SEED
)

torch.manual_seed(
    CNN_SEED
)

torch.cuda.manual_seed_all(
    CNN_SEED
)

# ============================================================
# 2. Configuration
# ============================================================

CNN_MAX_LEN = 800

CNN_MAX_VOCAB = 30000

CNN_MIN_FREQ = 2

CNN_BATCH_SIZE = 32

PAD_TOKEN = "<PAD>"

UNK_TOKEN = "<UNK>"

PAD_IDX = 0

UNK_IDX = 1

print("\nCNN Data Configuration")
print("-" * 60)

print(
    "Maximum sequence length :",
    CNN_MAX_LEN
)

print(
    "Maximum vocabulary size :",
    CNN_MAX_VOCAB
)

print(
    "Minimum token frequency :",
    CNN_MIN_FREQ
)

print(
    "Batch size              :",
    CNN_BATCH_SIZE
)

# ============================================================
# 3. Basic Tokenizer
# ============================================================

TOKEN_PATTERN = re.compile(
    r"[A-Za-z0-9']+"
)

def basic_tokenize(text):

    return TOKEN_PATTERN.findall(
        str(text).lower()
    )

# ============================================================
# 4. Build Vocabulary from TRAINING PATIENTS ONLY
# ============================================================

token_counter = Counter()

for document in train_text_df[
    "patient_document"
]:

    token_counter.update(
        basic_tokenize(
            document
        )
    )

vocab_tokens = [

    token

    for token, count in token_counter.most_common(
        CNN_MAX_VOCAB - 2
    )

    if count >= CNN_MIN_FREQ

]

word_to_idx = {

    PAD_TOKEN:
        PAD_IDX,

    UNK_TOKEN:
        UNK_IDX

}

for token in vocab_tokens:

    if token not in word_to_idx:

        word_to_idx[
            token
        ] = len(
            word_to_idx
        )

idx_to_word = {

    idx:
        word

    for word, idx in word_to_idx.items()

}

CNN_VOCAB_SIZE = len(
    word_to_idx
)

print(
    "\nVocabulary Size:",
    CNN_VOCAB_SIZE
)

# ============================================================
# 5. Encoding Function
# ============================================================

def encode_document(
    text,
    max_len
):

    tokens = basic_tokenize(
        text
    )

    token_ids = [

        word_to_idx.get(
            token,
            UNK_IDX
        )

        for token in tokens

    ]

    token_ids = token_ids[
        :max_len
    ]

    length = len(
        token_ids
    )

    if length < max_len:

        token_ids = (

            token_ids

            +

            [
                PAD_IDX
            ]
            *
            (
                max_len
                -
                length
            )

        )

    return (

        torch.tensor(
            token_ids,
            dtype=torch.long
        ),

        min(
            length,
            max_len
        )

    )

# ============================================================
# 6. Dataset
# ============================================================

class PatientTextDataset(
    Dataset
):

    def __init__(
        self,
        dataframe,
        max_len
    ):

        self.df = dataframe.reset_index(
            drop=True
        )

        self.max_len = max_len

    def __len__(
        self
    ):

        return len(
            self.df
        )

    def __getitem__(
        self,
        idx
    ):

        row = self.df.iloc[
            idx
        ]

        input_ids, length = encode_document(

            row[
                "patient_document"
            ],

            self.max_len

        )

        return {

            "patient_id":
                row[
                    "patient_id"
                ],

            "input_ids":
                input_ids,

            "length":
                torch.tensor(
                    length,
                    dtype=torch.long
                ),

            "label":
                torch.tensor(
                    int(
                        row[
                            "Disease_Label"
                        ]
                    ),
                    dtype=torch.long
                )

        }

# ============================================================
# 7. Create Dataset Objects
# ============================================================

train_cnn_dataset = PatientTextDataset(

    train_text_df,

    CNN_MAX_LEN

)

val_cnn_dataset = PatientTextDataset(

    val_text_df,

    CNN_MAX_LEN

)

test_cnn_dataset = PatientTextDataset(

    test_text_df,

    CNN_MAX_LEN

)

# ============================================================
# 8. Stable DataLoaders
# ============================================================

train_cnn_loader = DataLoader(

    train_cnn_dataset,

    batch_size=CNN_BATCH_SIZE,

    shuffle=True,

    num_workers=0,

    pin_memory=torch.cuda.is_available()

)

val_cnn_loader = DataLoader(

    val_cnn_dataset,

    batch_size=CNN_BATCH_SIZE,

    shuffle=False,

    num_workers=0,

    pin_memory=torch.cuda.is_available()

)

test_cnn_loader = DataLoader(

    test_cnn_dataset,

    batch_size=CNN_BATCH_SIZE,

    shuffle=False,

    num_workers=0,

    pin_memory=torch.cuda.is_available()

)

# ============================================================
# 9. Batch Verification
# ============================================================

sample_cnn = next(
    iter(
        train_cnn_loader
    )
)

print("\nBatch Verification")
print("-" * 60)

print(
    "Input IDs Shape :",
    sample_cnn[
        "input_ids"
    ].shape
)

print(
    "Labels Shape    :",
    sample_cnn[
        "label"
    ].shape
)

print(
    "Lengths Shape   :",
    sample_cnn[
        "length"
    ].shape
)

print(
    "Train Batches   :",
    len(
        train_cnn_loader
    )
)

print(
    "Val Batches     :",
    len(
        val_cnn_loader
    )
)

print(
    "Test Batches    :",
    len(
        test_cnn_loader
    )
)

assert (
    sample_cnn[
        "input_ids"
    ].shape[1]
    ==
    CNN_MAX_LEN
)

print("\n✓ TextCNN data pipeline verified.")

print("\n" + "=" * 70)
print("B2 Step 1 Completed Successfully")
print("=" * 70)

In [ ]:
# ============================================================
# STraT-Net Baseline B2
# TextCNN
# Model + Training + Validation + Final Test
# ============================================================

import os
import json
import numpy as np
import pandas as pd

import torch
import torch.nn as nn

from torch.optim import AdamW
from torch.optim.lr_scheduler import ReduceLROnPlateau

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report,
    roc_auc_score,
    average_precision_score
)

from sklearn.preprocessing import label_binarize

print("=" * 70)
print("Baseline B2 : TextCNN")
print("Model + Training + Final Evaluation")
print("=" * 70)

# ============================================================
# 1. Model Configuration
# ============================================================

CNN_EMBED_DIM = 200

CNN_NUM_FILTERS = 128

CNN_KERNEL_SIZES = [
    3,
    4,
    5
]

CNN_DROPOUT = 0.5

CNN_EPOCHS = 30

CNN_LR = 1e-3

CNN_WEIGHT_DECAY = 1e-4

CNN_PATIENCE = 6

CNN_DEVICE = torch.device(

    "cuda"
    if torch.cuda.is_available()
    else "cpu"

)

CNN_OUTPUT_DIR = (
    "outputs_baseline_textcnn"
)

os.makedirs(
    CNN_OUTPUT_DIR,
    exist_ok=True
)

CNN_BEST_PATH = os.path.join(

    CNN_OUTPUT_DIR,

    "best_textcnn.pth"

)

print("\nModel Configuration")
print("-" * 60)

print(
    "Embedding Dimension :",
    CNN_EMBED_DIM
)

print(
    "Filters per Kernel  :",
    CNN_NUM_FILTERS
)

print(
    "Kernel Sizes        :",
    CNN_KERNEL_SIZES
)

print(
    "Dropout             :",
    CNN_DROPOUT
)

print(
    "Device              :",
    CNN_DEVICE
)

# ============================================================
# 2. TextCNN Model
# ============================================================

class TextCNN(
    nn.Module
):

    def __init__(
        self,
        vocab_size,
        embed_dim,
        num_filters,
        kernel_sizes,
        num_classes,
        dropout,
        padding_idx=0
    ):

        super().__init__()

        self.embedding = nn.Embedding(

            vocab_size,

            embed_dim,

            padding_idx=padding_idx

        )

        self.convs = nn.ModuleList([

            nn.Conv1d(

                in_channels=embed_dim,

                out_channels=num_filters,

                kernel_size=k

            )

            for k in kernel_sizes

        ])

        self.dropout = nn.Dropout(
            dropout
        )

        self.classifier = nn.Linear(

            num_filters
            *
            len(
                kernel_sizes
            ),

            num_classes

        )

    def forward(
        self,
        input_ids
    ):

        # --------------------------------------------
        # [B, L] -> [B, L, E]
        # --------------------------------------------

        x = self.embedding(
            input_ids
        )

        # --------------------------------------------
        # [B, L, E] -> [B, E, L]
        # --------------------------------------------

        x = x.transpose(
            1,
            2
        )

        pooled_outputs = []

        for conv in self.convs:

            conv_out = torch.relu(
                conv(
                    x
                )
            )

            pooled = torch.max(

                conv_out,

                dim=2

            ).values

            pooled_outputs.append(
                pooled
            )

        features = torch.cat(

            pooled_outputs,

            dim=1

        )

        features = self.dropout(
            features
        )

        logits = self.classifier(
            features
        )

        return logits

# ============================================================
# 3. Fresh Model
# ============================================================

textcnn_model = TextCNN(

    vocab_size=(
        CNN_VOCAB_SIZE
    ),

    embed_dim=(
        CNN_EMBED_DIM
    ),

    num_filters=(
        CNN_NUM_FILTERS
    ),

    kernel_sizes=(
        CNN_KERNEL_SIZES
    ),

    num_classes=(
        NUM_DISEASES
    ),

    dropout=(
        CNN_DROPOUT
    ),

    padding_idx=(
        PAD_IDX
    )

).to(
    CNN_DEVICE
)

total_params_cnn = sum(

    p.numel()

    for p in textcnn_model.parameters()

)

print(
    "\nTextCNN Parameters:",
    f"{total_params_cnn:,}"
)

# ============================================================
# 4. Loss + Optimizer + Scheduler
# ============================================================

cnn_criterion = nn.CrossEntropyLoss(

    weight=class_weights.to(
        CNN_DEVICE
    )

)

cnn_optimizer = AdamW(

    textcnn_model.parameters(),

    lr=CNN_LR,

    weight_decay=(
        CNN_WEIGHT_DECAY
    )

)

cnn_scheduler = ReduceLROnPlateau(

    cnn_optimizer,

    mode="max",

    factor=0.5,

    patience=2

)

# ============================================================
# 5. Metric Helper
# ============================================================

def cnn_metrics(
    y_true,
    y_pred
):

    return {

        "accuracy":
            accuracy_score(
                y_true,
                y_pred
            ),

        "precision":
            precision_score(
                y_true,
                y_pred,
                average="macro",
                zero_division=0
            ),

        "recall":
            recall_score(
                y_true,
                y_pred,
                average="macro",
                zero_division=0
            ),

        "macro_f1":
            f1_score(
                y_true,
                y_pred,
                average="macro",
                zero_division=0
            )

    }

# ============================================================
# 6. Train One Epoch
# ============================================================

def train_cnn_epoch():

    textcnn_model.train()

    running_loss = 0.0

    all_labels = []

    all_predictions = []

    for batch in train_cnn_loader:

        input_ids = (
            batch[
                "input_ids"
            ]
            .long()
            .to(
                CNN_DEVICE
            )
        )

        labels = (
            batch[
                "label"
            ]
            .long()
            .to(
                CNN_DEVICE
            )
        )

        cnn_optimizer.zero_grad(
            set_to_none=True
        )

        logits = textcnn_model(
            input_ids
        )

        loss = cnn_criterion(
            logits,
            labels
        )

        loss.backward()

        torch.nn.utils.clip_grad_norm_(

            textcnn_model.parameters(),

            1.0

        )

        cnn_optimizer.step()

        running_loss += (
            loss.item()
        )

        predictions = torch.argmax(
            logits,
            dim=1
        )

        all_labels.extend(
            labels
            .detach()
            .cpu()
            .numpy()
            .tolist()
        )

        all_predictions.extend(
            predictions
            .detach()
            .cpu()
            .numpy()
            .tolist()
        )

    metrics = cnn_metrics(

        all_labels,

        all_predictions

    )

    return (

        running_loss
        /
        len(
            train_cnn_loader
        ),

        metrics

    )

# ============================================================
# 7. Validation
# ============================================================

def validate_cnn():

    textcnn_model.eval()

    running_loss = 0.0

    all_labels = []

    all_predictions = []

    with torch.no_grad():

        for batch in val_cnn_loader:

            input_ids = (
                batch[
                    "input_ids"
                ]
                .long()
                .to(
                    CNN_DEVICE
                )
            )

            labels = (
                batch[
                    "label"
                ]
                .long()
                .to(
                    CNN_DEVICE
                )
            )

            logits = textcnn_model(
                input_ids
            )

            loss = cnn_criterion(
                logits,
                labels
            )

            running_loss += (
                loss.item()
            )

            predictions = torch.argmax(
                logits,
                dim=1
            )

            all_labels.extend(
                labels
                .cpu()
                .numpy()
                .tolist()
            )

            all_predictions.extend(
                predictions
                .cpu()
                .numpy()
                .tolist()
            )

    metrics = cnn_metrics(

        all_labels,

        all_predictions

    )

    return (

        running_loss
        /
        len(
            val_cnn_loader
        ),

        metrics

    )

# ============================================================
# 8. Training Loop
# ============================================================

best_val_f1_cnn = -1.0

best_epoch_cnn = -1

cnn_early_counter = 0

cnn_history = []

for epoch in range(
    1,
    CNN_EPOCHS + 1
):

    train_loss, train_metrics = (
        train_cnn_epoch()
    )

    val_loss, val_metrics = (
        validate_cnn()
    )

    current_val_f1 = (
        val_metrics[
            "macro_f1"
        ]
    )

    cnn_scheduler.step(
        current_val_f1
    )

    cnn_history.append({

        "epoch":
            epoch,

        "train_loss":
            train_loss,

        "val_loss":
            val_loss,

        "train_accuracy":
            train_metrics[
                "accuracy"
            ],

        "val_accuracy":
            val_metrics[
                "accuracy"
            ],

        "train_macro_f1":
            train_metrics[
                "macro_f1"
            ],

        "val_macro_f1":
            val_metrics[
                "macro_f1"
            ]

    })

    print(

        f"Epoch {epoch:02d} | "
        f"Train Loss {train_loss:.4f} | "
        f"Val Loss {val_loss:.4f} | "
        f"Train F1 {train_metrics['macro_f1']:.4f} | "
        f"Val F1 {current_val_f1:.4f}"

    )

    if current_val_f1 > best_val_f1_cnn:

        best_val_f1_cnn = (
            current_val_f1
        )

        best_epoch_cnn = (
            epoch
        )

        cnn_early_counter = 0

        torch.save(

            {

                "epoch":
                    epoch,

                "val_macro_f1":
                    current_val_f1,

                "model_state_dict":
                    textcnn_model.state_dict(),

                "vocab_size":
                    CNN_VOCAB_SIZE,

                "max_len":
                    CNN_MAX_LEN

            },

            CNN_BEST_PATH

        )

    else:

        cnn_early_counter += 1

    if (
        cnn_early_counter
        >=
        CNN_PATIENCE
    ):

        print(
            "\nEarly stopping triggered."
        )

        break

# ============================================================
# 9. Save Training History
# ============================================================

cnn_history_df = pd.DataFrame(
    cnn_history
)

cnn_history_df.to_csv(

    os.path.join(

        CNN_OUTPUT_DIR,

        "TextCNN_training_history.csv"

    ),

    index=False

)

print("\nBest Epoch:", best_epoch_cnn)

print(
    "Best Validation Macro-F1:",
    f"{best_val_f1_cnn:.4f}"
)

# ============================================================
# 10. Load Best Checkpoint
# ============================================================

cnn_checkpoint = torch.load(

    CNN_BEST_PATH,

    map_location=(
        CNN_DEVICE
    )

)

textcnn_model.load_state_dict(

    cnn_checkpoint[
        "model_state_dict"
    ]

)

textcnn_model.eval()

# ============================================================
# 11. Final Test Inference
# ============================================================

cnn_test_labels = []

cnn_test_predictions = []

cnn_test_probabilities = []

cnn_test_patient_ids = []

with torch.no_grad():

    for batch in test_cnn_loader:

        input_ids = (
            batch[
                "input_ids"
            ]
            .long()
            .to(
                CNN_DEVICE
            )
        )

        labels = (
            batch[
                "label"
            ]
            .long()
            .to(
                CNN_DEVICE
            )
        )

        logits = textcnn_model(
            input_ids
        )

        probabilities = torch.softmax(

            logits,

            dim=1

        )

        predictions = torch.argmax(

            probabilities,

            dim=1

        )

        cnn_test_labels.extend(
            labels
            .cpu()
            .numpy()
            .tolist()
        )

        cnn_test_predictions.extend(
            predictions
            .cpu()
            .numpy()
            .tolist()
        )

        cnn_test_probabilities.extend(
            probabilities
            .cpu()
            .numpy()
        )

        cnn_test_patient_ids.extend(

            list(
                batch[
                    "patient_id"
                ]
            )

        )

y_true_cnn = np.asarray(
    cnn_test_labels
)

y_pred_cnn = np.asarray(
    cnn_test_predictions
)

y_prob_cnn = np.asarray(
    cnn_test_probabilities
)

# ============================================================
# 12. Overall Test Metrics
# ============================================================

cnn_accuracy = accuracy_score(
    y_true_cnn,
    y_pred_cnn
)

cnn_macro_precision = precision_score(
    y_true_cnn,
    y_pred_cnn,
    average="macro",
    zero_division=0
)

cnn_macro_recall = recall_score(
    y_true_cnn,
    y_pred_cnn,
    average="macro",
    zero_division=0
)

cnn_macro_f1 = f1_score(
    y_true_cnn,
    y_pred_cnn,
    average="macro",
    zero_division=0
)

cnn_error_rate = (
    1
    -
    cnn_accuracy
) * 100

cnn_y_binary = label_binarize(

    y_true_cnn,

    classes=np.arange(
        NUM_DISEASES
    )

)

cnn_macro_roc_auc = roc_auc_score(

    cnn_y_binary,

    y_prob_cnn,

    average="macro",

    multi_class="ovr"

)

cnn_macro_pr_auc = average_precision_score(

    cnn_y_binary,

    y_prob_cnn,

    average="macro"

)

print("\n" + "=" * 70)
print("TEXTCNN FINAL TEST METRICS")
print("=" * 70)

print(
    f"Accuracy        : {cnn_accuracy:.4f}"
)

print(
    f"Macro Precision : {cnn_macro_precision:.4f}"
)

print(
    f"Macro Recall    : {cnn_macro_recall:.4f}"
)

print(
    f"Macro F1        : {cnn_macro_f1:.4f}"
)

print(
    f"Macro ROC-AUC   : {cnn_macro_roc_auc:.4f}"
)

print(
    f"Macro PR-AUC    : {cnn_macro_pr_auc:.4f}"
)

print(
    f"Error Rate (%)  : {cnn_error_rate:.2f}"
)

# ============================================================
# 13. Confusion Matrix
# ============================================================

cm_cnn = confusion_matrix(

    y_true_cnn,

    y_pred_cnn

)

print("\nConfusion Matrix")
print("-" * 60)

print(
    cm_cnn
)

# ============================================================
# 14. Classification Report
# ============================================================

class_names = list(
    label_encoder.classes_
)

print("\nClassification Report")
print("-" * 60)

print(

    classification_report(

        y_true_cnn,

        y_pred_cnn,

        target_names=class_names,

        digits=4,

        zero_division=0

    )

)

# ============================================================
# 15. Per-Disease Metrics
# ============================================================

cnn_per_disease_rows = []

for class_index, disease_name in enumerate(
    class_names
):

    y_true_binary = (
        y_true_cnn
        ==
        class_index
    ).astype(int)

    y_pred_binary = (
        y_pred_cnn
        ==
        class_index
    ).astype(int)

    disease_f1 = f1_score(

        y_true_binary,

        y_pred_binary,

        zero_division=0

    )

    disease_recall = recall_score(

        y_true_binary,

        y_pred_binary,

        zero_division=0

    )

    disease_auc = roc_auc_score(

        y_true_binary,

        y_prob_cnn[
            :,
            class_index
        ]

    )

    cnn_per_disease_rows.append({

        "Neurological Disease":
            disease_name,

        "F1-score":
            disease_f1,

        "One-vs-Rest AUC":
            disease_auc,

        "Misdiagnosis Rate (%)":
            (
                1
                -
                disease_recall
            )
            *
            100

    })

cnn_per_disease_df = pd.DataFrame(
    cnn_per_disease_rows
)

print("\nPer-Disease Results")
print("-" * 60)

display(
    cnn_per_disease_df
)

# ============================================================
# 16. Save Results
# ============================================================

cnn_summary = {

    "Model":
        "TextCNN",

    "Best_Epoch":
        int(
            best_epoch_cnn
        ),

    "Best_Validation_Macro_F1":
        float(
            best_val_f1_cnn
        ),

    "Test_Accuracy":
        float(
            cnn_accuracy
        ),

    "Test_Macro_Precision":
        float(
            cnn_macro_precision
        ),

    "Test_Macro_Recall":
        float(
            cnn_macro_recall
        ),

    "Test_Macro_F1":
        float(
            cnn_macro_f1
        ),

    "Test_Macro_ROC_AUC":
        float(
            cnn_macro_roc_auc
        ),

    "Test_Macro_PR_AUC":
        float(
            cnn_macro_pr_auc
        ),

    "Error_Rate_Percent":
        float(
            cnn_error_rate
        )

}

with open(

    os.path.join(

        CNN_OUTPUT_DIR,

        "TextCNN_summary.json"

    ),

    "w"

) as f:

    json.dump(
        cnn_summary,
        f,
        indent=4
    )

cnn_per_disease_df.to_csv(

    os.path.join(

        CNN_OUTPUT_DIR,

        "TextCNN_per_disease_metrics.csv"

    ),

    index=False

)

cnn_predictions_df = pd.DataFrame({

    "patient_id":
        cnn_test_patient_ids,

    "true_label":
        y_true_cnn,

    "predicted_label":
        y_pred_cnn,

    "correct":
        (
            y_true_cnn
            ==
            y_pred_cnn
        )

})

cnn_predictions_df.to_csv(

    os.path.join(

        CNN_OUTPUT_DIR,

        "TextCNN_test_predictions.csv"

    ),

    index=False

)

# ============================================================
# 17. Baseline Comparison Preview
# ============================================================

baseline_preview_df = pd.DataFrame({

    "Model": [

        "TF-IDF + SVM",

        "TextCNN",

        "STraT-Net E3"

    ],

    "Accuracy": [

        0.397778,

        cnn_accuracy,

        0.518667

    ],

    "Macro-F1": [

        0.397438,

        cnn_macro_f1,

        0.517134

    ],

    "Macro ROC-AUC": [

        0.721580,

        cnn_macro_roc_auc,

        0.813035

    ],

    "Macro PR-AUC": [

        0.399809,

        cnn_macro_pr_auc,

        0.542316

    ],

    "Error Rate (%)": [

        60.222222,

        cnn_error_rate,

        48.1333

    ]

})

print("\n" + "=" * 70)
print("BASELINE COMPARISON PREVIEW")
print("=" * 70)

display(
    baseline_preview_df
)

print("\n" + "=" * 70)
print("TEXTCNN BASELINE COMPLETED")
print("=" * 70)

#3) Base Line: BiLSTM

In [ ]:
# ============================================================
# STraT-Net Baseline B3
# BiLSTM
# Model + Training + Validation + Final Test
# ============================================================

import os
import json
import random
import numpy as np
import pandas as pd

import torch
import torch.nn as nn

from torch.optim import AdamW
from torch.optim.lr_scheduler import ReduceLROnPlateau

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report,
    roc_auc_score,
    average_precision_score
)

from sklearn.preprocessing import label_binarize

print("=" * 70)
print("Baseline B3 : BiLSTM")
print("Model + Training + Final Evaluation")
print("=" * 70)

# ============================================================
# 1. Reproducibility
# ============================================================

BILSTM_SEED = 42

random.seed(
    BILSTM_SEED
)

np.random.seed(
    BILSTM_SEED
)

torch.manual_seed(
    BILSTM_SEED
)

torch.cuda.manual_seed_all(
    BILSTM_SEED
)

torch.backends.cudnn.deterministic = True

torch.backends.cudnn.benchmark = False

# ============================================================
# 2. Configuration
# ============================================================

BILSTM_EMBED_DIM = 200

BILSTM_HIDDEN_DIM = 128

BILSTM_NUM_LAYERS = 1

BILSTM_DROPOUT = 0.40

BILSTM_EPOCHS = 30

BILSTM_LR = 1e-3

BILSTM_WEIGHT_DECAY = 1e-4

BILSTM_PATIENCE = 6

BILSTM_GRAD_CLIP = 1.0

BILSTM_DEVICE = torch.device(

    "cuda"
    if torch.cuda.is_available()
    else "cpu"

)

BILSTM_OUTPUT_DIR = (
    "outputs_baseline_bilstm"
)

os.makedirs(
    BILSTM_OUTPUT_DIR,
    exist_ok=True
)

BILSTM_BEST_PATH = os.path.join(

    BILSTM_OUTPUT_DIR,

    "best_bilstm.pth"

)

print("\nBiLSTM Configuration")
print("-" * 60)

print(
    "Embedding Dimension :",
    BILSTM_EMBED_DIM
)

print(
    "Hidden Dimension    :",
    BILSTM_HIDDEN_DIM
)

print(
    "LSTM Layers         :",
    BILSTM_NUM_LAYERS
)

print(
    "Bidirectional       :",
    True
)

print(
    "Dropout             :",
    BILSTM_DROPOUT
)

print(
    "Device              :",
    BILSTM_DEVICE
)

# ============================================================
# 3. BiLSTM Model
#
# Important:
# Uses packed sequences so padded tokens do not influence
# the recurrent representation.
# ============================================================

class BiLSTMClassifier(
    nn.Module
):

    def __init__(
        self,
        vocab_size,
        embed_dim,
        hidden_dim,
        num_layers,
        num_classes,
        dropout,
        padding_idx
    ):

        super().__init__()

        self.embedding = nn.Embedding(

            vocab_size,

            embed_dim,

            padding_idx=padding_idx

        )

        self.lstm = nn.LSTM(

            input_size=embed_dim,

            hidden_size=hidden_dim,

            num_layers=num_layers,

            batch_first=True,

            bidirectional=True,

            dropout=(
                dropout
                if num_layers > 1
                else 0.0
            )

        )

        self.dropout = nn.Dropout(
            dropout
        )

        self.classifier = nn.Sequential(

            nn.Linear(
                hidden_dim * 2,
                hidden_dim
            ),

            nn.GELU(),

            nn.Dropout(
                dropout
            ),

            nn.Linear(
                hidden_dim,
                num_classes
            )

        )

    def forward(
        self,
        input_ids,
        lengths
    ):

        # --------------------------------------------
        # Embedding
        # [B, L] -> [B, L, E]
        # --------------------------------------------

        x = self.embedding(
            input_ids
        )

        # --------------------------------------------
        # Pack valid tokens only
        # lengths must be on CPU
        # --------------------------------------------

        packed = nn.utils.rnn.pack_padded_sequence(

            x,

            lengths.cpu(),

            batch_first=True,

            enforce_sorted=False

        )

        _, (
            hidden,
            _
        ) = self.lstm(
            packed
        )

        # --------------------------------------------
        # Final hidden states:
        #
        # hidden[-2] = forward direction
        # hidden[-1] = backward direction
        # --------------------------------------------

        forward_hidden = hidden[
            -2
        ]

        backward_hidden = hidden[
            -1
        ]

        patient_embedding = torch.cat(

            [
                forward_hidden,
                backward_hidden
            ],

            dim=1

        )

        patient_embedding = self.dropout(
            patient_embedding
        )

        logits = self.classifier(
            patient_embedding
        )

        return logits

# ============================================================
# 4. Fresh Model
# ============================================================

bilstm_model = BiLSTMClassifier(

    vocab_size=(
        CNN_VOCAB_SIZE
    ),

    embed_dim=(
        BILSTM_EMBED_DIM
    ),

    hidden_dim=(
        BILSTM_HIDDEN_DIM
    ),

    num_layers=(
        BILSTM_NUM_LAYERS
    ),

    num_classes=(
        NUM_DISEASES
    ),

    dropout=(
        BILSTM_DROPOUT
    ),

    padding_idx=(
        PAD_IDX
    )

).to(
    BILSTM_DEVICE
)

total_params_bilstm = sum(

    p.numel()

    for p in bilstm_model.parameters()

)

trainable_params_bilstm = sum(

    p.numel()

    for p in bilstm_model.parameters()

    if p.requires_grad

)

print("\nParameter Statistics")
print("-" * 60)

print(
    f"Total Parameters     : "
    f"{total_params_bilstm:,}"
)

print(
    f"Trainable Parameters : "
    f"{trainable_params_bilstm:,}"
)

# ============================================================
# 5. Loss + Optimizer + Scheduler
# ============================================================

bilstm_criterion = nn.CrossEntropyLoss(

    weight=class_weights.to(
        BILSTM_DEVICE
    )

)

bilstm_optimizer = AdamW(

    bilstm_model.parameters(),

    lr=BILSTM_LR,

    weight_decay=(
        BILSTM_WEIGHT_DECAY
    )

)

bilstm_scheduler = ReduceLROnPlateau(

    bilstm_optimizer,

    mode="max",

    factor=0.5,

    patience=2

)

# ============================================================
# 6. Metric Helper
# ============================================================

def bilstm_metrics(
    y_true,
    y_pred
):

    return {

        "accuracy":
            accuracy_score(
                y_true,
                y_pred
            ),

        "precision":
            precision_score(
                y_true,
                y_pred,
                average="macro",
                zero_division=0
            ),

        "recall":
            recall_score(
                y_true,
                y_pred,
                average="macro",
                zero_division=0
            ),

        "macro_f1":
            f1_score(
                y_true,
                y_pred,
                average="macro",
                zero_division=0
            )

    }

# ============================================================
# 7. Train One Epoch
# ============================================================

def train_bilstm_epoch():

    bilstm_model.train()

    running_loss = 0.0

    all_labels = []

    all_predictions = []

    for batch in train_cnn_loader:

        input_ids = (
            batch[
                "input_ids"
            ]
            .long()
            .to(
                BILSTM_DEVICE
            )
        )

        lengths = (
            batch[
                "length"
            ]
            .long()
        )

        labels = (
            batch[
                "label"
            ]
            .long()
            .to(
                BILSTM_DEVICE
            )
        )

        bilstm_optimizer.zero_grad(
            set_to_none=True
        )

        logits = bilstm_model(

            input_ids,

            lengths

        )

        loss = bilstm_criterion(

            logits,

            labels

        )

        loss.backward()

        torch.nn.utils.clip_grad_norm_(

            bilstm_model.parameters(),

            BILSTM_GRAD_CLIP

        )

        bilstm_optimizer.step()

        running_loss += (
            loss.item()
        )

        predictions = torch.argmax(

            logits,

            dim=1

        )

        all_labels.extend(
            labels
            .detach()
            .cpu()
            .numpy()
            .tolist()
        )

        all_predictions.extend(
            predictions
            .detach()
            .cpu()
            .numpy()
            .tolist()
        )

    metrics = bilstm_metrics(

        all_labels,

        all_predictions

    )

    return (

        running_loss
        /
        len(
            train_cnn_loader
        ),

        metrics

    )

# ============================================================
# 8. Validation
# ============================================================

def validate_bilstm():

    bilstm_model.eval()

    running_loss = 0.0

    all_labels = []

    all_predictions = []

    with torch.no_grad():

        for batch in val_cnn_loader:

            input_ids = (
                batch[
                    "input_ids"
                ]
                .long()
                .to(
                    BILSTM_DEVICE
                )
            )

            lengths = (
                batch[
                    "length"
                ]
                .long()
            )

            labels = (
                batch[
                    "label"
                ]
                .long()
                .to(
                    BILSTM_DEVICE
                )
            )

            logits = bilstm_model(

                input_ids,

                lengths

            )

            loss = bilstm_criterion(

                logits,

                labels

            )

            running_loss += (
                loss.item()
            )

            predictions = torch.argmax(

                logits,

                dim=1

            )

            all_labels.extend(
                labels
                .cpu()
                .numpy()
                .tolist()
            )

            all_predictions.extend(
                predictions
                .cpu()
                .numpy()
                .tolist()
            )

    metrics = bilstm_metrics(

        all_labels,

        all_predictions

    )

    return (

        running_loss
        /
        len(
            val_cnn_loader
        ),

        metrics

    )

# ============================================================
# 9. Training Loop
# ============================================================

best_val_f1_bilstm = -1.0

best_epoch_bilstm = -1

bilstm_early_counter = 0

bilstm_history = []

for epoch in range(
    1,
    BILSTM_EPOCHS + 1
):

    train_loss, train_metrics = (
        train_bilstm_epoch()
    )

    val_loss, val_metrics = (
        validate_bilstm()
    )

    current_val_f1 = (
        val_metrics[
            "macro_f1"
        ]
    )

    bilstm_scheduler.step(
        current_val_f1
    )

    current_lr = (
        bilstm_optimizer
        .param_groups[0]["lr"]
    )

    bilstm_history.append({

        "epoch":
            epoch,

        "train_loss":
            train_loss,

        "val_loss":
            val_loss,

        "train_accuracy":
            train_metrics[
                "accuracy"
            ],

        "val_accuracy":
            val_metrics[
                "accuracy"
            ],

        "train_macro_f1":
            train_metrics[
                "macro_f1"
            ],

        "val_macro_f1":
            val_metrics[
                "macro_f1"
            ],

        "learning_rate":
            current_lr

    })

    print(

        f"Epoch {epoch:02d} | "
        f"Train Loss {train_loss:.4f} | "
        f"Val Loss {val_loss:.4f} | "
        f"Train F1 {train_metrics['macro_f1']:.4f} | "
        f"Val F1 {current_val_f1:.4f} | "
        f"LR {current_lr:.6f}"

    )

    if current_val_f1 > best_val_f1_bilstm:

        best_val_f1_bilstm = (
            current_val_f1
        )

        best_epoch_bilstm = (
            epoch
        )

        bilstm_early_counter = 0

        torch.save(

            {

                "epoch":
                    epoch,

                "val_macro_f1":
                    current_val_f1,

                "model_state_dict":
                    bilstm_model.state_dict(),

                "vocab_size":
                    CNN_VOCAB_SIZE,

                "max_len":
                    CNN_MAX_LEN,

                "embedding_dim":
                    BILSTM_EMBED_DIM,

                "hidden_dim":
                    BILSTM_HIDDEN_DIM

            },

            BILSTM_BEST_PATH

        )

    else:

        bilstm_early_counter += 1

    if (
        bilstm_early_counter
        >=
        BILSTM_PATIENCE
    ):

        print(
            "\nEarly stopping triggered."
        )

        break

# ============================================================
# 10. Save Training History
# ============================================================

bilstm_history_df = pd.DataFrame(
    bilstm_history
)

bilstm_history_df.to_csv(

    os.path.join(

        BILSTM_OUTPUT_DIR,

        "BiLSTM_training_history.csv"

    ),

    index=False

)

print("\nBest Epoch:", best_epoch_bilstm)

print(
    "Best Validation Macro-F1:",
    f"{best_val_f1_bilstm:.4f}"
)

# ============================================================
# 11. Load Best Checkpoint
# ============================================================

bilstm_checkpoint = torch.load(

    BILSTM_BEST_PATH,

    map_location=(
        BILSTM_DEVICE
    )

)

bilstm_model.load_state_dict(

    bilstm_checkpoint[
        "model_state_dict"
    ]

)

bilstm_model.eval()

# ============================================================
# 12. Final Test Inference
# ============================================================

bilstm_test_labels = []

bilstm_test_predictions = []

bilstm_test_probabilities = []

bilstm_test_patient_ids = []

with torch.no_grad():

    for batch in test_cnn_loader:

        input_ids = (
            batch[
                "input_ids"
            ]
            .long()
            .to(
                BILSTM_DEVICE
            )
        )

        lengths = (
            batch[
                "length"
            ]
            .long()
        )

        labels = (
            batch[
                "label"
            ]
            .long()
            .to(
                BILSTM_DEVICE
            )
        )

        logits = bilstm_model(

            input_ids,

            lengths

        )

        probabilities = torch.softmax(

            logits,

            dim=1

        )

        predictions = torch.argmax(

            probabilities,

            dim=1

        )

        bilstm_test_labels.extend(
            labels
            .cpu()
            .numpy()
            .tolist()
        )

        bilstm_test_predictions.extend(
            predictions
            .cpu()
            .numpy()
            .tolist()
        )

        bilstm_test_probabilities.extend(
            probabilities
            .cpu()
            .numpy()
        )

        bilstm_test_patient_ids.extend(

            list(
                batch[
                    "patient_id"
                ]
            )

        )

y_true_bilstm = np.asarray(
    bilstm_test_labels
)

y_pred_bilstm = np.asarray(
    bilstm_test_predictions
)

y_prob_bilstm = np.asarray(
    bilstm_test_probabilities
)

# ============================================================
# 13. Overall Metrics
# ============================================================

bilstm_accuracy = accuracy_score(

    y_true_bilstm,

    y_pred_bilstm

)

bilstm_macro_precision = precision_score(

    y_true_bilstm,

    y_pred_bilstm,

    average="macro",

    zero_division=0

)

bilstm_macro_recall = recall_score(

    y_true_bilstm,

    y_pred_bilstm,

    average="macro",

    zero_division=0

)

bilstm_macro_f1 = f1_score(

    y_true_bilstm,

    y_pred_bilstm,

    average="macro",

    zero_division=0

)

bilstm_error_rate = (

    1.0
    -
    bilstm_accuracy

) * 100

bilstm_y_binary = label_binarize(

    y_true_bilstm,

    classes=np.arange(
        NUM_DISEASES
    )

)

bilstm_macro_roc_auc = roc_auc_score(

    bilstm_y_binary,

    y_prob_bilstm,

    average="macro",

    multi_class="ovr"

)

bilstm_macro_pr_auc = average_precision_score(

    bilstm_y_binary,

    y_prob_bilstm,

    average="macro"

)

print("\n" + "=" * 70)
print("BILSTM FINAL TEST METRICS")
print("=" * 70)

print(
    f"Accuracy        : {bilstm_accuracy:.4f}"
)

print(
    f"Macro Precision : {bilstm_macro_precision:.4f}"
)

print(
    f"Macro Recall    : {bilstm_macro_recall:.4f}"
)

print(
    f"Macro F1        : {bilstm_macro_f1:.4f}"
)

print(
    f"Macro ROC-AUC   : {bilstm_macro_roc_auc:.4f}"
)

print(
    f"Macro PR-AUC    : {bilstm_macro_pr_auc:.4f}"
)

print(
    f"Error Rate (%)  : {bilstm_error_rate:.2f}"
)

# ============================================================
# 14. Confusion Matrix
# ============================================================

cm_bilstm = confusion_matrix(

    y_true_bilstm,

    y_pred_bilstm

)

print("\nConfusion Matrix")
print("-" * 60)

print(
    cm_bilstm
)

# ============================================================
# 15. Classification Report
# ============================================================

class_names = list(
    label_encoder.classes_
)

print("\nClassification Report")
print("-" * 60)

print(

    classification_report(

        y_true_bilstm,

        y_pred_bilstm,

        target_names=class_names,

        digits=4,

        zero_division=0

    )

)

# ============================================================
# 16. Per-Disease Metrics
# ============================================================

bilstm_per_disease_rows = []

for class_index, disease_name in enumerate(
    class_names
):

    y_true_binary = (
        y_true_bilstm
        ==
        class_index
    ).astype(int)

    y_pred_binary = (
        y_pred_bilstm
        ==
        class_index
    ).astype(int)

    disease_f1 = f1_score(

        y_true_binary,

        y_pred_binary,

        zero_division=0

    )

    disease_recall = recall_score(

        y_true_binary,

        y_pred_binary,

        zero_division=0

    )

    disease_auc = roc_auc_score(

        y_true_binary,

        y_prob_bilstm[
            :,
            class_index
        ]

    )

    bilstm_per_disease_rows.append({

        "Neurological Disease":
            disease_name,

        "F1-score":
            disease_f1,

        "One-vs-Rest AUC":
            disease_auc,

        "Misdiagnosis Rate (%)":
            (
                1
                -
                disease_recall
            )
            *
            100

    })

bilstm_per_disease_df = pd.DataFrame(
    bilstm_per_disease_rows
)

print("\nPer-Disease Results")
print("-" * 60)

display(
    bilstm_per_disease_df
)

# ============================================================
# 17. Save Summary
# ============================================================

bilstm_summary = {

    "Model":
        "BiLSTM",

    "Best_Epoch":
        int(
            best_epoch_bilstm
        ),

    "Best_Validation_Macro_F1":
        float(
            best_val_f1_bilstm
        ),

    "Test_Accuracy":
        float(
            bilstm_accuracy
        ),

    "Test_Macro_Precision":
        float(
            bilstm_macro_precision
        ),

    "Test_Macro_Recall":
        float(
            bilstm_macro_recall
        ),

    "Test_Macro_F1":
        float(
            bilstm_macro_f1
        ),

    "Test_Macro_ROC_AUC":
        float(
            bilstm_macro_roc_auc
        ),

    "Test_Macro_PR_AUC":
        float(
            bilstm_macro_pr_auc
        ),

    "Error_Rate_Percent":
        float(
            bilstm_error_rate
        )

}

with open(

    os.path.join(

        BILSTM_OUTPUT_DIR,

        "BiLSTM_summary.json"

    ),

    "w"

) as f:

    json.dump(
        bilstm_summary,
        f,
        indent=4
    )

bilstm_per_disease_df.to_csv(

    os.path.join(

        BILSTM_OUTPUT_DIR,

        "BiLSTM_per_disease_metrics.csv"

    ),

    index=False

)

bilstm_predictions_df = pd.DataFrame({

    "patient_id":
        bilstm_test_patient_ids,

    "true_label":
        y_true_bilstm,

    "predicted_label":
        y_pred_bilstm,

    "correct":
        (
            y_true_bilstm
            ==
            y_pred_bilstm
        )

})

for i, disease_name in enumerate(
    class_names
):

    safe_name = (
        disease_name
        .lower()
        .replace(
            " ",
            "_"
        )
        .replace(
            "'",
            ""
        )
    )

    bilstm_predictions_df[
        f"prob_{safe_name}"
    ] = y_prob_bilstm[
        :,
        i
    ]

bilstm_predictions_df.to_csv(

    os.path.join(

        BILSTM_OUTPUT_DIR,

        "BiLSTM_test_predictions.csv"

    ),

    index=False

)

# ============================================================
# 18. Baseline Comparison Preview
# ============================================================

baseline_preview_df = pd.DataFrame({

    "Model": [

        "TF-IDF + SVM",

        "TextCNN",

        "BiLSTM",

        "STraT-Net E3"

    ],

    "Accuracy": [

        0.397778,

        0.384444,

        bilstm_accuracy,

        0.518667

    ],

    "Macro-F1": [

        0.397438,

        0.377259,

        bilstm_macro_f1,

        0.517134

    ],

    "Macro ROC-AUC": [

        0.721580,

        0.718228,

        bilstm_macro_roc_auc,

        0.813035

    ],

    "Macro PR-AUC": [

        0.399809,

        0.397138,

        bilstm_macro_pr_auc,

        0.542316

    ],

    "Error Rate (%)": [

        60.222222,

        61.555556,

        bilstm_error_rate,

        48.1333

    ]

})

print("\n" + "=" * 70)
print("BASELINE COMPARISON PREVIEW")
print("=" * 70)

display(
    baseline_preview_df
)

print("\n" + "=" * 70)
print("BILSTM BASELINE COMPLETED")
print("=" * 70)

#4)Baseline 4: ClinicalBERT

In [ ]:
# ============================================================
# ClinicalBERT
# FIXED Step 1
# Compatible Chunked Patient-Level Text Pipeline
# ============================================================

import os
import random
import numpy as np
import pandas as pd

import torch
from torch.utils.data import Dataset, DataLoader

from transformers import AutoTokenizer

print("=" * 70)
print("Baseline B4 : ClinicalBERT")
print("FIXED Chunked Patient-Level Text Pipeline")
print("=" * 70)

# ============================================================
# 1. Reproducibility
# ============================================================

CLINICALBERT_SEED = 42

random.seed(
    CLINICALBERT_SEED
)

np.random.seed(
    CLINICALBERT_SEED
)

torch.manual_seed(
    CLINICALBERT_SEED
)

torch.cuda.manual_seed_all(
    CLINICALBERT_SEED
)

torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

# ============================================================
# 2. Configuration
# ============================================================

CLINICALBERT_MODEL = (
    "emilyalsentzer/Bio_ClinicalBERT"
)

CLINICALBERT_MAX_LEN = 512

CLINICALBERT_BATCH_SIZE = 4

CLINICALBERT_DEVICE = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)

print("\nConfiguration")
print("-" * 60)

print(
    "Model:",
    CLINICALBERT_MODEL
)

print(
    "Maximum chunk length:",
    CLINICALBERT_MAX_LEN
)

print(
    "Batch size:",
    CLINICALBERT_BATCH_SIZE
)

print(
    "Device:",
    CLINICALBERT_DEVICE
)

# ============================================================
# 3. Load Tokenizer
# ============================================================

print(
    "\nLoading ClinicalBERT tokenizer..."
)

clinicalbert_tokenizer = (
    AutoTokenizer.from_pretrained(
        CLINICALBERT_MODEL,
        use_fast=False
    )
)

print(
    "✓ Tokenizer loaded:"
)

print(
    type(
        clinicalbert_tokenizer
    ).__name__
)

print(
    "CLS token ID:",
    clinicalbert_tokenizer.cls_token_id
)

print(
    "SEP token ID:",
    clinicalbert_tokenizer.sep_token_id
)

print(
    "PAD token ID:",
    clinicalbert_tokenizer.pad_token_id
)

# ============================================================
# 4. Corrected Chunking Function
#
# We:
# 1. Encode document WITHOUT special tokens.
# 2. Split into chunks of max_len - 2.
# 3. Manually add [CLS] and [SEP].
# 4. Pad to 512.
#
# This avoids prepare_for_model().
# ============================================================

def chunk_patient_document(
    text,
    tokenizer,
    max_len=512
):

    # --------------------------------------------
    # Convert whole document to token IDs
    # without CLS/SEP
    # --------------------------------------------

    token_ids = tokenizer.encode(
        str(text),
        add_special_tokens=False
    )

    # Reserve:
    # 1 position for [CLS]
    # 1 position for [SEP]

    chunk_content_length = (
        max_len - 2
    )

    chunks = []

    # --------------------------------------------
    # Split longitudinal document
    # --------------------------------------------

    for start in range(
        0,
        len(token_ids),
        chunk_content_length
    ):

        content_ids = token_ids[
            start:
            start + chunk_content_length
        ]

        # ----------------------------------------
        # Add BERT special tokens manually
        # ----------------------------------------

        chunk_ids = (

            [
                tokenizer.cls_token_id
            ]

            +

            content_ids

            +

            [
                tokenizer.sep_token_id
            ]

        )

        # ----------------------------------------
        # Attention mask before padding
        # ----------------------------------------

        attention_mask = [

            1

            for _ in chunk_ids

        ]

        # ----------------------------------------
        # Padding
        # ----------------------------------------

        padding_length = (
            max_len
            -
            len(chunk_ids)
        )

        if padding_length > 0:

            chunk_ids = (

                chunk_ids

                +

                [
                    tokenizer.pad_token_id
                ]
                *
                padding_length

            )

            attention_mask = (

                attention_mask

                +

                [0]
                *
                padding_length

            )

        # Safety checks

        assert len(
            chunk_ids
        ) == max_len

        assert len(
            attention_mask
        ) == max_len

        chunks.append({

            "input_ids":
                torch.tensor(
                    chunk_ids,
                    dtype=torch.long
                ),

            "attention_mask":
                torch.tensor(
                    attention_mask,
                    dtype=torch.long
                )

        })

    # --------------------------------------------
    # Safety fallback for empty text
    # --------------------------------------------

    if len(chunks) == 0:

        chunk_ids = [

            tokenizer.cls_token_id,

            tokenizer.sep_token_id

        ]

        attention_mask = [
            1,
            1
        ]

        padding_length = (
            max_len
            -
            len(chunk_ids)
        )

        chunk_ids += (

            [
                tokenizer.pad_token_id
            ]

            *
            padding_length

        )

        attention_mask += (

            [0]

            *
            padding_length

        )

        chunks.append({

            "input_ids":
                torch.tensor(
                    chunk_ids,
                    dtype=torch.long
                ),

            "attention_mask":
                torch.tensor(
                    attention_mask,
                    dtype=torch.long
                )

        })

    return chunks

# ============================================================
# 5. Test Chunking BEFORE Building DataLoaders
# ============================================================

test_document = train_text_df.loc[
    0,
    "patient_document"
]

test_chunks = chunk_patient_document(

    test_document,

    clinicalbert_tokenizer,

    CLINICALBERT_MAX_LEN

)

print("\nChunking Verification")
print("-" * 60)

print(
    "Example patient:",
    train_text_df.loc[
        0,
        "patient_id"
    ]
)

print(
    "Number of chunks:",
    len(
        test_chunks
    )
)

print(
    "First chunk input shape:",
    test_chunks[
        0
    ][
        "input_ids"
    ].shape
)

print(
    "First chunk mask shape:",
    test_chunks[
        0
    ][
        "attention_mask"
    ].shape
)

print(
    "Valid tokens in first chunk:",
    int(
        test_chunks[
            0
        ][
            "attention_mask"
        ].sum()
    )
)

print(
    "First token is CLS:",
    int(
        test_chunks[
            0
        ][
            "input_ids"
        ][0]
    )
    ==
    clinicalbert_tokenizer.cls_token_id
)

# ============================================================
# 6. Patient Dataset
# ============================================================

class ClinicalBERTPatientDataset(
    Dataset
):

    def __init__(
        self,
        dataframe,
        tokenizer,
        max_len
    ):

        self.df = dataframe.reset_index(
            drop=True
        )

        self.tokenizer = tokenizer

        self.max_len = max_len

    def __len__(
        self
    ):

        return len(
            self.df
        )

    def __getitem__(
        self,
        idx
    ):

        row = self.df.iloc[
            idx
        ]

        chunks = chunk_patient_document(

            row[
                "patient_document"
            ],

            self.tokenizer,

            self.max_len

        )

        input_ids = torch.stack([

            chunk[
                "input_ids"
            ]

            for chunk in chunks

        ])

        attention_mask = torch.stack([

            chunk[
                "attention_mask"
            ]

            for chunk in chunks

        ])

        return {

            "patient_id":
                row[
                    "patient_id"
                ],

            "input_ids":
                input_ids,

            "attention_mask":
                attention_mask,

            "num_chunks":
                torch.tensor(
                    len(
                        chunks
                    ),
                    dtype=torch.long
                ),

            "label":
                torch.tensor(

                    int(
                        row[
                            "Disease_Label"
                        ]
                    ),

                    dtype=torch.long

                )

        }

# ============================================================
# 7. Custom Collate Function
# ============================================================

def clinicalbert_collate_fn(
    batch
):

    batch_size = len(
        batch
    )

    max_chunks = max(

        int(
            item[
                "num_chunks"
            ].item()
        )

        for item in batch

    )

    seq_len = (
        CLINICALBERT_MAX_LEN
    )

    # Use tokenizer PAD id rather than zero assumption

    pad_id = (
        clinicalbert_tokenizer
        .pad_token_id
    )

    input_ids = torch.full(

        (
            batch_size,
            max_chunks,
            seq_len
        ),

        fill_value=pad_id,

        dtype=torch.long

    )

    attention_mask = torch.zeros(

        (
            batch_size,
            max_chunks,
            seq_len
        ),

        dtype=torch.long

    )

    chunk_mask = torch.zeros(

        (
            batch_size,
            max_chunks
        ),

        dtype=torch.bool

    )

    labels = []

    patient_ids = []

    num_chunks = []

    for i, item in enumerate(
        batch
    ):

        n = int(
            item[
                "num_chunks"
            ].item()
        )

        input_ids[
            i,
            :n
        ] = item[
            "input_ids"
        ]

        attention_mask[
            i,
            :n
        ] = item[
            "attention_mask"
        ]

        chunk_mask[
            i,
            :n
        ] = True

        labels.append(
            item[
                "label"
            ]
        )

        patient_ids.append(
            item[
                "patient_id"
            ]
        )

        num_chunks.append(
            n
        )

    return {

        "patient_id":
            patient_ids,

        "input_ids":
            input_ids,

        "attention_mask":
            attention_mask,

        "chunk_mask":
            chunk_mask,

        "num_chunks":
            torch.tensor(
                num_chunks,
                dtype=torch.long
            ),

        "label":
            torch.stack(
                labels
            )

    }

# ============================================================
# 8. Create Datasets
# ============================================================

train_cb_dataset = (
    ClinicalBERTPatientDataset(

        train_text_df,

        clinicalbert_tokenizer,

        CLINICALBERT_MAX_LEN

    )
)

val_cb_dataset = (
    ClinicalBERTPatientDataset(

        val_text_df,

        clinicalbert_tokenizer,

        CLINICALBERT_MAX_LEN

    )
)

test_cb_dataset = (
    ClinicalBERTPatientDataset(

        test_text_df,

        clinicalbert_tokenizer,

        CLINICALBERT_MAX_LEN

    )
)

# ============================================================
# 9. Create DataLoaders
# ============================================================

train_cb_loader = DataLoader(

    train_cb_dataset,

    batch_size=(
        CLINICALBERT_BATCH_SIZE
    ),

    shuffle=True,

    num_workers=0,

    pin_memory=torch.cuda.is_available(),

    collate_fn=(
        clinicalbert_collate_fn
    )

)

val_cb_loader = DataLoader(

    val_cb_dataset,

    batch_size=(
        CLINICALBERT_BATCH_SIZE
    ),

    shuffle=False,

    num_workers=0,

    pin_memory=torch.cuda.is_available(),

    collate_fn=(
        clinicalbert_collate_fn
    )

)

test_cb_loader = DataLoader(

    test_cb_dataset,

    batch_size=(
        CLINICALBERT_BATCH_SIZE
    ),

    shuffle=False,

    num_workers=0,

    pin_memory=torch.cuda.is_available(),

    collate_fn=(
        clinicalbert_collate_fn
    )

)

# ============================================================
# 10. Batch Verification
# ============================================================

sample_cb = next(
    iter(
        train_cb_loader
    )
)

print("\nBatch Verification")
print("-" * 60)

print(
    "Input IDs Shape      :",
    sample_cb[
        "input_ids"
    ].shape
)

print(
    "Attention Mask Shape :",
    sample_cb[
        "attention_mask"
    ].shape
)

print(
    "Chunk Mask Shape     :",
    sample_cb[
        "chunk_mask"
    ].shape
)

print(
    "Labels Shape         :",
    sample_cb[
        "label"
    ].shape
)

print(
    "Chunks per patient   :",
    sample_cb[
        "num_chunks"
    ].tolist()
)

print(
    "Train Batches        :",
    len(
        train_cb_loader
    )
)

print(
    "Validation Batches   :",
    len(
        val_cb_loader
    )
)

print(
    "Test Batches         :",
    len(
        test_cb_loader
    )
)

# ============================================================
# 11. Strong Assertions
# ============================================================

assert (
    sample_cb[
        "input_ids"
    ].shape[-1]
    ==
    512
)

assert (
    sample_cb[
        "attention_mask"
    ].shape
    ==
    sample_cb[
        "input_ids"
    ].shape
)

assert (
    sample_cb[
        "chunk_mask"
    ].shape[0]
    ==
    sample_cb[
        "input_ids"
    ].shape[0]
)

assert (
    sample_cb[
        "label"
    ].shape[0]
    ==
    sample_cb[
        "input_ids"
    ].shape[0]
)

print(
    "\n✓ ClinicalBERT chunking fixed."
)

print(
    "✓ Patient-level batches verified."
)

print("\n" + "=" * 70)
print("B4 FIXED Step 1 Completed Successfully")
print("=" * 70)

In [ ]:
CLINICALBERT_BATCH_SIZE = 2

In [ ]:
train_cb_loader = DataLoader(
    train_cb_dataset,
    batch_size=CLINICALBERT_BATCH_SIZE,
    shuffle=True,
    num_workers=0,
    pin_memory=torch.cuda.is_available(),
    collate_fn=clinicalbert_collate_fn
)

val_cb_loader = DataLoader(
    val_cb_dataset,
    batch_size=CLINICALBERT_BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    pin_memory=torch.cuda.is_available(),
    collate_fn=clinicalbert_collate_fn
)

test_cb_loader = DataLoader(
    test_cb_dataset,
    batch_size=CLINICALBERT_BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    pin_memory=torch.cuda.is_available(),
    collate_fn=clinicalbert_collate_fn
)

In [ ]:
print("Train batches:", len(train_cb_loader))
print("Val batches  :", len(val_cb_loader))
print("Test batches :", len(test_cb_loader))

In [ ]:
# ============================================================
# STraT-Net Research
# Baseline B4 — ClinicalBERT
#
# FINAL VERSION
#
# Patient-Level Chunked ClinicalBERT
# Training + Validation + Held-Out Test Evaluation
#
# Assumes these already exist:
#
# train_cb_loader
# val_cb_loader
# test_cb_loader
#
# CLINICALBERT_MODEL
# CLINICALBERT_DEVICE
#
# NUM_DISEASES
# class_weights
# label_encoder
#
# Verified batch size = 2
# ============================================================

import os
import gc
import json
import random
import warnings

import numpy as np
import pandas as pd

import torch
import torch.nn as nn

from transformers import AutoModel

from torch.optim import AdamW
from torch.optim.lr_scheduler import ReduceLROnPlateau

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report,
    roc_auc_score,
    average_precision_score
)

from sklearn.preprocessing import label_binarize

warnings.filterwarnings("ignore")

print("=" * 70)
print("Baseline B4 : ClinicalBERT")
print("FINAL TRAINING + HELD-OUT TEST EVALUATION")
print("=" * 70)


# ============================================================
# 1. Reproducibility
# ============================================================

CB_SEED = 42

random.seed(
    CB_SEED
)

np.random.seed(
    CB_SEED
)

torch.manual_seed(
    CB_SEED
)

torch.cuda.manual_seed_all(
    CB_SEED
)

os.environ[
    "PYTHONHASHSEED"
] = str(
    CB_SEED
)

torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False


# ============================================================
# 2. Clear GPU Cache Before Loading ClinicalBERT
# ============================================================

gc.collect()

if torch.cuda.is_available():

    torch.cuda.empty_cache()


# ============================================================
# 3. Training Configuration
#
# Batch size remains controlled by your already-created
# DataLoaders = 2 patients per batch.
# ============================================================

CB_EPOCHS = 12

CB_LEARNING_RATE = 2e-5

CB_WEIGHT_DECAY = 0.01

CB_DROPOUT = 0.30

CB_PATIENCE = 4

CB_GRAD_CLIP = 1.0

CB_DEVICE = CLINICALBERT_DEVICE

CB_OUTPUT_DIR = (
    "outputs_baseline_clinicalbert"
)

CB_CHECKPOINT_DIR = (
    "checkpoints_baseline_clinicalbert"
)

os.makedirs(
    CB_OUTPUT_DIR,
    exist_ok=True
)

os.makedirs(
    CB_CHECKPOINT_DIR,
    exist_ok=True
)

CB_BEST_MODEL_PATH = os.path.join(

    CB_CHECKPOINT_DIR,

    "best_clinicalbert_patient_model.pth"

)

CB_LAST_MODEL_PATH = os.path.join(

    CB_CHECKPOINT_DIR,

    "last_clinicalbert_patient_model.pth"

)

print("\nTraining Configuration")
print("-" * 60)

print(
    "Model                :",
    CLINICALBERT_MODEL
)

print(
    "Epochs               :",
    CB_EPOCHS
)

print(
    "Learning Rate        :",
    CB_LEARNING_RATE
)

print(
    "Weight Decay         :",
    CB_WEIGHT_DECAY
)

print(
    "Dropout              :",
    CB_DROPOUT
)

print(
    "Gradient Clip        :",
    CB_GRAD_CLIP
)

print(
    "Early Stop Patience  :",
    CB_PATIENCE
)

print(
    "Device               :",
    CB_DEVICE
)

print(
    "Train Batches        :",
    len(
        train_cb_loader
    )
)

print(
    "Validation Batches   :",
    len(
        val_cb_loader
    )
)

print(
    "Test Batches         :",
    len(
        test_cb_loader
    )
)


# ============================================================
# 4. ClinicalBERT Patient-Level Classifier
#
# Each patient:
#
# text
#   ↓
# multiple 512-token chunks
#   ↓
# ClinicalBERT [CLS] embedding per valid chunk
#   ↓
# mean aggregation across chunks
#   ↓
# patient-level disease classifier
#
# IMPORTANT:
# Only valid chunks are passed through BERT.
# Padded chunk slots are skipped entirely.
# ============================================================

class ClinicalBERTPatientClassifier(
    nn.Module
):

    def __init__(
        self,
        model_name,
        num_classes,
        dropout
    ):

        super().__init__()

        self.encoder = (
            AutoModel.from_pretrained(
                model_name
            )
        )

        hidden_size = int(
            self.encoder.config.hidden_size
        )

        self.dropout = nn.Dropout(
            dropout
        )

        self.classifier = nn.Sequential(

            nn.Linear(
                hidden_size,
                256
            ),

            nn.GELU(),

            nn.Dropout(
                dropout
            ),

            nn.Linear(
                256,
                num_classes
            )

        )

    def forward(
        self,
        input_ids,
        attention_mask,
        chunk_mask
    ):

        # --------------------------------------------
        # Shapes:
        #
        # input_ids:
        # [B, C, 512]
        #
        # chunk_mask:
        # [B, C]
        # --------------------------------------------

        batch_size = (
            input_ids.size(0)
        )

        num_chunks = (
            input_ids.size(1)
        )

        seq_len = (
            input_ids.size(2)
        )

        # --------------------------------------------
        # Flatten patient/chunk dimensions
        # --------------------------------------------

        flat_input_ids = (
            input_ids.reshape(
                batch_size
                *
                num_chunks,
                seq_len
            )
        )

        flat_attention_mask = (
            attention_mask.reshape(
                batch_size
                *
                num_chunks,
                seq_len
            )
        )

        flat_chunk_mask = (
            chunk_mask.reshape(
                -1
            )
        )

        # --------------------------------------------
        # Process ONLY real chunks
        # --------------------------------------------

        valid_input_ids = (
            flat_input_ids[
                flat_chunk_mask
            ]
        )

        valid_attention_mask = (
            flat_attention_mask[
                flat_chunk_mask
            ]
        )

        outputs = self.encoder(

            input_ids=(
                valid_input_ids
            ),

            attention_mask=(
                valid_attention_mask
            )

        )

        # --------------------------------------------
        # [CLS] embedding for each valid chunk
        # --------------------------------------------

        valid_cls = (
            outputs
            .last_hidden_state[
                :,
                0,
                :
            ]
        )

        hidden_size = (
            valid_cls.size(
                -1
            )
        )

        # --------------------------------------------
        # Restore padded chunk representation
        # --------------------------------------------

        flat_chunk_embeddings = torch.zeros(

            (
                batch_size
                *
                num_chunks,

                hidden_size
            ),

            device=(
                valid_cls.device
            ),

            dtype=(
                valid_cls.dtype
            )

        )

        flat_chunk_embeddings[
            flat_chunk_mask
        ] = valid_cls

        chunk_embeddings = (
            flat_chunk_embeddings.reshape(

                batch_size,

                num_chunks,

                hidden_size

            )
        )

        # --------------------------------------------
        # Mean pooling over valid chunks only
        # --------------------------------------------

        chunk_weights = (

            chunk_mask
            .unsqueeze(-1)
            .to(
                chunk_embeddings.dtype
            )

        )

        patient_embedding = (

            (
                chunk_embeddings
                *
                chunk_weights
            )
            .sum(
                dim=1
            )

            /

            chunk_weights
            .sum(
                dim=1
            )
            .clamp(
                min=1.0
            )

        )

        patient_embedding = self.dropout(
            patient_embedding
        )

        logits = self.classifier(
            patient_embedding
        )

        return logits


# ============================================================
# 5. Create Fresh ClinicalBERT Model
# ============================================================

print(
    "\nLoading pretrained ClinicalBERT..."
)

clinicalbert_model = (

    ClinicalBERTPatientClassifier(

        model_name=(
            CLINICALBERT_MODEL
        ),

        num_classes=(
            NUM_DISEASES
        ),

        dropout=(
            CB_DROPOUT
        )

    )

    .to(
        CB_DEVICE
    )

)

print(
    "✓ ClinicalBERT model loaded."
)


# ============================================================
# 6. Parameter Statistics
# ============================================================

cb_total_params = sum(

    p.numel()

    for p in clinicalbert_model.parameters()

)

cb_trainable_params = sum(

    p.numel()

    for p in clinicalbert_model.parameters()

    if p.requires_grad

)

print("\nParameter Statistics")
print("-" * 60)

print(
    f"Total Parameters     : "
    f"{cb_total_params:,}"
)

print(
    f"Trainable Parameters : "
    f"{cb_trainable_params:,}"
)


# ============================================================
# 7. Loss Function
# ============================================================

cb_criterion = nn.CrossEntropyLoss(

    weight=(
        class_weights
        .detach()
        .clone()
        .to(
            CB_DEVICE
        )
    )

)

print(
    "\n✓ CrossEntropyLoss created."
)


# ============================================================
# 8. Optimizer
# ============================================================

cb_optimizer = AdamW(

    clinicalbert_model.parameters(),

    lr=CB_LEARNING_RATE,

    weight_decay=(
        CB_WEIGHT_DECAY
    )

)

print(
    "✓ AdamW optimizer created."
)


# ============================================================
# 9. Scheduler
# ============================================================

cb_scheduler = ReduceLROnPlateau(

    cb_optimizer,

    mode="max",

    factor=0.5,

    patience=1,

    min_lr=1e-7

)

print(
    "✓ ReduceLROnPlateau scheduler created."
)


# ============================================================
# 10. Mixed Precision
# ============================================================

CB_USE_AMP = (
    torch.cuda.is_available()
)

if CB_USE_AMP:

    try:

        cb_scaler = (
            torch.amp.GradScaler(
                "cuda"
            )
        )

    except Exception:

        cb_scaler = (
            torch.cuda.amp.GradScaler()
        )

else:

    cb_scaler = None

print(
    "Mixed Precision:",
    CB_USE_AMP
)


# ============================================================
# 11. AMP Context Helper
# ============================================================

def cb_autocast():

    if CB_USE_AMP:

        try:

            return torch.autocast(

                device_type="cuda",

                dtype=torch.float16

            )

        except Exception:

            return torch.cuda.amp.autocast()

    return torch.autocast(

        device_type="cpu",

        enabled=False

    )


# ============================================================
# 12. Metric Function
# ============================================================

def compute_cb_metrics(
    y_true,
    y_pred
):

    return {

        "accuracy":
            accuracy_score(
                y_true,
                y_pred
            ),

        "macro_precision":
            precision_score(

                y_true,

                y_pred,

                average="macro",

                zero_division=0

            ),

        "macro_recall":
            recall_score(

                y_true,

                y_pred,

                average="macro",

                zero_division=0

            ),

        "macro_f1":
            f1_score(

                y_true,

                y_pred,

                average="macro",

                zero_division=0

            )

    }


# ============================================================
# 13. Train One Epoch
# ============================================================

def train_clinicalbert_epoch(
    epoch
):

    clinicalbert_model.train()

    running_loss = 0.0

    all_labels = []

    all_predictions = []

    for batch_index, batch in enumerate(
        train_cb_loader
    ):

        input_ids = (

            batch[
                "input_ids"
            ]

            .long()

            .to(
                CB_DEVICE,
                non_blocking=True
            )

        )

        attention_mask = (

            batch[
                "attention_mask"
            ]

            .long()

            .to(
                CB_DEVICE,
                non_blocking=True
            )

        )

        chunk_mask = (

            batch[
                "chunk_mask"
            ]

            .bool()

            .to(
                CB_DEVICE,
                non_blocking=True
            )

        )

        labels = (

            batch[
                "label"
            ]

            .long()

            .to(
                CB_DEVICE,
                non_blocking=True
            )

        )

        cb_optimizer.zero_grad(
            set_to_none=True
        )

        # --------------------------------------------
        # Forward with mixed precision
        # --------------------------------------------

        with cb_autocast():

            logits = clinicalbert_model(

                input_ids,

                attention_mask,

                chunk_mask

            )

            loss = cb_criterion(

                logits,

                labels

            )

        # --------------------------------------------
        # Backward
        # --------------------------------------------

        if cb_scaler is not None:

            cb_scaler.scale(
                loss
            ).backward()

            cb_scaler.unscale_(
                cb_optimizer
            )

            torch.nn.utils.clip_grad_norm_(

                clinicalbert_model.parameters(),

                CB_GRAD_CLIP

            )

            cb_scaler.step(
                cb_optimizer
            )

            cb_scaler.update()

        else:

            loss.backward()

            torch.nn.utils.clip_grad_norm_(

                clinicalbert_model.parameters(),

                CB_GRAD_CLIP

            )

            cb_optimizer.step()

        running_loss += float(
            loss.item()
        )

        predictions = torch.argmax(

            logits,

            dim=1

        )

        all_labels.extend(

            labels
            .detach()
            .cpu()
            .numpy()
            .tolist()

        )

        all_predictions.extend(

            predictions
            .detach()
            .cpu()
            .numpy()
            .tolist()

        )

        # --------------------------------------------
        # Progress output every 100 batches
        # --------------------------------------------

        if (
            batch_index + 1
        ) % 100 == 0:

            avg_loss = (

                running_loss

                /

                (
                    batch_index
                    +
                    1
                )

            )

            print(

                f"  Train Batch "
                f"{batch_index + 1:04d}/"
                f"{len(train_cb_loader)} | "
                f"Loss {avg_loss:.4f}"

            )

    metrics = compute_cb_metrics(

        all_labels,

        all_predictions

    )

    return {

        "loss":

            running_loss
            /
            len(
                train_cb_loader
            ),

        **metrics

    }


# ============================================================
# 14. Validation One Epoch
# ============================================================

def validate_clinicalbert_epoch():

    clinicalbert_model.eval()

    running_loss = 0.0

    all_labels = []

    all_predictions = []

    with torch.no_grad():

        for batch in val_cb_loader:

            input_ids = (

                batch[
                    "input_ids"
                ]

                .long()

                .to(
                    CB_DEVICE,
                    non_blocking=True
                )

            )

            attention_mask = (

                batch[
                    "attention_mask"
                ]

                .long()

                .to(
                    CB_DEVICE,
                    non_blocking=True
                )

            )

            chunk_mask = (

                batch[
                    "chunk_mask"
                ]

                .bool()

                .to(
                    CB_DEVICE,
                    non_blocking=True
                )

            )

            labels = (

                batch[
                    "label"
                ]

                .long()

                .to(
                    CB_DEVICE,
                    non_blocking=True
                )

            )

            with cb_autocast():

                logits = clinicalbert_model(

                    input_ids,

                    attention_mask,

                    chunk_mask

                )

                loss = cb_criterion(

                    logits,

                    labels

                )

            running_loss += float(
                loss.item()
            )

            predictions = torch.argmax(

                logits,

                dim=1

            )

            all_labels.extend(

                labels
                .cpu()
                .numpy()
                .tolist()

            )

            all_predictions.extend(

                predictions
                .cpu()
                .numpy()
                .tolist()

            )

    metrics = compute_cb_metrics(

        all_labels,

        all_predictions

    )

    return {

        "loss":

            running_loss
            /
            len(
                val_cb_loader
            ),

        **metrics

    }


# ============================================================
# 15. Checkpoint Helper
# ============================================================

def save_cb_checkpoint(
    path,
    epoch,
    val_macro_f1
):

    torch.save(

        {

            "baseline":
                "ClinicalBERT",

            "model_name":
                CLINICALBERT_MODEL,

            "seed":
                CB_SEED,

            "epoch":
                int(
                    epoch
                ),

            "val_macro_f1":
                float(
                    val_macro_f1
                ),

            "model_state_dict":
                clinicalbert_model.state_dict(),

            "optimizer_state_dict":
                cb_optimizer.state_dict(),

            "scheduler_state_dict":
                cb_scheduler.state_dict(),

            "num_classes":
                NUM_DISEASES,

            "label_classes":
                list(
                    label_encoder.classes_
                ),

            "aggregation":
                (
                    "Mean pooling of ClinicalBERT "
                    "[CLS] embeddings across "
                    "valid patient chunks"
                )

        },

        path

    )


# ============================================================
# 16. Training State
# ============================================================

best_val_f1_cb = -1.0

best_epoch_cb = -1

cb_early_stop_counter = 0

cb_history = []


# ============================================================
# 17. Full Training Loop
# ============================================================

print("\n" + "=" * 70)

print(
    "STARTING CLINICALBERT TRAINING"
)

print("=" * 70)

for epoch in range(
    1,
    CB_EPOCHS + 1
):

    print("\n" + "=" * 70)

    print(
        f"CLINICALBERT EPOCH "
        f"{epoch}/"
        f"{CB_EPOCHS}"
    )

    print("=" * 70)

    # --------------------------------------------
    # Train
    # --------------------------------------------

    train_result = (
        train_clinicalbert_epoch(
            epoch
        )
    )

    # --------------------------------------------
    # Validate
    # --------------------------------------------

    val_result = (
        validate_clinicalbert_epoch()
    )

    current_val_f1 = float(

        val_result[
            "macro_f1"
        ]

    )

    # --------------------------------------------
    # Scheduler
    # --------------------------------------------

    cb_scheduler.step(
        current_val_f1
    )

    current_lr = float(

        cb_optimizer
        .param_groups[
            0
        ][
            "lr"
        ]

    )

    # --------------------------------------------
    # Store history
    # --------------------------------------------

    cb_history.append({

        "epoch":
            epoch,

        "train_loss":
            train_result[
                "loss"
            ],

        "val_loss":
            val_result[
                "loss"
            ],

        "train_accuracy":
            train_result[
                "accuracy"
            ],

        "val_accuracy":
            val_result[
                "accuracy"
            ],

        "train_macro_f1":
            train_result[
                "macro_f1"
            ],

        "val_macro_f1":
            val_result[
                "macro_f1"
            ],

        "learning_rate":
            current_lr

    })

    # --------------------------------------------
    # Epoch Summary
    # --------------------------------------------

    print("\nClinicalBERT Epoch Summary")
    print("-" * 60)

    print(

        f"Train Loss       : "
        f"{train_result['loss']:.4f}"

    )

    print(

        f"Validation Loss  : "
        f"{val_result['loss']:.4f}"

    )

    print(

        f"Train Accuracy   : "
        f"{train_result['accuracy']:.4f}"

    )

    print(

        f"Val Accuracy     : "
        f"{val_result['accuracy']:.4f}"

    )

    print(

        f"Train Macro-F1   : "
        f"{train_result['macro_f1']:.4f}"

    )

    print(

        f"Val Macro-F1     : "
        f"{val_result['macro_f1']:.4f}"

    )

    print(

        f"Learning Rate    : "
        f"{current_lr:.8f}"

    )

    # --------------------------------------------
    # Best checkpoint based ONLY on validation F1
    # --------------------------------------------

    if (
        current_val_f1
        >
        best_val_f1_cb
    ):

        best_val_f1_cb = (
            current_val_f1
        )

        best_epoch_cb = (
            epoch
        )

        cb_early_stop_counter = 0

        save_cb_checkpoint(

            CB_BEST_MODEL_PATH,

            epoch,

            current_val_f1

        )

        print(
            "\n✓ New best ClinicalBERT checkpoint saved."
        )

    else:

        cb_early_stop_counter += 1

        print(
            "\nNo validation improvement."
        )

        print(

            "Early stopping counter:",

            f"{cb_early_stop_counter}/"
            f"{CB_PATIENCE}"

        )

    # --------------------------------------------
    # Save last checkpoint
    # --------------------------------------------

    save_cb_checkpoint(

        CB_LAST_MODEL_PATH,

        epoch,

        current_val_f1

    )

    # --------------------------------------------
    # Save history every epoch
    # --------------------------------------------

    pd.DataFrame(
        cb_history
    ).to_csv(

        os.path.join(

            CB_OUTPUT_DIR,

            "ClinicalBERT_training_history.csv"

        ),

        index=False

    )

    # --------------------------------------------
    # Early stopping
    # --------------------------------------------

    if (
        cb_early_stop_counter
        >=
        CB_PATIENCE
    ):

        print("\n" + "=" * 70)

        print(
            "CLINICALBERT EARLY STOPPING TRIGGERED"
        )

        print("=" * 70)

        break


# ============================================================
# 18. Training Completion
# ============================================================

print("\n" + "=" * 70)

print(
    "CLINICALBERT TRAINING COMPLETE"
)

print("=" * 70)

print(
    "Epochs Completed:",
    len(
        cb_history
    )
)

print(
    "Best Epoch:",
    best_epoch_cb
)

print(

    "Best Validation Macro-F1:",

    f"{best_val_f1_cb:.4f}"

)

print(

    "Best Checkpoint Exists:",

    os.path.exists(
        CB_BEST_MODEL_PATH
    )

)


# ============================================================
# 19. Load Best Validation-Selected Checkpoint
# ============================================================

print(
    "\nLoading best ClinicalBERT checkpoint..."
)

cb_best_checkpoint = torch.load(

    CB_BEST_MODEL_PATH,

    map_location=(
        CB_DEVICE
    ),

    weights_only=False

)

clinicalbert_model.load_state_dict(

    cb_best_checkpoint[
        "model_state_dict"
    ]

)

clinicalbert_model = (
    clinicalbert_model.to(
        CB_DEVICE
    )
)

clinicalbert_model.eval()

print(
    "✓ Best ClinicalBERT checkpoint loaded."
)

print(

    "Checkpoint Epoch:",

    cb_best_checkpoint[
        "epoch"
    ]

)

print(

    "Checkpoint Val Macro-F1:",

    cb_best_checkpoint[
        "val_macro_f1"
    ]

)


# ============================================================
# 20. Held-Out Test Inference
# ============================================================

print("\n" + "=" * 70)

print(
    "RUNNING FINAL HELD-OUT TEST EVALUATION"
)

print("=" * 70)

cb_test_labels = []

cb_test_predictions = []

cb_test_probabilities = []

cb_test_patient_ids = []


with torch.no_grad():

    for batch in test_cb_loader:

        input_ids = (

            batch[
                "input_ids"
            ]

            .long()

            .to(
                CB_DEVICE,
                non_blocking=True
            )

        )

        attention_mask = (

            batch[
                "attention_mask"
            ]

            .long()

            .to(
                CB_DEVICE,
                non_blocking=True
            )

        )

        chunk_mask = (

            batch[
                "chunk_mask"
            ]

            .bool()

            .to(
                CB_DEVICE,
                non_blocking=True
            )

        )

        labels = (

            batch[
                "label"
            ]

            .long()

            .to(
                CB_DEVICE,
                non_blocking=True
            )

        )

        with cb_autocast():

            logits = clinicalbert_model(

                input_ids,

                attention_mask,

                chunk_mask

            )

        probabilities = torch.softmax(

            logits.float(),

            dim=1

        )

        predictions = torch.argmax(

            probabilities,

            dim=1

        )

        cb_test_labels.extend(

            labels
            .cpu()
            .numpy()
            .tolist()

        )

        cb_test_predictions.extend(

            predictions
            .cpu()
            .numpy()
            .tolist()

        )

        cb_test_probabilities.extend(

            probabilities
            .cpu()
            .numpy()

        )

        cb_test_patient_ids.extend(

            list(
                batch[
                    "patient_id"
                ]
            )

        )


# ============================================================
# 21. Convert Test Outputs
# ============================================================

y_true_cb = np.asarray(

    cb_test_labels,

    dtype=int

)

y_pred_cb = np.asarray(

    cb_test_predictions,

    dtype=int

)

y_prob_cb = np.asarray(

    cb_test_probabilities,

    dtype=float

)

print(
    "\nTest Patients Evaluated:",
    len(
        y_true_cb
    )
)

assert (
    len(
        y_true_cb
    )
    ==
    450
)

print(
    "✓ All 450 held-out test patients evaluated."
)


# ============================================================
# 22. Overall Test Metrics
# ============================================================

cb_accuracy = accuracy_score(

    y_true_cb,

    y_pred_cb

)

cb_macro_precision = precision_score(

    y_true_cb,

    y_pred_cb,

    average="macro",

    zero_division=0

)

cb_macro_recall = recall_score(

    y_true_cb,

    y_pred_cb,

    average="macro",

    zero_division=0

)

cb_macro_f1 = f1_score(

    y_true_cb,

    y_pred_cb,

    average="macro",

    zero_division=0

)

cb_weighted_f1 = f1_score(

    y_true_cb,

    y_pred_cb,

    average="weighted",

    zero_division=0

)

cb_error_rate = (

    1.0
    -
    cb_accuracy

) * 100


# ============================================================
# 23. ROC-AUC / PR-AUC
# ============================================================

cb_y_binary = label_binarize(

    y_true_cb,

    classes=np.arange(
        NUM_DISEASES
    )

)

cb_macro_roc_auc = roc_auc_score(

    cb_y_binary,

    y_prob_cb,

    average="macro",

    multi_class="ovr"

)

cb_weighted_roc_auc = roc_auc_score(

    cb_y_binary,

    y_prob_cb,

    average="weighted",

    multi_class="ovr"

)

cb_macro_pr_auc = average_precision_score(

    cb_y_binary,

    y_prob_cb,

    average="macro"

)

cb_weighted_pr_auc = average_precision_score(

    cb_y_binary,

    y_prob_cb,

    average="weighted"

)


# ============================================================
# 24. Print Final Metrics
# ============================================================

print("\n" + "=" * 70)

print(
    "CLINICALBERT FINAL TEST METRICS"
)

print("=" * 70)

print(

    f"Accuracy          : "
    f"{cb_accuracy:.4f}"

)

print(

    f"Macro Precision   : "
    f"{cb_macro_precision:.4f}"

)

print(

    f"Macro Recall      : "
    f"{cb_macro_recall:.4f}"

)

print(

    f"Macro F1          : "
    f"{cb_macro_f1:.4f}"

)

print(

    f"Weighted F1       : "
    f"{cb_weighted_f1:.4f}"

)

print(

    f"Macro ROC-AUC     : "
    f"{cb_macro_roc_auc:.4f}"

)

print(

    f"Macro PR-AUC      : "
    f"{cb_macro_pr_auc:.4f}"

)

print(

    f"Error Rate (%)    : "
    f"{cb_error_rate:.2f}"

)


# ============================================================
# 25. Confusion Matrix
# ============================================================

cm_cb = confusion_matrix(

    y_true_cb,

    y_pred_cb,

    labels=np.arange(
        NUM_DISEASES
    )

)

print("\nConfusion Matrix")
print("-" * 60)

print(
    cm_cb
)


# ============================================================
# 26. Classification Report
# ============================================================

class_names = list(
    label_encoder.classes_
)

print("\nDisease Mapping")
print("-" * 60)

for index, disease in enumerate(
    class_names
):

    print(

        f"{index} -> "
        f"{disease}"

    )


print("\nClassification Report")
print("-" * 60)

print(

    classification_report(

        y_true_cb,

        y_pred_cb,

        labels=np.arange(
            NUM_DISEASES
        ),

        target_names=(
            class_names
        ),

        digits=4,

        zero_division=0

    )

)


# ============================================================
# 27. Per-Disease Metrics
#
# F1-score
# One-vs-Rest ROC-AUC
# Misdiagnosis Rate = 1 - sensitivity/recall
# ============================================================

cb_per_disease_rows = []


for class_index, disease_name in enumerate(
    class_names
):

    y_true_binary = (

        y_true_cb
        ==
        class_index

    ).astype(
        int
    )

    y_pred_binary = (

        y_pred_cb
        ==
        class_index

    ).astype(
        int
    )

    disease_f1 = f1_score(

        y_true_binary,

        y_pred_binary,

        zero_division=0

    )

    disease_recall = recall_score(

        y_true_binary,

        y_pred_binary,

        zero_division=0

    )

    disease_auc = roc_auc_score(

        y_true_binary,

        y_prob_cb[
            :,
            class_index
        ]

    )

    misdiagnosis_rate = (

        1.0
        -
        disease_recall

    ) * 100

    cb_per_disease_rows.append({

        "Neurological Disease":
            disease_name,

        "F1-score":
            disease_f1,

        "One-vs-Rest AUC":
            disease_auc,

        "Misdiagnosis Rate (%)":
            misdiagnosis_rate

    })


cb_per_disease_df = pd.DataFrame(

    cb_per_disease_rows

)

print("\nPer-Disease Results")
print("-" * 60)

display(
    cb_per_disease_df
)


# ============================================================
# 28. Patient-Level Prediction Table
# ============================================================

cb_predictions_df = pd.DataFrame({

    "patient_id":
        cb_test_patient_ids,

    "true_label":
        y_true_cb,

    "predicted_label":
        y_pred_cb,

    "true_disease":
        [
            class_names[
                i
            ]
            for i in y_true_cb
        ],

    "predicted_disease":
        [
            class_names[
                i
            ]
            for i in y_pred_cb
        ],

    "confidence":
        y_prob_cb.max(
            axis=1
        ),

    "correct":
        (
            y_true_cb
            ==
            y_pred_cb
        )

})


for i, disease_name in enumerate(
    class_names
):

    safe_name = (

        disease_name
        .lower()
        .replace(
            " ",
            "_"
        )
        .replace(
            "'",
            ""
        )
        .replace(
            "-",
            "_"
        )

    )

    cb_predictions_df[

        f"prob_{safe_name}"

    ] = y_prob_cb[
        :,
        i
    ]


# ============================================================
# 29. Save Per-Disease Metrics
# ============================================================

cb_per_disease_df.to_csv(

    os.path.join(

        CB_OUTPUT_DIR,

        "ClinicalBERT_per_disease_metrics.csv"

    ),

    index=False

)


# ============================================================
# 30. Save Predictions
# ============================================================

cb_predictions_df.to_csv(

    os.path.join(

        CB_OUTPUT_DIR,

        "ClinicalBERT_test_predictions.csv"

    ),

    index=False

)


# ============================================================
# 31. Save Confusion Matrix
# ============================================================

pd.DataFrame(

    cm_cb,

    index=(
        class_names
    ),

    columns=(
        class_names
    )

).to_csv(

    os.path.join(

        CB_OUTPUT_DIR,

        "ClinicalBERT_confusion_matrix.csv"

    )

)


# ============================================================
# 32. Save Final Summary JSON
# ============================================================

cb_summary = {

    "Model":
        "ClinicalBERT",

    "Pretrained_Model":
        CLINICALBERT_MODEL,

    "Patient_Level_Aggregation":
        (
            "Mean pooling of chunk-level "
            "[CLS] embeddings"
        ),

    "Seed":
        CB_SEED,

    "Best_Epoch":
        int(
            best_epoch_cb
        ),

    "Best_Validation_Macro_F1":
        float(
            best_val_f1_cb
        ),

    "Test_Accuracy":
        float(
            cb_accuracy
        ),

    "Test_Macro_Precision":
        float(
            cb_macro_precision
        ),

    "Test_Macro_Recall":
        float(
            cb_macro_recall
        ),

    "Test_Macro_F1":
        float(
            cb_macro_f1
        ),

    "Test_Weighted_F1":
        float(
            cb_weighted_f1
        ),

    "Test_Macro_ROC_AUC":
        float(
            cb_macro_roc_auc
        ),

    "Test_Weighted_ROC_AUC":
        float(
            cb_weighted_roc_auc
        ),

    "Test_Macro_PR_AUC":
        float(
            cb_macro_pr_auc
        ),

    "Test_Weighted_PR_AUC":
        float(
            cb_weighted_pr_auc
        ),

    "Error_Rate_Percent":
        float(
            cb_error_rate
        ),

    "Test_Patients":
        int(
            len(
                y_true_cb
            )
        )

}


with open(

    os.path.join(

        CB_OUTPUT_DIR,

        "ClinicalBERT_summary.json"

    ),

    "w"

) as f:

    json.dump(

        cb_summary,

        f,

        indent=4

    )


# ============================================================
# 33. Final Five-Model Comparison
# ============================================================

final_baseline_comparison_df = pd.DataFrame({

    "Model": [

        "TF-IDF + SVM",

        "TextCNN",

        "BiLSTM",

        "ClinicalBERT",

        "STraT-Net E3"

    ],

    "Accuracy": [

        0.397778,

        0.384444,

        0.222222,

        cb_accuracy,

        0.518667

    ],

    "Macro-F1": [

        0.397438,

        0.377259,

        0.222015,

        cb_macro_f1,

        0.517134

    ],

    "Macro ROC-AUC": [

        0.721580,

        0.718228,

        0.531080,

        cb_macro_roc_auc,

        0.813035

    ],

    "Macro PR-AUC": [

        0.399809,

        0.397138,

        0.227627,

        cb_macro_pr_auc,

        0.542316

    ],

    "Error Rate (%)": [

        60.222222,

        61.555556,

        77.777778,

        cb_error_rate,

        48.1333

    ]

})


print("\n" + "=" * 70)

print(
    "FINAL FIVE-MODEL BASELINE COMPARISON"
)

print("=" * 70)

display(
    final_baseline_comparison_df
)


# ============================================================
# 34. Save Final Comparison
# ============================================================

final_baseline_comparison_df.to_csv(

    os.path.join(

        CB_OUTPUT_DIR,

        "final_five_model_comparison.csv"

    ),

    index=False

)


# ============================================================
# 35. Final Status
# ============================================================

print("\n" + "=" * 70)

print(
    "CLINICALBERT BASELINE COMPLETED SUCCESSFULLY"
)

print("=" * 70)

print(

    "Best Epoch              :",

    best_epoch_cb

)

print(

    "Best Validation Macro-F1:",

    f"{best_val_f1_cb:.4f}"

)

print(

    "Test Accuracy           :",

    f"{cb_accuracy:.4f}"

)

print(

    "Test Macro-F1           :",

    f"{cb_macro_f1:.4f}"

)

print(

    "Test Macro ROC-AUC      :",

    f"{cb_macro_roc_auc:.4f}"

)

print(

    "Test Macro PR-AUC       :",

    f"{cb_macro_pr_auc:.4f}"

)

print(

    "Error Rate (%)          :",

    f"{cb_error_rate:.2f}"

)

print()

print(
    "Results saved to:"
)

print(
    CB_OUTPUT_DIR
)

print("=" * 70)

In [ ]:
import os
import json
import numpy as np
import pandas as pd
import torch

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report,
    roc_auc_score,
    average_precision_score
)

from sklearn.preprocessing import label_binarize

print("=" * 70)
print("CLINICALBERT : FINAL HELD-OUT TEST EVALUATION")
print("=" * 70)

# ============================================================
# 1. Paths
# ============================================================

# Corrected to load from the local directory where the model was saved during training
CB_CHECKPOINT_DIR = "checkpoints_baseline_clinicalbert"

# Corrected to save results to the local output directory
CB_OUTPUT_DIR = "outputs_baseline_clinicalbert"

CB_BEST_MODEL_PATH = os.path.join(
    CB_CHECKPOINT_DIR,
    "best_clinicalbert_patient_model.pth"
)

assert os.path.exists(
    CB_BEST_MODEL_PATH
), "Best ClinicalBERT checkpoint not found."

# ============================================================
# 2. Load Best Checkpoint
# ============================================================

checkpoint = torch.load(
    CB_BEST_MODEL_PATH,
    map_location=CB_DEVICE,
    weights_only=False
)

clinicalbert_model.load_state_dict(
    checkpoint[
        "model_state_dict"
    ]
)

clinicalbert_model = clinicalbert_model.to(
    CB_DEVICE
)

clinicalbert_model.eval()

print("\nBest Checkpoint")
print("-" * 60)

print(
    "Epoch:",
    checkpoint[
        "epoch"
    ]
)

print(
    "Validation Macro-F1:",
    checkpoint[
        "val_macro_f1"
    ]
)

# ============================================================
# 3. Test Inference
# ============================================================

all_labels = []
all_predictions = []
all_probabilities = []
all_patient_ids = []

with torch.no_grad():

    for batch in test_cb_loader:

        input_ids = (
            batch[
                "input_ids"
            ]
            .long()
            .to(
                CB_DEVICE
            )
        )

        attention_mask = (
            batch[
                "attention_mask"
            ]
            .long()
            .to(
                CB_DEVICE
            )
        )

        chunk_mask = (
            batch[
                "chunk_mask"
            ]
            .bool()
            .to(
                CB_DEVICE
            )
        )

        labels = (
            batch[
                "label"
            ]
            .long()
            .to(
                CB_DEVICE
            )
        )

        with cb_autocast():

            logits = clinicalbert_model(

                input_ids,

                attention_mask,

                chunk_mask

            )

        probabilities = torch.softmax(

            logits.float(),

            dim=1

        )

        predictions = torch.argmax(

            probabilities,

            dim=1

        )

        all_labels.extend(
            labels
            .cpu()
            .numpy()
            .tolist()
        )

        all_predictions.extend(
            predictions
            .cpu()
            .numpy()
            .tolist()
        )

        all_probabilities.extend(
            probabilities
            .cpu()
            .numpy()
        )

        all_patient_ids.extend(
            batch[
                "patient_id"
            ]
        )

# ============================================================
# 4. Convert to Arrays
# ============================================================

y_true_cb = np.asarray(
    all_labels,
    dtype=int
)

y_pred_cb = np.asarray(
    all_predictions,
    dtype=int
)

y_prob_cb = np.asarray(
    all_probabilities,
    dtype=float
)

print(
    "\nTest Patients Evaluated:",
    len(
        y_true_cb
    )
)

assert len(
    y_true_cb
) == 450

# ============================================================
# 5. Overall Metrics
# ============================================================

cb_accuracy = accuracy_score(
    y_true_cb,
    y_pred_cb
)

cb_macro_precision = precision_score(
    y_true_cb,
    y_pred_cb,
    average="macro",
    zero_division=0
)

cb_macro_recall = recall_score(
    y_true_cb,
    y_pred_cb,
    average="macro",
    zero_division=0
)

cb_macro_f1 = f1_score(
    y_true_cb,
    y_pred_cb,
    average="macro",
    zero_division=0
)

cb_weighted_f1 = f1_score(
    y_true_cb,
    y_pred_cb,
    average="weighted",
    zero_division=0
)

cb_error_rate = (
    1.0
    -
    cb_accuracy
) * 100

# ============================================================
# 6. ROC-AUC / PR-AUC
# ============================================================

y_binary_cb = label_binarize(

    y_true_cb,

    classes=np.arange(
        NUM_DISEASES
    )

)

cb_macro_roc_auc = roc_auc_score(

    y_binary_cb,

    y_prob_cb,

    average="macro",

    multi_class="ovr"

)

cb_macro_pr_auc = average_precision_score(

    y_binary_cb,

    y_prob_cb,

    average="macro"

)

# ============================================================
# 7. Print Overall Metrics
# ============================================================

print("\n" + "=" * 70)
print("CLINICALBERT FINAL TEST METRICS")
print("=" * 70)

print(
    f"Accuracy        : {cb_accuracy:.4f}"
)

print(
    f"Macro Precision : {cb_macro_precision:.4f}"
)

print(
    f"Macro Recall    : {cb_macro_recall:.4f}"
)

print(
    f"Macro F1        : {cb_macro_f1:.4f}"
)

print(
    f"Weighted F1     : {cb_weighted_f1:.4f}"
)

print(
    f"Macro ROC-AUC   : {cb_macro_roc_auc:.4f}"
)

print(
    f"Macro PR-AUC    : {cb_macro_pr_auc:.4f}"
)

print(
    f"Error Rate (%)  : {cb_error_rate:.2f}"
)

# ============================================================
# 8. Confusion Matrix
# ============================================================

cm_cb = confusion_matrix(
    y_true_cb,
    y_pred_cb
)

print("\nConfusion Matrix")
print("-" * 60)

print(
    cm_cb
)

# ============================================================
# 9. Classification Report
# ============================================================

class_names = list(
    label_encoder.classes_
)

print("\nClassification Report")
print("-" * 60)

print(

    classification_report(

        y_true_cb,

        y_pred_cb,

        target_names=class_names,

        digits=4,

        zero_division=0

    )

)

# ============================================================
# 10. Per-Disease Metrics
# ============================================================

per_disease_rows = []

for class_index, disease_name in enumerate(
    class_names
):

    y_true_binary = (
        y_true_cb
        ==
        class_index
    ).astype(int)

    y_pred_binary = (
        y_pred_cb
        ==
        class_index
    ).astype(int)

    disease_f1 = f1_score(

        y_true_binary,

        y_pred_binary,

        zero_division=0

    )

    disease_recall = recall_score(

        y_true_binary,

        y_pred_binary,

        zero_division=0

    )

    disease_auc = roc_auc_score(

        y_true_binary,

        y_prob_cb[
            :,
            class_index
        ]

    )

    per_disease_rows.append({

        "Neurological Disease":
            disease_name,

        "F1-score":
            disease_f1,

        "One-vs-Rest AUC":
            disease_auc,

        "Misdiagnosis Rate (%)":
            (
                1.0
                -
                disease_recall
            )
            *
            100

    })

cb_per_disease_df = pd.DataFrame(
    per_disease_rows
)

print("\nPer-Disease Results")
print("-" * 60)

display(
    cb_per_disease_df
)

# ============================================================
# 11. Save Predictions
# ============================================================

cb_predictions_df = pd.DataFrame({

    "patient_id":
        all_patient_ids,

    "true_label":
        y_true_cb,

    "predicted_label":
        y_pred_cb,

    "correct":
        (
            y_true_cb
            ==
            y_pred_cb
        ),

    "confidence":
        y_prob_cb.max(
            axis=1
        )

})

for i, disease_name in enumerate(
    class_names
):

    safe_name = (
        disease_name
        .lower()
        .replace(
            " ",
            "_"
        )
        .replace(
            "'",
            ""
        )
    )

    cb_predictions_df[
        f"prob_{safe_name}"
    ] = y_prob_cb[
        :,
        i
    ]

cb_predictions_df.to_csv(

    os.path.join(

        CB_OUTPUT_DIR,

        "ClinicalBERT_test_predictions.csv"

    ),

    index=False

)

# ============================================================
# 12. Save Per-Disease Metrics
# ============================================================

cb_per_disease_df.to_csv(

    os.path.join(

        CB_OUTPUT_DIR,

        "ClinicalBERT_per_disease_metrics.csv"

    ),

    index=False

)

# ============================================================
# 13. Save Summary
# ============================================================

cb_summary = {

    "Model":
        "ClinicalBERT",

    "Best_Epoch":
        int(
            checkpoint[
                "epoch"
            ]
        ),

    "Best_Validation_Macro_F1":
        float(
            checkpoint[
                "val_macro_f1"
            ]
        ),

    "Test_Accuracy":
        float(
            cb_accuracy
        ),

    "Test_Macro_Precision":
        float(
            cb_macro_precision
        ),

    "Test_Macro_Recall":
        float(
            cb_macro_recall
        ),

    "Test_Macro_F1":
        float(
            cb_macro_f1
        ),

    "Test_Weighted_F1":
        float(
            cb_weighted_f1
        ),

    "Test_Macro_ROC_AUC":
        float(
            cb_macro_roc_auc
        ),

    "Test_Macro_PR_AUC":
        float(
            cb_macro_pr_auc
        ),

    "Error_Rate_Percent":
        float(
            cb_error_rate
        )

}

with open(

    os.path.join(

        CB_OUTPUT_DIR,

        "ClinicalBERT_summary.json"

    ),

    "w"

) as f:

    json.dump(
        cb_summary,
        f,
        indent=4
    )

# ============================================================
# 14. Final Five-Model Comparison
# ============================================================

comparison_df = pd.DataFrame({

    "Model": [

        "TF-IDF + SVM",

        "TextCNN",

        "BiLSTM",

        "ClinicalBERT",

        "STraT-Net E3"

    ],

    "Accuracy": [

        0.397778,

        0.384444,

        0.222222,

        cb_accuracy,

        0.511111

    ],

    "Macro-F1": [

        0.397438,

        0.377968,

        0.222015,

        cb_macro_f1,

        0.509646

    ],

    "Macro ROC-AUC": [

        0.721580,

        0.718037,

        0.531080,

        cb_macro_roc_auc,

        0.810784

    ],

    "Macro PR-AUC": [

        0.399809,

        0.396817,

        0.227627,

        cb_macro_pr_auc,

        0.542041

    ],

    "Error Rate (%)": [

        60.222222,

        61.555556,

        77.777778,

        cb_error_rate,

        48.8889

    ]

})

print("\n" + "=" * 70)
print("FINAL FIVE-MODEL COMPARISON")
print("=" * 70)

display(
    comparison_df
)

comparison_df.to_csv(

    os.path.join(

        CB_OUTPUT_DIR,

        "final_five_model_comparison.csv"

    ),

    index=False

)

print("\n" + "=" * 70)
print("CLINICALBERT FINAL EVALUATION COMPLETE")
print("=" * 70)